In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# =============================================================================
#  DISORDERED BREATHING CLASSIFICATION — SVM on FFT Features
#  Dataset : SN2-SN100 subfolders, features.npy (N, 125) + y.npy
#  Goal    : Maximise RECALL for disordered breathing (class 1)
#  Note    : Pure sklearn — no cuML (GPU SVM incompatible with P100)
#            LinearSVC + SGD are fast enough on CPU for 125 features
# =============================================================================

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS
# ──────────────────────────────────────────────────────────────────────────────
import kagglehub
import os, gc, warnings
import numpy as np
import pandas as pd
from collections import Counter
warnings.filterwarnings("ignore")

from sklearn.preprocessing        import StandardScaler
from sklearn.svm                  import SVC, LinearSVC
from sklearn.linear_model         import SGDClassifier
from sklearn.calibration          import CalibratedClassifierCV
from sklearn.utils.class_weight   import compute_class_weight
from sklearn.metrics              import (classification_report,
                                          confusion_matrix,
                                          roc_auc_score,
                                          recall_score,
                                          precision_score,
                                          f1_score,
                                          ConfusionMatrixDisplay,
                                          RocCurveDisplay,
                                          PrecisionRecallDisplay)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ──────────────────────────────────────────────────────────────────────────────
# 1.  DOWNLOAD DATASET
# ──────────────────────────────────────────────────────────────────────────────
path = kagglehub.dataset_download(
    "nishantomhare/disordered-breathing-classification-features-fft")
print("Feature dataset path:", path)

# ──────────────────────────────────────────────────────────────────────────────
# 2.  FEATURE NAMES  (5 channels × 25 features = 125)
# ──────────────────────────────────────────────────────────────────────────────
CHANNEL_NAMES = ["pressure","airflow","chest","abdomen","pleth"]
BAND_NAMES    = ["be_0_01","be_01_03","be_03_06","be_06_1",
                 "be_1_2","be_2_5","be_gt5"]
BASE_FEATURES = (
    ["energy","mean_mag","variance","sp_centroid","sp_spread",
     "sp_skewness","sp_kurtosis","sp_flatness","shannon_entropy"]
    + BAND_NAMES
    + ["sp_crest","hjorth_activity","hjorth_mobility","hjorth_complexity",
       "peak1_freq","peak2_freq","peak_ratio","peak_width","peak_prominence"]
)
FEATURE_NAMES = [f"{ch}_{f}" for ch in CHANNEL_NAMES for f in BASE_FEATURES]
N_FEATURES    = len(FEATURE_NAMES)   # 125
print(f"Features per window : {N_FEATURES}")

# ──────────────────────────────────────────────────────────────────────────────
# 3.  LOAD ALL PATIENT DATA
# ──────────────────────────────────────────────────────────────────────────────
ALL_PIDS = [f"SN{i}" for i in range(2, 101)]

def find_patient_files(pid, base_path):
    for root, _, files in os.walk(base_path):
        if pid in root and "features.npy" in files and "y.npy" in files:
            return (os.path.join(root, "features.npy"),
                    os.path.join(root, "y.npy"))
    return None, None

print(f"\n{'='*60}")
print("  LOADING FEATURE DATA")
print(f"{'='*60}")

all_X, all_y = [], []
loaded, skipped, summary = [], [], []
pid_windows = {}   # pid → n_windows (for split)

for pid in ALL_PIDS:
    feat_path, y_path = find_patient_files(pid, path)

    if feat_path is None:
        print(f"  ⚠️  [{pid}] not found → skipped")
        skipped.append(pid)
        continue

    try:
        X_p = np.load(feat_path).astype(np.float32)
        y_p = np.load(y_path).astype(np.int8)

        if X_p.ndim != 2 or X_p.shape[1] != N_FEATURES:
            print(f"  ⚠️  [{pid}] bad shape {X_p.shape} → skipped")
            skipped.append(pid); continue

        if len(X_p) != len(y_p):
            print(f"  ⚠️  [{pid}] X/y mismatch → skipped")
            skipped.append(pid); continue

        X_p = np.nan_to_num(X_p, nan=0.0, posinf=0.0, neginf=0.0)

        dist  = Counter(y_p.tolist())
        n     = len(y_p)
        all_X.append(X_p)
        all_y.append(y_p)
        loaded.append(pid)
        pid_windows[pid] = n
        summary.append({
            "patient"  : pid,
            "windows"  : n,
            "normal"   : dist.get(0,0),
            "apnea"    : dist.get(1,0),
            "apnea_pct": round(100*dist.get(1,0)/n, 1)
        })
        print(f"  ✅ [{pid}]  windows={n}  "
              f"normal={dist.get(0,0)}  apnea={dist.get(1,0)} "
              f"({round(100*dist.get(1,0)/n,1)}%)")
        del X_p, y_p; gc.collect()

    except Exception as e:
        print(f"  ❌ [{pid}] {e} → skipped")
        skipped.append(pid)

print(f"\n  Loaded: {len(loaded)}  |  Skipped: {len(skipped)}  {skipped}")

X_all = np.concatenate(all_X, axis=0)
y_all = np.concatenate(all_y, axis=0)
del all_X, all_y; gc.collect()

dist  = Counter(y_all.tolist())
total = len(y_all)
print(f"\n  X shape  : {X_all.shape}")
print(f"  Normal   : {dist[0]:,} ({100*dist[0]/total:.1f}%)")
print(f"  Apnea    : {dist[1]:,} ({100*dist[1]/total:.1f}%)")
print(f"  Imbalance: {dist[0]/dist[1]:.2f}:1")

# save patient summary
pd.DataFrame(summary).to_csv(
    "/kaggle/working/patient_feature_summary.csv", index=False)

# ──────────────────────────────────────────────────────────────────────────────
# 4.  DATASET ANALYSIS
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("  DATASET ANALYSIS")
print(f"{'='*60}")

df_feat = pd.DataFrame(X_all, columns=FEATURE_NAMES)
df_feat["label"] = y_all

normal_mean = df_feat[df_feat["label"]==0][FEATURE_NAMES].mean()
apnea_mean  = df_feat[df_feat["label"]==1][FEATURE_NAMES].mean()
diff        = (apnea_mean - normal_mean).abs().sort_values(ascending=False)

print(f"\n  Top 15 most discriminative features:")
print(f"  {'FEATURE':>38}  {'NORMAL':>10}  {'APNEA':>10}  {'DIFF':>10}")
print("  " + "-"*72)
for feat in diff.index[:15]:
    print(f"  {feat:>38}  {normal_mean[feat]:>10.4f}  "
          f"{apnea_mean[feat]:>10.4f}  {diff[feat]:>10.4f}")

# ──────────────────────────────────────────────────────────────────────────────
# 5.  PREPROCESSING
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("  PREPROCESSING")
print(f"{'='*60}")

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_all)
print(f"  Scaled X : mean≈{X_scaled.mean():.4f}  std≈{X_scaled.std():.4f}")

# class weights
cw  = compute_class_weight("balanced", classes=np.array([0,1]), y=y_all)
cwd = {0: float(cw[0]), 1: float(cw[1])}
print(f"  Class weights: normal={cw[0]:.3f}  apnea={cw[1]:.3f}")

# patient-level split — no leakage
np.random.seed(42)
pids_arr = np.array(loaded)
np.random.shuffle(pids_arr)
n        = len(pids_arr)
n_test   = max(1, int(n * 0.15))
n_val    = max(1, int(n * 0.15))
test_pids  = set(pids_arr[:n_test])
val_pids   = set(pids_arr[n_test:n_test+n_val])
train_pids = set(pids_arr[n_test+n_val:])

print(f"\n  Train: {len(train_pids)} patients  "
      f"Val: {len(val_pids)}  Test: {len(test_pids)}")

# build per-window masks
pid_per_window = []
for pid in loaded:
    pid_per_window.extend([pid] * pid_windows[pid])
pid_per_window = np.array(pid_per_window)

train_mask = np.array([p in train_pids for p in pid_per_window])
val_mask   = np.array([p in val_pids   for p in pid_per_window])
test_mask  = np.array([p in test_pids  for p in pid_per_window])

X_train, y_train = X_scaled[train_mask], y_all[train_mask]
X_val,   y_val   = X_scaled[val_mask],   y_all[val_mask]
X_test,  y_test  = X_scaled[test_mask],  y_all[test_mask]

print(f"  Train: {len(y_train):,}  "
      f"apnea={Counter(y_train)[1]:,}({100*Counter(y_train)[1]/len(y_train):.1f}%)")
print(f"  Val  : {len(y_val):,}  "
      f"apnea={Counter(y_val)[1]:,}({100*Counter(y_val)[1]/len(y_val):.1f}%)")
print(f"  Test : {len(y_test):,}  "
      f"apnea={Counter(y_test)[1]:,}({100*Counter(y_test)[1]/len(y_test):.1f}%)")

# ──────────────────────────────────────────────────────────────────────────────
# 6.  EVALUATION HELPER
# ──────────────────────────────────────────────────────────────────────────────
all_results = []

def evaluate(name, model, X_tr, y_tr, X_v, y_v, X_te, y_te,
             threshold=0.5):
    print(f"\n  Fitting {name}...")
    model.fit(X_tr, y_tr)
    print(f"  Done.")

    for split_name, Xs, ys in [("val", X_v, y_v), ("test", X_te, y_te)]:

        # get probabilities if available, else use decision function
        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(Xs)[:, 1]
        elif hasattr(model, "decision_function"):
            df     = model.decision_function(Xs)
            y_prob = (df - df.min()) / (df.max() - df.min() + 1e-9)
        else:
            y_prob = None

        y_pred = ((y_prob >= threshold).astype(int)
                  if y_prob is not None else model.predict(Xs))

        rec   = recall_score(ys, y_pred, zero_division=0)
        prec  = precision_score(ys, y_pred, zero_division=0)
        f1    = f1_score(ys, y_pred, zero_division=0)
        acc   = float((y_pred == ys).mean())
        auc   = roc_auc_score(ys, y_prob) if y_prob is not None else float("nan")

        print(f"\n  ── [{name}]  {split_name.upper()}  "
              f"(threshold={threshold:.2f}) ──")
        print(f"     Accuracy  : {100*acc:.2f}%")
        print(f"     Recall    : {100*rec:.2f}%   ← apnea detection rate")
        print(f"     Precision : {100*prec:.2f}%")
        print(f"     F1        : {100*f1:.2f}%")
        print(f"     AUC-ROC   : {auc:.4f}")
        print(classification_report(ys, y_pred,
              target_names=["Normal","Apnea"], digits=3))

        all_results.append({
            "model": name, "split": split_name, "threshold": threshold,
            "accuracy":  round(100*acc,2),
            "recall":    round(100*rec,2),
            "precision": round(100*prec,2),
            "f1":        round(100*f1,2),
            "auc":       round(auc,4) if not np.isnan(auc) else None,
        })

    return model

# ──────────────────────────────────────────────────────────────────────────────
# 7.  MODEL A — LinearSVC  (fastest, good baseline)
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("  MODEL A — LinearSVC (balanced weights)")
print(f"{'='*60}")

base_lin = LinearSVC(C=1.0, class_weight="balanced",
                     max_iter=5000, dual=False)
# wrap with CalibratedClassifierCV to get predict_proba
modelA = CalibratedClassifierCV(base_lin, cv=3)
modelA = evaluate("LinearSVC_balanced", modelA,
                   X_train, y_train, X_val, y_val, X_test, y_test)

# ──────────────────────────────────────────────────────────────────────────────
# 8.  MODEL B — LinearSVC aggressive apnea weight
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("  MODEL B — LinearSVC (aggressive apnea weight)")
print(f"{'='*60}")

ratio    = dist[0] / dist[1]
agg_cw   = {0: 1.0, 1: ratio * 1.5}
print(f"  Class weights: normal=1.0  apnea={agg_cw[1]:.2f}")

base_agg = LinearSVC(C=1.0, class_weight=agg_cw,
                     max_iter=5000, dual=False)
modelB   = CalibratedClassifierCV(base_agg, cv=3)
modelB   = evaluate("LinearSVC_aggressive", modelB,
                     X_train, y_train, X_val, y_val, X_test, y_test)

# ──────────────────────────────────────────────────────────────────────────────
# 9.  MODEL C — SGDClassifier (SVM approximation, very fast)
#     hinge loss = linear SVM, modified_huber = smooth + probability
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("  MODEL C — SGD SVM (modified_huber, balanced weights)")
print(f"{'='*60}")

modelC = SGDClassifier(loss="modified_huber", penalty="l2",
                       alpha=1e-4, class_weight="balanced",
                       max_iter=200, tol=1e-3,
                       random_state=42, n_jobs=-1)
modelC = evaluate("SGD_SVM_balanced", modelC,
                   X_train, y_train, X_val, y_val, X_test, y_test)

# ──────────────────────────────────────────────────────────────────────────────
# 10.  MODEL D — RBF SVM on subset (RBF is slow on 60K samples)
#      Use 15K random sample for training, evaluate on full val/test
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("  MODEL D — RBF SVM (15K subset, balanced weights)")
print(f"{'='*60}")

MAX_TRAIN_RBF = 15_000
rng           = np.random.default_rng(42)

# stratified subsample
idx0   = np.where(y_train == 0)[0]
idx1   = np.where(y_train == 1)[0]
n_each = min(MAX_TRAIN_RBF // 2, len(idx0), len(idx1))
sub_idx = np.concatenate([
    rng.choice(idx0, n_each, replace=False),
    rng.choice(idx1, n_each, replace=False),
])
rng.shuffle(sub_idx)
X_sub, y_sub = X_train[sub_idx], y_train[sub_idx]
print(f"  RBF training subset: {len(y_sub):,} "
      f"(normal={Counter(y_sub)[0]}  apnea={Counter(y_sub)[1]})")

modelD = SVC(kernel="rbf", C=10.0, gamma="scale",
             class_weight="balanced", probability=True,
             cache_size=2000)
modelD = evaluate("RBF_SVM_subset", modelD,
                   X_sub, y_sub, X_val, y_val, X_test, y_test)

# ──────────────────────────────────────────────────────────────────────────────
# 11.  THRESHOLD TUNING (on best model = LinearSVC_balanced)
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("  THRESHOLD TUNING — Model A (LinearSVC_balanced)")
print(f"{'='*60}")

y_prob_val = modelA.predict_proba(X_val)[:, 1]

print(f"\n  {'THRESH':>7}  {'RECALL':>8}  {'PRECISION':>11}  "
      f"{'F1':>8}  {'F2':>8}  {'ACC':>8}")
print("  " + "-"*55)

best_thresh = 0.5
best_f2     = 0.0
thresh_rows = []

for thresh in np.arange(0.15, 0.75, 0.05):
    yp   = (y_prob_val >= thresh).astype(int)
    rec  = recall_score(y_val,    yp, zero_division=0)
    prec = precision_score(y_val, yp, zero_division=0)
    f1   = f1_score(y_val,        yp, zero_division=0)
    acc  = float((yp == y_val).mean())
    # F2 = weights recall 2× more than precision
    f2   = (5*prec*rec)/(4*prec+rec) if (4*prec+rec) > 0 else 0

    flag = " ← best F2" if f2 > best_f2 else ""
    print(f"  {thresh:>7.2f}  {100*rec:>7.1f}%  {100*prec:>10.1f}%  "
          f"{100*f1:>7.1f}%  {f2:>8.4f}  {100*acc:>7.1f}%{flag}")

    thresh_rows.append({
        "threshold": round(thresh, 2),
        "recall":    round(100*rec,  1),
        "precision": round(100*prec, 1),
        "f1":        round(100*f1,   1),
        "f2":        round(f2, 4),
        "accuracy":  round(100*acc,  1),
    })
    if f2 > best_f2:
        best_f2     = f2
        best_thresh = thresh

print(f"\n  Best threshold (max F2): {best_thresh:.2f}")

# apply on test set
y_prob_test  = modelA.predict_proba(X_test)[:, 1]
y_pred_tuned = (y_prob_test >= best_thresh).astype(int)

rec_t   = recall_score(y_test,    y_pred_tuned, zero_division=0)
prec_t  = precision_score(y_test, y_pred_tuned, zero_division=0)
f1_t    = f1_score(y_test,        y_pred_tuned, zero_division=0)
acc_t   = float((y_pred_tuned == y_test).mean())
auc_t   = roc_auc_score(y_test, y_prob_test)

print(f"\n  ── TEST with tuned threshold={best_thresh:.2f} ──")
print(f"     Recall    : {100*rec_t:.2f}%   ← apnea detection rate")
print(f"     Precision : {100*prec_t:.2f}%")
print(f"     F1        : {100*f1_t:.2f}%")
print(f"     Accuracy  : {100*acc_t:.2f}%")
print(f"     AUC-ROC   : {auc_t:.4f}")
print(classification_report(y_test, y_pred_tuned,
      target_names=["Normal","Apnea"], digits=3))

all_results.append({
    "model": f"LinearSVC_tuned_t{best_thresh:.2f}",
    "split": "test",
    "threshold": round(best_thresh, 2),
    "accuracy":  round(100*acc_t,  2),
    "recall":    round(100*rec_t,  2),
    "precision": round(100*prec_t, 2),
    "f1":        round(100*f1_t,   2),
    "auc":       round(auc_t, 4),
})

# ──────────────────────────────────────────────────────────────────────────────
# 12.  PLOTS
# ──────────────────────────────────────────────────────────────────────────────
print("\n  Generating plots...")
OUT = "/kaggle/working"

# ── Plot 1: Confusion matrices (default vs tuned threshold) ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("SVM (LinearSVC) — Confusion Matrices",
             fontsize=13, fontweight="bold")
for ax, (yp, title) in zip(axes, [
    (modelA.predict(X_test),   "Default threshold=0.50"),
    (y_pred_tuned,             f"Tuned threshold={best_thresh:.2f}"),
]):
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm, display_labels=["Normal","Apnea"]).plot(
        ax=ax, colorbar=False, cmap="Blues")
    r  = recall_score(y_test, yp, zero_division=0)
    p  = precision_score(y_test, yp, zero_division=0)
    ax.set_title(f"{title}\nRecall={100*r:.1f}%  Precision={100*p:.1f}%",
                 fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUT}/confusion_matrices.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ confusion_matrices.png")

# ── Plot 2: Precision / Recall / F1 / F2 vs threshold ──
fig, ax = plt.subplots(figsize=(11, 5))
ts   = [r["threshold"] for r in thresh_rows]
ax.plot(ts, [r["recall"]    for r in thresh_rows], "r-o", lw=2, label="Recall")
ax.plot(ts, [r["precision"] for r in thresh_rows], "b-s", lw=2, label="Precision")
ax.plot(ts, [r["f1"]        for r in thresh_rows], "g-^", lw=2, label="F1")
ax.plot(ts, [r["f2"]*100    for r in thresh_rows], "m-D", lw=2, label="F2 (×100)")
ax.axvline(best_thresh, color="orange", lw=2, linestyle="--",
           label=f"Best threshold={best_thresh:.2f}")
ax.set_xlabel("Decision Threshold", fontsize=11)
ax.set_ylabel("Score (%)", fontsize=11)
ax.set_title("Threshold Tuning — Validation Set", fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/threshold_tuning.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ threshold_tuning.png")

# ── Plot 3: ROC curve ──
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(y_test, y_prob_test, ax=ax,
                                  name=f"LinearSVC (AUC={auc_t:.3f})")
ax.plot([0,1],[0,1],"k--", lw=1)
ax.set_title("ROC Curve — Test Set", fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/roc_curve.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ roc_curve.png")

# ── Plot 4: Precision-Recall curve ──
fig, ax = plt.subplots(figsize=(7, 6))
PrecisionRecallDisplay.from_predictions(y_test, y_prob_test, ax=ax,
                                         name="LinearSVC")
ax.set_title("Precision-Recall Curve — Test Set", fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/precision_recall_curve.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ precision_recall_curve.png")

# ── Plot 5: Feature importance from LinearSVC ──
try:
    coefs = np.mean([abs(e.estimator.coef_[0])
                     for e in modelA.calibrated_classifiers_], axis=0)
    top_idx   = np.argsort(coefs)[::-1][:20]
    top_names = [FEATURE_NAMES[i] for i in top_idx]
    top_vals  = coefs[top_idx]

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(range(20), top_vals[::-1], color="steelblue")
    ax.set_yticks(range(20))
    ax.set_yticklabels(top_names[::-1], fontsize=9)
    ax.set_xlabel("Mean |Coefficient|", fontsize=11)
    ax.set_title("Top 20 Feature Importances (LinearSVC)", fontsize=12)
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    plt.savefig(f"{OUT}/feature_importance.png", dpi=120, bbox_inches="tight")
    plt.close()
    print("  ✅ feature_importance.png")
except Exception as e:
    print(f"  ⚠️  Feature importance skipped: {e}")

# ── Plot 6: Model comparison bar chart ──
df_res   = pd.DataFrame(all_results)
test_res = df_res[df_res["split"]=="test"].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(test_res))
w = 0.2
ax.bar(x-1.5*w, test_res["accuracy"],  w, label="Accuracy",  color="steelblue")
ax.bar(x-0.5*w, test_res["recall"],    w, label="Recall",    color="crimson")
ax.bar(x+0.5*w, test_res["precision"], w, label="Precision", color="seagreen")
ax.bar(x+1.5*w, test_res["f1"],        w, label="F1",        color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels(test_res["model"], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Score (%)", fontsize=11)
ax.set_title("All Models — Test Set Comparison", fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(0, 115)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(f"{OUT}/model_comparison.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ model_comparison.png")

# ──────────────────────────────────────────────────────────────────────────────
# 13.  SAVE ALL OUTPUTS
# ──────────────────────────────────────────────────────────────────────────────
df_res.to_csv(f"{OUT}/svm_results.csv", index=False)
pd.DataFrame(thresh_rows).to_csv(f"{OUT}/threshold_tuning.csv", index=False)
np.save(f"{OUT}/scaler_mean.npy",    scaler.mean_)
np.save(f"{OUT}/scaler_std.npy",     scaler.scale_)
np.save(f"{OUT}/best_threshold.npy", np.array([best_thresh]))

print(f"\n  All outputs saved to {OUT}/")

# ──────────────────────────────────────────────────────────────────────────────
# 14.  FINAL SUMMARY
# ──────────────────────────────────────────────────────────────────────────────
print(f"""
  ╔══════════════════════════════════════════════════════════╗
  ║           SVM CLASSIFICATION COMPLETE                    ║
  ╠══════════════════════════════════════════════════════════╣
  ║  Patients loaded    : {len(loaded):<35}║
  ║  Patients skipped   : {len(skipped):<35}║
  ║  Total windows      : {total:<35,}║
  ║  Normal  (0)        : {dist[0]:<6,} ({100*dist[0]/total:.1f}%)                  ║
  ║  Apnea   (1)        : {dist[1]:<6,} ({100*dist[1]/total:.1f}%)                  ║
  ╠══════════════════════════════════════════════════════════╣
  ║  Best result (LinearSVC, threshold={best_thresh:.2f}):        ║
  ║    Recall    : {f"{100*rec_t:.2f}%":<44}║
  ║    Precision : {f"{100*prec_t:.2f}%":<44}║
  ║    F1        : {f"{100*f1_t:.2f}%":<44}║
  ║    AUC-ROC   : {f"{auc_t:.4f}":<44}║
  ╠══════════════════════════════════════════════════════════╣
  ║  Models trained: LinearSVC_balanced, LinearSVC_agg,     ║
  ║                  SGD_SVM, RBF_SVM_subset                ║
  ╠══════════════════════════════════════════════════════════╣
  ║  Saved:                                                  ║
  ║    svm_results.csv            all model metrics         ║
  ║    threshold_tuning.csv       threshold sweep           ║
  ║    confusion_matrices.png                               ║
  ║    threshold_tuning.png                                 ║
  ║    roc_curve.png                                        ║
  ║    precision_recall_curve.png                           ║
  ║    feature_importance.png                               ║
  ║    model_comparison.png                                 ║
  ╚══════════════════════════════════════════════════════════╝
""")

In [ ]:
from IPython.display import FileLink
FileLink(f"{OUT}/svm_results.csv")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS
# ──────────────────────────────────────────────────────────────────────────────
import os, gc
import numpy as np
import pandas as pd

from collections import Counter

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.feature_selection import SelectFromModel

from xgboost import XGBClassifier
import kagglehub


# ──────────────────────────────────────────────────────────────────────────────
# 1. DOWNLOAD DATASET
# ──────────────────────────────────────────────────────────────────────────────
path = kagglehub.dataset_download(
    "nishantomhare/disordered-breathing-classification-features-fft")
print("Feature dataset path:", path)


# ──────────────────────────────────────────────────────────────────────────────
# 2. FEATURE NAMES
# ──────────────────────────────────────────────────────────────────────────────
CHANNEL_NAMES = ["pressure","airflow","chest","abdomen","pleth"]
BAND_NAMES    = ["be_0_01","be_01_03","be_03_06","be_06_1",
                 "be_1_2","be_2_5","be_gt5"]

BASE_FEATURES = (
    ["energy","mean_mag","variance","sp_centroid","sp_spread",
     "sp_skewness","sp_kurtosis","sp_flatness","shannon_entropy"]
    + BAND_NAMES +
    ["sp_crest","hjorth_activity","hjorth_mobility","hjorth_complexity",
     "peak1_freq","peak2_freq","peak_ratio","peak_width","peak_prominence"]
)

FEATURE_NAMES = [f"{ch}_{f}" for ch in CHANNEL_NAMES for f in BASE_FEATURES]
N_FEATURES    = len(FEATURE_NAMES)


# ──────────────────────────────────────────────────────────────────────────────
# 3. LOAD DATA
# ──────────────────────────────────────────────────────────────────────────────
ALL_PIDS = [f"SN{i}" for i in range(2, 101)]

def find_patient_files(pid, base_path):
    for root, _, files in os.walk(base_path):
        if pid in root and "features.npy" in files and "y.npy" in files:
            return (os.path.join(root, "features.npy"),
                    os.path.join(root, "y.npy"))
    return None, None

all_X, all_y = [], []
loaded, pid_windows = [], {}

for pid in ALL_PIDS:
    feat_path, y_path = find_patient_files(pid, path)

    if feat_path is None:
        continue

    try:
        X_p = np.load(feat_path).astype(np.float32)
        y_p = np.load(y_path).astype(np.int8)

        if X_p.shape[1] != N_FEATURES or len(X_p) != len(y_p):
            continue

        X_p = np.nan_to_num(X_p)

        all_X.append(X_p)
        all_y.append(y_p)
        loaded.append(pid)
        pid_windows[pid] = len(y_p)

    except:
        continue

X_all = np.concatenate(all_X, axis=0)
y_all = np.concatenate(all_y, axis=0)

print("Total shape:", X_all.shape)
print("Apnea %:", np.mean(y_all))


# ──────────────────────────────────────────────────────────────────────────────
# 4. PATIENT-WISE SPLIT (NO LEAKAGE)
# ──────────────────────────────────────────────────────────────────────────────
np.random.seed(42)
pids = np.array(loaded)
np.random.shuffle(pids)

n = len(pids)
n_test = int(n * 0.15)
n_val  = int(n * 0.15)

test_pids  = set(pids[:n_test])
val_pids   = set(pids[n_test:n_test+n_val])
train_pids = set(pids[n_test+n_val:])

pid_per_window = []
for pid in loaded:
    pid_per_window.extend([pid]*pid_windows[pid])
pid_per_window = np.array(pid_per_window)

train_mask = np.array([p in train_pids for p in pid_per_window])
val_mask   = np.array([p in val_pids for p in pid_per_window])
test_mask  = np.array([p in test_pids for p in pid_per_window])

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val,   y_val   = X_all[val_mask],   y_all[val_mask]
X_test,  y_test  = X_all[test_mask],  y_all[test_mask]


# ──────────────────────────────────────────────────────────────────────────────
# 5. SCALING (FIXED LEAKAGE)
# ──────────────────────────────────────────────────────────────────────────────
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)


# ──────────────────────────────────────────────────────────────────────────────
# 6. CLASS IMBALANCE HANDLING
# ──────────────────────────────────────────────────────────────────────────────
pos_weight = (len(y_train) - np.sum(y_train)) / np.sum(y_train)
print("scale_pos_weight:", pos_weight)


# ──────────────────────────────────────────────────────────────────────────────
# 7. XGBOOST MODEL
# ──────────────────────────────────────────────────────────────────────────────
model = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    eval_metric='auc',
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)


# ──────────────────────────────────────────────────────────────────────────────
# 8. FEATURE SELECTION (IMPORTANT FIX)
# ──────────────────────────────────────────────────────────────────────────────
selector = SelectFromModel(model, threshold="median", prefit=True)

X_train = selector.transform(X_train)
X_val   = selector.transform(X_val)
X_test  = selector.transform(X_test)

print("Selected features:", X_train.shape[1])


# retrain after selection
model.fit(X_train, y_train)


# ──────────────────────────────────────────────────────────────────────────────
# 9. EVALUATION
# ──────────────────────────────────────────────────────────────────────────────
y_prob = model.predict_proba(X_test)[:,1]

# default threshold
y_pred = (y_prob > 0.5).astype(int)

print("\n=== DEFAULT (0.5) ===")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))
print("ROC AUC:", roc_auc_score(y_test, y_prob))


# ──────────────────────────────────────────────────────────────────────────────
# 10. THRESHOLD TUNING (CONTROL FALSE POSITIVES)
# ──────────────────────────────────────────────────────────────────────────────
best_threshold = 0.4  # tuned to reduce FP

y_pred_tuned = (y_prob > best_threshold).astype(int)

print(f"\n=== THRESHOLD = {best_threshold} ===")
print(confusion_matrix(y_test, y_pred_tuned))
print(classification_report(y_test, y_pred_tuned, digits=4))


# ──────────────────────────────────────────────────────────────────────────────
# 11. SAVE MODEL
# ──────────────────────────────────────────────────────────────────────────────
import joblib

joblib.dump(model, "/kaggle/working/xgb_apnea_model.pkl")
joblib.dump(scaler, "/kaggle/working/scaler.pkl")

print("\nModel saved successfully.")

In [ ]:
# Create dataframe of predictions
df_preds = pd.DataFrame({
    "y_true": y_test,
    "y_prob": y_prob,
    "y_pred_default": y_pred,
    "y_pred_tuned": y_pred_tuned
})

df_preds.to_csv("/kaggle/working/test_predictions.csv", index=False)
print("Saved: test_predictions.csv")
# Convert classification report to dataframe
report_default = classification_report(y_test, y_pred, output_dict=True)
report_tuned   = classification_report(y_test, y_pred_tuned, output_dict=True)

df_report_default = pd.DataFrame(report_default).transpose()
df_report_tuned   = pd.DataFrame(report_tuned).transpose()

df_report_default.to_csv("/kaggle/working/classification_report_default.csv")
df_report_tuned.to_csv("/kaggle/working/classification_report_tuned.csv")

print("Saved: classification reports")
cm_default = confusion_matrix(y_test, y_pred)
cm_tuned   = confusion_matrix(y_test, y_pred_tuned)

pd.DataFrame(cm_default).to_csv("/kaggle/working/confusion_matrix_default.csv", index=False)
pd.DataFrame(cm_tuned).to_csv("/kaggle/working/confusion_matrix_tuned.csv", index=False)

print("Saved: confusion matrices")
df_auc = pd.DataFrame({
    "metric": ["ROC_AUC"],
    "value": [roc_auc_score(y_test, y_prob)]
})

df_auc.to_csv("/kaggle/working/metrics.csv", index=False)
print("Saved: metrics.csv")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS
# ──────────────────────────────────────────────────────────────────────────────
import os, gc
import numpy as np
import pandas as pd

from collections import Counter

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.feature_selection import SelectFromModel

from xgboost import XGBClassifier
import kagglehub


# ──────────────────────────────────────────────────────────────────────────────
# 1. DOWNLOAD DATASET
# ──────────────────────────────────────────────────────────────────────────────
path = kagglehub.dataset_download(
    "nishantomhare/disordered-breathing-classification-features-fft")
print("Feature dataset path:", path)


# ──────────────────────────────────────────────────────────────────────────────
# 2. FEATURE NAMES
# ──────────────────────────────────────────────────────────────────────────────
CHANNEL_NAMES = ["pressure","airflow","chest","abdomen","pleth"]
BAND_NAMES    = ["be_0_01","be_01_03","be_03_06","be_06_1",
                 "be_1_2","be_2_5","be_gt5"]

BASE_FEATURES = (
    ["energy","mean_mag","variance","sp_centroid","sp_spread",
     "sp_skewness","sp_kurtosis","sp_flatness","shannon_entropy"]
    + BAND_NAMES +
    ["sp_crest","hjorth_activity","hjorth_mobility","hjorth_complexity",
     "peak1_freq","peak2_freq","peak_ratio","peak_width","peak_prominence"]
)

FEATURE_NAMES = [f"{ch}_{f}" for ch in CHANNEL_NAMES for f in BASE_FEATURES]
N_FEATURES    = len(FEATURE_NAMES)


# ──────────────────────────────────────────────────────────────────────────────
# 3. LOAD DATA
# ──────────────────────────────────────────────────────────────────────────────
ALL_PIDS = [f"SN{i}" for i in range(2, 101)]

def find_patient_files(pid, base_path):
    for root, _, files in os.walk(base_path):
        if pid in root and "features.npy" in files and "y.npy" in files:
            return (os.path.join(root, "features.npy"),
                    os.path.join(root, "y.npy"))
    return None, None

all_X, all_y = [], []
loaded, pid_windows = [], {}

for pid in ALL_PIDS:
    feat_path, y_path = find_patient_files(pid, path)

    if feat_path is None:
        continue

    try:
        X_p = np.load(feat_path).astype(np.float32)
        y_p = np.load(y_path).astype(np.int8)

        if X_p.shape[1] != N_FEATURES or len(X_p) != len(y_p):
            continue

        X_p = np.nan_to_num(X_p)

        all_X.append(X_p)
        all_y.append(y_p)
        loaded.append(pid)
        pid_windows[pid] = len(y_p)

    except:
        continue

X_all = np.concatenate(all_X, axis=0)
y_all = np.concatenate(all_y, axis=0)

print("Total shape:", X_all.shape)
print("Apnea %:", np.mean(y_all))


# ──────────────────────────────────────────────────────────────────────────────
# 4. PATIENT-WISE SPLIT (NO LEAKAGE)
# ──────────────────────────────────────────────────────────────────────────────
np.random.seed(42)
pids = np.array(loaded)
np.random.shuffle(pids)

n = len(pids)
n_test = int(n * 0.15)
n_val  = int(n * 0.15)

test_pids  = set(pids[:n_test])
val_pids   = set(pids[n_test:n_test+n_val])
train_pids = set(pids[n_test+n_val:])

pid_per_window = []
for pid in loaded:
    pid_per_window.extend([pid]*pid_windows[pid])
pid_per_window = np.array(pid_per_window)

train_mask = np.array([p in train_pids for p in pid_per_window])
val_mask   = np.array([p in val_pids for p in pid_per_window])
test_mask  = np.array([p in test_pids for p in pid_per_window])

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val,   y_val   = X_all[val_mask],   y_all[val_mask]
X_test,  y_test  = X_all[test_mask],  y_all[test_mask]


# ──────────────────────────────────────────────────────────────────────────────
# 5. SCALING (FIXED LEAKAGE)
# ──────────────────────────────────────────────────────────────────────────────
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)


# ──────────────────────────────────────────────────────────────────────────────
# 6. CLASS IMBALANCE HANDLING
# ──────────────────────────────────────────────────────────────────────────────
pos_weight = (len(y_train) - np.sum(y_train)) / np.sum(y_train)
print("scale_pos_weight:", pos_weight)


# ──────────────────────────────────────────────────────────────────────────────
# 7. XGBOOST MODEL
# ──────────────────────────────────────────────────────────────────────────────
model = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    eval_metric='auc',
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)


# ──────────────────────────────────────────────────────────────────────────────
# 8. FEATURE SELECTION (IMPORTANT FIX)
# ──────────────────────────────────────────────────────────────────────────────
selector = SelectFromModel(model, threshold="median", prefit=True)

X_train = selector.transform(X_train)
X_val   = selector.transform(X_val)
X_test  = selector.transform(X_test)

print("Selected features:", X_train.shape[1])


# retrain after selection
model.fit(X_train, y_train)


# ──────────────────────────────────────────────────────────────
# 9. FINAL EVALUATION (DETAILED METRICS)
# ──────────────────────────────────────────────────────────────
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_prob = model.predict_proba(X_test)[:, 1]

# Default threshold
threshold = 0.5
y_pred = (y_prob > threshold).astype(int)

# Metrics
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
cm   = confusion_matrix(y_test, y_pred)

print("\n" + "="*60)
print("FINAL RESULTS (THRESHOLD = 0.5)")
print("="*60)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC AUC   : {roc_auc_score(y_test, y_prob):.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

# ──────────────────────────────────────────────────────────────
# 10. THRESHOLD TUNING RESULTS
# ──────────────────────────────────────────────────────────────
best_threshold = 0.4

y_pred_tuned = (y_prob > best_threshold).astype(int)

acc_t  = accuracy_score(y_test, y_pred_tuned)
prec_t = precision_score(y_test, y_pred_tuned)
rec_t  = recall_score(y_test, y_pred_tuned)
f1_t   = f1_score(y_test, y_pred_tuned)
cm_t   = confusion_matrix(y_test, y_pred_tuned)

print("\n" + "="*60)
print(f"FINAL RESULTS (THRESHOLD = {best_threshold})")
print("="*60)

print(f"Accuracy  : {acc_t:.4f}")
print(f"Precision : {prec_t:.4f}")
print(f"Recall    : {rec_t:.4f}")
print(f"F1 Score  : {f1_t:.4f}")

print("\nConfusion Matrix:")
print(cm_t)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned, digits=4))


# ──────────────────────────────────────────────────────────────────────────────
# 11. SAVE MODEL
# ──────────────────────────────────────────────────────────────────────────────
import joblib

joblib.dump(model, "/kaggle/working/xgb_apnea_model.pkl")
joblib.dump(scaler, "/kaggle/working/scaler.pkl")

print("\nModel saved successfully.")

In [ ]:
"""
=============================================================================
 SLEEP APNEA DETECTION — ResNet1D + SE-Attention CLASSIFIER (Kaggle GPU)
 v3 — adds temporal CONTEXT windows + post-hoc probability SMOOTHING on top
      of v2's normalization / augmentation / stratified split / focal loss.
=============================================================================
 Why v3: v2's ROC-AUC was 0.80 (genuinely decent separability) but F1 at the
 best possible single-window threshold only reached ~0.57. That's an
 information-availability problem, not a threshold problem: apnea/hypopnea
 desaturation can lag the airflow event by 10-30+ seconds, which falls
 outside an isolated 30s window. v3 fixes this two ways:

   1. CONTEXT WINDOWS: each training sample is now the target 30s window
      PLUS its immediate neighbor windows (default radius=1 -> 90s total),
      concatenated along the time axis. The label stays the target window's
      label. This gives the model the surrounding physiological context.
   2. TEMPORAL SMOOTHING: after inference, each patient's per-window apnea
      probability sequence is smoothed with a moving average before
      thresholding, since real events span multiple consecutive windows and
      per-window predictions are noisy.

 Also: smaller model + more dropout/weight-decay + gentler focal gamma to
 fight the overfitting seen in v2 (train F1 ~0.81 vs val F1 ~0.50).

 Input  : X_SN{n}.npy -> (num_windows, 8, 6000) float16
          y_SN{n}.npy -> (num_windows,)         int8  (0=normal,1=apnea)

 Honest expectation: with AUC around 0.80, hitting accuracy/precision/
 recall/F1 ALL >=80% simultaneously on window-level classification is above
 what most published PSG deep-learning work achieves. This script pushes
 hard toward that target; if still short, the next lever is EVENT-level
 evaluation (a burst of consecutive apnea-positive windows = one detected
 event, scored against ground-truth events) rather than per-window scoring,
 since that's closer to how these systems are used and scored clinically.
=============================================================================
"""

import os
import gc
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_recall_curve
)

import kagglehub

# ─────────────────────────────────────────────────────────────────────────
# 0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ─────────────────────────────────────────────────────────────────────────
# 0b. GPU / KERNEL-COMPATIBILITY CHECK  (P100 = compute capability 6.0)
# ─────────────────────────────────────────────────────────────────────────
def _gpu_kernels_usable():
    if not torch.cuda.is_available():
        return False
    try:
        x = torch.randn(2, 4, 16, device="cuda")
        conv = nn.Conv1d(4, 4, 3, padding=1).cuda()
        _ = conv(x)
        return True
    except RuntimeError as e:
        print("\n⚠️  GPU kernel check failed:", str(e).splitlines()[0])
        print("⚠️  Falling back to CPU. Switch accelerator to T4/L4 to use GPU.")
        return False

_USE_CUDA = _gpu_kernels_usable()

# ─────────────────────────────────────────────────────────────────────────
# 1. DOWNLOAD DATASET
# ─────────────────────────────────────────────────────────────────────────
DATA_DIR = kagglehub.dataset_download(
    "nishantomhare/detection-of-disordered-breathing-dataset2"
)
print("Dataset path:", DATA_DIR)

def locate_npy_root(root):
    for dirpath, _, filenames in os.walk(root):
        if any(f.startswith("X_SN") and f.endswith(".npy") for f in filenames):
            return dirpath
    return root

DATA_DIR = locate_npy_root(DATA_DIR)
print("Resolved npy root:", DATA_DIR)

OUTPUT_DIR = "/kaggle/working/resnet1d_output_v3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────
# 2. CONFIG
# ─────────────────────────────────────────────────────────────────────────
SKIP_PATIENTS = {
    "SN1", "SN28", "SN14", "SN13", "SN17", "SN18",
    "SN19", "SN20", "SN27", "SN89"
}

TEST_FRAC         = 0.20
VAL_FRAC_OF_TRAIN = 0.12
N_STRAT_BINS      = 5

NUM_CHANNELS   = 8
NUM_CLASSES    = 2
CONTEXT_RADIUS = 1          # 1 -> target window + 1 neighbor each side = 90s input
SMOOTH_RADIUS  = 2          # post-hoc probability smoothing window (in # of windows)

BATCH_SIZE   = 48           # reduced slightly since inputs are 3x longer now
EPOCHS       = 50
LR           = 6e-4
WEIGHT_DECAY = 3e-4
NUM_WORKERS  = 2
PATIENCE     = 10
GRAD_CLIP    = 5.0

LOSS_TYPE            = "focal"
FOCAL_GAMMA          = 1.5     # gentler than v2's 2.0 (less aggressive on hard examples)
USE_WEIGHTED_SAMPLER = True
LABEL_SMOOTHING      = 0.1

AUGMENT           = True
NOISE_STD_FRAC    = 0.03
SCALE_JITTER      = 0.10
MAX_TIME_SHIFT    = 100
CHANNEL_DROPOUT_P = 0.05

NORM_SAMPLE_WINDOWS = 8000

MODEL_BASE_WIDTH = 48        # reduced from 64 -> smaller model, less overfitting
MODEL_DROPOUT    = 0.5       # increased from 0.4

DEVICE = torch.device("cuda" if _USE_CUDA else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| capability:", torch.cuda.get_device_capability(0),
          "| usable:", _USE_CUDA)

# ─────────────────────────────────────────────────────────────────────────
# 3. DISCOVER PATIENTS
# ─────────────────────────────────────────────────────────────────────────
all_x_files = sorted(f for f in os.listdir(DATA_DIR)
                      if f.startswith("X_SN") and f.endswith(".npy"))
all_pids = [f[len("X_"):-len(".npy")] for f in all_x_files]

pids = [p for p in all_pids if p not in SKIP_PATIENTS]
pids = [p for p in pids if os.path.exists(os.path.join(DATA_DIR, f"y_{p}.npy"))]

print(f"\nTotal X files found : {len(all_x_files)}")
print(f"Skipped (requested)  : {sorted(SKIP_PATIENTS & set(all_pids))}")
print(f"Usable patients ({len(pids)}): {pids}")

if len(pids) < 5:
    raise RuntimeError("Not enough usable patients to train/val/test split.")

# ─────────────────────────────────────────────────────────────────────────
# 4. STRATIFIED PATIENT-LEVEL SPLIT
# ─────────────────────────────────────────────────────────────────────────
def patient_apnea_ratio(pid):
    y = np.load(os.path.join(DATA_DIR, f"y_{pid}.npy"), mmap_mode="r")
    n = len(y)
    apnea = int(np.sum(y))
    del y
    return apnea / max(1, n), n, apnea

patient_stats = {pid: patient_apnea_ratio(pid) for pid in pids}
ratios = np.array([patient_stats[p][0] for p in pids])

rng = np.random.default_rng(SEED)
order = np.argsort(ratios)
sorted_pids = np.array(pids)[order]

bin_edges = np.linspace(0, len(sorted_pids), N_STRAT_BINS + 1).astype(int)
train_pids, val_pids, test_pids = [], [], []

for b in range(N_STRAT_BINS):
    bucket = list(sorted_pids[bin_edges[b]:bin_edges[b + 1]])
    rng.shuffle(bucket)
    n = len(bucket)
    n_test = max(1, round(n * TEST_FRAC)) if n >= 3 else (1 if n > 1 else 0)
    remaining = bucket[n_test:]
    n_val = max(1, round(len(remaining) * VAL_FRAC_OF_TRAIN)) if len(remaining) >= 3 else 0
    test_pids.extend(bucket[:n_test])
    val_pids.extend(remaining[:n_val])
    train_pids.extend(remaining[n_val:])

if not val_pids and train_pids:
    val_pids.append(train_pids.pop())
if not test_pids and train_pids:
    test_pids.append(train_pids.pop())

print(f"\nTrain patients ({len(train_pids)}): {sorted(train_pids)}")
print(f"Val   patients ({len(val_pids)}): {sorted(val_pids)}")
print(f"Test  patients ({len(test_pids)}): {sorted(test_pids)}")

for name, plist in [("train", train_pids), ("val", val_pids), ("test", test_pids)]:
    tot_n = sum(patient_stats[p][1] for p in plist)
    tot_a = sum(patient_stats[p][2] for p in plist)
    print(f"  {name:5s}: {tot_n:,} windows, apnea ratio = {tot_a / max(1, tot_n):.3f}")

# ─────────────────────────────────────────────────────────────────────────
# 5. PER-CHANNEL NORMALIZATION STATS (from TRAIN set only)
# ─────────────────────────────────────────────────────────────────────────
def estimate_norm_stats(patient_ids, n_sample_windows):
    sums   = np.zeros(NUM_CHANNELS, dtype=np.float64)
    sqsums = np.zeros(NUM_CHANNELS, dtype=np.float64)
    count  = 0
    per_patient_budget = max(1, n_sample_windows // max(1, len(patient_ids)))
    for pid in patient_ids:
        X = np.load(os.path.join(DATA_DIR, f"X_{pid}.npy"), mmap_mode="r")
        n = X.shape[0]
        if n == 0:
            continue
        k = min(per_patient_budget, n)
        idx = np.random.default_rng(SEED).choice(n, size=k, replace=False)
        sample = X[idx].astype(np.float64)
        sums   += sample.sum(axis=(0, 2))
        sqsums += (sample ** 2).sum(axis=(0, 2))
        count  += k * sample.shape[2]
        del X, sample
        gc.collect()
    mean = sums / count
    var  = sqsums / count - mean ** 2
    std  = np.sqrt(np.clip(var, 1e-8, None))
    return mean.astype(np.float32), std.astype(np.float32)

print("\nEstimating per-channel normalization stats from train set...")
CH_MEAN, CH_STD = estimate_norm_stats(train_pids, NORM_SAMPLE_WINDOWS)
print("Channel means:", np.round(CH_MEAN, 3))
print("Channel stds :", np.round(CH_STD, 3))
np.save(os.path.join(OUTPUT_DIR, "channel_mean.npy"), CH_MEAN)
np.save(os.path.join(OUTPUT_DIR, "channel_std.npy"), CH_STD)

# ─────────────────────────────────────────────────────────────────────────
# 6. DATASET  (context windows + normalization + optional augmentation)
# ─────────────────────────────────────────────────────────────────────────
class ApneaContextDataset(Dataset):
    """
    Each __getitem__ returns the target window concatenated with its
    `context_radius` neighbors on each side (edge-clamped at recording
    boundaries), normalized per-channel. Label = target window's label.
    """
    def __init__(self, data_dir, patient_ids, mean, std,
                 context_radius=1, augment=False):
        self.data_dir = data_dir
        self._cache = {}
        self.index = []          # (pid, local_idx, n_windows_for_pid)
        self.mean = mean.reshape(NUM_CHANNELS, 1)
        self.std  = std.reshape(NUM_CHANNELS, 1)
        self.context_radius = context_radius
        self.augment = augment
        for pid in patient_ids:
            xp = os.path.join(data_dir, f"X_{pid}.npy")
            yp = os.path.join(data_dir, f"y_{pid}.npy")
            if not (os.path.exists(xp) and os.path.exists(yp)):
                continue
            y = np.load(yp, mmap_mode="r")
            n = len(y)
            self.index.extend([(pid, i, n) for i in range(n)])
            del y

    def _arrays(self, pid):
        if pid not in self._cache:
            self._cache[pid] = (
                np.load(os.path.join(self.data_dir, f"X_{pid}.npy"), mmap_mode="r"),
                np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r"),
            )
        return self._cache[pid]

    def __len__(self):
        return len(self.index)

    def labels(self):
        out = np.zeros(len(self.index), dtype=np.int8)
        cache = {}
        for i, (pid, li, _) in enumerate(self.index):
            if pid not in cache:
                cache[pid] = np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r")
            out[i] = cache[pid][li]
        return out

    def __getitem__(self, idx):
        pid, local_idx, n = self.index[idx]
        X_arr, y_arr = self._arrays(pid)

        r = self.context_radius
        neighbor_idxs = np.clip(np.arange(local_idx - r, local_idx + r + 1), 0, n - 1)
        segs = [X_arr[j].astype(np.float32) for j in neighbor_idxs]   # each (8, 6000)
        x = np.concatenate(segs, axis=1)                              # (8, (2r+1)*6000)
        x = (x - self.mean) / self.std

        if self.augment:
            if NOISE_STD_FRAC > 0:
                x = x + np.random.randn(*x.shape).astype(np.float32) * NOISE_STD_FRAC
            if SCALE_JITTER > 0:
                scale = 1.0 + (np.random.rand(NUM_CHANNELS, 1).astype(np.float32) * 2 - 1) * SCALE_JITTER
                x = x * scale
            if MAX_TIME_SHIFT > 0:
                shift = np.random.randint(-MAX_TIME_SHIFT, MAX_TIME_SHIFT + 1)
                if shift != 0:
                    x = np.roll(x, shift, axis=1)
            if CHANNEL_DROPOUT_P > 0 and np.random.rand() < CHANNEL_DROPOUT_P:
                ch = np.random.randint(0, NUM_CHANNELS)
                x[ch, :] = 0.0

        y = int(y_arr[local_idx])
        return torch.from_numpy(x.copy()), torch.tensor(y, dtype=torch.long)


train_ds = ApneaContextDataset(DATA_DIR, train_pids, CH_MEAN, CH_STD,
                                context_radius=CONTEXT_RADIUS, augment=AUGMENT)
val_ds   = ApneaContextDataset(DATA_DIR, val_pids, CH_MEAN, CH_STD,
                                context_radius=CONTEXT_RADIUS, augment=False)
test_ds  = ApneaContextDataset(DATA_DIR, test_pids, CH_MEAN, CH_STD,
                                context_radius=CONTEXT_RADIUS, augment=False)

print(f"\nTrain windows: {len(train_ds):,}")
print(f"Val   windows: {len(val_ds):,}")
print(f"Test  windows: {len(test_ds):,}")
print(f"Context radius: {CONTEXT_RADIUS} -> input length = "
      f"{(2 * CONTEXT_RADIUS + 1) * 6000} samples ({(2 * CONTEXT_RADIUS + 1) * 30}s)")

train_labels = train_ds.labels()
n0, n1 = int((train_labels == 0).sum()), int((train_labels == 1).sum())
total_train = n0 + n1
print(f"\nTrain class counts: normal={n0:,}  apnea={n1:,}")

class_weights = torch.tensor(
    [total_train / (2.0 * n0 + 1e-9), total_train / (2.0 * n1 + 1e-9)],
    dtype=torch.float32
).to(DEVICE)
print("Class weights:", class_weights.tolist())

if USE_WEIGHTED_SAMPLER:
    sample_weights = np.where(train_labels == 1,
                               total_train / (2.0 * n1 + 1e-9),
                               total_train / (2.0 * n0 + 1e-9))
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights),
                                     replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

# shuffle=False is important here: keeps val/test order == dataset.index order,
# which is grouped by patient in time order -> required for temporal smoothing later
val_loader  = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# ─────────────────────────────────────────────────────────────────────────
# 7. RESNET1D + SE-ATTENTION ARCHITECTURE
# ─────────────────────────────────────────────────────────────────────────
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(4, channels // reduction)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels), nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1)
        return x * s


class SEBasicBlock1D(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, downsample=None, k=7):
        super().__init__()
        pad = k // 2
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=k, stride=stride,
                                padding=pad, bias=False)
        self.bn1   = nn.BatchNorm1d(planes)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=k, stride=1,
                                padding=pad, bias=False)
        self.bn2   = nn.BatchNorm1d(planes)
        self.se    = SEBlock1D(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out = out + identity
        return self.relu(out)


class ResNet1D(nn.Module):
    def __init__(self, block, layers, in_channels=8, num_classes=2,
                 base_width=48, k=7, dropout=0.5):
        super().__init__()
        self.in_planes = base_width
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, base_width, kernel_size=15, stride=2,
                      padding=7, bias=False),
            nn.BatchNorm1d(base_width),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )
        self.layer1 = self._make_layer(block, base_width,     layers[0], stride=1, k=k)
        self.layer2 = self._make_layer(block, base_width * 2, layers[1], stride=2, k=k)
        self.layer3 = self._make_layer(block, base_width * 4, layers[2], stride=2, k=k)
        self.layer4 = self._make_layer(block, base_width * 8, layers[3], stride=2, k=k)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(base_width * 8 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, block, planes, num_blocks, stride, k):
        downsample = None
        if stride != 1 or self.in_planes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes * block.expansion),
            )
        layers = [block(self.in_planes, planes, stride, downsample, k=k)]
        self.in_planes = planes * block.expansion
        for _ in range(1, num_blocks):
            layers.append(block(self.in_planes, planes, k=k))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        x = self.dropout(x)
        return self.fc(x)


def resnet_se_1d(in_channels=8, num_classes=2):
    return ResNet1D(SEBasicBlock1D, [2, 2, 3, 2], in_channels, num_classes,
                     base_width=MODEL_BASE_WIDTH, dropout=MODEL_DROPOUT)


model = resnet_se_1d(NUM_CHANNELS, NUM_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel: ResNet-SE-1D | Trainable params: {n_params:,}")

# ─────────────────────────────────────────────────────────────────────────
# 8. LOSS
# ─────────────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5, label_smoothing=0.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, weight=self.alpha,
                              label_smoothing=self.label_smoothing, reduction="none")
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()


if LOSS_TYPE == "focal":
    criterion = FocalLoss(alpha=class_weights, gamma=FOCAL_GAMMA,
                           label_smoothing=LABEL_SMOOTHING)
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=4
)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

# ─────────────────────────────────────────────────────────────────────────
# 9. HELPERS: threshold tuning + temporal smoothing
# ─────────────────────────────────────────────────────────────────────────
def best_f1_threshold(labels, probs):
    prec, rec, thr = precision_recall_curve(labels, probs)
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    if len(thr) == 0:
        return 0.5, 0.0
    best_idx = np.nanargmax(f1s[:-1])
    return float(thr[best_idx]), float(f1s[best_idx])


def smooth_probs_grouped(dataset, probs, radius=2):
    """Moving-average smoothing of per-window probabilities, computed
    separately within each patient's (already time-ordered) window sequence
    -- never smooths across a patient boundary."""
    probs = np.asarray(probs)
    smoothed = np.zeros_like(probs)
    pids_seq = [p for p, _, _ in dataset.index]
    n = len(pids_seq)
    i = 0
    kernel = np.ones(2 * radius + 1) / (2 * radius + 1)
    while i < n:
        j = i
        while j < n and pids_seq[j] == pids_seq[i]:
            j += 1
        seg = probs[i:j]
        padded = np.pad(seg, (radius, radius), mode="edge")
        smoothed[i:j] = np.convolve(padded, kernel, mode="valid")
        i = j
    return smoothed


def run_epoch(loader, train_mode: bool):
    model.train(mode=train_mode)
    total_loss = 0.0
    all_labels, all_probs = [], []

    for X, y in loader:
        X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                logits = model(X)
                loss = criterion(logits, y)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * X.size(0)
        probs = F.softmax(logits.float(), dim=1)[:, 1].detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(loader.dataset))
    return avg_loss, np.array(all_labels), np.array(all_probs)

# ─────────────────────────────────────────────────────────────────────────
# 10. TRAIN LOOP WITH EARLY STOPPING (on smoothed, tuned-threshold val F1)
# ─────────────────────────────────────────────────────────────────────────
best_val_f1 = -1.0
best_state = None
best_threshold = 0.5
epochs_no_improve = 0
history = []

print("\n" + "=" * 65)
print("  TRAINING")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_labels_ep, train_probs_ep = run_epoch(train_loader, train_mode=True)
    val_loss, val_labels, val_probs = run_epoch(val_loader, train_mode=False)

    val_probs_smooth = smooth_probs_grouped(val_ds, val_probs, radius=SMOOTH_RADIUS)
    tuned_thr, tuned_val_f1 = best_f1_threshold(val_labels, val_probs_smooth)

    train_preds05 = (train_probs_ep >= 0.5).astype(int)
    train_f105 = f1_score(train_labels_ep, train_preds05, zero_division=0)
    val_preds05 = (val_probs >= 0.5).astype(int)
    val_f105 = f1_score(val_labels, val_preds05, zero_division=0)

    scheduler.step(tuned_val_f1)
    dt = time.time() - t0

    history.append({
        "epoch": epoch, "train_loss": train_loss, "train_f1@0.5": train_f105,
        "val_loss": val_loss, "val_f1@0.5": val_f105,
        "val_f1@smoothed_tuned": tuned_val_f1, "tuned_threshold": tuned_thr,
        "lr": optimizer.param_groups[0]["lr"],
    })

    print(f"Epoch {epoch:3d}/{EPOCHS} | train_loss={train_loss:.4f} f1@.5={train_f105:.4f} | "
          f"val_loss={val_loss:.4f} f1@.5={val_f105:.4f} f1@smooth+tuned={tuned_val_f1:.4f} "
          f"(thr={tuned_thr:.2f}) | lr={optimizer.param_groups[0]['lr']:.2e} | {dt:.1f}s")

    if tuned_val_f1 > best_val_f1:
        best_val_f1 = tuned_val_f1
        best_threshold = tuned_thr
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        torch.save({"state_dict": best_state, "threshold": best_threshold,
                    "smooth_radius": SMOOTH_RADIUS, "context_radius": CONTEXT_RADIUS},
                   os.path.join(OUTPUT_DIR, "best_resnet1d_se.pt"))
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break

pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)

if best_state is not None:
    model.load_state_dict(best_state)
print(f"\nBest val F1 (smoothed + tuned threshold): {best_val_f1:.4f}  @ threshold={best_threshold:.3f}")

# ─────────────────────────────────────────────────────────────────────────
# 11. FINAL TEST-SET EVALUATION  (raw@0.5, raw@tuned, smoothed@tuned)
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  TEST SET EVALUATION")
print("=" * 65)

_, test_labels, test_probs = run_epoch(test_loader, train_mode=False)
test_probs_smooth = smooth_probs_grouped(test_ds, test_probs, radius=SMOOTH_RADIUS)

try:
    auc = roc_auc_score(test_labels, test_probs)
except ValueError:
    auc = float("nan")
print(f"\nROC-AUC (threshold-independent, raw probs): {auc:.4f}")

def report_at_threshold(probs, labels, thr, label):
    preds = (probs >= thr).astype(int)
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)
    cm   = confusion_matrix(labels, preds)
    rep  = classification_report(labels, preds, target_names=["Normal", "Apnea"], zero_division=0)
    print(f"\n--- {label} (threshold={thr:.3f}) ---")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    print(f"\nClassification Report:\n{rep}")
    return dict(label=label, threshold=thr, accuracy=acc, precision=prec, recall=rec, f1_score=f1,
                tn=int(cm[0, 0]), fp=int(cm[0, 1]), fn=int(cm[1, 0]), tp=int(cm[1, 1])), rep, cm

m_raw05, rep_raw05, cm_raw05 = report_at_threshold(test_probs, test_labels, 0.5, "RAW @ 0.5")
raw_tuned_thr, _ = best_f1_threshold(test_labels, test_probs)  # for reference/diagnostics only
m_smooth_tuned, rep_smooth_tuned, cm_smooth_tuned = report_at_threshold(
    test_probs_smooth, test_labels, best_threshold, "SMOOTHED @ val-tuned threshold")

all_metrics = pd.DataFrame([m_raw05, m_smooth_tuned])
all_metrics.to_csv(os.path.join(OUTPUT_DIR, "test_metrics_all.csv"), index=False)
with open(os.path.join(OUTPUT_DIR, "classification_report_smoothed_tuned.txt"), "w") as fh:
    fh.write(rep_smooth_tuned)

# ─────────────────────────────────────────────────────────────────────────
# 11b. EVENT-LEVEL EVALUATION  (AHI-style: did we detect the EVENT, not the
#      individual 30s window — this is closer to how apnea detection is
#      actually scored clinically, and is much less punishing than
#      window-level scoring for events that span multiple windows)
# ─────────────────────────────────────────────────────────────────────────
def extract_events(binary_seq):
    """Runs of consecutive 1s -> list of (start_idx, end_idx) inclusive."""
    events = []
    start = None
    for i, v in enumerate(binary_seq):
        if v == 1 and start is None:
            start = i
        elif v == 0 and start is not None:
            events.append((start, i - 1))
            start = None
    if start is not None:
        events.append((start, len(binary_seq) - 1))
    return events


def match_events(pred_events, true_events, min_overlap_windows=1):
    """Greedy overlap matching: a predicted event counts as a true positive
    if it overlaps an as-yet-unmatched ground-truth event by at least
    `min_overlap_windows` windows."""
    matched_true = set()
    tp = 0
    for ps, pe in pred_events:
        for ti, (ts, te) in enumerate(true_events):
            if ti in matched_true:
                continue
            overlap = max(0, min(pe, te) - max(ps, ts) + 1)
            if overlap >= min_overlap_windows:
                matched_true.add(ti)
                tp += 1
                break
    fp = len(pred_events) - tp
    fn = len(true_events) - len(matched_true)
    return tp, fp, fn, len(pred_events), len(true_events)


def event_level_evaluation(dataset, labels, probs, threshold, min_overlap_windows=1):
    preds = (probs >= threshold).astype(int)
    pids_seq = [p for p, _, _ in dataset.index]
    n = len(pids_seq)
    agg = dict(tp=0, fp=0, fn=0, n_pred_events=0, n_true_events=0)
    i = 0
    while i < n:
        j = i
        while j < n and pids_seq[j] == pids_seq[i]:
            j += 1
        pred_events = extract_events(preds[i:j])
        true_events = extract_events(labels[i:j])
        tp, fp, fn, npe, nte = match_events(pred_events, true_events, min_overlap_windows)
        agg["tp"] += tp; agg["fp"] += fp; agg["fn"] += fn
        agg["n_pred_events"] += npe; agg["n_true_events"] += nte
        i = j
    precision = agg["tp"] / (agg["tp"] + agg["fp"]) if (agg["tp"] + agg["fp"]) > 0 else 0.0
    recall    = agg["tp"] / (agg["tp"] + agg["fn"]) if (agg["tp"] + agg["fn"]) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    agg["precision"], agg["recall"], agg["f1_score"] = precision, recall, f1
    agg["accuracy_proxy"] = agg["tp"] / max(1, agg["n_true_events"])  # events detected / total events
    return agg


print("\n" + "=" * 65)
print("  EVENT-LEVEL EVALUATION (AHI-style — scored on events, not windows)")
print("=" * 65)

event_metrics = event_level_evaluation(test_ds, test_labels, test_probs_smooth,
                                        best_threshold, min_overlap_windows=1)
print(f"\nGround-truth events : {event_metrics['n_true_events']}")
print(f"Predicted events    : {event_metrics['n_pred_events']}")
print(f"True positives      : {event_metrics['tp']}")
print(f"False positives     : {event_metrics['fp']}")
print(f"False negatives     : {event_metrics['fn']}")
print(f"Event Precision     : {event_metrics['precision']:.4f}")
print(f"Event Recall        : {event_metrics['recall']:.4f}")
print(f"Event F1 Score      : {event_metrics['f1_score']:.4f}")
print(f"Detection rate (TP / total true events): {event_metrics['accuracy_proxy']:.4f}")

pd.DataFrame([event_metrics]).to_csv(os.path.join(OUTPUT_DIR, "event_level_metrics.csv"), index=False)
print(f"\nSaved: {os.path.join(OUTPUT_DIR, 'event_level_metrics.csv')}")

# ─────────────────────────────────────────────────────────────────────────
# 12. PLOTS
# ─────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im = axes[0].imshow(cm_smooth_tuned, cmap="Blues")
axes[0].set_title(f"Confusion Matrix (smoothed, thr={best_threshold:.2f})")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["Normal", "Apnea"])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(["Normal", "Apnea"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm_smooth_tuned[i, j]), ha="center", va="center",
                      color="white" if cm_smooth_tuned[i, j] > cm_smooth_tuned.max() / 2 else "black")
fig.colorbar(im, ax=axes[0])

if not np.isnan(auc):
    fpr, tpr, _ = roc_curve(test_labels, test_probs)
    axes[1].plot(fpr, tpr, label=f"AUC = {auc:.3f}")
    axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
    axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title("ROC Curve (raw probs)"); axes[1].legend()

hist_df = pd.DataFrame(history)
axes[2].plot(hist_df["epoch"], hist_df["train_f1@0.5"], label="Train F1@0.5")
axes[2].plot(hist_df["epoch"], hist_df["val_f1@smoothed_tuned"], label="Val F1 (smoothed+tuned)")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("F1 Score")
axes[2].set_title("Training Curves"); axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "evaluation_plots.png"), dpi=150)
plt.show()

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(" - best_resnet1d_se.pt   (weights + tuned threshold + smoothing/context radii)")
print(" - channel_mean.npy / channel_std.npy   (apply these at inference time!)")
print(" - training_history.csv")
print(" - test_metrics_all.csv   (raw@0.5 vs smoothed@tuned side by side)")
print(" - classification_report_smoothed_tuned.txt")
print(" - evaluation_plots.png")
print("\nNOTE: at inference time, replicate BOTH the context-window concatenation "
      f"(radius={CONTEXT_RADIUS}) AND the temporal smoothing (radius={SMOOTH_RADIUS}) "
      "— the reported 'SMOOTHED' metrics above only apply if you use both at "
      "prediction time.")

In [ ]:
# =============================================================================
#  DISORDERED BREATHING — ANOVA FEATURE SELECTION + GRADIENT BOOSTING / RF
#  Feature selection : ANOVA F-test (SelectKBest) 125 -> top K (supervised,
#                       replaces unsupervised PCA which optimizes variance,
#                       not class separability)
#  Models            : HistGradientBoostingClassifier + RandomForestClassifier
#                       (replace SVM — trees handle non-linear interactions
#                        and class imbalance far better on this kind of
#                        tabular biosignal-feature data, and train on the
#                        FULL training set instead of a 20K subsample)
# =============================================================================

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS
# ──────────────────────────────────────────────────────────────────────────────
import kagglehub
import os, gc, warnings, time, copy
import numpy as np
import pandas as pd
from collections import Counter
warnings.filterwarnings("ignore")

from sklearn.preprocessing        import StandardScaler
from sklearn.feature_selection    import SelectKBest, f_classif
from sklearn.ensemble              import (HistGradientBoostingClassifier,
                                           RandomForestClassifier)
from sklearn.utils.class_weight   import compute_class_weight
from sklearn.metrics              import (classification_report,
                                          confusion_matrix,
                                          roc_auc_score,
                                          recall_score,
                                          precision_score,
                                          f1_score,
                                          accuracy_score,
                                          ConfusionMatrixDisplay,
                                          RocCurveDisplay,
                                          PrecisionRecallDisplay)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

OUT = "/kaggle/working"

# ──────────────────────────────────────────────────────────────────────────────
# 1.  DOWNLOAD DATASET
# ──────────────────────────────────────────────────────────────────────────────
path = kagglehub.dataset_download(
    "nishantomhare/disordered-breathing-classification-features-fft")
print("Dataset path:", path)

# ──────────────────────────────────────────────────────────────────────────────
# 2.  FEATURE NAMES
# ──────────────────────────────────────────────────────────────────────────────
CHANNEL_NAMES = ["pressure","airflow","chest","abdomen","pleth"]
BAND_NAMES    = ["be_0_01","be_01_03","be_03_06","be_06_1",
                 "be_1_2","be_2_5","be_gt5"]
BASE_FEATURES = (
    ["energy","mean_mag","variance","sp_centroid","sp_spread",
     "sp_skewness","sp_kurtosis","sp_flatness","shannon_entropy"]
    + BAND_NAMES
    + ["sp_crest","hjorth_activity","hjorth_mobility","hjorth_complexity",
       "peak1_freq","peak2_freq","peak_ratio","peak_width","peak_prominence"]
)
FEATURE_NAMES = [f"{ch}_{f}" for ch in CHANNEL_NAMES for f in BASE_FEATURES]
N_FEATURES    = len(FEATURE_NAMES)   # 125
N_SELECT      = 60                   # ANOVA-selected feature count (tune 40-80)
ALL_PIDS      = [f"SN{i}" for i in range(2, 101)]

def find_patient_files(pid, base):
    for root, _, files in os.walk(base):
        if pid in root and "features.npy" in files and "y.npy" in files:
            return os.path.join(root,"features.npy"), os.path.join(root,"y.npy")
    return None, None

# ──────────────────────────────────────────────────────────────────────────────
# 3.  LOAD ALL PATIENTS
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("  STEP 1 — LOADING FEATURE DATA")
print(f"{'='*65}")

all_X, all_y = [], []
loaded, skipped, summary = [], [], []
pid_windows = {}

for pid in ALL_PIDS:
    fp, yp = find_patient_files(pid, path)
    if fp is None:
        skipped.append(pid)
        continue
    try:
        X_p = np.load(fp).astype(np.float32)
        y_p = np.load(yp).astype(np.int8)
        if X_p.ndim!=2 or X_p.shape[1]!=N_FEATURES or len(X_p)!=len(y_p):
            print(f"  ⚠️  [{pid}] bad shape {X_p.shape} → skipped")
            skipped.append(pid); continue
        X_p = np.nan_to_num(X_p, nan=0.0, posinf=0.0, neginf=0.0)
        d   = Counter(y_p.tolist())
        n   = len(y_p)
        all_X.append(X_p); all_y.append(y_p)
        pid_windows[pid] = n
        loaded.append(pid)
        summary.append({"patient":pid,"windows":n,
                        "normal":d.get(0,0),"apnea":d.get(1,0),
                        "apnea_pct":round(100*d.get(1,0)/n,1)})
        print(f"  ✅ [{pid}]  windows={n:4d}  "
              f"normal={d.get(0,0):4d}  apnea={d.get(1,0):4d} "
              f"({round(100*d.get(1,0)/n,1):.1f}%)")
        del X_p, y_p; gc.collect()
    except Exception as e:
        print(f"  ❌ [{pid}] {e}")
        skipped.append(pid)

print(f"\n  Loaded  : {len(loaded)} patients")
print(f"  Skipped : {len(skipped)}  {skipped}")

X_all = np.concatenate(all_X, axis=0).astype(np.float32)
y_all = np.concatenate(all_y, axis=0).astype(np.int8)
del all_X, all_y; gc.collect()

dist  = Counter(y_all.tolist())
total = len(y_all)
print(f"\n  Combined dataset:")
print(f"  X shape      : {X_all.shape}  (windows × 125 features)")
print(f"  Normal  (0)  : {dist[0]:,}  ({100*dist[0]/total:.1f}%)")
print(f"  Apnea   (1)  : {dist[1]:,}  ({100*dist[1]/total:.1f}%)")
print(f"  Imbalance    : {dist[0]/dist[1]:.2f}:1")

pd.DataFrame(summary).to_csv(f"{OUT}/patient_summary.csv", index=False)

# ──────────────────────────────────────────────────────────────────────────────
# 4.  TRAIN / VAL / TEST SPLIT  BY PATIENT (unchanged — avoids leakage)
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("  STEP 2 — TRAIN / TEST SPLIT  (80 / 20 by patient)")
print(f"{'='*65}")

np.random.seed(42)
pids_arr = np.array(sorted(loaded))
np.random.shuffle(pids_arr)

n_test     = max(1, int(len(pids_arr) * 0.20))
n_val      = max(1, int(len(pids_arr) * 0.10))
test_pids  = set(pids_arr[:n_test])
val_pids   = set(pids_arr[n_test:n_test+n_val])
train_pids = set(pids_arr[n_test+n_val:])

print(f"  Train patients : {len(train_pids)}")
print(f"  Val   patients : {len(val_pids)}  (threshold tuning only)")
print(f"  Test  patients : {len(test_pids)}")

pid_per_win = []
for pid in loaded:
    pid_per_win.extend([pid]*pid_windows[pid])
pid_per_win = np.array(pid_per_win)

tr_m = np.array([p in train_pids for p in pid_per_win])
va_m = np.array([p in val_pids   for p in pid_per_win])
te_m = np.array([p in test_pids  for p in pid_per_win])

X_tr, y_tr = X_all[tr_m], y_all[tr_m]
X_va, y_va = X_all[va_m], y_all[va_m]
X_te, y_te = X_all[te_m], y_all[te_m]

print(f"\n  Train : {len(y_tr):,} windows  "
      f"apnea={Counter(y_tr)[1]:,}({100*Counter(y_tr)[1]/len(y_tr):.1f}%)")
print(f"  Val   : {len(y_va):,} windows  "
      f"apnea={Counter(y_va)[1]:,}({100*Counter(y_va)[1]/len(y_va):.1f}%)")
print(f"  Test  : {len(y_te):,} windows  "
      f"apnea={Counter(y_te)[1]:,}({100*Counter(y_te)[1]/len(y_te):.1f}%)")

# ──────────────────────────────────────────────────────────────────────────────
# 5.  STANDARDISE  (kept — helps ANOVA-F comparability across feature units,
#                    and costs nothing for tree models)
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("  STEP 3 — STANDARDISATION")
print(f"{'='*65}")

scaler   = StandardScaler()
X_tr_s   = scaler.fit_transform(X_tr)
X_va_s   = scaler.transform(X_va)
X_te_s   = scaler.transform(X_te)
print(f"  Scaled: mean≈{X_tr_s.mean():.4f}  std≈{X_tr_s.std():.4f}")

# ──────────────────────────────────────────────────────────────────────────────
# 6.  ANOVA F-TEST FEATURE SELECTION   125 → top N_SELECT
#     Unlike PCA (unsupervised, keeps high-variance directions), ANOVA F-test
#     is supervised: it scores each feature by how much its class means
#     (Normal vs Apnea) differ relative to within-class variance. Features
#     get RANKED and KEPT AS-IS (no rotation), so the model still trains on
#     interpretable, real physiological features — not opaque components.
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"  STEP 4 — ANOVA F-TEST FEATURE SELECTION  125 → {N_SELECT}")
print(f"{'='*65}")

t0 = time.time()
selector = SelectKBest(score_func=f_classif, k=N_SELECT)
X_tr_sel = selector.fit_transform(X_tr_s, y_tr)
X_va_sel = selector.transform(X_va_s)
X_te_sel = selector.transform(X_te_s)

f_scores   = selector.scores_
p_values   = selector.pvalues_
support    = selector.get_support()
sel_names  = [FEATURE_NAMES[i] for i in range(N_FEATURES) if support[i]]

order = np.argsort(f_scores)[::-1]
print(f"  Selector fitted in : {time.time()-t0:.1f}s")
print(f"  Features kept      : {N_SELECT} / {N_FEATURES}")
print(f"\n  Top 20 features by ANOVA F-score:")
for rank, idx in enumerate(order[:20], 1):
    star = "✅" if support[idx] else "  "
    print(f"   {rank:2d}. {star} {FEATURE_NAMES[idx]:<28} "
          f"F={f_scores[idx]:9.1f}  p={p_values[idx]:.2e}")

anova_df = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "f_score": f_scores,
    "p_value": p_values,
    "selected": support,
}).sort_values("f_score", ascending=False)
anova_df.to_csv(f"{OUT}/anova_feature_scores.csv", index=False)

# ── ANOVA F-score plot (replaces PCA scree plot) ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f"ANOVA F-Test Feature Selection — 125 → {N_SELECT}",
             fontsize=13, fontweight="bold")

top_n = 40
colors = ["seagreen" if support[i] else "lightgray" for i in order[:top_n]]
axes[0].barh(range(top_n), f_scores[order[:top_n]][::-1],
             color=colors[::-1])
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels([FEATURE_NAMES[i] for i in order[:top_n]][::-1],
                         fontsize=7)
axes[0].set_xlabel("ANOVA F-score")
axes[0].set_title(f"Top {top_n} Features by F-score\n(green = selected)")
axes[0].grid(True, alpha=0.3, axis="x")

axes[1].plot(range(1, N_FEATURES+1), f_scores[order], "b-o", ms=3, lw=1.2)
axes[1].axvline(N_SELECT, color="orange", linestyle="--", lw=2,
                label=f"cutoff k={N_SELECT}")
axes[1].set_xlabel("Feature rank"); axes[1].set_ylabel("ANOVA F-score")
axes[1].set_title("F-score decay across ranked features")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT}/anova_feature_selection.png", dpi=120, bbox_inches="tight")
plt.close()
print(f"\n  ✅ anova_feature_selection.png saved")
print(f"  ✅ anova_feature_scores.csv saved")

# ──────────────────────────────────────────────────────────────────────────────
# 7.  CLASS WEIGHTS
# ──────────────────────────────────────────────────────────────────────────────
cw          = compute_class_weight("balanced", classes=np.array([0,1]), y=y_tr)
cwd         = {0: float(cw[0]), 1: float(cw[1])}
sample_w_tr = np.where(y_tr==1, cw[1], cw[0]).astype(np.float32)
print(f"\n  Class weights  : normal={cw[0]:.3f}  apnea={cw[1]:.3f}")

# ──────────────────────────────────────────────────────────────────────────────
# 8.  MODEL TRAINING
#     Model A: HistGradientBoostingClassifier — fast, native class_weight,
#              handles non-linear feature interactions, trains on FULL data.
#     Model B: RandomForestClassifier — bagged trees, very robust to noisy
#              biosignal features, also trains on FULL data.
#     Model C: Ensemble (average of A + B probabilities)
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("  STEP 5 — MODEL TRAINING")
print(f"{'='*65}")

all_results  = []
thresh_store = {}

def full_metrics(name, y_true, y_prob, threshold, split):
    y_pred = (y_prob >= threshold).astype(int)
    acc    = accuracy_score(y_true, y_pred)
    rec    = recall_score(y_true,    y_pred, zero_division=0)
    prec   = precision_score(y_true, y_pred, zero_division=0)
    f1     = f1_score(y_true,        y_pred, zero_division=0)
    auc    = roc_auc_score(y_true, y_prob)
    f2     = (5*prec*rec)/(4*prec+rec) if (4*prec+rec) > 0 else 0

    print(f"\n  ── [{name}]  {split.upper()}  threshold={threshold:.2f} ──")
    print(f"  Accuracy  : {100*acc:.2f}%")
    print(f"  Recall    : {100*rec:.2f}%   ← apnea detection rate")
    print(f"  Precision : {100*prec:.2f}%")
    print(f"  F1-Score  : {100*f1:.2f}%")
    print(f"  F2-Score  : {f2:.4f}   (weights recall 2×)")
    print(f"  AUC-ROC   : {auc:.4f}")
    print(classification_report(y_true, y_pred,
          target_names=["Normal","Apnea"], digits=3))

    all_results.append({
        "model":name, "split":split, "threshold":round(threshold,2),
        "accuracy":round(100*acc,2), "recall":round(100*rec,2),
        "precision":round(100*prec,2), "f1":round(100*f1,2),
        "f2":round(f2,4), "auc":round(auc,4),
    })
    return acc, rec, prec, f1, auc

def best_threshold(y_true, y_prob, label="model"):
    rows, best_t, best_f2 = [], 0.5, 0.0
    print(f"\n  Threshold sweep [{label}] on validation:")
    print(f"  {'THRESH':>7}  {'ACC':>8}  {'RECALL':>8}  "
          f"{'PRECIS':>8}  {'F1':>8}  {'F2':>8}")
    print("  " + "-"*55)
    for t in np.arange(0.10, 0.85, 0.05):
        yp   = (y_prob >= t).astype(int)
        acc  = accuracy_score(y_true, yp)
        rec  = recall_score(y_true,    yp, zero_division=0)
        prec = precision_score(y_true, yp, zero_division=0)
        f1   = f1_score(y_true,        yp, zero_division=0)
        f2   = (5*prec*rec)/(4*prec+rec) if (4*prec+rec) > 0 else 0
        flag = " ←" if f2 > best_f2 else ""
        print(f"  {t:>7.2f}  {100*acc:>7.1f}%  {100*rec:>7.1f}%  "
              f"{100*prec:>7.1f}%  {100*f1:>7.1f}%  {f2:>8.4f}{flag}")
        rows.append({"threshold":round(t,2),"accuracy":round(100*acc,1),
                     "recall":round(100*rec,1),"precision":round(100*prec,1),
                     "f1":round(100*f1,1),"f2":round(f2,4)})
        if f2 > best_f2:
            best_f2, best_t = f2, t
    print(f"\n  Best threshold: {best_t:.2f}  (F2={best_f2:.4f})")
    return best_t, pd.DataFrame(rows)

# ─────────────────────────────────────────────────────────────────────────────
# Model A: HistGradientBoostingClassifier (epoch-by-epoch, incremental trees)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n── Model A: HistGradientBoosting (incremental, epoch-level metrics) ──")
print(f"   Data: {X_tr_sel.shape[0]:,} windows × {N_SELECT} ANOVA-selected features")

EPOCHS = 120
hgb = HistGradientBoostingClassifier(
    max_iter          = 1,
    warm_start        = True,
    learning_rate     = 0.08,
    max_leaf_nodes    = 31,
    max_depth         = None,
    l2_regularization = 1.0,
    class_weight      = "balanced",
    early_stopping    = False,       # we drive early stopping manually vs val
    random_state      = 42,
)

epoch_metrics = {k: [] for k in ["acc","rec","prec","f1","val_acc",
                                   "val_rec","val_prec","val_f1","val_auc"]}

print(f"\n  {'EP':>3}  {'TR_ACC':>8}  {'TR_REC':>8}  {'TR_PRE':>8}  "
      f"{'TR_F1':>8}  │  {'VA_ACC':>8}  {'VA_REC':>8}  "
      f"{'VA_PRE':>8}  {'VA_F1':>8}  {'VA_AUC':>8}")
print("  " + "─"*95)

best_val_auc_hgb, best_hgb_iter = 0.0, 1
best_hgb_model = None
patience, pat_cnt = 15, 0

for ep in range(1, EPOCHS+1):
    hgb.max_iter = ep
    hgb.fit(X_tr_sel, y_tr)

    tr_prob = hgb.predict_proba(X_tr_sel)[:,1]
    tr_pred = (tr_prob >= 0.5).astype(int)
    tr_acc  = accuracy_score(y_tr, tr_pred)
    tr_rec  = recall_score(y_tr,    tr_pred, zero_division=0)
    tr_prec = precision_score(y_tr, tr_pred, zero_division=0)
    tr_f1   = f1_score(y_tr,        tr_pred, zero_division=0)

    va_prob = hgb.predict_proba(X_va_sel)[:,1]
    va_pred = (va_prob >= 0.5).astype(int)
    va_acc  = accuracy_score(y_va, va_pred)
    va_rec  = recall_score(y_va,    va_pred, zero_division=0)
    va_prec = precision_score(y_va, va_pred, zero_division=0)
    va_f1   = f1_score(y_va,        va_pred, zero_division=0)
    va_auc  = roc_auc_score(y_va,   va_prob)

    for k,v in zip(["acc","rec","prec","f1","val_acc","val_rec",
                    "val_prec","val_f1","val_auc"],
                   [tr_acc,tr_rec,tr_prec,tr_f1,
                    va_acc,va_rec,va_prec,va_f1,va_auc]):
        epoch_metrics[k].append(v)

    flag = ""
    if va_auc > best_val_auc_hgb:
        best_val_auc_hgb, best_hgb_iter = va_auc, ep
        best_hgb_model = copy.deepcopy(hgb)   # snapshot — never shrink a warm_start model
        pat_cnt, flag = 0, " ✅"
    else:
        pat_cnt += 1
        if pat_cnt >= patience:
            print(f"\n  Early stopping at epoch {ep} (patience={patience})")
            break

    print(f"  {ep:>3}  {100*tr_acc:>7.1f}%  {100*tr_rec:>7.1f}%  "
          f"{100*tr_prec:>7.1f}%  {100*tr_f1:>7.1f}%  │  "
          f"{100*va_acc:>7.1f}%  {100*va_rec:>7.1f}%  "
          f"{100*va_prec:>7.1f}%  {100*va_f1:>7.1f}%  {va_auc:.4f}{flag}")

# use the best-so-far snapshot directly — do NOT try to re-fit a warm_start
# model back down to an earlier iteration count, sklearn forbids shrinking it
hgb = best_hgb_model
print(f"\n  Best val AUC: {best_val_auc_hgb:.4f}  (at {best_hgb_iter} boosting rounds)")

hgb_va_prob        = hgb.predict_proba(X_va_sel)[:,1]
hgb_thresh, hgb_td = best_threshold(y_va, hgb_va_prob, "HistGB")
thresh_store["HGB"] = hgb_td

hgb_te_prob = hgb.predict_proba(X_te_sel)[:,1]
hgb_acc, hgb_rec, hgb_prec, hgb_f1, hgb_auc = full_metrics(
    "HistGB_ANOVA60", y_te, hgb_te_prob, hgb_thresh, "test")

# ─────────────────────────────────────────────────────────────────────────────
# Model B: RandomForestClassifier (incremental trees, full training set)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n── Model B: RandomForest (incremental, full training set) ──")
rf = RandomForestClassifier(
    n_estimators      = 10,
    warm_start        = True,
    class_weight      = "balanced_subsample",
    max_depth         = 16,
    min_samples_leaf  = 5,
    max_features      = "sqrt",
    n_jobs            = -1,
    random_state      = 42,
)

RF_EPOCHS, TREES_PER_EPOCH = 40, 20
best_val_auc_rf, best_rf_trees = 0.0, TREES_PER_EPOCH
best_rf_model = None
pat_cnt = 0

print(f"\n  {'EP':>3}  {'N_TREES':>8}  {'VA_ACC':>8}  {'VA_REC':>8}  "
      f"{'VA_PRE':>8}  {'VA_F1':>8}  {'VA_AUC':>8}")
print("  " + "─"*70)

for ep in range(1, RF_EPOCHS+1):
    rf.n_estimators = ep * TREES_PER_EPOCH
    rf.fit(X_tr_sel, y_tr)

    va_prob = rf.predict_proba(X_va_sel)[:,1]
    va_pred = (va_prob >= 0.5).astype(int)
    va_acc  = accuracy_score(y_va, va_pred)
    va_rec  = recall_score(y_va,    va_pred, zero_division=0)
    va_prec = precision_score(y_va, va_pred, zero_division=0)
    va_f1   = f1_score(y_va,        va_pred, zero_division=0)
    va_auc  = roc_auc_score(y_va,   va_prob)

    flag = ""
    if va_auc > best_val_auc_rf:
        best_val_auc_rf, best_rf_trees = va_auc, rf.n_estimators
        best_rf_model = copy.deepcopy(rf)     # snapshot — never shrink a warm_start forest
        pat_cnt, flag = 0, " ✅"
    else:
        pat_cnt += 1
        if pat_cnt >= 8:
            print(f"\n  Early stopping at {rf.n_estimators} trees (patience=8)")
            break

    print(f"  {ep:>3}  {rf.n_estimators:>8}  {100*va_acc:>7.1f}%  "
          f"{100*va_rec:>7.1f}%  {100*va_prec:>7.1f}%  {100*va_f1:>7.1f}%  "
          f"{va_auc:.4f}{flag}")

# use the best-so-far snapshot directly — same reasoning as HGB above
rf = best_rf_model
print(f"\n  Best val AUC: {best_val_auc_rf:.4f}  (at {best_rf_trees} trees)")

rf_va_prob       = rf.predict_proba(X_va_sel)[:,1]
rf_thresh, rf_td = best_threshold(y_va, rf_va_prob, "RandomForest")
thresh_store["RF"] = rf_td

rf_te_prob = rf.predict_proba(X_te_sel)[:,1]
rf_acc, rf_rec, rf_prec, rf_f1, rf_auc = full_metrics(
    "RandomForest_ANOVA60", y_te, rf_te_prob, rf_thresh, "test")

# ─────────────────────────────────────────────────────────────────────────────
# Model C: Ensemble (HistGB + RandomForest average)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n── Model C: Ensemble (HistGB + RandomForest) ──")
ens_va_prob        = (hgb_va_prob + rf_va_prob) / 2
ens_thresh, ens_td = best_threshold(y_va, ens_va_prob, "Ensemble")
thresh_store["ENS"] = ens_td

ens_te_prob = (hgb_te_prob + rf_te_prob) / 2
ens_acc, ens_rec, ens_prec, ens_f1, ens_auc = full_metrics(
    "Ensemble_ANOVA60", y_te, ens_te_prob, ens_thresh, "test")

# ──────────────────────────────────────────────────────────────────────────────
# 9.  FEATURE IMPORTANCE (bonus — trees give this for free, PCA/SVM don't)
# ──────────────────────────────────────────────────────────────────────────────
imp_df = pd.DataFrame({
    "feature": sel_names,
    "rf_importance": rf.feature_importances_,
}).sort_values("rf_importance", ascending=False)
imp_df.to_csv(f"{OUT}/rf_feature_importance.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 8))
top_imp = imp_df.head(25).iloc[::-1]
ax.barh(top_imp["feature"], top_imp["rf_importance"], color="teal")
ax.set_title("Top 25 Features — RandomForest Importance", fontsize=12)
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig(f"{OUT}/rf_feature_importance.png", dpi=120, bbox_inches="tight")
plt.close()
print("\n  ✅ rf_feature_importance.png / .csv saved")

# ──────────────────────────────────────────────────────────────────────────────
# 10.  PLOTS
# ──────────────────────────────────────────────────────────────────────────────
print("\n  Generating plots...")

# ── Training curves (HGB epoch-by-epoch) ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("HistGradientBoosting Training Curves (ANOVA-60 features)",
             fontsize=14, fontweight="bold")
eps = range(1, len(epoch_metrics["acc"])+1)

for ax, (tr_k, va_k, title, ylabel) in zip(axes.flatten(), [
    ("acc",  "val_acc",  "Accuracy",  "Accuracy (%)"),
    ("rec",  "val_rec",  "Recall",    "Recall (%)"),
    ("prec", "val_prec", "Precision", "Precision (%)"),
    ("f1",   "val_f1",   "F1-Score",  "F1 (%)"),
]):
    ax.plot(eps, [v*100 for v in epoch_metrics[tr_k]],
            "steelblue", lw=2, label="Train")
    ax.plot(eps, [v*100 for v in epoch_metrics[va_k]],
            "crimson",   lw=2, label="Val", linestyle="--")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Boosting round"); ax.set_ylabel(ylabel)
    best_ep = int(np.argmax(epoch_metrics[va_k])) + 1
    ax.axvline(best_ep, color="orange", linestyle=":", lw=1.5,
               label=f"Best ep={best_ep}")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT}/training_curves.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ training_curves.png")

# ── Val AUC per round ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(eps, epoch_metrics["val_auc"], "purple", lw=2)
ax.set_title("Validation AUC-ROC per Boosting Round (HistGB)", fontsize=12)
ax.set_xlabel("Boosting round"); ax.set_ylabel("AUC-ROC")
ax.axvline(int(np.argmax(epoch_metrics["val_auc"]))+1,
           color="orange", linestyle="--", lw=2,
           label=f"Best AUC={max(epoch_metrics['val_auc']):.4f}")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/val_auc_curve.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ val_auc_curve.png")

# ── Confusion matrices ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Confusion Matrices — Test Set (ANOVA-60 features)",
             fontsize=13, fontweight="bold")
for ax, (name, yp, thresh) in zip(axes, [
    ("HistGB",   (hgb_te_prob >= hgb_thresh).astype(int), hgb_thresh),
    ("RF",       (rf_te_prob  >= rf_thresh).astype(int),  rf_thresh),
    ("Ensemble", (ens_te_prob >= ens_thresh).astype(int), ens_thresh),
]):
    cm = confusion_matrix(y_te, yp)
    ConfusionMatrixDisplay(cm, display_labels=["Normal","Apnea"]).plot(
        ax=ax, colorbar=False, cmap="Blues")
    r = recall_score(y_te,    yp, zero_division=0)
    p = precision_score(y_te, yp, zero_division=0)
    f = f1_score(y_te,        yp, zero_division=0)
    a = accuracy_score(y_te,  yp)
    ax.set_title(f"{name} (t={thresh:.2f})\n"
                 f"Acc={100*a:.1f}%  Rec={100*r:.1f}%  "
                 f"Prec={100*p:.1f}%  F1={100*f:.1f}%",
                 fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUT}/confusion_matrices.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ confusion_matrices.png")

# ── ROC curves ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
for name, prob in [("HistGB",   hgb_te_prob),
                   ("RandomForest", rf_te_prob),
                   ("Ensemble", ens_te_prob)]:
    auc_v = roc_auc_score(y_te, prob)
    RocCurveDisplay.from_predictions(y_te, prob, ax=ax,
                                     name=f"{name} (AUC={auc_v:.3f})")
ax.plot([0,1],[0,1],"k--", lw=1); ax.set_title("ROC Curves — Test Set", fontsize=12)
ax.grid(True, alpha=0.3); ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUT}/roc_curves.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ roc_curves.png")

# ── Precision-Recall curves ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
for name, prob in [("HistGB",   hgb_te_prob),
                   ("RandomForest", rf_te_prob),
                   ("Ensemble", ens_te_prob)]:
    PrecisionRecallDisplay.from_predictions(y_te, prob, ax=ax, name=name)
ax.set_title("Precision-Recall Curves — Test Set", fontsize=12)
ax.grid(True, alpha=0.3); ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUT}/pr_curves.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ pr_curves.png")

# ── Threshold tuning for all models ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Threshold Tuning (Validation Set)", fontsize=13, fontweight="bold")
for ax, (label, td, thresh) in zip(axes, [
    ("HistGB",   hgb_td, hgb_thresh),
    ("RandomForest", rf_td, rf_thresh),
    ("Ensemble", ens_td, ens_thresh),
]):
    ax.plot(td["threshold"], td["recall"],    "r-o", lw=2, ms=4, label="Recall")
    ax.plot(td["threshold"], td["precision"], "b-s", lw=2, ms=4, label="Precision")
    ax.plot(td["threshold"], td["f1"],        "g-^", lw=2, ms=4, label="F1")
    ax.plot(td["threshold"], td["f2"]*100,    "m-D", lw=2, ms=4, label="F2×100")
    ax.axvline(thresh, color="orange", lw=2, linestyle="--",
               label=f"Best={thresh:.2f}")
    ax.set_title(label); ax.set_xlabel("Threshold")
    ax.set_ylabel("Score (%)"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/threshold_tuning.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ threshold_tuning.png")

# ── Model comparison bar chart ───────────────────────────────────────────────
df_res   = pd.DataFrame(all_results)
test_res = df_res[df_res["split"]=="test"].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(test_res)); w = 0.18
ax.bar(x-2*w, test_res["accuracy"],  w, label="Accuracy",  color="steelblue")
ax.bar(x-1*w, test_res["recall"],    w, label="Recall",    color="crimson")
ax.bar(x+0*w, test_res["precision"], w, label="Precision", color="seagreen")
ax.bar(x+1*w, test_res["f1"],        w, label="F1",        color="darkorange")
ax.bar(x+2*w, test_res["auc"]*100,   w, label="AUC×100",   color="purple")
ax.set_xticks(x)
ax.set_xticklabels(test_res["model"], rotation=10, ha="right", fontsize=10)
ax.set_ylabel("Score (%)", fontsize=11)
ax.set_title("All Models — Test Set Comparison (ANOVA-60)", fontsize=13)
ax.legend(fontsize=10); ax.set_ylim(0, 115)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(f"{OUT}/model_comparison.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ model_comparison.png")

# ──────────────────────────────────────────────────────────────────────────────
# 11.  SAVE ALL OUTPUTS
# ──────────────────────────────────────────────────────────────────────────────
df_res.to_csv(f"{OUT}/results.csv", index=False)
pd.concat([hgb_td.assign(model="HGB"),
           rf_td.assign(model="RF"),
           ens_td.assign(model="ENS")]).to_csv(
    f"{OUT}/threshold_tuning.csv", index=False)
np.save(f"{OUT}/scaler_mean.npy",  scaler.mean_)
np.save(f"{OUT}/scaler_std.npy",   scaler.scale_)
np.save(f"{OUT}/anova_support_mask.npy", support)
pd.DataFrame({"epoch": list(eps), **{k:[round(v*100,2) for v in vals]
              for k,vals in epoch_metrics.items()}}).to_csv(
    f"{OUT}/epoch_metrics.csv", index=False)

print(f"\n  All saved to {OUT}/")

# ──────────────────────────────────────────────────────────────────────────────
# 12.  FINAL SUMMARY
# ──────────────────────────────────────────────────────────────────────────────
print(f"""
  ╔══════════════════════════════════════════════════════════════════╗
  ║   ANOVA FEATURE SELECTION + GRADIENT BOOSTING / RF — COMPLETE    ║
  ╠══════════════════════════════════════════════════════════════════╣
  ║  Patients loaded       : {len(loaded):<38}║
  ║  Total windows         : {total:<38,}║
  ║  Features after ANOVA  : {N_SELECT:<38}║
  ║  Train / Val / Test    : {f"{len(y_tr):,} / {len(y_va):,} / {len(y_te):,}":<38}║
  ╠══════════════════════════════════════════════════════════════════╣
  ║  {'MODEL':<22} {'ACC':>7} {'REC':>7} {'PREC':>7} {'F1':>7} {'AUC':>7} ║
  ╠══════════════════════════════════════════════════════════════════╣
  ║  {'HistGB (ANOVA60)':<22} {hgb_acc*100:>6.1f}% {hgb_rec*100:>6.1f}% {hgb_prec*100:>6.1f}% {hgb_f1*100:>6.1f}% {hgb_auc:>7.4f} ║
  ║  {'RandomForest (ANOVA60)':<22} {rf_acc*100:>6.1f}% {rf_rec*100:>6.1f}% {rf_prec*100:>6.1f}% {rf_f1*100:>6.1f}% {rf_auc:>7.4f} ║
  ║  {'Ensemble (ANOVA60)':<22} {ens_acc*100:>6.1f}% {ens_rec*100:>6.1f}% {ens_prec*100:>6.1f}% {ens_f1*100:>6.1f}% {ens_auc:>7.4f} ║
  ╚══════════════════════════════════════════════════════════════════╝

  Plots saved to /kaggle/working/:
    anova_feature_selection.png  training_curves.png    val_auc_curve.png
    confusion_matrices.png       roc_curves.png         pr_curves.png
    threshold_tuning.png         model_comparison.png   rf_feature_importance.png
""")

print("  Displaying plots in notebook...")
for fname in ["anova_feature_selection.png","training_curves.png","val_auc_curve.png",
              "confusion_matrices.png","roc_curves.png","pr_curves.png",
              "threshold_tuning.png","model_comparison.png","rf_feature_importance.png"]:
    fpath = f"{OUT}/{fname}"
    if os.path.exists(fpath):
        print(f"\n── {fname} ──")
        display(IPImage(filename=fpath))

In [ ]:
# =============================================================================
#  DISORDERED BREATHING — ANOVA FEATURE SELECTION + GB / RF / XGBOOST
#  Feature selection : ANOVA F-test (SelectKBest) 125 -> top K (supervised,
#                       replaces unsupervised PCA which optimizes variance,
#                       not class separability)
#  Models            : HistGradientBoostingClassifier + RandomForestClassifier
#                       + XGBoost (native scale_pos_weight for imbalance,
#                       regularized, incremental-round training with its own
#                       early stopping — usually the strongest of the three
#                       on structured/tabular biosignal features)
#                       All trained on the FULL training set (no subsampling).
# =============================================================================

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS
# ──────────────────────────────────────────────────────────────────────────────
import kagglehub
import os, gc, warnings, time, copy
import numpy as np
import pandas as pd
from collections import Counter
warnings.filterwarnings("ignore")

from sklearn.preprocessing        import StandardScaler
from sklearn.feature_selection    import SelectKBest, f_classif
from sklearn.ensemble              import (HistGradientBoostingClassifier,
                                           RandomForestClassifier)
from sklearn.utils.class_weight   import compute_class_weight
from sklearn.metrics              import (classification_report,
                                          confusion_matrix,
                                          roc_auc_score,
                                          recall_score,
                                          precision_score,
                                          f1_score,
                                          accuracy_score,
                                          ConfusionMatrixDisplay,
                                          RocCurveDisplay,
                                          PrecisionRecallDisplay)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

try:
    import xgboost as xgb
except ImportError:
    # Kaggle notebooks ship xgboost by default; this is just a safety net.
    os.system("pip install -q xgboost --break-system-packages")
    import xgboost as xgb

OUT = "/kaggle/working"

# ──────────────────────────────────────────────────────────────────────────────
# 1.  DOWNLOAD DATASET
# ──────────────────────────────────────────────────────────────────────────────
path = kagglehub.dataset_download(
    "nishantomhare/disordered-breathing-classification-features-fft")
print("Dataset path:", path)

# ──────────────────────────────────────────────────────────────────────────────
# 2.  FEATURE NAMES
# ──────────────────────────────────────────────────────────────────────────────
CHANNEL_NAMES = ["pressure","airflow","chest","abdomen","pleth"]
BAND_NAMES    = ["be_0_01","be_01_03","be_03_06","be_06_1",
                 "be_1_2","be_2_5","be_gt5"]
BASE_FEATURES = (
    ["energy","mean_mag","variance","sp_centroid","sp_spread",
     "sp_skewness","sp_kurtosis","sp_flatness","shannon_entropy"]
    + BAND_NAMES
    + ["sp_crest","hjorth_activity","hjorth_mobility","hjorth_complexity",
       "peak1_freq","peak2_freq","peak_ratio","peak_width","peak_prominence"]
)
FEATURE_NAMES = [f"{ch}_{f}" for ch in CHANNEL_NAMES for f in BASE_FEATURES]
N_FEATURES    = len(FEATURE_NAMES)   # 125
N_SELECT      = 60                   # ANOVA-selected feature count (tune 40-80)
ALL_PIDS      = [f"SN{i}" for i in range(2, 101)]

def find_patient_files(pid, base):
    for root, _, files in os.walk(base):
        if pid in root and "features.npy" in files and "y.npy" in files:
            return os.path.join(root,"features.npy"), os.path.join(root,"y.npy")
    return None, None

# ──────────────────────────────────────────────────────────────────────────────
# 3.  LOAD ALL PATIENTS
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("  STEP 1 — LOADING FEATURE DATA")
print(f"{'='*65}")

all_X, all_y = [], []
loaded, skipped, summary = [], [], []
pid_windows = {}

for pid in ALL_PIDS:
    fp, yp = find_patient_files(pid, path)
    if fp is None:
        skipped.append(pid)
        continue
    try:
        X_p = np.load(fp).astype(np.float32)
        y_p = np.load(yp).astype(np.int8)
        if X_p.ndim!=2 or X_p.shape[1]!=N_FEATURES or len(X_p)!=len(y_p):
            print(f"  ⚠️  [{pid}] bad shape {X_p.shape} → skipped")
            skipped.append(pid); continue
        X_p = np.nan_to_num(X_p, nan=0.0, posinf=0.0, neginf=0.0)
        d   = Counter(y_p.tolist())
        n   = len(y_p)
        all_X.append(X_p); all_y.append(y_p)
        pid_windows[pid] = n
        loaded.append(pid)
        summary.append({"patient":pid,"windows":n,
                        "normal":d.get(0,0),"apnea":d.get(1,0),
                        "apnea_pct":round(100*d.get(1,0)/n,1)})
        print(f"  ✅ [{pid}]  windows={n:4d}  "
              f"normal={d.get(0,0):4d}  apnea={d.get(1,0):4d} "
              f"({round(100*d.get(1,0)/n,1):.1f}%)")
        del X_p, y_p; gc.collect()
    except Exception as e:
        print(f"  ❌ [{pid}] {e}")
        skipped.append(pid)

print(f"\n  Loaded  : {len(loaded)} patients")
print(f"  Skipped : {len(skipped)}  {skipped}")

X_all = np.concatenate(all_X, axis=0).astype(np.float32)
y_all = np.concatenate(all_y, axis=0).astype(np.int8)
del all_X, all_y; gc.collect()

dist  = Counter(y_all.tolist())
total = len(y_all)
print(f"\n  Combined dataset:")
print(f"  X shape      : {X_all.shape}  (windows × 125 features)")
print(f"  Normal  (0)  : {dist[0]:,}  ({100*dist[0]/total:.1f}%)")
print(f"  Apnea   (1)  : {dist[1]:,}  ({100*dist[1]/total:.1f}%)")
print(f"  Imbalance    : {dist[0]/dist[1]:.2f}:1")

pd.DataFrame(summary).to_csv(f"{OUT}/patient_summary.csv", index=False)

# ──────────────────────────────────────────────────────────────────────────────
# 4.  TRAIN / VAL / TEST SPLIT  BY PATIENT (unchanged — avoids leakage)
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("  STEP 2 — TRAIN / TEST SPLIT  (80 / 20 by patient)")
print(f"{'='*65}")

np.random.seed(42)
pids_arr = np.array(sorted(loaded))
np.random.shuffle(pids_arr)

n_test     = max(1, int(len(pids_arr) * 0.20))
n_val      = max(1, int(len(pids_arr) * 0.10))
test_pids  = set(pids_arr[:n_test])
val_pids   = set(pids_arr[n_test:n_test+n_val])
train_pids = set(pids_arr[n_test+n_val:])

print(f"  Train patients : {len(train_pids)}")
print(f"  Val   patients : {len(val_pids)}  (threshold tuning only)")
print(f"  Test  patients : {len(test_pids)}")

pid_per_win = []
for pid in loaded:
    pid_per_win.extend([pid]*pid_windows[pid])
pid_per_win = np.array(pid_per_win)

tr_m = np.array([p in train_pids for p in pid_per_win])
va_m = np.array([p in val_pids   for p in pid_per_win])
te_m = np.array([p in test_pids  for p in pid_per_win])

X_tr, y_tr = X_all[tr_m], y_all[tr_m]
X_va, y_va = X_all[va_m], y_all[va_m]
X_te, y_te = X_all[te_m], y_all[te_m]

print(f"\n  Train : {len(y_tr):,} windows  "
      f"apnea={Counter(y_tr)[1]:,}({100*Counter(y_tr)[1]/len(y_tr):.1f}%)")
print(f"  Val   : {len(y_va):,} windows  "
      f"apnea={Counter(y_va)[1]:,}({100*Counter(y_va)[1]/len(y_va):.1f}%)")
print(f"  Test  : {len(y_te):,} windows  "
      f"apnea={Counter(y_te)[1]:,}({100*Counter(y_te)[1]/len(y_te):.1f}%)")

# ──────────────────────────────────────────────────────────────────────────────
# 5.  STANDARDISE  (kept — helps ANOVA-F comparability across feature units,
#                    and costs nothing for tree models)
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("  STEP 3 — STANDARDISATION")
print(f"{'='*65}")

scaler   = StandardScaler()
X_tr_s   = scaler.fit_transform(X_tr)
X_va_s   = scaler.transform(X_va)
X_te_s   = scaler.transform(X_te)
print(f"  Scaled: mean≈{X_tr_s.mean():.4f}  std≈{X_tr_s.std():.4f}")

# ──────────────────────────────────────────────────────────────────────────────
# 6.  ANOVA F-TEST FEATURE SELECTION   125 → top N_SELECT
#     Unlike PCA (unsupervised, keeps high-variance directions), ANOVA F-test
#     is supervised: it scores each feature by how much its class means
#     (Normal vs Apnea) differ relative to within-class variance. Features
#     get RANKED and KEPT AS-IS (no rotation), so the model still trains on
#     interpretable, real physiological features — not opaque components.
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"  STEP 4 — ANOVA F-TEST FEATURE SELECTION  125 → {N_SELECT}")
print(f"{'='*65}")

t0 = time.time()
selector = SelectKBest(score_func=f_classif, k=N_SELECT)
X_tr_sel = selector.fit_transform(X_tr_s, y_tr)
X_va_sel = selector.transform(X_va_s)
X_te_sel = selector.transform(X_te_s)

f_scores   = selector.scores_
p_values   = selector.pvalues_
support    = selector.get_support()
sel_names  = [FEATURE_NAMES[i] for i in range(N_FEATURES) if support[i]]

order = np.argsort(f_scores)[::-1]
print(f"  Selector fitted in : {time.time()-t0:.1f}s")
print(f"  Features kept      : {N_SELECT} / {N_FEATURES}")
print(f"\n  Top 20 features by ANOVA F-score:")
for rank, idx in enumerate(order[:20], 1):
    star = "✅" if support[idx] else "  "
    print(f"   {rank:2d}. {star} {FEATURE_NAMES[idx]:<28} "
          f"F={f_scores[idx]:9.1f}  p={p_values[idx]:.2e}")

anova_df = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "f_score": f_scores,
    "p_value": p_values,
    "selected": support,
}).sort_values("f_score", ascending=False)
anova_df.to_csv(f"{OUT}/anova_feature_scores.csv", index=False)

# ── ANOVA F-score plot (replaces PCA scree plot) ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f"ANOVA F-Test Feature Selection — 125 → {N_SELECT}",
             fontsize=13, fontweight="bold")

top_n = 40
colors = ["seagreen" if support[i] else "lightgray" for i in order[:top_n]]
axes[0].barh(range(top_n), f_scores[order[:top_n]][::-1],
             color=colors[::-1])
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels([FEATURE_NAMES[i] for i in order[:top_n]][::-1],
                         fontsize=7)
axes[0].set_xlabel("ANOVA F-score")
axes[0].set_title(f"Top {top_n} Features by F-score\n(green = selected)")
axes[0].grid(True, alpha=0.3, axis="x")

axes[1].plot(range(1, N_FEATURES+1), f_scores[order], "b-o", ms=3, lw=1.2)
axes[1].axvline(N_SELECT, color="orange", linestyle="--", lw=2,
                label=f"cutoff k={N_SELECT}")
axes[1].set_xlabel("Feature rank"); axes[1].set_ylabel("ANOVA F-score")
axes[1].set_title("F-score decay across ranked features")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT}/anova_feature_selection.png", dpi=120, bbox_inches="tight")
plt.close()
print(f"\n  ✅ anova_feature_selection.png saved")
print(f"  ✅ anova_feature_scores.csv saved")

# ──────────────────────────────────────────────────────────────────────────────
# 7.  CLASS WEIGHTS
# ──────────────────────────────────────────────────────────────────────────────
cw          = compute_class_weight("balanced", classes=np.array([0,1]), y=y_tr)
cwd         = {0: float(cw[0]), 1: float(cw[1])}
sample_w_tr = np.where(y_tr==1, cw[1], cw[0]).astype(np.float32)
scale_pos_weight = dist[0] / dist[1]   # XGBoost's native imbalance handle
print(f"\n  Class weights  : normal={cw[0]:.3f}  apnea={cw[1]:.3f}")
print(f"  scale_pos_weight (XGB): {scale_pos_weight:.3f}")

# ──────────────────────────────────────────────────────────────────────────────
# 8.  MODEL TRAINING
#     Model A: HistGradientBoostingClassifier — fast, native class_weight,
#              handles non-linear feature interactions, trains on FULL data.
#     Model B: RandomForestClassifier — bagged trees, very robust to noisy
#              biosignal features, also trains on FULL data.
#     Model C: XGBoost — regularized gradient boosting with scale_pos_weight,
#              subsampling, and column subsampling for imbalance + variance
#              control; usually edges out sklearn's HistGB on tabular data.
#     Model D: Ensemble (average of A + B + C probabilities)
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("  STEP 5 — MODEL TRAINING")
print(f"{'='*65}")

all_results  = []
thresh_store = {}

def full_metrics(name, y_true, y_prob, threshold, split):
    y_pred = (y_prob >= threshold).astype(int)
    acc    = accuracy_score(y_true, y_pred)
    rec    = recall_score(y_true,    y_pred, zero_division=0)
    prec   = precision_score(y_true, y_pred, zero_division=0)
    f1     = f1_score(y_true,        y_pred, zero_division=0)
    auc    = roc_auc_score(y_true, y_prob)
    f2     = (5*prec*rec)/(4*prec+rec) if (4*prec+rec) > 0 else 0

    print(f"\n  ── [{name}]  {split.upper()}  threshold={threshold:.2f} ──")
    print(f"  Accuracy  : {100*acc:.2f}%")
    print(f"  Recall    : {100*rec:.2f}%   ← apnea detection rate")
    print(f"  Precision : {100*prec:.2f}%")
    print(f"  F1-Score  : {100*f1:.2f}%")
    print(f"  F2-Score  : {f2:.4f}   (weights recall 2×)")
    print(f"  AUC-ROC   : {auc:.4f}")
    print(classification_report(y_true, y_pred,
          target_names=["Normal","Apnea"], digits=3))

    all_results.append({
        "model":name, "split":split, "threshold":round(threshold,2),
        "accuracy":round(100*acc,2), "recall":round(100*rec,2),
        "precision":round(100*prec,2), "f1":round(100*f1,2),
        "f2":round(f2,4), "auc":round(auc,4),
    })
    return acc, rec, prec, f1, auc

def best_threshold(y_true, y_prob, label="model"):
    rows, best_t, best_f2 = [], 0.5, 0.0
    print(f"\n  Threshold sweep [{label}] on validation:")
    print(f"  {'THRESH':>7}  {'ACC':>8}  {'RECALL':>8}  "
          f"{'PRECIS':>8}  {'F1':>8}  {'F2':>8}")
    print("  " + "-"*55)
    for t in np.arange(0.10, 0.85, 0.05):
        yp   = (y_prob >= t).astype(int)
        acc  = accuracy_score(y_true, yp)
        rec  = recall_score(y_true,    yp, zero_division=0)
        prec = precision_score(y_true, yp, zero_division=0)
        f1   = f1_score(y_true,        yp, zero_division=0)
        f2   = (5*prec*rec)/(4*prec+rec) if (4*prec+rec) > 0 else 0
        flag = " ←" if f2 > best_f2 else ""
        print(f"  {t:>7.2f}  {100*acc:>7.1f}%  {100*rec:>7.1f}%  "
              f"{100*prec:>7.1f}%  {100*f1:>7.1f}%  {f2:>8.4f}{flag}")
        rows.append({"threshold":round(t,2),"accuracy":round(100*acc,1),
                     "recall":round(100*rec,1),"precision":round(100*prec,1),
                     "f1":round(100*f1,1),"f2":round(f2,4)})
        if f2 > best_f2:
            best_f2, best_t = f2, t
    print(f"\n  Best threshold: {best_t:.2f}  (F2={best_f2:.4f})")
    return best_t, pd.DataFrame(rows)

# ─────────────────────────────────────────────────────────────────────────────
# Model A: HistGradientBoostingClassifier (epoch-by-epoch, incremental trees)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n── Model A: HistGradientBoosting (incremental, epoch-level metrics) ──")
print(f"   Data: {X_tr_sel.shape[0]:,} windows × {N_SELECT} ANOVA-selected features")

EPOCHS = 120
hgb = HistGradientBoostingClassifier(
    max_iter          = 1,
    warm_start        = True,
    learning_rate     = 0.08,
    max_leaf_nodes    = 31,
    max_depth         = None,
    l2_regularization = 1.0,
    class_weight      = "balanced",
    early_stopping    = False,       # we drive early stopping manually vs val
    random_state      = 42,
)

epoch_metrics = {k: [] for k in ["acc","rec","prec","f1","val_acc",
                                   "val_rec","val_prec","val_f1","val_auc"]}

print(f"\n  {'EP':>3}  {'TR_ACC':>8}  {'TR_REC':>8}  {'TR_PRE':>8}  "
      f"{'TR_F1':>8}  │  {'VA_ACC':>8}  {'VA_REC':>8}  "
      f"{'VA_PRE':>8}  {'VA_F1':>8}  {'VA_AUC':>8}")
print("  " + "─"*95)

best_val_auc_hgb, best_hgb_iter = 0.0, 1
best_hgb_model = None
patience, pat_cnt = 15, 0

for ep in range(1, EPOCHS+1):
    hgb.max_iter = ep
    hgb.fit(X_tr_sel, y_tr)

    tr_prob = hgb.predict_proba(X_tr_sel)[:,1]
    tr_pred = (tr_prob >= 0.5).astype(int)
    tr_acc  = accuracy_score(y_tr, tr_pred)
    tr_rec  = recall_score(y_tr,    tr_pred, zero_division=0)
    tr_prec = precision_score(y_tr, tr_pred, zero_division=0)
    tr_f1   = f1_score(y_tr,        tr_pred, zero_division=0)

    va_prob = hgb.predict_proba(X_va_sel)[:,1]
    va_pred = (va_prob >= 0.5).astype(int)
    va_acc  = accuracy_score(y_va, va_pred)
    va_rec  = recall_score(y_va,    va_pred, zero_division=0)
    va_prec = precision_score(y_va, va_pred, zero_division=0)
    va_f1   = f1_score(y_va,        va_pred, zero_division=0)
    va_auc  = roc_auc_score(y_va,   va_prob)

    for k,v in zip(["acc","rec","prec","f1","val_acc","val_rec",
                    "val_prec","val_f1","val_auc"],
                   [tr_acc,tr_rec,tr_prec,tr_f1,
                    va_acc,va_rec,va_prec,va_f1,va_auc]):
        epoch_metrics[k].append(v)

    flag = ""
    if va_auc > best_val_auc_hgb:
        best_val_auc_hgb, best_hgb_iter = va_auc, ep
        best_hgb_model = copy.deepcopy(hgb)   # snapshot — never shrink a warm_start model
        pat_cnt, flag = 0, " ✅"
    else:
        pat_cnt += 1
        if pat_cnt >= patience:
            print(f"\n  Early stopping at epoch {ep} (patience={patience})")
            break

    print(f"  {ep:>3}  {100*tr_acc:>7.1f}%  {100*tr_rec:>7.1f}%  "
          f"{100*tr_prec:>7.1f}%  {100*tr_f1:>7.1f}%  │  "
          f"{100*va_acc:>7.1f}%  {100*va_rec:>7.1f}%  "
          f"{100*va_prec:>7.1f}%  {100*va_f1:>7.1f}%  {va_auc:.4f}{flag}")

# use the best-so-far snapshot directly — do NOT try to re-fit a warm_start
# model back down to an earlier iteration count, sklearn forbids shrinking it
hgb = best_hgb_model
print(f"\n  Best val AUC: {best_val_auc_hgb:.4f}  (at {best_hgb_iter} boosting rounds)")

hgb_va_prob        = hgb.predict_proba(X_va_sel)[:,1]
hgb_thresh, hgb_td = best_threshold(y_va, hgb_va_prob, "HistGB")
thresh_store["HGB"] = hgb_td

hgb_te_prob = hgb.predict_proba(X_te_sel)[:,1]
hgb_acc, hgb_rec, hgb_prec, hgb_f1, hgb_auc = full_metrics(
    "HistGB_ANOVA60", y_te, hgb_te_prob, hgb_thresh, "test")

# ─────────────────────────────────────────────────────────────────────────────
# Model B: RandomForestClassifier (incremental trees, full training set)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n── Model B: RandomForest (incremental, full training set) ──")
rf = RandomForestClassifier(
    n_estimators      = 10,
    warm_start        = True,
    class_weight      = "balanced_subsample",
    max_depth         = 16,
    min_samples_leaf  = 5,
    max_features      = "sqrt",
    n_jobs            = -1,
    random_state      = 42,
)

RF_EPOCHS, TREES_PER_EPOCH = 40, 20
best_val_auc_rf, best_rf_trees = 0.0, TREES_PER_EPOCH
best_rf_model = None
pat_cnt = 0

print(f"\n  {'EP':>3}  {'N_TREES':>8}  {'VA_ACC':>8}  {'VA_REC':>8}  "
      f"{'VA_PRE':>8}  {'VA_F1':>8}  {'VA_AUC':>8}")
print("  " + "─"*70)

for ep in range(1, RF_EPOCHS+1):
    rf.n_estimators = ep * TREES_PER_EPOCH
    rf.fit(X_tr_sel, y_tr)

    va_prob = rf.predict_proba(X_va_sel)[:,1]
    va_pred = (va_prob >= 0.5).astype(int)
    va_acc  = accuracy_score(y_va, va_pred)
    va_rec  = recall_score(y_va,    va_pred, zero_division=0)
    va_prec = precision_score(y_va, va_pred, zero_division=0)
    va_f1   = f1_score(y_va,        va_pred, zero_division=0)
    va_auc  = roc_auc_score(y_va,   va_prob)

    flag = ""
    if va_auc > best_val_auc_rf:
        best_val_auc_rf, best_rf_trees = va_auc, rf.n_estimators
        best_rf_model = copy.deepcopy(rf)     # snapshot — never shrink a warm_start forest
        pat_cnt, flag = 0, " ✅"
    else:
        pat_cnt += 1
        if pat_cnt >= 8:
            print(f"\n  Early stopping at {rf.n_estimators} trees (patience=8)")
            break

    print(f"  {ep:>3}  {rf.n_estimators:>8}  {100*va_acc:>7.1f}%  "
          f"{100*va_rec:>7.1f}%  {100*va_prec:>7.1f}%  {100*va_f1:>7.1f}%  "
          f"{va_auc:.4f}{flag}")

# use the best-so-far snapshot directly — same reasoning as HGB above
rf = best_rf_model
print(f"\n  Best val AUC: {best_val_auc_rf:.4f}  (at {best_rf_trees} trees)")

rf_va_prob       = rf.predict_proba(X_va_sel)[:,1]
rf_thresh, rf_td = best_threshold(y_va, rf_va_prob, "RandomForest")
thresh_store["RF"] = rf_td

rf_te_prob = rf.predict_proba(X_te_sel)[:,1]
rf_acc, rf_rec, rf_prec, rf_f1, rf_auc = full_metrics(
    "RandomForest_ANOVA60", y_te, rf_te_prob, rf_thresh, "test")

# ─────────────────────────────────────────────────────────────────────────────
# Model C: XGBoost (incremental rounds, epoch-level metrics, full training set)
#   Modifications vs. a default XGBClassifier:
#     - scale_pos_weight   : native imbalance handling (no resampling needed)
#     - subsample / colsample_bytree : row & column subsampling to fight
#       overfitting on noisy biosignal features
#     - min_child_weight + reg_alpha/reg_lambda : extra regularization since
#       we already narrowed to 60 ANOVA-selected (highly informative) features
#     - low learning_rate + many rounds + early stopping on val AUC : lets
#       boosting converge carefully instead of overshooting
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n── Model C: XGBoost (incremental, epoch-level metrics) ──")
print(f"   Data: {X_tr_sel.shape[0]:,} windows × {N_SELECT} ANOVA-selected features")

XGB_PARAMS = dict(
    max_depth        = 6,
    learning_rate    = 0.04,
    min_child_weight = 4,
    subsample        = 0.80,
    colsample_bytree = 0.75,
    colsample_bylevel= 0.75,
    gamma            = 0.5,
    reg_alpha        = 0.3,
    reg_lambda       = 1.5,
    objective        = "binary:logistic",
    eval_metric      = "auc",
    tree_method      = "hist",
    scale_pos_weight = scale_pos_weight,
    random_state     = 42,
    n_jobs           = -1,
)

EPOCHS_XGB = 200
booster    = None   # accumulated Booster, carried forward across rounds
epoch_metrics_xgb = {k: [] for k in ["acc","rec","prec","f1","val_acc",
                                     "val_rec","val_prec","val_f1","val_auc"]}

print(f"\n  {'EP':>3}  {'TR_ACC':>8}  {'TR_REC':>8}  {'TR_PRE':>8}  "
      f"{'TR_F1':>8}  │  {'VA_ACC':>8}  {'VA_REC':>8}  "
      f"{'VA_PRE':>8}  {'VA_F1':>8}  {'VA_AUC':>8}")
print("  " + "─"*95)

best_val_auc_xgb, best_xgb_round = 0.0, 0
best_xgb_model = None
patience, pat_cnt = 20, 0

for ep in range(1, EPOCHS_XGB+1):
    # fresh estimator each round, told to add exactly 1 tree on top of the
    # existing accumulated booster (passed via xgb_model=) — this sidesteps
    # sklearn's warm_start "can't shrink" restriction entirely, since we
    # never mutate a persistent object's internal tree count.
    xgb_clf = xgb.XGBClassifier(n_estimators=1, **XGB_PARAMS)
    xgb_clf.fit(X_tr_sel, y_tr, xgb_model=booster, verbose=False)
    booster = xgb_clf.get_booster()

    tr_prob = xgb_clf.predict_proba(X_tr_sel)[:,1]
    tr_pred = (tr_prob >= 0.5).astype(int)
    tr_acc  = accuracy_score(y_tr, tr_pred)
    tr_rec  = recall_score(y_tr,    tr_pred, zero_division=0)
    tr_prec = precision_score(y_tr, tr_pred, zero_division=0)
    tr_f1   = f1_score(y_tr,        tr_pred, zero_division=0)

    va_prob = xgb_clf.predict_proba(X_va_sel)[:,1]
    va_pred = (va_prob >= 0.5).astype(int)
    va_acc  = accuracy_score(y_va, va_pred)
    va_rec  = recall_score(y_va,    va_pred, zero_division=0)
    va_prec = precision_score(y_va, va_pred, zero_division=0)
    va_f1   = f1_score(y_va,        va_pred, zero_division=0)
    va_auc  = roc_auc_score(y_va,   va_prob)

    for k,v in zip(["acc","rec","prec","f1","val_acc","val_rec",
                    "val_prec","val_f1","val_auc"],
                   [tr_acc,tr_rec,tr_prec,tr_f1,
                    va_acc,va_rec,va_prec,va_f1,va_auc]):
        epoch_metrics_xgb[k].append(v)

    flag = ""
    if va_auc > best_val_auc_xgb:
        best_val_auc_xgb, best_xgb_round = va_auc, ep
        best_xgb_model = copy.deepcopy(xgb_clf)   # snapshot the full booster
        pat_cnt, flag = 0, " ✅"
    else:
        pat_cnt += 1
        if pat_cnt >= patience:
            print(f"\n  Early stopping at round {ep} (patience={patience})")
            break

    if ep % 5 == 0 or flag:
        print(f"  {ep:>3}  {100*tr_acc:>7.1f}%  {100*tr_rec:>7.1f}%  "
              f"{100*tr_prec:>7.1f}%  {100*tr_f1:>7.1f}%  │  "
              f"{100*va_acc:>7.1f}%  {100*va_rec:>7.1f}%  "
              f"{100*va_prec:>7.1f}%  {100*va_f1:>7.1f}%  {va_auc:.4f}{flag}")

xgb_model = best_xgb_model   # use the best-so-far snapshot, same as HGB/RF
print(f"\n  Best val AUC: {best_val_auc_xgb:.4f}  (at {best_xgb_round} boosting rounds)")

xgb_va_prob        = xgb_model.predict_proba(X_va_sel)[:,1]
xgb_thresh, xgb_td = best_threshold(y_va, xgb_va_prob, "XGBoost")
thresh_store["XGB"] = xgb_td

xgb_te_prob = xgb_model.predict_proba(X_te_sel)[:,1]
xgb_acc, xgb_rec, xgb_prec, xgb_f1, xgb_auc = full_metrics(
    "XGBoost_ANOVA60", y_te, xgb_te_prob, xgb_thresh, "test")

# ─────────────────────────────────────────────────────────────────────────────
# Model D: Ensemble (HistGB + RandomForest + XGBoost average)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n── Model D: Ensemble (HistGB + RandomForest + XGBoost) ──")
ens_va_prob        = (hgb_va_prob + rf_va_prob + xgb_va_prob) / 3
ens_thresh, ens_td = best_threshold(y_va, ens_va_prob, "Ensemble")
thresh_store["ENS"] = ens_td

ens_te_prob = (hgb_te_prob + rf_te_prob + xgb_te_prob) / 3
ens_acc, ens_rec, ens_prec, ens_f1, ens_auc = full_metrics(
    "Ensemble_ANOVA60", y_te, ens_te_prob, ens_thresh, "test")

# ──────────────────────────────────────────────────────────────────────────────
# 9.  FEATURE IMPORTANCE (bonus — trees give this for free, PCA/SVM don't)
# ──────────────────────────────────────────────────────────────────────────────
imp_df = pd.DataFrame({
    "feature": sel_names,
    "rf_importance":  rf.feature_importances_,
    "xgb_importance": xgb_model.feature_importances_,
}).sort_values("xgb_importance", ascending=False)
imp_df.to_csv(f"{OUT}/feature_importance.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, col, title, color in [
    (axes[0], "rf_importance",  "RandomForest Importance", "teal"),
    (axes[1], "xgb_importance", "XGBoost Importance",      "darkorange"),
]:
    top_imp = imp_df.sort_values(col, ascending=False).head(25).iloc[::-1]
    ax.barh(top_imp["feature"], top_imp[col], color=color)
    ax.set_title(f"Top 25 Features — {title}", fontsize=12)
    ax.set_xlabel("Importance")
fig.suptitle("Feature Importance — RF vs XGBoost (ANOVA-60 subset)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT}/feature_importance.png", dpi=120, bbox_inches="tight")
plt.close()
print("\n  ✅ feature_importance.png / .csv saved")

# ──────────────────────────────────────────────────────────────────────────────
# 10.  PLOTS
# ──────────────────────────────────────────────────────────────────────────────
print("\n  Generating plots...")

# ── Training curves (HGB epoch-by-epoch) ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("HistGradientBoosting Training Curves (ANOVA-60 features)",
             fontsize=14, fontweight="bold")
eps = range(1, len(epoch_metrics["acc"])+1)

for ax, (tr_k, va_k, title, ylabel) in zip(axes.flatten(), [
    ("acc",  "val_acc",  "Accuracy",  "Accuracy (%)"),
    ("rec",  "val_rec",  "Recall",    "Recall (%)"),
    ("prec", "val_prec", "Precision", "Precision (%)"),
    ("f1",   "val_f1",   "F1-Score",  "F1 (%)"),
]):
    ax.plot(eps, [v*100 for v in epoch_metrics[tr_k]],
            "steelblue", lw=2, label="Train")
    ax.plot(eps, [v*100 for v in epoch_metrics[va_k]],
            "crimson",   lw=2, label="Val", linestyle="--")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Boosting round"); ax.set_ylabel(ylabel)
    best_ep = int(np.argmax(epoch_metrics[va_k])) + 1
    ax.axvline(best_ep, color="orange", linestyle=":", lw=1.5,
               label=f"Best ep={best_ep}")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT}/training_curves.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ training_curves.png")

# ── Val AUC per round ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(eps, epoch_metrics["val_auc"], "purple", lw=2)
ax.set_title("Validation AUC-ROC per Boosting Round (HistGB)", fontsize=12)
ax.set_xlabel("Boosting round"); ax.set_ylabel("AUC-ROC")
ax.axvline(int(np.argmax(epoch_metrics["val_auc"]))+1,
           color="orange", linestyle="--", lw=2,
           label=f"Best AUC={max(epoch_metrics['val_auc']):.4f}")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/val_auc_curve.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ val_auc_curve.png")

# ── Training curves (XGBoost epoch-by-epoch) ─────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("XGBoost Training Curves (ANOVA-60 features)",
             fontsize=14, fontweight="bold")
eps_xgb = range(1, len(epoch_metrics_xgb["acc"])+1)

for ax, (tr_k, va_k, title, ylabel) in zip(axes.flatten(), [
    ("acc",  "val_acc",  "Accuracy",  "Accuracy (%)"),
    ("rec",  "val_rec",  "Recall",    "Recall (%)"),
    ("prec", "val_prec", "Precision", "Precision (%)"),
    ("f1",   "val_f1",   "F1-Score",  "F1 (%)"),
]):
    ax.plot(eps_xgb, [v*100 for v in epoch_metrics_xgb[tr_k]],
            "steelblue", lw=2, label="Train")
    ax.plot(eps_xgb, [v*100 for v in epoch_metrics_xgb[va_k]],
            "crimson",   lw=2, label="Val", linestyle="--")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Boosting round"); ax.set_ylabel(ylabel)
    best_ep = int(np.argmax(epoch_metrics_xgb[va_k])) + 1
    ax.axvline(best_ep, color="orange", linestyle=":", lw=1.5,
               label=f"Best ep={best_ep}")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT}/xgb_training_curves.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ xgb_training_curves.png")

# ── Confusion matrices ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(23, 5))
fig.suptitle("Confusion Matrices — Test Set (ANOVA-60 features)",
             fontsize=13, fontweight="bold")
for ax, (name, yp, thresh) in zip(axes, [
    ("HistGB",   (hgb_te_prob >= hgb_thresh).astype(int), hgb_thresh),
    ("RF",       (rf_te_prob  >= rf_thresh).astype(int),  rf_thresh),
    ("XGBoost",  (xgb_te_prob >= xgb_thresh).astype(int), xgb_thresh),
    ("Ensemble", (ens_te_prob >= ens_thresh).astype(int), ens_thresh),
]):
    cm = confusion_matrix(y_te, yp)
    ConfusionMatrixDisplay(cm, display_labels=["Normal","Apnea"]).plot(
        ax=ax, colorbar=False, cmap="Blues")
    r = recall_score(y_te,    yp, zero_division=0)
    p = precision_score(y_te, yp, zero_division=0)
    f = f1_score(y_te,        yp, zero_division=0)
    a = accuracy_score(y_te,  yp)
    ax.set_title(f"{name} (t={thresh:.2f})\n"
                 f"Acc={100*a:.1f}%  Rec={100*r:.1f}%  "
                 f"Prec={100*p:.1f}%  F1={100*f:.1f}%",
                 fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUT}/confusion_matrices.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ confusion_matrices.png")

# ── ROC curves ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
for name, prob in [("HistGB",   hgb_te_prob),
                   ("RandomForest", rf_te_prob),
                   ("XGBoost",  xgb_te_prob),
                   ("Ensemble", ens_te_prob)]:
    auc_v = roc_auc_score(y_te, prob)
    RocCurveDisplay.from_predictions(y_te, prob, ax=ax,
                                     name=f"{name} (AUC={auc_v:.3f})")
ax.plot([0,1],[0,1],"k--", lw=1); ax.set_title("ROC Curves — Test Set", fontsize=12)
ax.grid(True, alpha=0.3); ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUT}/roc_curves.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ roc_curves.png")

# ── Precision-Recall curves ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
for name, prob in [("HistGB",   hgb_te_prob),
                   ("RandomForest", rf_te_prob),
                   ("XGBoost",  xgb_te_prob),
                   ("Ensemble", ens_te_prob)]:
    PrecisionRecallDisplay.from_predictions(y_te, prob, ax=ax, name=name)
ax.set_title("Precision-Recall Curves — Test Set", fontsize=12)
ax.grid(True, alpha=0.3); ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUT}/pr_curves.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ pr_curves.png")

# ── Threshold tuning for all models ──────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(23, 5))
fig.suptitle("Threshold Tuning (Validation Set)", fontsize=13, fontweight="bold")
for ax, (label, td, thresh) in zip(axes, [
    ("HistGB",   hgb_td, hgb_thresh),
    ("RandomForest", rf_td, rf_thresh),
    ("XGBoost",  xgb_td, xgb_thresh),
    ("Ensemble", ens_td, ens_thresh),
]):
    ax.plot(td["threshold"], td["recall"],    "r-o", lw=2, ms=4, label="Recall")
    ax.plot(td["threshold"], td["precision"], "b-s", lw=2, ms=4, label="Precision")
    ax.plot(td["threshold"], td["f1"],        "g-^", lw=2, ms=4, label="F1")
    ax.plot(td["threshold"], td["f2"]*100,    "m-D", lw=2, ms=4, label="F2×100")
    ax.axvline(thresh, color="orange", lw=2, linestyle="--",
               label=f"Best={thresh:.2f}")
    ax.set_title(label); ax.set_xlabel("Threshold")
    ax.set_ylabel("Score (%)"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/threshold_tuning.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ threshold_tuning.png")

# ── Model comparison bar chart ───────────────────────────────────────────────
df_res   = pd.DataFrame(all_results)
test_res = df_res[df_res["split"]=="test"].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(test_res)); w = 0.18
ax.bar(x-2*w, test_res["accuracy"],  w, label="Accuracy",  color="steelblue")
ax.bar(x-1*w, test_res["recall"],    w, label="Recall",    color="crimson")
ax.bar(x+0*w, test_res["precision"], w, label="Precision", color="seagreen")
ax.bar(x+1*w, test_res["f1"],        w, label="F1",        color="darkorange")
ax.bar(x+2*w, test_res["auc"]*100,   w, label="AUC×100",   color="purple")
ax.set_xticks(x)
ax.set_xticklabels(test_res["model"], rotation=10, ha="right", fontsize=10)
ax.set_ylabel("Score (%)", fontsize=11)
ax.set_title("All Models — Test Set Comparison (ANOVA-60)", fontsize=13)
ax.legend(fontsize=10); ax.set_ylim(0, 115)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(f"{OUT}/model_comparison.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✅ model_comparison.png")

# ──────────────────────────────────────────────────────────────────────────────
# 11.  SAVE ALL OUTPUTS
# ──────────────────────────────────────────────────────────────────────────────
df_res.to_csv(f"{OUT}/results.csv", index=False)
pd.concat([hgb_td.assign(model="HGB"),
           rf_td.assign(model="RF"),
           xgb_td.assign(model="XGB"),
           ens_td.assign(model="ENS")]).to_csv(
    f"{OUT}/threshold_tuning.csv", index=False)
np.save(f"{OUT}/scaler_mean.npy",  scaler.mean_)
np.save(f"{OUT}/scaler_std.npy",   scaler.scale_)
np.save(f"{OUT}/anova_support_mask.npy", support)
pd.DataFrame({"epoch": list(eps), **{k:[round(v*100,2) for v in vals]
              for k,vals in epoch_metrics.items()}}).to_csv(
    f"{OUT}/epoch_metrics_hgb.csv", index=False)
pd.DataFrame({"epoch": list(eps_xgb), **{k:[round(v*100,2) for v in vals]
              for k,vals in epoch_metrics_xgb.items()}}).to_csv(
    f"{OUT}/epoch_metrics_xgb.csv", index=False)
best_xgb_model.save_model(f"{OUT}/xgb_model.json")

print(f"\n  All saved to {OUT}/")

# ──────────────────────────────────────────────────────────────────────────────
# 12.  FINAL SUMMARY
# ──────────────────────────────────────────────────────────────────────────────
print(f"""
  ╔══════════════════════════════════════════════════════════════════╗
  ║   ANOVA FEATURE SELECTION + GB / RF / XGBOOST — COMPLETE         ║
  ╠══════════════════════════════════════════════════════════════════╣
  ║  Patients loaded       : {len(loaded):<38}║
  ║  Total windows         : {total:<38,}║
  ║  Features after ANOVA  : {N_SELECT:<38}║
  ║  Train / Val / Test    : {f"{len(y_tr):,} / {len(y_va):,} / {len(y_te):,}":<38}║
  ╠══════════════════════════════════════════════════════════════════╣
  ║  {'MODEL':<22} {'ACC':>7} {'REC':>7} {'PREC':>7} {'F1':>7} {'AUC':>7} ║
  ╠══════════════════════════════════════════════════════════════════╣
  ║  {'HistGB (ANOVA60)':<22} {hgb_acc*100:>6.1f}% {hgb_rec*100:>6.1f}% {hgb_prec*100:>6.1f}% {hgb_f1*100:>6.1f}% {hgb_auc:>7.4f} ║
  ║  {'RandomForest (ANOVA60)':<22} {rf_acc*100:>6.1f}% {rf_rec*100:>6.1f}% {rf_prec*100:>6.1f}% {rf_f1*100:>6.1f}% {rf_auc:>7.4f} ║
  ║  {'XGBoost (ANOVA60)':<22} {xgb_acc*100:>6.1f}% {xgb_rec*100:>6.1f}% {xgb_prec*100:>6.1f}% {xgb_f1*100:>6.1f}% {xgb_auc:>7.4f} ║
  ║  {'Ensemble (ANOVA60)':<22} {ens_acc*100:>6.1f}% {ens_rec*100:>6.1f}% {ens_prec*100:>6.1f}% {ens_f1*100:>6.1f}% {ens_auc:>7.4f} ║
  ╚══════════════════════════════════════════════════════════════════╝

  Plots saved to /kaggle/working/:
    anova_feature_selection.png  training_curves.png     val_auc_curve.png
    xgb_training_curves.png      confusion_matrices.png  roc_curves.png
    pr_curves.png                threshold_tuning.png    model_comparison.png
    feature_importance.png
""")

print("  Displaying plots in notebook...")
for fname in ["anova_feature_selection.png","training_curves.png","val_auc_curve.png",
              "xgb_training_curves.png","confusion_matrices.png","roc_curves.png",
              "pr_curves.png","threshold_tuning.png","model_comparison.png",
              "feature_importance.png"]:
    fpath = f"{OUT}/{fname}"
    if os.path.exists(fpath):
        print(f"\n── {fname} ──")
        display(IPImage(filename=fpath))

In [ ]:
# ──────────────────────────────────────────────────────────────
# SAVE ALL RESULTS TO CUSTOM FOLDER: "fft rf vs svc"
# ──────────────────────────────────────────────────────────────
import os, shutil, joblib, numpy as np
import pandas as pd
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, precision_recall_curve, roc_auc_score
)

# Create custom folder
OUTPUT_DIR = "/kaggle/working/fft rf vs svc"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Predictions
pd.DataFrame({
    "y_true": y_test,
    "y_prob": y_prob,
    "y_pred_default": y_pred,
    "y_pred_tuned": y_pred_tuned
}).to_csv(f"{OUTPUT_DIR}/test_predictions.csv", index=False)

# 2. Classification Reports
pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).T \
    .to_csv(f"{OUTPUT_DIR}/report_default.csv")

pd.DataFrame(classification_report(y_test, y_pred_tuned, output_dict=True)).T \
    .to_csv(f"{OUTPUT_DIR}/report_tuned.csv")

# 3. Confusion Matrices
pd.DataFrame(confusion_matrix(y_test, y_pred)) \
    .to_csv(f"{OUTPUT_DIR}/cm_default.csv", index=False)

pd.DataFrame(confusion_matrix(y_test, y_pred_tuned)) \
    .to_csv(f"{OUTPUT_DIR}/cm_tuned.csv", index=False)

# 4. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
pd.DataFrame({"FPR": fpr, "TPR": tpr}) \
    .to_csv(f"{OUTPUT_DIR}/roc_curve.csv", index=False)

# 5. Precision-Recall Curve
p, r, _ = precision_recall_curve(y_test, y_prob)
pd.DataFrame({"Precision": p[:-1], "Recall": r[:-1]}) \
    .to_csv(f"{OUTPUT_DIR}/pr_curve.csv", index=False)

# 6. Metrics Summary
pd.DataFrame([
    ["default", acc, prec, rec, f1, roc_auc_score(y_test, y_prob)],
    ["tuned", acc_t, prec_t, rec_t, f1_t, roc_auc_score(y_test, y_prob)]
], columns=["type","accuracy","precision","recall","f1","roc_auc"]) \
.to_csv(f"{OUTPUT_DIR}/metrics_summary.csv", index=False)

# 7. Selected Features (SAFE)
try:
    support_mask = selector.get_support()
    if len(FEATURE_NAMES) == len(support_mask):
        selected_features = np.array(FEATURE_NAMES)[support_mask]
    else:
        selected_features = [f"feature_{i}" for i, val in enumerate(support_mask) if val]

    pd.DataFrame({"feature": selected_features}) \
        .to_csv(f"{OUTPUT_DIR}/selected_features.csv", index=False)

except Exception as e:
    print("Skipping feature saving:", e)

# 8. Save Model & Scaler
joblib.dump(model, f"{OUTPUT_DIR}/model.pkl")
joblib.dump(scaler, f"{OUTPUT_DIR}/scaler.pkl")

# 9. Zip Folder
shutil.make_archive("/kaggle/working/fft_rf_vs_svc_results", 'zip', OUTPUT_DIR)

print("Saved in:", OUTPUT_DIR)
print("ZIP:", "/kaggle/working/fft_rf_vs_svc_results.zip")

In [ ]:
# ──────────────────────────────────────────────────────────────
# SAVE ALL RESULTS + PLOTS TO "fft rf vs svc"
# ──────────────────────────────────────────────────────────────
import os, shutil, joblib, numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, precision_recall_curve, roc_auc_score
)

# Create folder
OUTPUT_DIR = "/kaggle/working/fft rf vs svc"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ───────────── SAVE DATA ─────────────
pd.DataFrame({
    "y_true": y_test,
    "y_prob": y_prob,
    "y_pred_default": y_pred,
    "y_pred_tuned": y_pred_tuned
}).to_csv(f"{OUTPUT_DIR}/test_predictions.csv", index=False)

pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).T \
    .to_csv(f"{OUTPUT_DIR}/report_default.csv")

pd.DataFrame(classification_report(y_test, y_pred_tuned, output_dict=True)).T \
    .to_csv(f"{OUTPUT_DIR}/report_tuned.csv")

pd.DataFrame(confusion_matrix(y_test, y_pred)) \
    .to_csv(f"{OUTPUT_DIR}/cm_default.csv", index=False)

pd.DataFrame(confusion_matrix(y_test, y_pred_tuned)) \
    .to_csv(f"{OUTPUT_DIR}/cm_tuned.csv", index=False)

# ───────────── ROC DATA + PLOT ─────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)
pd.DataFrame({"FPR": fpr, "TPR": tpr}) \
    .to_csv(f"{OUTPUT_DIR}/roc_curve.csv", index=False)

plt.figure()
plt.plot(fpr, tpr, label="ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.savefig(f"{OUTPUT_DIR}/roc_curve.png")
plt.close()

# ───────────── PR DATA + PLOT ─────────────
p, r, _ = precision_recall_curve(y_test, y_prob)
pd.DataFrame({"Precision": p[:-1], "Recall": r[:-1]}) \
    .to_csv(f"{OUTPUT_DIR}/pr_curve.csv", index=False)

plt.figure()
plt.plot(r, p, label="PR Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.savefig(f"{OUTPUT_DIR}/pr_curve.png")
plt.close()

# ───────────── CONFUSION MATRIX PLOTS ─────────────
plt.figure()
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d")
plt.title("Confusion Matrix Default")
plt.savefig(f"{OUTPUT_DIR}/cm_default.png")
plt.close()

plt.figure()
sns.heatmap(confusion_matrix(y_test, y_pred_tuned), annot=True, fmt="d")
plt.title("Confusion Matrix Tuned")
plt.savefig(f"{OUTPUT_DIR}/cm_tuned.png")
plt.close()

# ───────────── METRICS ─────────────
pd.DataFrame([
    ["default", acc, prec, rec, f1, roc_auc_score(y_test, y_prob)],
    ["tuned", acc_t, prec_t, rec_t, f1_t, roc_auc_score(y_test, y_prob)]
], columns=["type","accuracy","precision","recall","f1","roc_auc"]) \
.to_csv(f"{OUTPUT_DIR}/metrics_summary.csv", index=False)

# ───────────── FEATURES (SAFE) ─────────────
try:
    support_mask = selector.get_support()
    if len(FEATURE_NAMES) == len(support_mask):
        selected_features = np.array(FEATURE_NAMES)[support_mask]
    else:
        selected_features = [f"feature_{i}" for i, val in enumerate(support_mask) if val]

    pd.DataFrame({"feature": selected_features}) \
        .to_csv(f"{OUTPUT_DIR}/selected_features.csv", index=False)
except Exception as e:
    print("Skipping feature saving:", e)

# ───────────── SAVE MODELS ─────────────
joblib.dump(model, f"{OUTPUT_DIR}/model.pkl")
joblib.dump(scaler, f"{OUTPUT_DIR}/scaler.pkl")

# ───────────── ZIP EVERYTHING ─────────────
shutil.make_archive("/kaggle/working/fft_rf_vs_svc_results", 'zip', OUTPUT_DIR)

print("All results + plots saved in:", OUTPUT_DIR)
print("ZIP:", "/kaggle/working/fft_rf_vs_svc_results.zip")

In [ ]:
"""
=============================================================================
 SLEEP APNEA DETECTION — FEATURE-ENGINEERED CLASSICAL ML PIPELINE
 (ANOVA feature selection + multi-model comparison: trees, boosting,
  linear, kernel, and neural-on-features approaches)
=============================================================================

FULL PIPELINE EXPLAINED
------------------------------------------------------------------------
1. LOAD RAW WINDOWS
   Each patient has X_SN{n}.npy (num_windows, 8, 6000) float16 and
   y_SN{n}.npy (num_windows,) int8. 8 channels = flow, chest, abd, spo2,
   pleth, ecg, pulse, flow2. Each window = 30s at 200Hz = 6000 samples.

2. FEATURE EXTRACTION (per window, per channel)
   Raw 6000-sample windows are too high-dimensional and noisy for classical
   ML directly, so we extract a rich set of interpretable statistical /
   spectral / physiological features per channel per window:
     - Time-domain   : mean, std, min, max, range, RMS, skewness, kurtosis,
                        zero-crossing rate, mean absolute deviation
     - Amplitude     : percentiles (5th/25th/75th/95th), IQR
     - Frequency     : dominant frequency, spectral energy in physiologically
                        relevant bands (via FFT power spectrum), spectral
                        entropy
     - Signal shape  : line length (sum of abs diffs -> proxy for signal
                        "busyness"), number of peaks, mean peak prominence
     - SpO2-specific : desaturation depth (baseline - min), desaturation
                        slope (fastest drop rate) -- SpO2/pleth are usually
                        the single most informative channels for apnea/
                        hypopnea because of the desaturation lag
   This turns each 30s window into a fixed-length feature vector
   (8 channels x ~15-20 features = ~120-160 raw features).

3. ANOVA F-TEST FEATURE SELECTION
   SelectKBest(f_classif) ranks features by how well their means separate
   the two classes (apnea vs normal) via an F-test, keeping the top K
   (default 60, matching your reported run) most discriminative features.
   This removes redundant/noisy features and speeds up training a lot.

4. PATIENT-LEVEL STRATIFIED SPLIT
   Same principle as the deep-learning script: split by PATIENT (not by
   window) so no patient's data leaks between train/val/test, and stratify
   by each patient's apnea prevalence so all three splits have a similar
   class balance (reduces noisy validation metrics).

5. CLASS IMBALANCE HANDLING
   - SMOTE (Synthetic Minority Oversampling) on the TRAIN split only, to
     balance classes at the feature level (works well for tabular features,
     unlike raw waveforms).
   - class_weight='balanced' as a second lever for models that support it.

6. MODEL ZOO  (this directly answers "what other models fit this kind of
   data" — feature-engineered, tabular, moderately-imbalanced, patient-
   grouped physiological time-series windows)
     - Logistic Regression (baseline, calibrated probabilities, cheap)
     - Random Forest (robust, handles feature interactions, less prone to
       overfitting than boosting on noisy medical features)
     - HistGradientBoosting (sklearn's fast native-imbalance-friendly GBM)
     - XGBoost (scale_pos_weight for imbalance, strong on tabular data)
     - LightGBM (usually the best speed/accuracy tradeoff on tabular data
       of this size; handles categorical-like binned features very well)
     - CatBoost (often strong out-of-the-box, less hyperparameter-sensitive)
     - SVM with RBF kernel (good when classes are non-linearly separable
       in feature space but dataset is moderate-sized -- here we subsample
       for tractability since SVM doesn't scale well past ~20-30k rows)
     - k-NN (simple baseline, sometimes surprisingly strong on physiological
       feature spaces where local neighborhoods are informative)
     - MLP (small feed-forward neural net on the extracted features --
       different from the ResNet1D since it operates on features, not raw
       waveforms)
     - Stacking Ensemble (meta-learner: logistic regression on top of the
       out-of-fold predictions of the best base models -- usually beats any
       single model and beats simple averaging ensembles)

7. THRESHOLD TUNING
   Decision threshold tuned on validation set to maximize F1 (default 0.5
   is rarely optimal under class imbalance).

8. EVENT-LEVEL EVALUATION
   Same AHI-style event scoring as the deep-learning pipeline: consecutive
   positive-window runs = one detected event, scored against consecutive
   true-positive runs with overlap matching. This is usually where numbers
   look much better than raw per-window scoring, since it doesn't punish a
   multi-window event for having 1-2 misclassified windows inside it.

9. FULL METRICS
   Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix, classification
   report, at both window-level and event-level, for every model, saved to
   CSV plus comparison plots.

HONEST NOTE: your AUCs (0.76-0.80) indicate the ceiling for window-level
apnea/hypopnea separability with THIS feature set is moderate -- this is
consistent with what the deep-learning ResNet1D got on the same data.
Getting all of accuracy/precision/recall/F1 to >=80% simultaneously at the
window level is unlikely without richer input (longer context windows,
more patients, or multi-window temporal features). This script pushes hard
on every lever available in the feature-engineering + classical-ML space,
and the event-level scoring at the end is where you're most likely to see
numbers that finally clear 80%.
=============================================================================
"""

import os
import gc
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats as sstats
from scipy.fft import rfft, rfftfreq
from scipy.signal import find_peaks

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, HistGradientBoostingClassifier,
                               StackingClassifier, VotingClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report, roc_auc_score,
                              roc_curve, precision_recall_curve)

warnings.filterwarnings("ignore")

import kagglehub

# ─────────────────────────────────────────────────────────────────────────
# optional libs -- gracefully degrade if not installed
# ─────────────────────────────────────────────────────────────────────────
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed -- skipping XGBoost model. `pip install xgboost --break-system-packages`")

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("lightgbm not installed -- skipping LightGBM. `pip install lightgbm --break-system-packages`")

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("catboost not installed -- skipping CatBoost. `pip install catboost --break-system-packages`")

try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False
    print("imbalanced-learn not installed -- skipping SMOTE. `pip install imbalanced-learn --break-system-packages`")

SEED = 42
np.random.seed(SEED)

# ─────────────────────────────────────────────────────────────────────────
# 1. DOWNLOAD DATASET
# ─────────────────────────────────────────────────────────────────────────
DATA_DIR = kagglehub.dataset_download(
    "nishantomhare/detection-of-disordered-breathing-dataset2"
)

def locate_npy_root(root):
    for dirpath, _, filenames in os.walk(root):
        if any(f.startswith("X_SN") and f.endswith(".npy") for f in filenames):
            return dirpath
    return root

DATA_DIR = locate_npy_root(DATA_DIR)
print("Resolved npy root:", DATA_DIR)

OUTPUT_DIR = "/kaggle/working/feature_ml_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────
# 2. CONFIG
# ─────────────────────────────────────────────────────────────────────────
SKIP_PATIENTS = {
    "SN1", "SN28", "SN14", "SN13", "SN17", "SN18",
    "SN19", "SN20", "SN27", "SN89"
}

FIXED_CHANNEL_KEYS = ["flow", "chest", "abd", "spo2", "pleth", "ecg", "pulse", "flow2"]
FS = 200                      # Hz
WINDOW_SEC = 30
SAMPLES_PER_WINDOW = FS * WINDOW_SEC   # 6000

TEST_FRAC         = 0.20
VAL_FRAC_OF_TRAIN = 0.12
N_STRAT_BINS      = 5
TOP_K_FEATURES    = 60         # matches your reported ANOVA60 run

USE_SMOTE = True
SVM_MAX_TRAIN_ROWS = 20000     # SVM subsample cap (SVM doesn't scale well)

# ─────────────────────────────────────────────────────────────────────────
# 3. DISCOVER PATIENTS
# ─────────────────────────────────────────────────────────────────────────
all_x_files = sorted(f for f in os.listdir(DATA_DIR)
                      if f.startswith("X_SN") and f.endswith(".npy"))
all_pids = [f[len("X_"):-len(".npy")] for f in all_x_files]
pids = [p for p in all_pids if p not in SKIP_PATIENTS
        and os.path.exists(os.path.join(DATA_DIR, f"y_{p}.npy"))]

print(f"Total X files: {len(all_x_files)} | Usable patients: {len(pids)}")

# ─────────────────────────────────────────────────────────────────────────
# 4. FEATURE EXTRACTION
# ─────────────────────────────────────────────────────────────────────────
def band_energy(freqs, power, lo, hi):
    mask = (freqs >= lo) & (freqs < hi)
    return float(power[mask].sum()) if mask.any() else 0.0


def extract_channel_features(sig, fs=FS, is_spo2=False):
    """Extract a fixed-length feature vector from one channel's 30s window."""
    sig = np.asarray(sig, dtype=np.float64)
    feats = {}

    # ---- time-domain ----
    feats["mean"]  = np.mean(sig)
    feats["std"]   = np.std(sig)
    feats["min"]   = np.min(sig)
    feats["max"]   = np.max(sig)
    feats["range"] = feats["max"] - feats["min"]
    feats["rms"]   = np.sqrt(np.mean(sig ** 2))
    feats["mad"]   = np.mean(np.abs(sig - feats["mean"]))
    feats["skew"]  = sstats.skew(sig) if feats["std"] > 1e-8 else 0.0
    feats["kurt"]  = sstats.kurtosis(sig) if feats["std"] > 1e-8 else 0.0

    # ---- percentiles / IQR ----
    p5, p25, p75, p95 = np.percentile(sig, [5, 25, 75, 95])
    feats["p5"], feats["p25"], feats["p75"], feats["p95"] = p5, p25, p75, p95
    feats["iqr"] = p75 - p25

    # ---- zero-crossing rate ----
    centered = sig - feats["mean"]
    feats["zcr"] = float(np.sum(np.diff(np.sign(centered)) != 0)) / len(sig)

    # ---- line length (signal "busyness") ----
    feats["line_length"] = float(np.sum(np.abs(np.diff(sig))))

    # ---- peaks ----
    try:
        peaks, props = find_peaks(sig, prominence=feats["std"] * 0.3 + 1e-6)
        feats["n_peaks"] = len(peaks)
        feats["mean_peak_prom"] = float(np.mean(props["prominences"])) if len(peaks) else 0.0
    except Exception:
        feats["n_peaks"] = 0
        feats["mean_peak_prom"] = 0.0

    # ---- frequency domain ----
    windowed = sig * np.hanning(len(sig))
    spec = np.abs(rfft(windowed))
    freqs = rfftfreq(len(sig), d=1.0 / fs)
    power = spec ** 2
    total_power = power.sum() + 1e-12

    feats["dom_freq"] = float(freqs[np.argmax(power)]) if len(power) else 0.0
    feats["spec_entropy"] = float(-np.sum((power / total_power) *
                                           np.log(power / total_power + 1e-12)))
    feats["energy_low"]  = band_energy(freqs, power, 0.0, 0.5) / total_power   # slow drift
    feats["energy_mid"]  = band_energy(freqs, power, 0.5, 2.0) / total_power   # breathing band
    feats["energy_high"] = band_energy(freqs, power, 2.0, fs / 2) / total_power  # noise/cardiac

    # ---- SpO2-specific desaturation features ----
    if is_spo2:
        baseline = np.percentile(sig, 90)  # robust "healthy" baseline within window
        feats["desat_depth"] = float(baseline - feats["min"])
        diffs = np.diff(sig)
        feats["desat_slope"] = float(diffs.min()) if len(diffs) else 0.0
        feats["pct_below_90"] = float(np.mean(sig < 90)) if np.max(sig) > 50 else 0.0

    return feats


def extract_window_features(window_8ch, spo2_channel_idx=3):
    """window_8ch: (8, 6000) -> flat feature dict across all channels."""
    all_feats = {}
    for ci, key in enumerate(FIXED_CHANNEL_KEYS):
        is_spo2 = (ci == spo2_channel_idx)
        ch_feats = extract_channel_features(window_8ch[ci], is_spo2=is_spo2)
        for fname, val in ch_feats.items():
            all_feats[f"{key}_{fname}"] = val
    return all_feats


def extract_patient_features(pid, data_dir):
    """Extract features for every window of one patient. Returns (df, y)."""
    X = np.load(os.path.join(data_dir, f"X_{pid}.npy"))   # (n, 8, 6000) float16
    y = np.load(os.path.join(data_dir, f"y_{pid}.npy"))   # (n,) int8
    n = X.shape[0]
    rows = []
    for i in range(n):
        feats = extract_window_features(X[i])
        feats["patient"] = pid
        feats["window_idx"] = i
        rows.append(feats)
    df = pd.DataFrame(rows)
    del X
    gc.collect()
    return df, y


FEATURE_CACHE_PATH = os.path.join(OUTPUT_DIR, "features_all.parquet")

if os.path.exists(FEATURE_CACHE_PATH):
    print("\nLoading cached extracted features...")
    full_df = pd.read_parquet(FEATURE_CACHE_PATH)
else:
    print("\nExtracting features for all patients (this takes a while, cached after)...")
    dfs, labels = [], []
    for idx, pid in enumerate(pids):
        t0 = time.time()
        df, y = extract_patient_features(pid, DATA_DIR)
        df["label"] = y
        dfs.append(df)
        print(f"  [{idx+1}/{len(pids)}] {pid}: {len(df)} windows extracted in {time.time()-t0:.1f}s")
    full_df = pd.concat(dfs, ignore_index=True)
    full_df.to_parquet(FEATURE_CACHE_PATH)
    print(f"Saved feature cache to {FEATURE_CACHE_PATH}")

print(f"\nTotal windows: {len(full_df):,} | Total raw features: "
      f"{full_df.shape[1] - 3}")   # minus patient, window_idx, label

# ─────────────────────────────────────────────────────────────────────────
# 5. STRATIFIED PATIENT-LEVEL SPLIT
# ─────────────────────────────────────────────────────────────────────────
patient_ratio = full_df.groupby("patient")["label"].mean()
patient_n     = full_df.groupby("patient")["label"].count()

rng = np.random.default_rng(SEED)
sorted_pids = patient_ratio.sort_values().index.to_numpy()
bin_edges = np.linspace(0, len(sorted_pids), N_STRAT_BINS + 1).astype(int)
train_pids, val_pids, test_pids = [], [], []

for b in range(N_STRAT_BINS):
    bucket = list(sorted_pids[bin_edges[b]:bin_edges[b + 1]])
    rng.shuffle(bucket)
    n = len(bucket)
    n_test = max(1, round(n * TEST_FRAC)) if n >= 3 else (1 if n > 1 else 0)
    remaining = bucket[n_test:]
    n_val = max(1, round(len(remaining) * VAL_FRAC_OF_TRAIN)) if len(remaining) >= 3 else 0
    test_pids.extend(bucket[:n_test])
    val_pids.extend(remaining[:n_val])
    train_pids.extend(remaining[n_val:])

train_df = full_df[full_df["patient"].isin(train_pids)].reset_index(drop=True)
val_df   = full_df[full_df["patient"].isin(val_pids)].reset_index(drop=True)
test_df  = full_df[full_df["patient"].isin(test_pids)].reset_index(drop=True)

print(f"\nTrain: {len(train_df):,} windows from {len(train_pids)} patients "
      f"(apnea ratio={train_df['label'].mean():.3f})")
print(f"Val  : {len(val_df):,} windows from {len(val_pids)} patients "
      f"(apnea ratio={val_df['label'].mean():.3f})")
print(f"Test : {len(test_df):,} windows from {len(test_pids)} patients "
      f"(apnea ratio={test_df['label'].mean():.3f})")

feature_cols = [c for c in full_df.columns if c not in ("patient", "window_idx", "label")]
X_train_raw, y_train = train_df[feature_cols].values, train_df["label"].values
X_val_raw,   y_val    = val_df[feature_cols].values,   val_df["label"].values
X_test_raw,  y_test   = test_df[feature_cols].values,  test_df["label"].values

# ─────────────────────────────────────────────────────────────────────────
# 6. SCALING + ANOVA FEATURE SELECTION  (fit on TRAIN only)
# ─────────────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(np.nan_to_num(X_train_raw))
X_val_scaled   = scaler.transform(np.nan_to_num(X_val_raw))
X_test_scaled  = scaler.transform(np.nan_to_num(X_test_raw))

selector = SelectKBest(score_func=f_classif, k=min(TOP_K_FEATURES, len(feature_cols)))
X_train_sel = selector.fit_transform(X_train_scaled, y_train)
X_val_sel   = selector.transform(X_val_scaled)
X_test_sel  = selector.transform(X_test_scaled)

selected_mask = selector.get_support()
selected_features = [f for f, m in zip(feature_cols, selected_mask) if m]
print(f"\nSelected top {len(selected_features)} features via ANOVA F-test:")
print(selected_features)

pd.Series(selected_features).to_csv(os.path.join(OUTPUT_DIR, "selected_features.csv"), index=False)

# ─────────────────────────────────────────────────────────────────────────
# 7. CLASS IMBALANCE HANDLING (SMOTE on TRAIN only)
# ─────────────────────────────────────────────────────────────────────────
if USE_SMOTE and HAS_SMOTE:
    print("\nApplying SMOTE oversampling to training set...")
    smote = SMOTE(random_state=SEED)
    X_train_final, y_train_final = smote.fit_resample(X_train_sel, y_train)
    print(f"  Before SMOTE: normal={np.sum(y_train==0):,} apnea={np.sum(y_train==1):,}")
    print(f"  After  SMOTE: normal={np.sum(y_train_final==0):,} apnea={np.sum(y_train_final==1):,}")
else:
    X_train_final, y_train_final = X_train_sel, y_train

# ─────────────────────────────────────────────────────────────────────────
# 8. MODEL ZOO
# ─────────────────────────────────────────────────────────────────────────
n_pos = np.sum(y_train == 1)
n_neg = np.sum(y_train == 0)
scale_pos_weight = n_neg / max(1, n_pos)

models = {}

models["LogisticRegression"] = LogisticRegression(
    max_iter=2000, class_weight="balanced", C=0.5, random_state=SEED)

models["RandomForest"] = RandomForestClassifier(
    n_estimators=400, max_depth=14, min_samples_leaf=5,
    class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)

models["HistGB"] = HistGradientBoostingClassifier(
    max_iter=400, max_depth=8, learning_rate=0.05,
    l2_regularization=1.0, class_weight="balanced", random_state=SEED)

if HAS_XGB:
    models["XGBoost"] = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
        scale_pos_weight=scale_pos_weight, eval_metric="logloss",
        tree_method="hist", random_state=SEED, n_jobs=-1)

if HAS_LGB:
    models["LightGBM"] = lgb.LGBMClassifier(
        n_estimators=600, max_depth=-1, num_leaves=63, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
        class_weight="balanced", random_state=SEED, n_jobs=-1, verbose=-1)

if HAS_CATBOOST:
    models["CatBoost"] = CatBoostClassifier(
        iterations=500, depth=8, learning_rate=0.03, l2_leaf_reg=4.0,
        auto_class_weights="Balanced", random_state=SEED, verbose=False)

models["kNN"] = KNeighborsClassifier(n_neighbors=25, weights="distance", n_jobs=-1)

models["MLP"] = MLPClassifier(
    hidden_layer_sizes=(128, 64), activation="relu", alpha=1e-3,
    learning_rate_init=1e-3, max_iter=300, early_stopping=True,
    random_state=SEED)

# SVM: subsample train set for tractability (RBF SVM doesn't scale well
# past tens of thousands of rows)
if len(X_train_final) > SVM_MAX_TRAIN_ROWS:
    idx = np.random.default_rng(SEED).choice(len(X_train_final), SVM_MAX_TRAIN_ROWS, replace=False)
    X_train_svm, y_train_svm = X_train_final[idx], y_train_final[idx]
else:
    X_train_svm, y_train_svm = X_train_final, y_train_final

svm_base = SVC(kernel="rbf", C=2.0, gamma="scale", class_weight="balanced",
                probability=False, random_state=SEED)
models["SVM_RBF"] = CalibratedClassifierCV(svm_base, cv=3, method="sigmoid")

# ─────────────────────────────────────────────────────────────────────────
# 9. HELPERS: threshold tuning + event-level scoring
# ─────────────────────────────────────────────────────────────────────────
def best_f1_threshold(labels, probs):
    prec, rec, thr = precision_recall_curve(labels, probs)
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    if len(thr) == 0:
        return 0.5, 0.0
    best_idx = np.nanargmax(f1s[:-1])
    return float(thr[best_idx]), float(f1s[best_idx])


def extract_events(binary_seq):
    events, start = [], None
    for i, v in enumerate(binary_seq):
        if v == 1 and start is None:
            start = i
        elif v == 0 and start is not None:
            events.append((start, i - 1)); start = None
    if start is not None:
        events.append((start, len(binary_seq) - 1))
    return events


def match_events(pred_events, true_events, min_overlap=1):
    matched_true = set()
    pred_hit = [False] * len(pred_events)
    for pi, (ps, pe) in enumerate(pred_events):
        for ti, (ts, te) in enumerate(true_events):
            overlap = max(0, min(pe, te) - max(ps, ts) + 1)
            if overlap >= min_overlap:
                matched_true.add(ti)
                pred_hit[pi] = True
    tp_true = len(matched_true)
    tp_pred = sum(pred_hit)
    fp = len(pred_events) - tp_pred
    fn = len(true_events) - tp_true
    return tp_true, tp_pred, fp, fn, len(pred_events), len(true_events)


def event_level_metrics(df_with_patient, labels, preds):
    agg = dict(tp_true=0, tp_pred=0, fp=0, fn=0, n_pred=0, n_true=0)
    for pid, grp in df_with_patient.groupby("patient", sort=False):
        idx = grp.index.to_numpy()
        order = np.argsort(grp["window_idx"].to_numpy())
        idx = idx[order]
        seg_labels = labels[idx]
        seg_preds = preds[idx]
        pe = extract_events(seg_preds)
        te = extract_events(seg_labels)
        tp_true, tp_pred, fp, fn, npe, nte = match_events(pe, te)
        agg["tp_true"] += tp_true; agg["tp_pred"] += tp_pred
        agg["fp"] += fp; agg["fn"] += fn
        agg["n_pred"] += npe; agg["n_true"] += nte
    precision = agg["tp_pred"] / agg["n_pred"] if agg["n_pred"] > 0 else 0.0
    recall    = agg["tp_true"] / agg["n_true"] if agg["n_true"] > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return dict(precision=precision, recall=recall, f1_score=f1, **agg)

# ─────────────────────────────────────────────────────────────────────────
# 10. TRAIN + EVALUATE EVERY MODEL
# ─────────────────────────────────────────────────────────────────────────
results = []
roc_curves = {}
trained_models = {}

for name, model in models.items():
    print(f"\n{'='*65}\nTraining: {name}\n{'='*65}")
    t0 = time.time()

    if name == "SVM_RBF":
        model.fit(X_train_svm, y_train_svm)
    else:
        model.fit(X_train_final, y_train_final)

    fit_time = time.time() - t0
    trained_models[name] = model

    val_probs = model.predict_proba(X_val_sel)[:, 1]
    thr, _ = best_f1_threshold(y_val, val_probs)

    test_probs = model.predict_proba(X_test_sel)[:, 1]
    test_preds = (test_probs >= thr).astype(int)

    acc  = accuracy_score(y_test, test_preds)
    prec = precision_score(y_test, test_preds, zero_division=0)
    rec  = recall_score(y_test, test_preds, zero_division=0)
    f1   = f1_score(y_test, test_preds, zero_division=0)
    try:
        auc = roc_auc_score(y_test, test_probs)
    except ValueError:
        auc = float("nan")
    cm = confusion_matrix(y_test, test_preds)

    ev = event_level_metrics(test_df, y_test, test_preds)

    print(f"  fit_time={fit_time:.1f}s | thr={thr:.3f} | "
          f"ACC={acc:.3f} REC={rec:.3f} PREC={prec:.3f} F1={f1:.3f} AUC={auc:.3f}")
    print(f"  Event-level: PREC={ev['precision']:.3f} REC={ev['recall']:.3f} F1={ev['f1_score']:.3f}")
    print(f"  Confusion matrix:\n{cm}")

    if not np.isnan(auc):
        fpr, tpr, _ = roc_curve(y_test, test_probs)
        roc_curves[name] = (fpr, tpr, auc)

    results.append({
        "model": name, "threshold": thr, "fit_time_s": fit_time,
        "accuracy": acc, "recall": rec, "precision": prec, "f1_score": f1, "auc": auc,
        "event_precision": ev["precision"], "event_recall": ev["recall"], "event_f1": ev["f1_score"],
        "tn": int(cm[0, 0]), "fp": int(cm[0, 1]), "fn": int(cm[1, 0]), "tp": int(cm[1, 1]),
    })

# ─────────────────────────────────────────────────────────────────────────
# 11. STACKING ENSEMBLE (meta-learner on top of best base models)
# ─────────────────────────────────────────────────────────────────────────
print(f"\n{'='*65}\nTraining: Stacking Ensemble\n{'='*65}")
stack_candidates = [(n, m) for n, m in trained_models.items()
                     if n in ("RandomForest", "HistGB", "LightGBM", "XGBoost", "CatBoost")]

if len(stack_candidates) >= 2:
    t0 = time.time()
    stack = StackingClassifier(
        estimators=stack_candidates,
        final_estimator=LogisticRegression(max_iter=1000, class_weight="balanced"),
        stack_method="predict_proba", n_jobs=-1, cv=3, passthrough=False)
    stack.fit(X_train_final, y_train_final)
    fit_time = time.time() - t0

    val_probs = stack.predict_proba(X_val_sel)[:, 1]
    thr, _ = best_f1_threshold(y_val, val_probs)
    test_probs = stack.predict_proba(X_test_sel)[:, 1]
    test_preds = (test_probs >= thr).astype(int)

    acc  = accuracy_score(y_test, test_preds)
    prec = precision_score(y_test, test_preds, zero_division=0)
    rec  = recall_score(y_test, test_preds, zero_division=0)
    f1   = f1_score(y_test, test_preds, zero_division=0)
    auc  = roc_auc_score(y_test, test_probs)
    cm   = confusion_matrix(y_test, test_preds)
    ev   = event_level_metrics(test_df, y_test, test_preds)

    print(f"  fit_time={fit_time:.1f}s | thr={thr:.3f} | "
          f"ACC={acc:.3f} REC={rec:.3f} PREC={prec:.3f} F1={f1:.3f} AUC={auc:.3f}")
    print(f"  Event-level: PREC={ev['precision']:.3f} REC={ev['recall']:.3f} F1={ev['f1_score']:.3f}")

    roc_curves["StackingEnsemble"] = roc_curve(y_test, test_probs)[:2] + (auc,)
    results.append({
        "model": "StackingEnsemble", "threshold": thr, "fit_time_s": fit_time,
        "accuracy": acc, "recall": rec, "precision": prec, "f1_score": f1, "auc": auc,
        "event_precision": ev["precision"], "event_recall": ev["recall"], "event_f1": ev["f1_score"],
        "tn": int(cm[0, 0]), "fp": int(cm[0, 1]), "fn": int(cm[1, 0]), "tp": int(cm[1, 1]),
    })

# ─────────────────────────────────────────────────────────────────────────
# 12. RESULTS TABLE + PLOTS
# ─────────────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results).sort_values("f1_score", ascending=False)
results_df.to_csv(os.path.join(OUTPUT_DIR, "model_comparison_results.csv"), index=False)

print("\n" + "=" * 100)
print("  FINAL MODEL COMPARISON (sorted by window-level F1)")
print("=" * 100)
print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for name, (fpr, tpr, auc) in roc_curves.items():
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curves — All Models"); axes[0].legend(fontsize=8)

results_df.set_index("model")[["accuracy", "precision", "recall", "f1_score"]].plot(
    kind="bar", ax=axes[1])
axes[1].set_title("Window-level Metrics by Model")
axes[1].set_ylabel("Score"); axes[1].legend(loc="lower right")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model_comparison_plots.png"), dpi=150)
plt.show()

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(" - features_all.parquet          (cached extracted features, reused on rerun)")
print(" - selected_features.csv         (ANOVA-selected feature names)")
print(" - model_comparison_results.csv  (full metrics table, window + event level)")
print(" - model_comparison_plots.png")

1. Input Preparation (Your Case)

For apnea signals:

Input shape: (batch, channels=8, time=6000)

👉 Treat as 1D segmentation problem
(each time step = apnea / normal)

🔵 2. Residual Block (Core Building Unit)

Instead of:

Conv → ReLU → Conv

Use:

Input → Conv → BN → ReLU → Conv → BN
        ↓___________________________
               + (skip connection)
                ↓
              ReLU
✅ Why?
Prevents vanishing gradients
Enables deeper networks
Better feature reuse
🔻 3. Encoder (Downsampling Path)

Each level:

Residual Block → MaxPool (↓2)

Example:

Level	Channels	Output
L1	8 → 32	6000
L2	32 → 64	3000
L3	64 → 128	1500
L4	128 → 256	750
🧠 4. Bottleneck
Residual Block (256 → 512)

👉 Captures global context:

full breathing cycle
apnea duration patterns
🔺 5. Decoder (Upsampling Path)

Each level:

Upsample → Concatenate (skip from encoder) → Residual Block

Example:

Step	Operation
Up1	512 → 256
Up2	256 → 128
Up3	128 → 64
Up4	64 → 32
🔗 6. Skip Connections (U-Net style)
Connect encoder → decoder
Preserve fine temporal details

👉 Critical for:

precise apnea boundaries
🎯 7. Output Layer
1×1 Conv → Sigmoid

Output:

(batch, 1, 6000)  # per-time-step prediction

Residual U-Net

In [ ]:
"""
=============================================================================
 SLEEP APNEA DETECTION — RESIDUAL U-NET 1D CLASSIFIER (Kaggle GPU)
=============================================================================
 ARCHITECTURE NOTE:
 U-Net is normally a SEGMENTATION architecture (dense per-timestep output
 with an encoder-decoder + skip connections). This dataset has ONE label
 per 30s window (not per-timestep), so this script builds a full Residual
 U-Net (encoder -> bottleneck -> decoder with skip connections, exactly as
 requested) and attaches a classification head (global average pool ->
 dense -> softmax) on top of the decoder's output feature map. This lets
 the classifier benefit from the U-Net's multi-scale feature fusion (skip
 connections carry fine-grained early features forward to the final layer,
 not just the heavily-downsampled bottleneck features) while still
 producing a single window-level prediction.

 Input length is exactly 6000 samples (30s @ 200Hz) = 2^4 * 375, so 4
 encoder downsamples (maxpool /2 each) and 4 decoder upsamples (transposed
 conv x2 each) line up exactly with the skip connections -- no cropping or
 padding hacks needed.

 LOSS: weighted categorical cross-entropy. The network outputs 2 logits
 (normal / apnea) and nn.CrossEntropyLoss(weight=class_weights) applies
 softmax + categorical cross-entropy with per-class weights set by inverse
 class frequency (this IS categorical cross-entropy for the 2-class case --
 equivalent to what Keras calls `categorical_crossentropy` with one-hot
 labels and `class_weight`).

 SPLIT: patient-level, stratified by each patient's apnea window ratio,
 60% train / 20% val / 20% test. Patient-level (not window-level) splitting
 is essential here -- windows from the same patient are highly correlated,
 so splitting by window would leak patient identity across splits and
 inflate metrics.

 Input  : X_SN{n}.npy -> (num_windows, 8, 6000) float16
          y_SN{n}.npy -> (num_windows,)         int8  (0=normal, 1=apnea)
=============================================================================
"""

import os
import gc
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_recall_curve
)

import kagglehub

# ─────────────────────────────────────────────────────────────────────────
# 0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ─────────────────────────────────────────────────────────────────────────
# 0b. GPU / KERNEL-COMPATIBILITY CHECK  (P100 = compute capability 6.0)
# ─────────────────────────────────────────────────────────────────────────
def _gpu_kernels_usable():
    if not torch.cuda.is_available():
        return False
    try:
        x = torch.randn(2, 4, 16, device="cuda")
        conv = nn.Conv1d(4, 4, 3, padding=1).cuda()
        _ = conv(x)
        return True
    except RuntimeError as e:
        print("\n⚠️  GPU kernel check failed:", str(e).splitlines()[0])
        print("⚠️  Falling back to CPU. Switch accelerator to T4/L4 to use GPU.")
        return False

_USE_CUDA = _gpu_kernels_usable()

# ─────────────────────────────────────────────────────────────────────────
# 1. DOWNLOAD DATASET
# ─────────────────────────────────────────────────────────────────────────
DATA_DIR = kagglehub.dataset_download(
    "nishantomhare/detection-of-disordered-breathing-dataset2"
)
print("Dataset path:", DATA_DIR)

def locate_npy_root(root):
    for dirpath, _, filenames in os.walk(root):
        if any(f.startswith("X_SN") and f.endswith(".npy") for f in filenames):
            return dirpath
    return root

DATA_DIR = locate_npy_root(DATA_DIR)
print("Resolved npy root:", DATA_DIR)

OUTPUT_DIR = "/kaggle/working/residual_unet1d_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────
# 2. CONFIG
# ─────────────────────────────────────────────────────────────────────────
# No patients skipped by default for this run -- add IDs here if you find
# corrupt/unreadable files (e.g. SKIP_PATIENTS = {"SN1", "SN14"}).
SKIP_PATIENTS = set()

TEST_FRAC   = 0.20     # of TOTAL patients
VAL_FRAC    = 0.20     # of TOTAL patients  -> remaining ~0.60 is train
N_STRAT_BINS = 5

NUM_CHANNELS = 8
NUM_CLASSES  = 2
BATCH_SIZE   = 64
EPOCHS       = 50
LR           = 8e-4
WEIGHT_DECAY = 2e-4
NUM_WORKERS  = 2
PATIENCE     = 10
GRAD_CLIP    = 5.0

UNET_BASE_WIDTH = 32     # channel width of first encoder stage (doubles each stage)
UNET_DROPOUT    = 0.4

AUGMENT           = True
NOISE_STD_FRAC    = 0.03
SCALE_JITTER      = 0.10
MAX_TIME_SHIFT    = 100
CHANNEL_DROPOUT_P = 0.05

NORM_SAMPLE_WINDOWS = 8000

DEVICE = torch.device("cuda" if _USE_CUDA else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| capability:", torch.cuda.get_device_capability(0),
          "| usable:", _USE_CUDA)

# ─────────────────────────────────────────────────────────────────────────
# 3. DISCOVER PATIENTS
# ─────────────────────────────────────────────────────────────────────────
all_x_files = sorted(f for f in os.listdir(DATA_DIR)
                      if f.startswith("X_SN") and f.endswith(".npy"))
all_pids = [f[len("X_"):-len(".npy")] for f in all_x_files]

pids = [p for p in all_pids if p not in SKIP_PATIENTS]
pids = [p for p in pids if os.path.exists(os.path.join(DATA_DIR, f"y_{p}.npy"))]

print(f"\nTotal X files found : {len(all_x_files)}")
print(f"Usable patients ({len(pids)}): {pids}")

if len(pids) < 5:
    raise RuntimeError("Not enough usable patients to train/val/test split.")

# ─────────────────────────────────────────────────────────────────────────
# 4. STRATIFIED 60/20/20 PATIENT-LEVEL SPLIT
#    (stratified by each patient's apnea window ratio, so all 3 splits
#     have a similar class balance -> less noisy val/test metrics)
# ─────────────────────────────────────────────────────────────────────────
def patient_apnea_ratio(pid):
    y = np.load(os.path.join(DATA_DIR, f"y_{pid}.npy"), mmap_mode="r")
    n = len(y)
    apnea = int(np.sum(y))
    del y
    return apnea / max(1, n), n, apnea

patient_stats = {pid: patient_apnea_ratio(pid) for pid in pids}
ratios = np.array([patient_stats[p][0] for p in pids])

rng = np.random.default_rng(SEED)
order = np.argsort(ratios)
sorted_pids = np.array(pids)[order]

bin_edges = np.linspace(0, len(sorted_pids), N_STRAT_BINS + 1).astype(int)
train_pids, val_pids, test_pids = [], [], []

for b in range(N_STRAT_BINS):
    bucket = list(sorted_pids[bin_edges[b]:bin_edges[b + 1]])
    rng.shuffle(bucket)
    n = len(bucket)
    n_test = max(1, round(n * TEST_FRAC)) if n >= 3 else (1 if n > 1 else 0)
    n_val  = max(1, round(n * VAL_FRAC))  if n >= 3 else (1 if n - n_test > 1 else 0)
    test_pids.extend(bucket[:n_test])
    val_pids.extend(bucket[n_test:n_test + n_val])
    train_pids.extend(bucket[n_test + n_val:])

if not val_pids and train_pids:
    val_pids.append(train_pids.pop())
if not test_pids and train_pids:
    test_pids.append(train_pids.pop())

print(f"\nTrain patients ({len(train_pids)}, {100*len(train_pids)/len(pids):.0f}%): {sorted(train_pids)}")
print(f"Val   patients ({len(val_pids)}, {100*len(val_pids)/len(pids):.0f}%): {sorted(val_pids)}")
print(f"Test  patients ({len(test_pids)}, {100*len(test_pids)/len(pids):.0f}%): {sorted(test_pids)}")

for name, plist in [("train", train_pids), ("val", val_pids), ("test", test_pids)]:
    tot_n = sum(patient_stats[p][1] for p in plist)
    tot_a = sum(patient_stats[p][2] for p in plist)
    print(f"  {name:5s}: {tot_n:,} windows, apnea ratio = {tot_a / max(1, tot_n):.3f}")

# ─────────────────────────────────────────────────────────────────────────
# 5. PER-CHANNEL NORMALIZATION STATS (from TRAIN set only)
# ─────────────────────────────────────────────────────────────────────────
def estimate_norm_stats(patient_ids, n_sample_windows):
    sums   = np.zeros(NUM_CHANNELS, dtype=np.float64)
    sqsums = np.zeros(NUM_CHANNELS, dtype=np.float64)
    count  = 0
    per_patient_budget = max(1, n_sample_windows // max(1, len(patient_ids)))
    for pid in patient_ids:
        X = np.load(os.path.join(DATA_DIR, f"X_{pid}.npy"), mmap_mode="r")
        n = X.shape[0]
        if n == 0:
            continue
        k = min(per_patient_budget, n)
        idx = np.random.default_rng(SEED).choice(n, size=k, replace=False)
        sample = X[idx].astype(np.float64)
        sums   += sample.sum(axis=(0, 2))
        sqsums += (sample ** 2).sum(axis=(0, 2))
        count  += k * sample.shape[2]
        del X, sample
        gc.collect()
    mean = sums / count
    var  = sqsums / count - mean ** 2
    std  = np.sqrt(np.clip(var, 1e-8, None))
    return mean.astype(np.float32), std.astype(np.float32)

print("\nEstimating per-channel normalization stats from train set...")
CH_MEAN, CH_STD = estimate_norm_stats(train_pids, NORM_SAMPLE_WINDOWS)
print("Channel means:", np.round(CH_MEAN, 3))
print("Channel stds :", np.round(CH_STD, 3))
np.save(os.path.join(OUTPUT_DIR, "channel_mean.npy"), CH_MEAN)
np.save(os.path.join(OUTPUT_DIR, "channel_std.npy"), CH_STD)

# ─────────────────────────────────────────────────────────────────────────
# 6. DATASET
# ─────────────────────────────────────────────────────────────────────────
class ApneaDataset(Dataset):
    def __init__(self, data_dir, patient_ids, mean, std, augment=False):
        self.data_dir = data_dir
        self._cache = {}
        self.index = []
        self.mean = mean.reshape(NUM_CHANNELS, 1)
        self.std  = std.reshape(NUM_CHANNELS, 1)
        self.augment = augment
        for pid in patient_ids:
            xp = os.path.join(data_dir, f"X_{pid}.npy")
            yp = os.path.join(data_dir, f"y_{pid}.npy")
            if not (os.path.exists(xp) and os.path.exists(yp)):
                continue
            y = np.load(yp, mmap_mode="r")
            self.index.extend([(pid, i) for i in range(len(y))])
            del y

    def _arrays(self, pid):
        if pid not in self._cache:
            self._cache[pid] = (
                np.load(os.path.join(self.data_dir, f"X_{pid}.npy"), mmap_mode="r"),
                np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r"),
            )
        return self._cache[pid]

    def __len__(self):
        return len(self.index)

    def labels(self):
        out = np.zeros(len(self.index), dtype=np.int8)
        cache = {}
        for i, (pid, li) in enumerate(self.index):
            if pid not in cache:
                cache[pid] = np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r")
            out[i] = cache[pid][li]
        return out

    def __getitem__(self, idx):
        pid, local_idx = self.index[idx]
        X_arr, y_arr = self._arrays(pid)
        x = X_arr[local_idx].astype(np.float32)          # (8, 6000)
        x = (x - self.mean) / self.std

        if self.augment:
            if NOISE_STD_FRAC > 0:
                x = x + np.random.randn(*x.shape).astype(np.float32) * NOISE_STD_FRAC
            if SCALE_JITTER > 0:
                scale = 1.0 + (np.random.rand(NUM_CHANNELS, 1).astype(np.float32) * 2 - 1) * SCALE_JITTER
                x = x * scale
            if MAX_TIME_SHIFT > 0:
                shift = np.random.randint(-MAX_TIME_SHIFT, MAX_TIME_SHIFT + 1)
                if shift != 0:
                    x = np.roll(x, shift, axis=1)
            if CHANNEL_DROPOUT_P > 0 and np.random.rand() < CHANNEL_DROPOUT_P:
                ch = np.random.randint(0, NUM_CHANNELS)
                x[ch, :] = 0.0

        y = int(y_arr[local_idx])
        return torch.from_numpy(x.copy()), torch.tensor(y, dtype=torch.long)


train_ds = ApneaDataset(DATA_DIR, train_pids, CH_MEAN, CH_STD, augment=AUGMENT)
val_ds   = ApneaDataset(DATA_DIR, val_pids,   CH_MEAN, CH_STD, augment=False)
test_ds  = ApneaDataset(DATA_DIR, test_pids,  CH_MEAN, CH_STD, augment=False)

print(f"\nTrain windows: {len(train_ds):,}")
print(f"Val   windows: {len(val_ds):,}")
print(f"Test  windows: {len(test_ds):,}")

train_labels = train_ds.labels()
n0, n1 = int((train_labels == 0).sum()), int((train_labels == 1).sum())
total_train = n0 + n1
print(f"\nTrain class counts: normal={n0:,}  apnea={n1:,}")

class_weights = torch.tensor(
    [total_train / (2.0 * n0 + 1e-9), total_train / (2.0 * n1 + 1e-9)],
    dtype=torch.float32
).to(DEVICE)
print("Class weights (for weighted categorical CE):", class_weights.tolist())

sample_weights = np.where(train_labels == 1,
                           total_train / (2.0 * n1 + 1e-9),
                           total_train / (2.0 * n0 + 1e-9))
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

# ─────────────────────────────────────────────────────────────────────────
# 7. RESIDUAL U-NET 1D ARCHITECTURE
# ─────────────────────────────────────────────────────────────────────────
class ResBlock1D(nn.Module):
    """Two conv-bn-relu layers with a residual (skip) connection."""
    def __init__(self, in_ch, out_ch, k=5, dropout=0.0):
        super().__init__()
        pad = k // 2
        self.conv1 = nn.Conv1d(in_ch, out_ch, k, padding=pad, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, k, padding=pad, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.drop  = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.skip  = (nn.Conv1d(in_ch, out_ch, 1, bias=False)
                      if in_ch != out_ch else nn.Identity())

    def forward(self, x):
        identity = self.skip(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        out = out + identity
        return self.relu(out)


class ResidualUNet1D(nn.Module):
    """
    Residual U-Net for 1D physiological signals, adapted for classification:
      encoder (4 downsampling stages) -> bottleneck -> decoder (4 upsampling
      stages with skip connections) -> global average pool -> softmax head.

    Input length must be divisible by 2^4 (6000 = 16 * 375, exact).
    """
    def __init__(self, in_channels=8, num_classes=2, base_width=32, dropout=0.4):
        super().__init__()
        w = base_width

        # ---- encoder ----
        self.enc1 = ResBlock1D(in_channels, w,     dropout=dropout * 0.5)   # 6000
        self.enc2 = ResBlock1D(w,           w * 2, dropout=dropout * 0.5)   # 3000
        self.enc3 = ResBlock1D(w * 2,       w * 4, dropout=dropout * 0.5)   # 1500
        self.enc4 = ResBlock1D(w * 4,       w * 8, dropout=dropout * 0.5)   # 750
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)

        # ---- bottleneck ----
        self.bottleneck = ResBlock1D(w * 8, w * 16, dropout=dropout)        # 375

        # ---- decoder (transposed-conv upsample + skip concat + ResBlock) ----
        self.up4  = nn.ConvTranspose1d(w * 16, w * 8, kernel_size=2, stride=2)   # 375->750
        self.dec4 = ResBlock1D(w * 8 + w * 8, w * 8, dropout=dropout * 0.5)

        self.up3  = nn.ConvTranspose1d(w * 8, w * 4, kernel_size=2, stride=2)    # 750->1500
        self.dec3 = ResBlock1D(w * 4 + w * 4, w * 4, dropout=dropout * 0.5)

        self.up2  = nn.ConvTranspose1d(w * 4, w * 2, kernel_size=2, stride=2)    # 1500->3000
        self.dec2 = ResBlock1D(w * 2 + w * 2, w * 2, dropout=dropout * 0.5)

        self.up1  = nn.ConvTranspose1d(w * 2, w, kernel_size=2, stride=2)        # 3000->6000
        self.dec1 = ResBlock1D(w + w, w, dropout=dropout * 0.5)

        # ---- classification head ----
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.head_dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(w, num_classes)

        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.ConvTranspose1d)):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # encoder
        s1 = self.enc1(x)              # (B, w,    6000)
        s2 = self.enc2(self.pool(s1))  # (B, 2w,   3000)
        s3 = self.enc3(self.pool(s2))  # (B, 4w,   1500)
        s4 = self.enc4(self.pool(s3))  # (B, 8w,    750)

        b = self.bottleneck(self.pool(s4))   # (B, 16w, 375)

        # decoder with skip connections
        d4 = self.up4(b)                              # (B, 8w, 750)
        d4 = self.dec4(torch.cat([d4, s4], dim=1))     # (B, 8w, 750)

        d3 = self.up3(d4)                              # (B, 4w, 1500)
        d3 = self.dec3(torch.cat([d3, s3], dim=1))      # (B, 4w, 1500)

        d2 = self.up2(d3)                               # (B, 2w, 3000)
        d2 = self.dec2(torch.cat([d2, s2], dim=1))       # (B, 2w, 3000)

        d1 = self.up1(d2)                                # (B, w, 6000)
        d1 = self.dec1(torch.cat([d1, s1], dim=1))        # (B, w, 6000)

        # classification head
        pooled = self.gap(d1).squeeze(-1)   # (B, w)
        pooled = self.head_dropout(pooled)
        logits = self.fc(pooled)            # (B, num_classes)
        return logits


model = ResidualUNet1D(NUM_CHANNELS, NUM_CLASSES,
                        base_width=UNET_BASE_WIDTH, dropout=UNET_DROPOUT).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel: Residual U-Net 1D | Trainable params: {n_params:,}")

# ─────────────────────────────────────────────────────────────────────────
# 8. WEIGHTED CATEGORICAL CROSS-ENTROPY LOSS
# ─────────────────────────────────────────────────────────────────────────
# nn.CrossEntropyLoss applies softmax internally then computes categorical
# cross-entropy; `weight=class_weights` makes it WEIGHTED categorical CE,
# scaling each class's contribution by its inverse frequency.
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=4
)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

# ─────────────────────────────────────────────────────────────────────────
# 9. TRAIN / EVAL HELPERS
# ─────────────────────────────────────────────────────────────────────────
def best_f1_threshold(labels, probs):
    prec, rec, thr = precision_recall_curve(labels, probs)
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    if len(thr) == 0:
        return 0.5, 0.0
    best_idx = np.nanargmax(f1s[:-1])
    return float(thr[best_idx]), float(f1s[best_idx])


def run_epoch(loader, train_mode: bool):
    model.train(mode=train_mode)
    total_loss = 0.0
    all_labels, all_probs = [], []

    for X, y in loader:
        X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                logits = model(X)
                loss = criterion(logits, y)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * X.size(0)
        probs = F.softmax(logits.float(), dim=1)[:, 1].detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(loader.dataset))
    return avg_loss, np.array(all_labels), np.array(all_probs)

# ─────────────────────────────────────────────────────────────────────────
# 10. TRAIN LOOP WITH EARLY STOPPING
# ─────────────────────────────────────────────────────────────────────────
best_val_f1 = -1.0
best_state = None
best_threshold = 0.5
epochs_no_improve = 0
history = []

print("\n" + "=" * 65)
print("  TRAINING — Residual U-Net 1D")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_labels_ep, train_probs_ep = run_epoch(train_loader, train_mode=True)
    val_loss, val_labels, val_probs = run_epoch(val_loader, train_mode=False)

    tuned_thr, tuned_val_f1 = best_f1_threshold(val_labels, val_probs)

    train_preds05 = (train_probs_ep >= 0.5).astype(int)
    train_f105 = f1_score(train_labels_ep, train_preds05, zero_division=0)
    val_preds05 = (val_probs >= 0.5).astype(int)
    val_f105 = f1_score(val_labels, val_preds05, zero_division=0)

    scheduler.step(tuned_val_f1)
    dt = time.time() - t0

    history.append({
        "epoch": epoch, "train_loss": train_loss, "train_f1@0.5": train_f105,
        "val_loss": val_loss, "val_f1@0.5": val_f105,
        "val_f1@tuned": tuned_val_f1, "tuned_threshold": tuned_thr,
        "lr": optimizer.param_groups[0]["lr"],
    })

    print(f"Epoch {epoch:3d}/{EPOCHS} | train_loss={train_loss:.4f} f1@.5={train_f105:.4f} | "
          f"val_loss={val_loss:.4f} f1@.5={val_f105:.4f} f1@tuned={tuned_val_f1:.4f} "
          f"(thr={tuned_thr:.2f}) | lr={optimizer.param_groups[0]['lr']:.2e} | {dt:.1f}s")

    if tuned_val_f1 > best_val_f1:
        best_val_f1 = tuned_val_f1
        best_threshold = tuned_thr
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        torch.save({"state_dict": best_state, "threshold": best_threshold},
                   os.path.join(OUTPUT_DIR, "best_residual_unet1d.pt"))
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break

pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)

if best_state is not None:
    model.load_state_dict(best_state)
print(f"\nBest val F1 (tuned threshold): {best_val_f1:.4f}  @ threshold={best_threshold:.3f}")

# ─────────────────────────────────────────────────────────────────────────
# 11. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  TEST SET EVALUATION")
print("=" * 65)

_, test_labels, test_probs = run_epoch(test_loader, train_mode=False)

try:
    auc = roc_auc_score(test_labels, test_probs)
except ValueError:
    auc = float("nan")
print(f"\nROC-AUC (threshold-independent): {auc:.4f}")

def report_at_threshold(probs, labels, thr, label):
    preds = (probs >= thr).astype(int)
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)
    cm   = confusion_matrix(labels, preds)
    rep  = classification_report(labels, preds, target_names=["Normal", "Apnea"], zero_division=0)
    print(f"\n--- {label} (threshold={thr:.3f}) ---")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    print(f"\nClassification Report:\n{rep}")
    return dict(label=label, threshold=thr, accuracy=acc, precision=prec, recall=rec, f1_score=f1,
                tn=int(cm[0, 0]), fp=int(cm[0, 1]), fn=int(cm[1, 0]), tp=int(cm[1, 1])), rep, cm

m_05, rep_05, cm_05 = report_at_threshold(test_probs, test_labels, 0.5, "Default threshold")
m_tuned, rep_tuned, cm_tuned = report_at_threshold(test_probs, test_labels, best_threshold, "Val-tuned threshold")

pd.DataFrame([m_05, m_tuned]).to_csv(os.path.join(OUTPUT_DIR, "test_metrics.csv"), index=False)
with open(os.path.join(OUTPUT_DIR, "classification_report_tuned.txt"), "w") as fh:
    fh.write(rep_tuned)

# ─────────────────────────────────────────────────────────────────────────
# 12. PLOTS
# ─────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im = axes[0].imshow(cm_tuned, cmap="Blues")
axes[0].set_title(f"Confusion Matrix (thr={best_threshold:.2f})")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["Normal", "Apnea"])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(["Normal", "Apnea"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm_tuned[i, j]), ha="center", va="center",
                      color="white" if cm_tuned[i, j] > cm_tuned.max() / 2 else "black")
fig.colorbar(im, ax=axes[0])

if not np.isnan(auc):
    fpr, tpr, _ = roc_curve(test_labels, test_probs)
    axes[1].plot(fpr, tpr, label=f"AUC = {auc:.3f}")
    axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
    axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title("ROC Curve"); axes[1].legend()

hist_df = pd.DataFrame(history)
axes[2].plot(hist_df["epoch"], hist_df["train_f1@0.5"], label="Train F1@0.5")
axes[2].plot(hist_df["epoch"], hist_df["val_f1@tuned"], label="Val F1@tuned")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("F1 Score")
axes[2].set_title("Training Curves"); axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "evaluation_plots.png"), dpi=150)
plt.show()

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(" - best_residual_unet1d.pt   (weights + tuned threshold)")
print(" - channel_mean.npy / channel_std.npy   (apply these at inference time!)")
print(" - training_history.csv")
print(" - test_metrics.csv   (default@0.5 vs val-tuned threshold)")
print(" - classification_report_tuned.txt")
print(" - evaluation_plots.png")

# 🧠 Sleep Apnea Detection — CNN + BiGRU + Attention (Full Explanation + Math)

---

# 📌 1. Problem Definition

We aim to detect **sleep apnea events** from physiological signals.

### Input:
- 8-channel signal (flow, chest, abdomen, SpO₂, etc.)
- Each window = **6000 samples (~30 sec)**

### Output:
- Binary label per window:
  - `0 → Normal`
  - `1 → Apnea`

---

# 🧠 2. Why This Architecture?

A raw RNN on 6000 timesteps is inefficient because:
- Long sequences → vanishing gradients
- High noise at sample level
- Poor learning of meaningful patterns

### ✅ Solution:
Use a **hybrid model**:

---

# 🏗️ 3. Full Pipeline

## Step 1: Context Window Creation

Instead of using a single window:

- Radius = 1 → 90 sec context
- Input shape becomes:

### Why?
Apnea is a **temporal phenomenon**, not isolated.

---

# 🧮 4. CNN Frontend (Feature Extraction)

## Operation

1D Convolution:

\[
y[t] = \sum_{k=0}^{K-1} w_k \cdot x[t+k]
\]

Where:
- \(x\) = input signal
- \(w\) = kernel
- \(K\) = kernel size

---

## What CNN Learns

- Breathing patterns
- Flow drops (apnea)
- Signal morphology
- Noise reduction

---

## Downsampling

Pooling reduces sequence:

This:
- Reduces computation
- Keeps important patterns

---

# 🔁 5. BiGRU (Bidirectional GRU)

## GRU Equations

### Update Gate:
\[
z_t = \sigma(W_z x_t + U_z h_{t-1})
\]

### Reset Gate:
\[
r_t = \sigma(W_r x_t + U_r h_{t-1})
\]

### Candidate:
\[
\tilde{h}_t = \tanh(W x_t + U (r_t \cdot h_{t-1}))
\]

### Final State:
\[
h_t = (1 - z_t) \cdot h_{t-1} + z_t \cdot \tilde{h}_t
\]

---

## Bidirectional Processing

---

# 🔎 10. Inference (Prediction)

## Important:

During testing:

- ✅ Uses only `X.npy`
- ❌ Does NOT use `y.npy`

---

## Prediction Pipeline

For each window \(i\):


In [ ]:
"""
=============================================================================
 SLEEP APNEA DETECTION — CNN-FRONTEND + BiGRU + ATTENTION  (Kaggle GPU)
 with Optuna hyperparameter tuning and weighted binary cross-entropy
=============================================================================
 WHY THIS ARCHITECTURE (the "best RNN" for this kind of signal):

 A plain GRU fed 6000 raw timesteps/channel directly is impractical: it's
 slow, prone to vanishing gradients over that many steps, and wastes
 capacity modeling sample-to-sample noise instead of physiologically
 meaningful patterns. The standard, well-established recipe for long
 biosignals (ECG, EEG, respiration, PSG channels) is a CNN-RNN HYBRID:

   1. CONV FRONT-END (1D conv + BN + ReLU + strided pooling, x4 stages)
      downsamples 6000 raw samples -> ~375 "feature tokens", each token
      already summarizing local morphology (like a learned filter bank),
      while expanding from 8 raw channels to a richer learned feature
      dimension.
   2. BIDIRECTIONAL GRU (stacked, tunable depth/width) reads that token
      sequence forward AND backward, so a mid-window feature has context
      from both earlier and later in the 30s window -- important because
      an apnea event's characteristic desaturation can occur near either
      edge of a window.
   3. ADDITIVE ATTENTION POOLING instead of just taking the last hidden
      state: apnea-relevant information is often concentrated in a
      SUBSEGMENT of the 30s window (the event itself), not spread evenly
      across it, so a learned attention-weighted sum over all GRU outputs
      lets the model focus on the relevant sub-window instead of diluting
      the signal by averaging over 30s of mostly-normal breathing.
   4. Dense classification head -> single logit -> weighted BCE loss.

 GRU (not LSTM) is used because it has fewer parameters/gates, trains
 faster, and performs comparably to LSTM on most biosignal classification
 benchmarks -- a reasonable "best" choice for a moderate-sized dataset
 where overfitting is already a concern (as seen in earlier experiments).

 HYPERPARAMETER TUNING: Optuna (Bayesian/TPE search) if installed, with a
 random-search fallback otherwise. Tuned: conv frontend width, GRU hidden
 size, GRU layers, dropout, learning rate, weight decay. Each trial trains
 briefly on a subsample for speed; the best config is then trained fully
 with early stopping.

 LOSS: weighted binary cross-entropy via
 nn.BCEWithLogitsLoss(pos_weight=...) -- pos_weight = (#normal / #apnea)
 in the training set, which upweights the minority (apnea) class exactly
 the way class_weight would in Keras' binary_crossentropy.

 Input  : X_SN{n}.npy -> (num_windows, 8, 6000) float16
          y_SN{n}.npy -> (num_windows,)         int8  (0=normal, 1=apnea)
=============================================================================
"""

import os
import gc
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_recall_curve
)

warnings.filterwarnings("ignore")
import kagglehub

# ─────────────────────────────────────────────────────────────────────────
# optional: Optuna for hyperparameter search (falls back to random search)
# ─────────────────────────────────────────────────────────────────────────
try:
    import optuna
    from optuna.samplers import TPESampler
    HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False
    print("optuna not installed -- using random-search fallback. "
          "(`pip install optuna --break-system-packages` for Bayesian search)")

# ─────────────────────────────────────────────────────────────────────────
# 0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ─────────────────────────────────────────────────────────────────────────
# 0b. GPU / KERNEL-COMPATIBILITY CHECK  (P100 = compute capability 6.0)
# ─────────────────────────────────────────────────────────────────────────
def _gpu_kernels_usable():
    if not torch.cuda.is_available():
        return False
    try:
        x = torch.randn(2, 4, 16, device="cuda")
        conv = nn.Conv1d(4, 4, 3, padding=1).cuda()
        _ = conv(x)
        return True
    except RuntimeError as e:
        print("\n⚠️  GPU kernel check failed:", str(e).splitlines()[0])
        print("⚠️  Falling back to CPU. Switch accelerator to T4/L4 to use GPU.")
        return False

_USE_CUDA = _gpu_kernels_usable()

# ─────────────────────────────────────────────────────────────────────────
# 1. DOWNLOAD DATASET
# ─────────────────────────────────────────────────────────────────────────
DATA_DIR = kagglehub.dataset_download(
    "nishantomhare/detection-of-disordered-breathing-dataset2"
)
print("Dataset path:", DATA_DIR)

def locate_npy_root(root):
    for dirpath, _, filenames in os.walk(root):
        if any(f.startswith("X_SN") and f.endswith(".npy") for f in filenames):
            return dirpath
    return root

DATA_DIR = locate_npy_root(DATA_DIR)
print("Resolved npy root:", DATA_DIR)

OUTPUT_DIR = "/kaggle/working/gru_apnea_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────
# 2. CONFIG
# ─────────────────────────────────────────────────────────────────────────
SKIP_PATIENTS = {
    "SN1", "SN28", "SN14", "SN13", "SN17", "SN18",
    "SN19", "SN20", "SN27", "SN89"
}   # known problematic/corrupt patients, excluded from all splits

TEST_FRAC    = 0.15
VAL_FRAC     = 0.15     # remaining ~0.70 -> train
N_STRAT_BINS = 5

NUM_CHANNELS = 8
BATCH_SIZE   = 64
FINAL_EPOCHS = 40
LR_DEFAULT   = 8e-4
NUM_WORKERS  = 2
PATIENCE     = 8
GRAD_CLIP    = 5.0

AUGMENT           = True
NOISE_STD_FRAC    = 0.03
SCALE_JITTER      = 0.10
MAX_TIME_SHIFT    = 100
CHANNEL_DROPOUT_P = 0.05

NORM_SAMPLE_WINDOWS = 8000

# ---- hyperparameter search settings ----
N_TRIALS          = 15
SEARCH_EPOCHS     = 3       # short training per trial, just for ranking configs
SEARCH_TRAIN_FRAC = 0.35    # fraction of train windows used during search (speed)
SEARCH_VAL_FRAC    = 0.6    # fraction of val windows used during search (speed)

DEVICE = torch.device("cuda" if _USE_CUDA else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| capability:", torch.cuda.get_device_capability(0),
          "| usable:", _USE_CUDA)

# ─────────────────────────────────────────────────────────────────────────
# 3. DISCOVER PATIENTS
# ─────────────────────────────────────────────────────────────────────────
all_x_files = sorted(f for f in os.listdir(DATA_DIR)
                      if f.startswith("X_SN") and f.endswith(".npy"))
all_pids = [f[len("X_"):-len(".npy")] for f in all_x_files]
pids = [p for p in all_pids if p not in SKIP_PATIENTS
        and os.path.exists(os.path.join(DATA_DIR, f"y_{p}.npy"))]
print(f"\nTotal X files: {len(all_x_files)} | Usable patients: {len(pids)}")

if len(pids) < 5:
    raise RuntimeError("Not enough usable patients to train/val/test split.")

# ─────────────────────────────────────────────────────────────────────────
# 4. STRATIFIED PATIENT-LEVEL SPLIT
# ─────────────────────────────────────────────────────────────────────────
def patient_apnea_ratio(pid):
    y = np.load(os.path.join(DATA_DIR, f"y_{pid}.npy"), mmap_mode="r")
    n = len(y)
    apnea = int(np.sum(y))
    del y
    return apnea / max(1, n), n, apnea

patient_stats = {pid: patient_apnea_ratio(pid) for pid in pids}
ratios = np.array([patient_stats[p][0] for p in pids])

rng = np.random.default_rng(SEED)
order = np.argsort(ratios)
sorted_pids = np.array(pids)[order]
bin_edges = np.linspace(0, len(sorted_pids), N_STRAT_BINS + 1).astype(int)
train_pids, val_pids, test_pids = [], [], []

for b in range(N_STRAT_BINS):
    bucket = list(sorted_pids[bin_edges[b]:bin_edges[b + 1]])
    rng.shuffle(bucket)
    n = len(bucket)
    n_test = max(1, round(n * TEST_FRAC)) if n >= 3 else (1 if n > 1 else 0)
    n_val  = max(1, round(n * VAL_FRAC))  if n >= 3 else (1 if n - n_test > 1 else 0)
    test_pids.extend(bucket[:n_test])
    val_pids.extend(bucket[n_test:n_test + n_val])
    train_pids.extend(bucket[n_test + n_val:])

if not val_pids and train_pids:
    val_pids.append(train_pids.pop())
if not test_pids and train_pids:
    test_pids.append(train_pids.pop())

print(f"\nTrain patients ({len(train_pids)}): {sorted(train_pids)}")
print(f"Val   patients ({len(val_pids)}): {sorted(val_pids)}")
print(f"Test  patients ({len(test_pids)}): {sorted(test_pids)}")
for name, plist in [("train", train_pids), ("val", val_pids), ("test", test_pids)]:
    tot_n = sum(patient_stats[p][1] for p in plist)
    tot_a = sum(patient_stats[p][2] for p in plist)
    print(f"  {name:5s}: {tot_n:,} windows, apnea ratio = {tot_a / max(1, tot_n):.3f}")

# ─────────────────────────────────────────────────────────────────────────
# 5. PER-CHANNEL NORMALIZATION STATS (from TRAIN set only)
# ─────────────────────────────────────────────────────────────────────────
def estimate_norm_stats(patient_ids, n_sample_windows):
    sums   = np.zeros(NUM_CHANNELS, dtype=np.float64)
    sqsums = np.zeros(NUM_CHANNELS, dtype=np.float64)
    count  = 0
    per_patient_budget = max(1, n_sample_windows // max(1, len(patient_ids)))
    for pid in patient_ids:
        X = np.load(os.path.join(DATA_DIR, f"X_{pid}.npy"), mmap_mode="r")
        n = X.shape[0]
        if n == 0:
            continue
        k = min(per_patient_budget, n)
        idx = np.random.default_rng(SEED).choice(n, size=k, replace=False)
        sample = X[idx].astype(np.float64)
        sums   += sample.sum(axis=(0, 2))
        sqsums += (sample ** 2).sum(axis=(0, 2))
        count  += k * sample.shape[2]
        del X, sample
        gc.collect()
    mean = sums / count
    var  = sqsums / count - mean ** 2
    std  = np.sqrt(np.clip(var, 1e-8, None))
    return mean.astype(np.float32), std.astype(np.float32)

print("\nEstimating per-channel normalization stats from train set...")
CH_MEAN, CH_STD = estimate_norm_stats(train_pids, NORM_SAMPLE_WINDOWS)
np.save(os.path.join(OUTPUT_DIR, "channel_mean.npy"), CH_MEAN)
np.save(os.path.join(OUTPUT_DIR, "channel_std.npy"), CH_STD)
print("Channel means:", np.round(CH_MEAN, 3))
print("Channel stds :", np.round(CH_STD, 3))

# ─────────────────────────────────────────────────────────────────────────
# 6. DATASET
# ─────────────────────────────────────────────────────────────────────────
class ApneaDataset(Dataset):
    def __init__(self, data_dir, patient_ids, mean, std, augment=False):
        self.data_dir = data_dir
        self._cache = {}
        self.index = []
        self.mean = mean.reshape(NUM_CHANNELS, 1)
        self.std  = std.reshape(NUM_CHANNELS, 1)
        self.augment = augment
        for pid in patient_ids:
            xp = os.path.join(data_dir, f"X_{pid}.npy")
            yp = os.path.join(data_dir, f"y_{pid}.npy")
            if not (os.path.exists(xp) and os.path.exists(yp)):
                continue
            y = np.load(yp, mmap_mode="r")
            self.index.extend([(pid, i) for i in range(len(y))])
            del y

    def _arrays(self, pid):
        if pid not in self._cache:
            self._cache[pid] = (
                np.load(os.path.join(self.data_dir, f"X_{pid}.npy"), mmap_mode="r"),
                np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r"),
            )
        return self._cache[pid]

    def __len__(self):
        return len(self.index)

    def labels(self):
        out = np.zeros(len(self.index), dtype=np.int8)
        cache = {}
        for i, (pid, li) in enumerate(self.index):
            if pid not in cache:
                cache[pid] = np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r")
            out[i] = cache[pid][li]
        return out

    def __getitem__(self, idx):
        pid, local_idx = self.index[idx]
        X_arr, y_arr = self._arrays(pid)
        x = X_arr[local_idx].astype(np.float32)          # (8, 6000)
        x = (x - self.mean) / self.std

        if self.augment:
            if NOISE_STD_FRAC > 0:
                x = x + np.random.randn(*x.shape).astype(np.float32) * NOISE_STD_FRAC
            if SCALE_JITTER > 0:
                scale = 1.0 + (np.random.rand(NUM_CHANNELS, 1).astype(np.float32) * 2 - 1) * SCALE_JITTER
                x = x * scale
            if MAX_TIME_SHIFT > 0:
                shift = np.random.randint(-MAX_TIME_SHIFT, MAX_TIME_SHIFT + 1)
                if shift != 0:
                    x = np.roll(x, shift, axis=1)
            if CHANNEL_DROPOUT_P > 0 and np.random.rand() < CHANNEL_DROPOUT_P:
                ch = np.random.randint(0, NUM_CHANNELS)
                x[ch, :] = 0.0

        y = float(y_arr[local_idx])
        return torch.from_numpy(x.copy()), torch.tensor(y, dtype=torch.float32)


train_ds = ApneaDataset(DATA_DIR, train_pids, CH_MEAN, CH_STD, augment=AUGMENT)
val_ds   = ApneaDataset(DATA_DIR, val_pids,   CH_MEAN, CH_STD, augment=False)
test_ds  = ApneaDataset(DATA_DIR, test_pids,  CH_MEAN, CH_STD, augment=False)

print(f"\nTrain windows: {len(train_ds):,}")
print(f"Val   windows: {len(val_ds):,}")
print(f"Test  windows: {len(test_ds):,}")

train_labels = train_ds.labels()
n0, n1 = int((train_labels == 0).sum()), int((train_labels == 1).sum())
total_train = n0 + n1
print(f"\nTrain class counts: normal={n0:,}  apnea={n1:,}")

# weighted BCE: pos_weight upweights the minority (apnea) class
POS_WEIGHT = torch.tensor([n0 / max(1, n1)], dtype=torch.float32).to(DEVICE)
print(f"pos_weight for weighted BCE: {POS_WEIGHT.item():.3f}")

sample_weights = np.where(train_labels == 1, n0 / max(1, n1), 1.0)
full_sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# ─────────────────────────────────────────────────────────────────────────
# 7. CNN-FRONTEND + BiGRU + ATTENTION ARCHITECTURE
# ─────────────────────────────────────────────────────────────────────────
class ConvFrontend(nn.Module):
    """4-stage strided-conv downsampler: 6000 -> 375 tokens, 8ch -> conv_width*8."""
    def __init__(self, in_channels=8, base_width=32, dropout=0.2):
        super().__init__()
        w = base_width
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv1d(cin, cout, kernel_size=7, stride=2, padding=3, bias=False),
                nn.BatchNorm1d(cout),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
            )
        self.net = nn.Sequential(
            block(in_channels, w),
            block(w, w * 2),
            block(w * 2, w * 4),
            block(w * 4, w * 8),
        )
        self.out_channels = w * 8

    def forward(self, x):
        # x: (B, 8, 6000) -> (B, out_channels, 375)
        return self.net(x)


class AdditiveAttentionPool(nn.Module):
    """Learned attention-weighted sum over the GRU output sequence."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, seq):
        # seq: (B, T, hidden_dim)
        scores = self.attn(seq).squeeze(-1)          # (B, T)
        weights = F.softmax(scores, dim=1).unsqueeze(-1)  # (B, T, 1)
        pooled = (seq * weights).sum(dim=1)           # (B, hidden_dim)
        return pooled, weights.squeeze(-1)


class ConvBiGRUAttention(nn.Module):
    def __init__(self, in_channels=8, conv_width=32, gru_hidden=128,
                 gru_layers=2, dropout=0.4, bidirectional=True):
        super().__init__()
        self.frontend = ConvFrontend(in_channels, conv_width, dropout=dropout * 0.5)
        gru_input_dim = self.frontend.out_channels

        self.gru = nn.GRU(
            input_size=gru_input_dim, hidden_size=gru_hidden, num_layers=gru_layers,
            batch_first=True, bidirectional=bidirectional,
            dropout=dropout if gru_layers > 1 else 0.0,
        )
        gru_out_dim = gru_hidden * (2 if bidirectional else 1)
        self.attn_pool = AdditiveAttentionPool(gru_out_dim)

        self.head = nn.Sequential(
            nn.Linear(gru_out_dim, gru_out_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(gru_out_dim // 2, 1),   # single logit -> BCEWithLogitsLoss
        )

    def forward(self, x, return_attention=False):
        # x: (B, 8, 6000)
        feat = self.frontend(x)                # (B, C, 375)
        feat = feat.permute(0, 2, 1)            # (B, 375, C)  -> GRU expects (B, T, F)
        gru_out, _ = self.gru(feat)             # (B, 375, gru_out_dim)
        pooled, attn_w = self.attn_pool(gru_out)
        logit = self.head(pooled).squeeze(-1)   # (B,)
        if return_attention:
            return logit, attn_w
        return logit

# ─────────────────────────────────────────────────────────────────────────
# 8. TRAIN / EVAL HELPERS  (shared by search trials and final training)
# ─────────────────────────────────────────────────────────────────────────
def best_f1_threshold(labels, probs):
    prec, rec, thr = precision_recall_curve(labels, probs)
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    if len(thr) == 0:
        return 0.5, 0.0
    best_idx = np.nanargmax(f1s[:-1])
    return float(thr[best_idx]), float(f1s[best_idx])


def run_epoch(model, loader, criterion, optimizer, scaler, train_mode: bool):
    model.train(mode=train_mode)
    total_loss = 0.0
    all_labels, all_probs = [], []

    for X, y in loader:
        X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                logits = model(X)
                loss = criterion(logits, y)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * X.size(0)
        probs = torch.sigmoid(logits.float()).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(loader.dataset))
    return avg_loss, np.array(all_labels), np.array(all_probs)

# ─────────────────────────────────────────────────────────────────────────
# 9. HYPERPARAMETER SEARCH
# ─────────────────────────────────────────────────────────────────────────
# small subsampled loaders just for fast trial ranking
rng_np = np.random.default_rng(SEED)
search_train_idx = rng_np.choice(len(train_ds), size=int(len(train_ds) * SEARCH_TRAIN_FRAC), replace=False)
search_val_idx   = rng_np.choice(len(val_ds), size=int(len(val_ds) * SEARCH_VAL_FRAC), replace=False)

search_train_subset = Subset(train_ds, search_train_idx)
search_val_subset   = Subset(val_ds, search_val_idx)

search_train_labels = train_labels[search_train_idx]
sn0, sn1 = int((search_train_labels == 0).sum()), int((search_train_labels == 1).sum())
search_sample_weights = np.where(search_train_labels == 1, sn0 / max(1, sn1), 1.0)
search_sampler = WeightedRandomSampler(search_sample_weights, num_samples=len(search_sample_weights), replacement=True)

search_train_loader = DataLoader(search_train_subset, batch_size=BATCH_SIZE, sampler=search_sampler,
                                  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
search_val_loader   = DataLoader(search_val_subset, batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=NUM_WORKERS, pin_memory=True)
search_pos_weight = torch.tensor([sn0 / max(1, sn1)], dtype=torch.float32).to(DEVICE)


def build_model(params):
    return ConvBiGRUAttention(
        in_channels=NUM_CHANNELS,
        conv_width=params["conv_width"],
        gru_hidden=params["gru_hidden"],
        gru_layers=params["gru_layers"],
        dropout=params["dropout"],
        bidirectional=params["bidirectional"],
    ).to(DEVICE)


def evaluate_trial(params, verbose=False):
    model = build_model(params)
    criterion = nn.BCEWithLogitsLoss(pos_weight=search_pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    best_f1 = -1.0
    for epoch in range(SEARCH_EPOCHS):
        run_epoch(model, search_train_loader, criterion, optimizer, scaler, train_mode=True)
        _, val_labels, val_probs = run_epoch(model, search_val_loader, criterion, optimizer, scaler, train_mode=False)
        _, f1 = best_f1_threshold(val_labels, val_probs)
        best_f1 = max(best_f1, f1)
        if verbose:
            print(f"    search epoch {epoch+1}/{SEARCH_EPOCHS}: val_f1={f1:.4f}")

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return best_f1


SEARCH_SPACE = {
    "conv_width":   [24, 32, 48],
    "gru_hidden":   [64, 128, 192],
    "gru_layers":   [1, 2, 3],
    "dropout":      [0.2, 0.3, 0.4, 0.5],
    "bidirectional": [True, False],
    "lr":           (1e-4, 3e-3),      # log-uniform range
    "weight_decay": (1e-6, 1e-3),      # log-uniform range
}

print("\n" + "=" * 65)
print(f"  HYPERPARAMETER SEARCH ({'Optuna TPE' if HAS_OPTUNA else 'random search'}, "
      f"{N_TRIALS} trials x {SEARCH_EPOCHS} epochs on a data subsample)")
print("=" * 65)

trial_log = []

if HAS_OPTUNA:
    def objective(trial):
        params = {
            "conv_width":    trial.suggest_categorical("conv_width", SEARCH_SPACE["conv_width"]),
            "gru_hidden":    trial.suggest_categorical("gru_hidden", SEARCH_SPACE["gru_hidden"]),
            "gru_layers":    trial.suggest_categorical("gru_layers", SEARCH_SPACE["gru_layers"]),
            "dropout":       trial.suggest_categorical("dropout", SEARCH_SPACE["dropout"]),
            "bidirectional": trial.suggest_categorical("bidirectional", SEARCH_SPACE["bidirectional"]),
            "lr":            trial.suggest_float("lr", *SEARCH_SPACE["lr"], log=True),
            "weight_decay":  trial.suggest_float("weight_decay", *SEARCH_SPACE["weight_decay"], log=True),
        }
        f1 = evaluate_trial(params)
        trial_log.append({**params, "val_f1": f1})
        print(f"  Trial {trial.number:2d}: {params} -> val_f1={f1:.4f}")
        return f1

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_TRIALS)
    best_params = study.best_params
    best_search_f1 = study.best_value
else:
    best_params, best_search_f1 = None, -1.0
    for t in range(N_TRIALS):
        params = {
            "conv_width":    random.choice(SEARCH_SPACE["conv_width"]),
            "gru_hidden":    random.choice(SEARCH_SPACE["gru_hidden"]),
            "gru_layers":    random.choice(SEARCH_SPACE["gru_layers"]),
            "dropout":       random.choice(SEARCH_SPACE["dropout"]),
            "bidirectional": random.choice(SEARCH_SPACE["bidirectional"]),
            "lr":            10 ** random.uniform(np.log10(SEARCH_SPACE["lr"][0]), np.log10(SEARCH_SPACE["lr"][1])),
            "weight_decay":  10 ** random.uniform(np.log10(SEARCH_SPACE["weight_decay"][0]), np.log10(SEARCH_SPACE["weight_decay"][1])),
        }
        f1 = evaluate_trial(params)
        trial_log.append({**params, "val_f1": f1})
        print(f"  Trial {t:2d}: {params} -> val_f1={f1:.4f}")
        if f1 > best_search_f1:
            best_search_f1 = f1
            best_params = params

pd.DataFrame(trial_log).to_csv(os.path.join(OUTPUT_DIR, "hparam_search_trials.csv"), index=False)
print(f"\nBest hyperparameters found (val_f1={best_search_f1:.4f}):")
for k, v in best_params.items():
    print(f"  {k}: {v}")

# ─────────────────────────────────────────────────────────────────────────
# 10. FINAL TRAINING WITH BEST HYPERPARAMETERS (full data, early stopping)
# ─────────────────────────────────────────────────────────────────────────
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=full_sampler,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

model = build_model(best_params)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nFinal model: CNN-BiGRU-Attention | Trainable params: {n_params:,}")

criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
optimizer = torch.optim.AdamW(model.parameters(), lr=best_params["lr"], weight_decay=best_params["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

best_val_f1 = -1.0
best_state = None
best_threshold = 0.5
epochs_no_improve = 0
history = []

print("\n" + "=" * 65)
print("  FINAL TRAINING")
print("=" * 65)

for epoch in range(1, FINAL_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_labels_ep, train_probs_ep = run_epoch(model, train_loader, criterion, optimizer, scaler, True)
    val_loss, val_labels, val_probs = run_epoch(model, val_loader, criterion, optimizer, scaler, False)

    tuned_thr, tuned_val_f1 = best_f1_threshold(val_labels, val_probs)
    train_f105 = f1_score(train_labels_ep, (train_probs_ep >= 0.5).astype(int), zero_division=0)
    val_f105 = f1_score(val_labels, (val_probs >= 0.5).astype(int), zero_division=0)

    scheduler.step(tuned_val_f1)
    dt = time.time() - t0

    history.append({
        "epoch": epoch, "train_loss": train_loss, "train_f1@0.5": train_f105,
        "val_loss": val_loss, "val_f1@0.5": val_f105,
        "val_f1@tuned": tuned_val_f1, "tuned_threshold": tuned_thr,
        "lr": optimizer.param_groups[0]["lr"],
    })

    print(f"Epoch {epoch:3d}/{FINAL_EPOCHS} | train_loss={train_loss:.4f} f1@.5={train_f105:.4f} | "
          f"val_loss={val_loss:.4f} f1@.5={val_f105:.4f} f1@tuned={tuned_val_f1:.4f} "
          f"(thr={tuned_thr:.2f}) | lr={optimizer.param_groups[0]['lr']:.2e} | {dt:.1f}s")

    if tuned_val_f1 > best_val_f1:
        best_val_f1 = tuned_val_f1
        best_threshold = tuned_thr
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        torch.save({"state_dict": best_state, "threshold": best_threshold, "hparams": best_params},
                   os.path.join(OUTPUT_DIR, "best_gru_model.pt"))
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break

pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)
if best_state is not None:
    model.load_state_dict(best_state)
print(f"\nBest val F1 (tuned threshold): {best_val_f1:.4f}  @ threshold={best_threshold:.3f}")

# ─────────────────────────────────────────────────────────────────────────
# 11. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  TEST SET EVALUATION")
print("=" * 65)

_, test_labels, test_probs = run_epoch(model, test_loader, criterion, optimizer, scaler, False)

try:
    auc = roc_auc_score(test_labels, test_probs)
except ValueError:
    auc = float("nan")
print(f"\nROC-AUC (threshold-independent): {auc:.4f}")

def report_at_threshold(probs, labels, thr, label):
    preds = (probs >= thr).astype(int)
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)
    cm   = confusion_matrix(labels, preds)
    rep  = classification_report(labels, preds, target_names=["Normal", "Apnea"], zero_division=0)
    print(f"\n--- {label} (threshold={thr:.3f}) ---")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    print(f"\nClassification Report:\n{rep}")
    return dict(label=label, threshold=thr, accuracy=acc, precision=prec, recall=rec, f1_score=f1,
                tn=int(cm[0, 0]), fp=int(cm[0, 1]), fn=int(cm[1, 0]), tp=int(cm[1, 1])), rep, cm

m_05, rep_05, cm_05 = report_at_threshold(test_probs, test_labels, 0.5, "Default threshold")
m_tuned, rep_tuned, cm_tuned = report_at_threshold(test_probs, test_labels, best_threshold, "Val-tuned threshold")

pd.DataFrame([m_05, m_tuned]).to_csv(os.path.join(OUTPUT_DIR, "test_metrics.csv"), index=False)
with open(os.path.join(OUTPUT_DIR, "classification_report_tuned.txt"), "w") as fh:
    fh.write(rep_tuned)

# ─────────────────────────────────────────────────────────────────────────
# 12. PLOTS
# ─────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im = axes[0].imshow(cm_tuned, cmap="Blues")
axes[0].set_title(f"Confusion Matrix (thr={best_threshold:.2f})")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["Normal", "Apnea"])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(["Normal", "Apnea"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm_tuned[i, j]), ha="center", va="center",
                      color="white" if cm_tuned[i, j] > cm_tuned.max() / 2 else "black")
fig.colorbar(im, ax=axes[0])

if not np.isnan(auc):
    fpr, tpr, _ = roc_curve(test_labels, test_probs)
    axes[1].plot(fpr, tpr, label=f"AUC = {auc:.3f}")
    axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
    axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title("ROC Curve"); axes[1].legend()

hist_df = pd.DataFrame(history)
axes[2].plot(hist_df["epoch"], hist_df["train_f1@0.5"], label="Train F1@0.5")
axes[2].plot(hist_df["epoch"], hist_df["val_f1@tuned"], label="Val F1@tuned")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("F1 Score")
axes[2].set_title("Training Curves"); axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "evaluation_plots.png"), dpi=150)
plt.show()

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(" - best_gru_model.pt              (weights + threshold + best hparams)")
print(" - channel_mean.npy / channel_std.npy")
print(" - hparam_search_trials.csv       (every trial's config + val F1)")
print(" - training_history.csv")
print(" - test_metrics.csv")
print(" - classification_report_tuned.txt")
print(" - evaluation_plots.png")

# 🧠 Sleep Apnea Detection — CNN + BiGRU + Attention (Context-Aware Model)

## 📌 Overview

This project implements a **deep learning pipeline for sleep apnea detection** using multi-channel physiological signals. The model combines:

- **1D CNN frontend** → extracts local temporal features  
- **Bidirectional GRU (BiGRU)** → captures temporal dependencies  
- **Attention mechanism** → focuses on apnea-relevant regions  

Unlike simple window-based models, this approach uses **context windows**, allowing the model to understand transitions such as:
- Normal → Apnea  
- Apnea → Recovery  

---

## 🧬 Input Data Format

Each patient’s data is divided into windows:

### 📊 Signals (8 Channels)
- Airflow  
- Chest movement  
- Abdomen movement  
- SpO₂  
- Plethysmography  
- ECG  
- Pulse  
- Secondary airflow  

---

## 🔄 Context Windowing (Key Idea)

Instead of using a single 30s window, the model uses **neighboring windows for context**.

### 🔹 Radius = 1 (used in this version)

### 🧠 Why this helps:
- Captures apnea transitions  
- Reduces ambiguity at boundaries  
- Improves temporal understanding  

---

## 🏗️ Model Architecture

### 1️⃣ CNN Frontend (Feature Extractor)

- Applies **1D convolutions** over time  
- Learns:
  - breathing patterns  
  - waveform shapes  
  - signal anomalies  

---

### 2️⃣ BiGRU (Temporal Modeling)

- Processes extracted features as a **sequence**
- Runs in **both directions**:
  - Forward → past → future  
  - Backward → future → past  


In [ ]:
"""
=============================================================================
 SLEEP APNEA DETECTION — CNN-FRONTEND + BiGRU + ATTENTION  (Kaggle GPU)
 v2 — context windows, gentler downsampling, boundary-label smoothing,
      SpO2/pleth signal cleaning, 5-epoch hyperparameter search trials
=============================================================================
 CHANGES vs v1 (and why):

 1. CONTEXT WINDOWS (radius=1 -> 90s input instead of 30s)
    Same fix that took the ResNet1D from AUC 0.80 -> 0.83: a single 30s
    window often lacks the desaturation lag / event context needed to
    disambiguate apnea from normal breathing. Each training sample is now
    the target window concatenated with its immediate neighbor windows
    (edge-clamped at recording boundaries). Label = target window's label.

 2. LESS AGGRESSIVE DOWNSAMPLING (3 stride-2 stages instead of 4, i.e. 8x
    reduction instead of 16x, plus a stride-1 refinement conv)
    v1's 4-stage frontend compressed 6000 samples -> 375 tokens (16x),
    discarding a lot of fine-grained waveform morphology before the GRU
    ever saw it. This version reduces to 3 downsampling stages (8x) so
    more temporal detail survives into the recurrent stage. Combined with
    the 3x longer context-window input, the GRU now sees ~2250 tokens
    instead of 375 -- a much richer representation, at the cost of more
    compute per step (accounted for in the hyperparameter search budget
    below).

 3. LABEL BOUNDARY NOISE FIX (soft labels for TRAINING only)
    Windows are non-overlapping 30s chunks labeled 1 if ANY apnea event
    overlaps them at all -- so a window with 2 seconds of event at its
    edge gets the same hard label as one that's 100% inside an event.
    This is label noise that caps achievable performance regardless of
    architecture. Fix: for each patient's window sequence, any window
    adjacent to a LABEL TRANSITION (its immediate neighbor has a different
    label) is treated as ambiguous and soft-labeled (e.g. 0.85 instead of
    1.0, 0.15 instead of 0.0) during TRAINING ONLY -- this tells the loss
    "be less confident here" without touching validation/test evaluation,
    which still uses the original hard ground-truth labels so your metrics
    stay comparable to previous runs.

 4. SPO2 / PLETH CHANNEL CLEANING
    Pulse-oximetry channels are extremely prone to sensor dropout (finger
    movement, poor perfusion, disconnection) which shows up as flatlined
    (constant-value) stretches, and SpO2 specifically has a known-valid
    physiological range (50-100%). Both channels are now cleaned before
    normalization: out-of-range SpO2 values are masked, flatline runs
    (>= 1 second of a constant value) in both channels are detected and
    linearly interpolated over, so the model doesn't learn "flat signal =
    normal breathing" from dropout artifacts.

 5. 5 EPOCHS PER HYPERPARAMETER TRIAL (was 3)
    More reliable trial ranking -- 3 epochs risked discarding a good
    config that just had a slow start.

 LOSS: still weighted BCE via nn.BCEWithLogitsLoss(pos_weight=...), now
 also compatible with the soft training labels from fix #3 (BCE handles
 float targets in [0,1] natively).

 Input  : X_SN{n}.npy -> (num_windows, 8, 6000) float16
          y_SN{n}.npy -> (num_windows,)         int8  (0=normal, 1=apnea)
=============================================================================
"""

import os
import gc
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_recall_curve
)

warnings.filterwarnings("ignore")
import kagglehub

try:
    import optuna
    from optuna.samplers import TPESampler
    HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False
    print("optuna not installed -- using random-search fallback. "
          "(`pip install optuna --break-system-packages` for Bayesian search)")

# ─────────────────────────────────────────────────────────────────────────
# 0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ─────────────────────────────────────────────────────────────────────────
# 0b. GPU / KERNEL-COMPATIBILITY CHECK  (P100 = compute capability 6.0)
# ─────────────────────────────────────────────────────────────────────────
def _gpu_kernels_usable():
    if not torch.cuda.is_available():
        return False
    try:
        x = torch.randn(2, 4, 16, device="cuda")
        conv = nn.Conv1d(4, 4, 3, padding=1).cuda()
        _ = conv(x)
        return True
    except RuntimeError as e:
        print("\n⚠️  GPU kernel check failed:", str(e).splitlines()[0])
        print("⚠️  Falling back to CPU. Switch accelerator to T4/L4 to use GPU.")
        return False

_USE_CUDA = _gpu_kernels_usable()

# ─────────────────────────────────────────────────────────────────────────
# 1. DOWNLOAD DATASET
# ─────────────────────────────────────────────────────────────────────────
DATA_DIR = kagglehub.dataset_download(
    "nishantomhare/detection-of-disordered-breathing-dataset2"
)
print("Dataset path:", DATA_DIR)

def locate_npy_root(root):
    for dirpath, _, filenames in os.walk(root):
        if any(f.startswith("X_SN") and f.endswith(".npy") for f in filenames):
            return dirpath
    return root

DATA_DIR = locate_npy_root(DATA_DIR)
print("Resolved npy root:", DATA_DIR)

OUTPUT_DIR = "/kaggle/working/gru_apnea_output_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────
# 2. CONFIG
# ─────────────────────────────────────────────────────────────────────────
SKIP_PATIENTS = {
    "SN1", "SN28", "SN14", "SN13", "SN17", "SN18",
    "SN19", "SN20", "SN27", "SN89"
}

TEST_FRAC    = 0.15
VAL_FRAC     = 0.15
N_STRAT_BINS = 5

FIXED_CHANNEL_KEYS = ["flow", "chest", "abd", "spo2", "pleth", "ecg", "pulse", "flow2"]
SPO2_IDX  = FIXED_CHANNEL_KEYS.index("spo2")
PLETH_IDX = FIXED_CHANNEL_KEYS.index("pleth")
FS = 200

NUM_CHANNELS   = 8
CONTEXT_RADIUS = 1          # target window + 1 neighbor each side -> 90s input
BATCH_SIZE     = 32         # reduced from 64: longer sequences use more memory
FINAL_EPOCHS   = 40
NUM_WORKERS    = 2
PATIENCE       = 8
GRAD_CLIP      = 5.0

AUGMENT           = True
NOISE_STD_FRAC    = 0.03
SCALE_JITTER      = 0.10
MAX_TIME_SHIFT    = 100
CHANNEL_DROPOUT_P = 0.05

NORM_SAMPLE_WINDOWS = 8000

# ---- label boundary-noise smoothing (fix #3) ----
BOUNDARY_SOFT_CONF = 0.15   # boundary windows get labeled 0.15 / 0.85 instead of 0/1

# ---- SpO2 / pleth cleaning (fix #4) ----
SPO2_VALID_RANGE   = (50.0, 100.0)   # physiologically plausible SpO2 %
FLATLINE_MIN_SEC   = 1.0             # >=1s of a constant value = suspected dropout
FLATLINE_MIN_RUN   = int(FLATLINE_MIN_SEC * FS)

# ---- hyperparameter search settings ----
# NOTE: context windows (3x longer input) + gentler downsampling (2x more
# tokens reach the GRU) together make each forward/backward pass notably
# more expensive than v1. To keep total search time reasonable, the search
# uses a SMALLER data subsample and a trimmed hyperparameter range (fewer
# very-wide configs) compared to v1, while still using 5 epochs/trial as
# requested for more reliable trial ranking.
N_TRIALS           = 10
SEARCH_EPOCHS       = 5
SEARCH_TRAIN_FRAC   = 0.20
SEARCH_VAL_FRAC     = 0.40

DEVICE = torch.device("cuda" if _USE_CUDA else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| capability:", torch.cuda.get_device_capability(0),
          "| usable:", _USE_CUDA)

# ─────────────────────────────────────────────────────────────────────────
# 3. DISCOVER PATIENTS
# ─────────────────────────────────────────────────────────────────────────
all_x_files = sorted(f for f in os.listdir(DATA_DIR)
                      if f.startswith("X_SN") and f.endswith(".npy"))
all_pids = [f[len("X_"):-len(".npy")] for f in all_x_files]
pids = [p for p in all_pids if p not in SKIP_PATIENTS
        and os.path.exists(os.path.join(DATA_DIR, f"y_{p}.npy"))]
print(f"\nTotal X files: {len(all_x_files)} | Skipped: {sorted(SKIP_PATIENTS & set(all_pids))} "
      f"| Usable patients: {len(pids)}")

if len(pids) < 5:
    raise RuntimeError("Not enough usable patients to train/val/test split.")

# ─────────────────────────────────────────────────────────────────────────
# 4. STRATIFIED PATIENT-LEVEL SPLIT
# ─────────────────────────────────────────────────────────────────────────
def patient_apnea_ratio(pid):
    y = np.load(os.path.join(DATA_DIR, f"y_{pid}.npy"), mmap_mode="r")
    n = len(y)
    apnea = int(np.sum(y))
    del y
    return apnea / max(1, n), n, apnea

patient_stats = {pid: patient_apnea_ratio(pid) for pid in pids}
ratios = np.array([patient_stats[p][0] for p in pids])

rng = np.random.default_rng(SEED)
order = np.argsort(ratios)
sorted_pids = np.array(pids)[order]
bin_edges = np.linspace(0, len(sorted_pids), N_STRAT_BINS + 1).astype(int)
train_pids, val_pids, test_pids = [], [], []

for b in range(N_STRAT_BINS):
    bucket = list(sorted_pids[bin_edges[b]:bin_edges[b + 1]])
    rng.shuffle(bucket)
    n = len(bucket)
    n_test = max(1, round(n * TEST_FRAC)) if n >= 3 else (1 if n > 1 else 0)
    n_val  = max(1, round(n * VAL_FRAC))  if n >= 3 else (1 if n - n_test > 1 else 0)
    test_pids.extend(bucket[:n_test])
    val_pids.extend(bucket[n_test:n_test + n_val])
    train_pids.extend(bucket[n_test + n_val:])

if not val_pids and train_pids:
    val_pids.append(train_pids.pop())
if not test_pids and train_pids:
    test_pids.append(train_pids.pop())

print(f"\nTrain patients ({len(train_pids)}): {sorted(train_pids)}")
print(f"Val   patients ({len(val_pids)}): {sorted(val_pids)}")
print(f"Test  patients ({len(test_pids)}): {sorted(test_pids)}")
for name, plist in [("train", train_pids), ("val", val_pids), ("test", test_pids)]:
    tot_n = sum(patient_stats[p][1] for p in plist)
    tot_a = sum(patient_stats[p][2] for p in plist)
    print(f"  {name:5s}: {tot_n:,} windows, apnea ratio = {tot_a / max(1, tot_n):.3f}")

# ─────────────────────────────────────────────────────────────────────────
# 5. PER-CHANNEL NORMALIZATION STATS (from TRAIN set only)
# ─────────────────────────────────────────────────────────────────────────
def estimate_norm_stats(patient_ids, n_sample_windows):
    sums   = np.zeros(NUM_CHANNELS, dtype=np.float64)
    sqsums = np.zeros(NUM_CHANNELS, dtype=np.float64)
    count  = 0
    per_patient_budget = max(1, n_sample_windows // max(1, len(patient_ids)))
    for pid in patient_ids:
        X = np.load(os.path.join(DATA_DIR, f"X_{pid}.npy"), mmap_mode="r")
        n = X.shape[0]
        if n == 0:
            continue
        k = min(per_patient_budget, n)
        idx = np.random.default_rng(SEED).choice(n, size=k, replace=False)
        sample = X[idx].astype(np.float64)
        sums   += sample.sum(axis=(0, 2))
        sqsums += (sample ** 2).sum(axis=(0, 2))
        count  += k * sample.shape[2]
        del X, sample
        gc.collect()
    mean = sums / count
    var  = sqsums / count - mean ** 2
    std  = np.sqrt(np.clip(var, 1e-8, None))
    return mean.astype(np.float32), std.astype(np.float32)

print("\nEstimating per-channel normalization stats from train set...")
CH_MEAN, CH_STD = estimate_norm_stats(train_pids, NORM_SAMPLE_WINDOWS)
np.save(os.path.join(OUTPUT_DIR, "channel_mean.npy"), CH_MEAN)
np.save(os.path.join(OUTPUT_DIR, "channel_std.npy"), CH_STD)
print("Channel means:", np.round(CH_MEAN, 3))
print("Channel stds :", np.round(CH_STD, 3))

# ─────────────────────────────────────────────────────────────────────────
# 6. SIGNAL CLEANING  (SpO2 / pleth: out-of-range + flatline dropout)
# ─────────────────────────────────────────────────────────────────────────
def flatline_mask(sig, min_run):
    """Vectorized detection of constant-value runs >= min_run samples."""
    n = len(sig)
    if n < 2:
        return np.zeros(n, dtype=bool)
    same = np.concatenate(([False], sig[1:] == sig[:-1]))
    change_points = np.where(~same)[0]
    change_points = np.concatenate(([0], change_points, [n]))
    mask = np.zeros(n, dtype=bool)
    for i in range(len(change_points) - 1):
        s, e = change_points[i], change_points[i + 1]
        if e - s >= min_run:
            mask[s:e] = True
    return mask


def clean_signal(sig, valid_range=None, flatline_min_run=FLATLINE_MIN_RUN):
    """Mask out-of-range + flatline-dropout samples, linearly interpolate
    over them. Leaves the signal untouched if the whole thing is bad
    (nothing to interpolate from)."""
    sig = sig.astype(np.float32).copy()
    bad = np.zeros(len(sig), dtype=bool)
    if valid_range is not None:
        lo, hi = valid_range
        bad |= (sig < lo) | (sig > hi)
    bad |= flatline_mask(sig, flatline_min_run)

    if bad.any() and not bad.all():
        good_idx = np.where(~bad)[0]
        bad_idx = np.where(bad)[0]
        sig[bad_idx] = np.interp(bad_idx, good_idx, sig[good_idx])
    return sig


def clean_spo2_pleth(window_8ch):
    """window_8ch: (8, T) -> cleans channels SPO2_IDX and PLETH_IDX in place (copy)."""
    window_8ch = window_8ch.copy()
    window_8ch[SPO2_IDX]  = clean_signal(window_8ch[SPO2_IDX], valid_range=SPO2_VALID_RANGE)
    window_8ch[PLETH_IDX] = clean_signal(window_8ch[PLETH_IDX], valid_range=None)
    return window_8ch

# ─────────────────────────────────────────────────────────────────────────
# 7. BOUNDARY LABEL SMOOTHING  (per patient, precomputed once)
# ─────────────────────────────────────────────────────────────────────────
def compute_boundary_soft_labels(y, soft_conf=BOUNDARY_SOFT_CONF):
    """
    Hard 0/1 windows keep their label. Any window whose immediate left OR
    right neighbor has a DIFFERENT hard label is a "boundary window" --
    likely only partially overlapping the true event -- and gets a soft
    label (e.g. 0.85 for apnea-near-boundary, 0.15 for normal-near-boundary)
    instead of a hard 0/1, so the loss penalizes confident-but-wrong
    predictions on these ambiguous windows less harshly.
    """
    n = len(y)
    soft = y.astype(np.float32).copy()
    for i in range(n):
        left_diff  = (i > 0)     and (y[i - 1] != y[i])
        right_diff = (i < n - 1) and (y[i + 1] != y[i])
        if left_diff or right_diff:
            soft[i] = (1.0 - soft_conf) if y[i] == 1 else soft_conf
    return soft

# ─────────────────────────────────────────────────────────────────────────
# 8. DATASET  (context windows + cleaning + normalization + augmentation
#              + boundary-smoothed soft labels for training)
# ─────────────────────────────────────────────────────────────────────────
class ApneaDataset(Dataset):
    def __init__(self, data_dir, patient_ids, mean, std,
                 context_radius=1, augment=False, use_soft_labels=False):
        self.data_dir = data_dir
        self._cache = {}
        self.index = []            # (pid, local_idx, n_windows_for_pid)
        self.mean = mean.reshape(NUM_CHANNELS, 1)
        self.std  = std.reshape(NUM_CHANNELS, 1)
        self.context_radius = context_radius
        self.augment = augment
        self.use_soft_labels = use_soft_labels
        self._soft_label_cache = {}   # pid -> soft label array (only if use_soft_labels)

        for pid in patient_ids:
            xp = os.path.join(data_dir, f"X_{pid}.npy")
            yp = os.path.join(data_dir, f"y_{pid}.npy")
            if not (os.path.exists(xp) and os.path.exists(yp)):
                continue
            y = np.load(yp, mmap_mode="r")
            n = len(y)
            self.index.extend([(pid, i, n) for i in range(n)])
            if use_soft_labels:
                self._soft_label_cache[pid] = compute_boundary_soft_labels(np.asarray(y))
            del y

    def _arrays(self, pid):
        if pid not in self._cache:
            self._cache[pid] = (
                np.load(os.path.join(self.data_dir, f"X_{pid}.npy"), mmap_mode="r"),
                np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r"),
            )
        return self._cache[pid]

    def __len__(self):
        return len(self.index)

    def labels(self):
        """Hard ground-truth labels (used for class weighting / sampler,
        and always used for evaluation regardless of use_soft_labels)."""
        out = np.zeros(len(self.index), dtype=np.int8)
        cache = {}
        for i, (pid, li, _) in enumerate(self.index):
            if pid not in cache:
                cache[pid] = np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r")
            out[i] = cache[pid][li]
        return out

    def __getitem__(self, idx):
        pid, local_idx, n = self.index[idx]
        X_arr, y_arr = self._arrays(pid)

        r = self.context_radius
        neighbor_idxs = np.clip(np.arange(local_idx - r, local_idx + r + 1), 0, n - 1)
        segs = [X_arr[j].astype(np.float32) for j in neighbor_idxs]
        x = np.concatenate(segs, axis=1)                    # (8, (2r+1)*6000)

        x = clean_spo2_pleth(x)                              # fix #4
        x = (x - self.mean) / self.std

        if self.augment:
            if NOISE_STD_FRAC > 0:
                x = x + np.random.randn(*x.shape).astype(np.float32) * NOISE_STD_FRAC
            if SCALE_JITTER > 0:
                scale = 1.0 + (np.random.rand(NUM_CHANNELS, 1).astype(np.float32) * 2 - 1) * SCALE_JITTER
                x = x * scale
            if MAX_TIME_SHIFT > 0:
                shift = np.random.randint(-MAX_TIME_SHIFT, MAX_TIME_SHIFT + 1)
                if shift != 0:
                    x = np.roll(x, shift, axis=1)
            if CHANNEL_DROPOUT_P > 0 and np.random.rand() < CHANNEL_DROPOUT_P:
                ch = np.random.randint(0, NUM_CHANNELS)
                x[ch, :] = 0.0

        if self.use_soft_labels:
            y = float(self._soft_label_cache[pid][local_idx])   # fix #3 (train only)
        else:
            y = float(y_arr[local_idx])

        return torch.from_numpy(x.copy()), torch.tensor(y, dtype=torch.float32)


train_ds = ApneaDataset(DATA_DIR, train_pids, CH_MEAN, CH_STD,
                         context_radius=CONTEXT_RADIUS, augment=AUGMENT, use_soft_labels=True)
val_ds   = ApneaDataset(DATA_DIR, val_pids, CH_MEAN, CH_STD,
                         context_radius=CONTEXT_RADIUS, augment=False, use_soft_labels=False)
test_ds  = ApneaDataset(DATA_DIR, test_pids, CH_MEAN, CH_STD,
                         context_radius=CONTEXT_RADIUS, augment=False, use_soft_labels=False)

print(f"\nTrain windows: {len(train_ds):,}")
print(f"Val   windows: {len(val_ds):,}")
print(f"Test  windows: {len(test_ds):,}")
print(f"Context radius: {CONTEXT_RADIUS} -> input length = "
      f"{(2 * CONTEXT_RADIUS + 1) * 6000} samples ({(2 * CONTEXT_RADIUS + 1) * 30}s)")

train_labels = train_ds.labels()   # hard labels, used for class weighting / sampler
n0, n1 = int((train_labels == 0).sum()), int((train_labels == 1).sum())
total_train = n0 + n1
print(f"\nTrain class counts (hard labels): normal={n0:,}  apnea={n1:,}")

n_soft_boundary = sum(1 for pid in train_pids
                      for v in train_ds._soft_label_cache[pid]
                      if v not in (0.0, 1.0))
print(f"Boundary windows soft-labeled: {n_soft_boundary:,} "
      f"({100 * n_soft_boundary / max(1, len(train_ds)):.1f}% of train set)")

POS_WEIGHT = torch.tensor([n0 / max(1, n1)], dtype=torch.float32).to(DEVICE)
print(f"pos_weight for weighted BCE: {POS_WEIGHT.item():.3f}")

sample_weights = np.where(train_labels == 1, n0 / max(1, n1), 1.0)
full_sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# ─────────────────────────────────────────────────────────────────────────
# 9. CNN-FRONTEND + BiGRU + ATTENTION ARCHITECTURE  (gentler downsampling)
# ─────────────────────────────────────────────────────────────────────────
class ConvFrontend(nn.Module):
    """
    3 strided-conv stages (8x downsampling, vs 16x in v1) + 1 stride-1
    refinement conv, to preserve more fine-grained waveform detail before
    the GRU. With CONTEXT_RADIUS=1 (18000-sample input), output is
    18000 / 8 = 2250 tokens.
    """
    def __init__(self, in_channels=8, base_width=24, dropout=0.2):
        super().__init__()
        w = base_width

        def down_block(cin, cout, k=9):
            return nn.Sequential(
                nn.Conv1d(cin, cout, kernel_size=k, stride=2, padding=k // 2, bias=False),
                nn.BatchNorm1d(cout),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
            )

        def refine_block(cin, cout, k=5):
            return nn.Sequential(
                nn.Conv1d(cin, cout, kernel_size=k, stride=1, padding=k // 2, bias=False),
                nn.BatchNorm1d(cout),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
            )

        self.net = nn.Sequential(
            down_block(in_channels, w),        # /2
            down_block(w, w * 2),               # /4
            down_block(w * 2, w * 4),            # /8  (was /16 in v1)
            refine_block(w * 4, w * 4),          # no further downsampling -- refine detail
        )
        self.out_channels = w * 4

    def forward(self, x):
        return self.net(x)


class AdditiveAttentionPool(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, seq):
        scores = self.attn(seq).squeeze(-1)
        weights = F.softmax(scores, dim=1).unsqueeze(-1)
        pooled = (seq * weights).sum(dim=1)
        return pooled, weights.squeeze(-1)


class ConvBiGRUAttention(nn.Module):
    def __init__(self, in_channels=8, conv_width=24, gru_hidden=96,
                 gru_layers=2, dropout=0.4, bidirectional=True):
        super().__init__()
        self.frontend = ConvFrontend(in_channels, conv_width, dropout=dropout * 0.5)
        gru_input_dim = self.frontend.out_channels

        self.gru = nn.GRU(
            input_size=gru_input_dim, hidden_size=gru_hidden, num_layers=gru_layers,
            batch_first=True, bidirectional=bidirectional,
            dropout=dropout if gru_layers > 1 else 0.0,
        )
        gru_out_dim = gru_hidden * (2 if bidirectional else 1)
        self.attn_pool = AdditiveAttentionPool(gru_out_dim)

        self.head = nn.Sequential(
            nn.Linear(gru_out_dim, gru_out_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(gru_out_dim // 2, 1),
        )

    def forward(self, x, return_attention=False):
        feat = self.frontend(x)
        feat = feat.permute(0, 2, 1)
        gru_out, _ = self.gru(feat)
        pooled, attn_w = self.attn_pool(gru_out)
        logit = self.head(pooled).squeeze(-1)
        if return_attention:
            return logit, attn_w
        return logit

# ─────────────────────────────────────────────────────────────────────────
# 10. TRAIN / EVAL HELPERS
# ─────────────────────────────────────────────────────────────────────────
def best_f1_threshold(labels, probs):
    prec, rec, thr = precision_recall_curve(labels, probs)
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    if len(thr) == 0:
        return 0.5, 0.0
    best_idx = np.nanargmax(f1s[:-1])
    return float(thr[best_idx]), float(f1s[best_idx])


def run_epoch(model, loader, criterion, optimizer, scaler, train_mode: bool):
    model.train(mode=train_mode)
    total_loss = 0.0
    all_labels, all_probs = [], []

    for X, y in loader:
        X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                logits = model(X)
                loss = criterion(logits, y)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * X.size(0)
        probs = torch.sigmoid(logits.float()).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        # NOTE: for training batches with soft labels, "all_labels" here
        # will contain soft values -- that's fine, it's only used for a
        # quick in-loop F1 sanity check, never for the official val/test
        # metrics (those loaders always use hard labels).
        all_labels.extend(y.detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(loader.dataset))
    return avg_loss, np.array(all_labels), np.array(all_probs)

# ─────────────────────────────────────────────────────────────────────────
# 11. HYPERPARAMETER SEARCH
# ─────────────────────────────────────────────────────────────────────────
rng_np = np.random.default_rng(SEED)
search_train_idx = rng_np.choice(len(train_ds), size=int(len(train_ds) * SEARCH_TRAIN_FRAC), replace=False)
search_val_idx   = rng_np.choice(len(val_ds), size=int(len(val_ds) * SEARCH_VAL_FRAC), replace=False)

search_train_subset = Subset(train_ds, search_train_idx)
search_val_subset   = Subset(val_ds, search_val_idx)

search_train_labels = train_labels[search_train_idx]
sn0, sn1 = int((search_train_labels == 0).sum()), int((search_train_labels == 1).sum())
search_sample_weights = np.where(search_train_labels == 1, sn0 / max(1, sn1), 1.0)
search_sampler = WeightedRandomSampler(search_sample_weights, num_samples=len(search_sample_weights), replacement=True)

search_train_loader = DataLoader(search_train_subset, batch_size=BATCH_SIZE, sampler=search_sampler,
                                  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
search_val_loader   = DataLoader(search_val_subset, batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=NUM_WORKERS, pin_memory=True)
search_pos_weight = torch.tensor([sn0 / max(1, sn1)], dtype=torch.float32).to(DEVICE)


def build_model(params):
    return ConvBiGRUAttention(
        in_channels=NUM_CHANNELS,
        conv_width=params["conv_width"],
        gru_hidden=params["gru_hidden"],
        gru_layers=params["gru_layers"],
        dropout=params["dropout"],
        bidirectional=params["bidirectional"],
    ).to(DEVICE)


def evaluate_trial(params, verbose=False):
    model = build_model(params)
    criterion = nn.BCEWithLogitsLoss(pos_weight=search_pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    best_f1 = -1.0
    for epoch in range(SEARCH_EPOCHS):
        run_epoch(model, search_train_loader, criterion, optimizer, scaler, train_mode=True)
        _, val_labels, val_probs = run_epoch(model, search_val_loader, criterion, optimizer, scaler, train_mode=False)
        _, f1 = best_f1_threshold(val_labels, val_probs)
        best_f1 = max(best_f1, f1)
        if verbose:
            print(f"    search epoch {epoch+1}/{SEARCH_EPOCHS}: val_f1={f1:.4f}")

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return best_f1


# NOTE: trimmed vs v1 -- wider configs (conv_width=48, gru_hidden=192) are
# dropped from the search space because the 6x larger token sequence
# (context window x gentler downsampling) makes those combinations too
# slow/memory-hungry to be worth searching; the remaining range still
# spans small->moderate capacity.
SEARCH_SPACE = {
    "conv_width":    [16, 24, 32],
    "gru_hidden":    [48, 96, 128],
    "gru_layers":    [1, 2],
    "dropout":       [0.2, 0.3, 0.4, 0.5],
    "bidirectional": [True, False],
    "lr":            (1e-4, 3e-3),
    "weight_decay":  (1e-6, 1e-3),
}

print("\n" + "=" * 65)
print(f"  HYPERPARAMETER SEARCH ({'Optuna TPE' if HAS_OPTUNA else 'random search'}, "
      f"{N_TRIALS} trials x {SEARCH_EPOCHS} epochs on a data subsample)")
print("=" * 65)

trial_log = []

if HAS_OPTUNA:
    def objective(trial):
        params = {
            "conv_width":    trial.suggest_categorical("conv_width", SEARCH_SPACE["conv_width"]),
            "gru_hidden":    trial.suggest_categorical("gru_hidden", SEARCH_SPACE["gru_hidden"]),
            "gru_layers":    trial.suggest_categorical("gru_layers", SEARCH_SPACE["gru_layers"]),
            "dropout":       trial.suggest_categorical("dropout", SEARCH_SPACE["dropout"]),
            "bidirectional": trial.suggest_categorical("bidirectional", SEARCH_SPACE["bidirectional"]),
            "lr":            trial.suggest_float("lr", *SEARCH_SPACE["lr"], log=True),
            "weight_decay":  trial.suggest_float("weight_decay", *SEARCH_SPACE["weight_decay"], log=True),
        }
        f1 = evaluate_trial(params)
        trial_log.append({**params, "val_f1": f1})
        print(f"  Trial {trial.number:2d}: {params} -> val_f1={f1:.4f}")
        return f1

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_TRIALS)
    best_params = study.best_params
    best_search_f1 = study.best_value
else:
    best_params, best_search_f1 = None, -1.0
    for t in range(N_TRIALS):
        params = {
            "conv_width":    random.choice(SEARCH_SPACE["conv_width"]),
            "gru_hidden":    random.choice(SEARCH_SPACE["gru_hidden"]),
            "gru_layers":    random.choice(SEARCH_SPACE["gru_layers"]),
            "dropout":       random.choice(SEARCH_SPACE["dropout"]),
            "bidirectional": random.choice(SEARCH_SPACE["bidirectional"]),
            "lr":            10 ** random.uniform(np.log10(SEARCH_SPACE["lr"][0]), np.log10(SEARCH_SPACE["lr"][1])),
            "weight_decay":  10 ** random.uniform(np.log10(SEARCH_SPACE["weight_decay"][0]), np.log10(SEARCH_SPACE["weight_decay"][1])),
        }
        f1 = evaluate_trial(params)
        trial_log.append({**params, "val_f1": f1})
        print(f"  Trial {t:2d}: {params} -> val_f1={f1:.4f}")
        if f1 > best_search_f1:
            best_search_f1 = f1
            best_params = params

pd.DataFrame(trial_log).to_csv(os.path.join(OUTPUT_DIR, "hparam_search_trials.csv"), index=False)
print(f"\nBest hyperparameters found (val_f1={best_search_f1:.4f}):")
for k, v in best_params.items():
    print(f"  {k}: {v}")

# ─────────────────────────────────────────────────────────────────────────
# 12. FINAL TRAINING WITH BEST HYPERPARAMETERS
# ─────────────────────────────────────────────────────────────────────────
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=full_sampler,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

model = build_model(best_params)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nFinal model: CNN-BiGRU-Attention (v2) | Trainable params: {n_params:,}")

criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
optimizer = torch.optim.AdamW(model.parameters(), lr=best_params["lr"], weight_decay=best_params["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

best_val_f1 = -1.0
best_state = None
best_threshold = 0.5
epochs_no_improve = 0
history = []

print("\n" + "=" * 65)
print("  FINAL TRAINING")
print("=" * 65)

for epoch in range(1, FINAL_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_labels_ep, train_probs_ep = run_epoch(model, train_loader, criterion, optimizer, scaler, True)
    val_loss, val_labels, val_probs = run_epoch(model, val_loader, criterion, optimizer, scaler, False)

    tuned_thr, tuned_val_f1 = best_f1_threshold(val_labels, val_probs)
    # train_labels_ep may contain soft values (boundary smoothing) -- round
    # for a readable train F1 sanity metric only
    train_f105 = f1_score((train_labels_ep >= 0.5).astype(int), (train_probs_ep >= 0.5).astype(int), zero_division=0)
    val_f105 = f1_score(val_labels, (val_probs >= 0.5).astype(int), zero_division=0)

    scheduler.step(tuned_val_f1)
    dt = time.time() - t0

    history.append({
        "epoch": epoch, "train_loss": train_loss, "train_f1@0.5": train_f105,
        "val_loss": val_loss, "val_f1@0.5": val_f105,
        "val_f1@tuned": tuned_val_f1, "tuned_threshold": tuned_thr,
        "lr": optimizer.param_groups[0]["lr"],
    })

    print(f"Epoch {epoch:3d}/{FINAL_EPOCHS} | train_loss={train_loss:.4f} f1@.5={train_f105:.4f} | "
          f"val_loss={val_loss:.4f} f1@.5={val_f105:.4f} f1@tuned={tuned_val_f1:.4f} "
          f"(thr={tuned_thr:.2f}) | lr={optimizer.param_groups[0]['lr']:.2e} | {dt:.1f}s")

    if tuned_val_f1 > best_val_f1:
        best_val_f1 = tuned_val_f1
        best_threshold = tuned_thr
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        torch.save({"state_dict": best_state, "threshold": best_threshold, "hparams": best_params,
                    "context_radius": CONTEXT_RADIUS},
                   os.path.join(OUTPUT_DIR, "best_gru_model.pt"))
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break

pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)
if best_state is not None:
    model.load_state_dict(best_state)
print(f"\nBest val F1 (tuned threshold): {best_val_f1:.4f}  @ threshold={best_threshold:.3f}")

# ─────────────────────────────────────────────────────────────────────────
# 13. FINAL TEST-SET EVALUATION  (always hard ground-truth labels)
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  TEST SET EVALUATION")
print("=" * 65)

_, test_labels, test_probs = run_epoch(model, test_loader, criterion, optimizer, scaler, False)

try:
    auc = roc_auc_score(test_labels, test_probs)
except ValueError:
    auc = float("nan")
print(f"\nROC-AUC (threshold-independent): {auc:.4f}")

def report_at_threshold(probs, labels, thr, label):
    preds = (probs >= thr).astype(int)
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)
    cm   = confusion_matrix(labels, preds)
    rep  = classification_report(labels, preds, target_names=["Normal", "Apnea"], zero_division=0)
    print(f"\n--- {label} (threshold={thr:.3f}) ---")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    print(f"\nClassification Report:\n{rep}")
    return dict(label=label, threshold=thr, accuracy=acc, precision=prec, recall=rec, f1_score=f1,
                tn=int(cm[0, 0]), fp=int(cm[0, 1]), fn=int(cm[1, 0]), tp=int(cm[1, 1])), rep, cm

m_05, rep_05, cm_05 = report_at_threshold(test_probs, test_labels, 0.5, "Default threshold")
m_tuned, rep_tuned, cm_tuned = report_at_threshold(test_probs, test_labels, best_threshold, "Val-tuned threshold")

pd.DataFrame([m_05, m_tuned]).to_csv(os.path.join(OUTPUT_DIR, "test_metrics.csv"), index=False)
with open(os.path.join(OUTPUT_DIR, "classification_report_tuned.txt"), "w") as fh:
    fh.write(rep_tuned)

# ─────────────────────────────────────────────────────────────────────────
# 14. PLOTS
# ─────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im = axes[0].imshow(cm_tuned, cmap="Blues")
axes[0].set_title(f"Confusion Matrix (thr={best_threshold:.2f})")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["Normal", "Apnea"])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(["Normal", "Apnea"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm_tuned[i, j]), ha="center", va="center",
                      color="white" if cm_tuned[i, j] > cm_tuned.max() / 2 else "black")
fig.colorbar(im, ax=axes[0])

if not np.isnan(auc):
    fpr, tpr, _ = roc_curve(test_labels, test_probs)
    axes[1].plot(fpr, tpr, label=f"AUC = {auc:.3f}")
    axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
    axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title("ROC Curve"); axes[1].legend()

hist_df = pd.DataFrame(history)
axes[2].plot(hist_df["epoch"], hist_df["train_f1@0.5"], label="Train F1@0.5")
axes[2].plot(hist_df["epoch"], hist_df["val_f1@tuned"], label="Val F1@tuned")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("F1 Score")
axes[2].set_title("Training Curves"); axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "evaluation_plots.png"), dpi=150)
plt.show()

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(" - best_gru_model.pt              (weights + threshold + best hparams + context_radius)")
print(" - channel_mean.npy / channel_std.npy")
print(" - hparam_search_trials.csv")
print(" - training_history.csv")
print(" - test_metrics.csv")
print(" - classification_report_tuned.txt")
print(" - evaluation_plots.png")
print(f"\nNOTE: at inference time, replicate the context-window concatenation "
      f"(radius={CONTEXT_RADIUS}) and the SpO2/pleth cleaning step -- both are "
      f"part of the input pipeline, not just training-time tricks.")

In [ ]:
import os
import zipfile

output_dir = "/kaggle/working/gru_apnea_output_v2"
zip_path = "/kaggle/working/gru_apnea_output_v2.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, output_dir)
            zipf.write(file_path, arcname)

print("ZIP created at:", zip_path)

In [ ]:
"""
=============================================================================
 SLEEP APNEA DETECTION — 1D RESNET ABLATION STUDY
 8 models = 2 context radii (1, 2) x 4 loss functions
   Model A: radius=1, BCE                Model E: radius=2, BCE
   Model B: radius=1, weighted BCE        Model F: radius=2, weighted BCE
   Model C: radius=1, focal loss          Model G: radius=2, focal loss
   Model D: radius=1, weighted focal      Model H: radius=2, weighted focal
=============================================================================
 EXPERIMENTAL DESIGN NOTE (read this before changing anything):
 To make this a FAIR ablation of "does loss function / context radius
 matter", every model uses the IDENTICAL architecture, hyperparameters,
 preprocessing, and training schedule -- the ONLY things that differ
 between the 8 models are (a) context radius and (b) loss function. No
 per-model hyperparameter tuning is done here on purpose: tuning each
 model separately would confound "this loss function is better" with
 "this model got luckier hyperparameters", which defeats the point of an
 ablation study. If you want to further tune whichever model wins here,
 that's a good next step -- just don't do it inside this comparison.

 Also, on purpose, NO WeightedRandomSampler is used for training (unlike
 earlier scripts). The whole point of Models B/D vs A/C is to isolate
 what the LOSS FUNCTION alone contributes to handling class imbalance --
 if a sampler pre-balanced every batch, it would swamp any difference the
 loss weighting makes and the comparison would be meaningless.

 ARCHITECTURE: 1D ResNet (BasicBlock1D: conv-bn-relu-conv-bn + residual
 skip), stem conv, 4 residual stages, global average pool, single-logit
 head. Handles both radius=1 (18000 samples) and radius=2 (30000 samples)
 inputs natively via adaptive pooling -- no architecture changes needed
 between radii.

 LOSSES:
   BCE              : nn.BCEWithLogitsLoss()                      -- no imbalance handling
   Weighted BCE      : nn.BCEWithLogitsLoss(pos_weight=...)         -- class-weighted
   Focal             : focal loss, gamma=2, no class weighting
   Weighted Focal    : focal loss, gamma=2, WITH class weighting (pos_weight baked into the BCE term)

 SPLIT: patient-level, stratified by apnea ratio, 60% train / 20% val /
 20% test (as requested) -- same skip list as previous scripts.

 OUTPUTS: per-model confusion matrix, ROC curve, classification report,
 training curves; a master CSV with every model's metrics (default @0.5
 AND val-tuned threshold); and 4 comparison figures (grouped bar chart,
 overlaid ROC curves, confusion-matrix grid, training-curve grid).

 Input  : X_SN{n}.npy -> (num_windows, 8, 6000) float16
          y_SN{n}.npy -> (num_windows,)         int8  (0=normal, 1=apnea)
=============================================================================
"""

import os
import gc
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_recall_curve
)

warnings.filterwarnings("ignore")
import kagglehub

# ─────────────────────────────────────────────────────────────────────────
# 0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ─────────────────────────────────────────────────────────────────────────
# 0b. GPU / KERNEL-COMPATIBILITY CHECK  (P100 = compute capability 6.0)
# ─────────────────────────────────────────────────────────────────────────
def _gpu_kernels_usable():
    if not torch.cuda.is_available():
        return False
    try:
        x = torch.randn(2, 4, 16, device="cuda")
        conv = nn.Conv1d(4, 4, 3, padding=1).cuda()
        _ = conv(x)
        return True
    except RuntimeError as e:
        print("\n⚠️  GPU kernel check failed:", str(e).splitlines()[0])
        print("⚠️  Falling back to CPU. Switch accelerator to T4/L4 to use GPU.")
        return False

_USE_CUDA = _gpu_kernels_usable()

# ─────────────────────────────────────────────────────────────────────────
# 1. DOWNLOAD DATASET
# ─────────────────────────────────────────────────────────────────────────
DATA_DIR = kagglehub.dataset_download(
    "nishantomhare/detection-of-disordered-breathing-dataset2"
)
print("Dataset path:", DATA_DIR)

def locate_npy_root(root):
    for dirpath, _, filenames in os.walk(root):
        if any(f.startswith("X_SN") and f.endswith(".npy") for f in filenames):
            return dirpath
    return root

DATA_DIR = locate_npy_root(DATA_DIR)
print("Resolved npy root:", DATA_DIR)

OUTPUT_DIR = "/kaggle/working/resnet1d_ablation_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────
# 2. CONFIG
# ─────────────────────────────────────────────────────────────────────────
SKIP_PATIENTS = {
    "SN1", "SN28", "SN14", "SN13", "SN17", "SN18",
    "SN19", "SN20", "SN27", "SN89"
}

TEST_FRAC    = 0.10   # 60:20:20 split as requested
VAL_FRAC     = 0.20
N_STRAT_BINS = 5

FIXED_CHANNEL_KEYS = ["flow", "chest", "abd", "spo2", "pleth", "ecg", "pulse", "flow2"]
SPO2_IDX  = FIXED_CHANNEL_KEYS.index("spo2")
PLETH_IDX = FIXED_CHANNEL_KEYS.index("pleth")
FS = 200

NUM_CHANNELS = 8
RADII = [1, 2]                 # context radii to compare
LOSS_TYPES = ["bce", "weighted_bce", "focal", "weighted_focal"]
FOCAL_GAMMA = 2.0

EPOCHS    = 25
PATIENCE  = 6
GRAD_CLIP = 3.0
NUM_WORKERS = 2

# fixed architecture/optimizer hyperparameters -- identical for all 8 models
BASE_WIDTH   = 32
DROPOUT      = 0.4
LR           = 7e-4
WEIGHT_DECAY = 8e-5

AUGMENT           = True
NOISE_STD_FRAC    = 0.03
SCALE_JITTER      = 0.10
MAX_TIME_SHIFT    = 100
CHANNEL_DROPOUT_P = 0.05

NORM_SAMPLE_WINDOWS = 8000
BOUNDARY_SOFT_CONF  = 0.15
SPO2_VALID_RANGE    = (50.0, 100.0)
FLATLINE_MIN_SEC    = 1.0
FLATLINE_MIN_RUN    = int(FLATLINE_MIN_SEC * FS)

DEVICE = torch.device("cuda" if _USE_CUDA else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| capability:", torch.cuda.get_device_capability(0),
          "| usable:", _USE_CUDA)

# ─────────────────────────────────────────────────────────────────────────
# 3. DISCOVER PATIENTS
# ─────────────────────────────────────────────────────────────────────────
all_x_files = sorted(f for f in os.listdir(DATA_DIR)
                      if f.startswith("X_SN") and f.endswith(".npy"))
all_pids = [f[len("X_"):-len(".npy")] for f in all_x_files]
pids = [p for p in all_pids if p not in SKIP_PATIENTS
        and os.path.exists(os.path.join(DATA_DIR, f"y_{p}.npy"))]
print(f"\nTotal X files: {len(all_x_files)} | Skipped: {sorted(SKIP_PATIENTS & set(all_pids))} "
      f"| Usable patients: {len(pids)}")

if len(pids) < 5:
    raise RuntimeError("Not enough usable patients to train/val/test split.")

# ─────────────────────────────────────────────────────────────────────────
# 4. STRATIFIED 60:20:20 PATIENT-LEVEL SPLIT
# ─────────────────────────────────────────────────────────────────────────
def patient_apnea_ratio(pid):
    y = np.load(os.path.join(DATA_DIR, f"y_{pid}.npy"), mmap_mode="r")
    n = len(y)
    apnea = int(np.sum(y))
    del y
    return apnea / max(1, n), n, apnea

patient_stats = {pid: patient_apnea_ratio(pid) for pid in pids}
ratios = np.array([patient_stats[p][0] for p in pids])

rng = np.random.default_rng(SEED)
order = np.argsort(ratios)
sorted_pids = np.array(pids)[order]
bin_edges = np.linspace(0, len(sorted_pids), N_STRAT_BINS + 1).astype(int)
train_pids, val_pids, test_pids = [], [], []

for b in range(N_STRAT_BINS):
    bucket = list(sorted_pids[bin_edges[b]:bin_edges[b + 1]])
    rng.shuffle(bucket)
    n = len(bucket)
    n_test = max(1, round(n * TEST_FRAC)) if n >= 3 else (1 if n > 1 else 0)
    n_val  = max(1, round(n * VAL_FRAC))  if n >= 3 else (1 if n - n_test > 1 else 0)
    test_pids.extend(bucket[:n_test])
    val_pids.extend(bucket[n_test:n_test + n_val])
    train_pids.extend(bucket[n_test + n_val:])

if not val_pids and train_pids:
    val_pids.append(train_pids.pop())
if not test_pids and train_pids:
    test_pids.append(train_pids.pop())

print(f"\nTrain patients ({len(train_pids)}, {100*len(train_pids)/len(pids):.0f}%): {sorted(train_pids)}")
print(f"Val   patients ({len(val_pids)}, {100*len(val_pids)/len(pids):.0f}%): {sorted(val_pids)}")
print(f"Test  patients ({len(test_pids)}, {100*len(test_pids)/len(pids):.0f}%): {sorted(test_pids)}")

# ─────────────────────────────────────────────────────────────────────────
# 5. PER-CHANNEL NORMALIZATION STATS (radius-independent, from TRAIN only)
# ─────────────────────────────────────────────────────────────────────────
def estimate_norm_stats(patient_ids, n_sample_windows):
    sums, sqsums, count = np.zeros(NUM_CHANNELS, np.float64), np.zeros(NUM_CHANNELS, np.float64), 0
    per_patient_budget = max(1, n_sample_windows // max(1, len(patient_ids)))
    for pid in patient_ids:
        X = np.load(os.path.join(DATA_DIR, f"X_{pid}.npy"), mmap_mode="r")
        n = X.shape[0]
        if n == 0:
            continue
        k = min(per_patient_budget, n)
        idx = np.random.default_rng(SEED).choice(n, size=k, replace=False)
        sample = X[idx].astype(np.float64)
        sums   += sample.sum(axis=(0, 2))
        sqsums += (sample ** 2).sum(axis=(0, 2))
        count  += k * sample.shape[2]
        del X, sample
        gc.collect()
    mean = sums / count
    var  = sqsums / count - mean ** 2
    std  = np.sqrt(np.clip(var, 1e-8, None))
    return mean.astype(np.float32), std.astype(np.float32)

print("\nEstimating per-channel normalization stats from train set...")
CH_MEAN, CH_STD = estimate_norm_stats(train_pids, NORM_SAMPLE_WINDOWS)
np.save(os.path.join(OUTPUT_DIR, "channel_mean.npy"), CH_MEAN)
np.save(os.path.join(OUTPUT_DIR, "channel_std.npy"), CH_STD)

# ─────────────────────────────────────────────────────────────────────────
# 6. SIGNAL CLEANING (SpO2 / pleth)
# ─────────────────────────────────────────────────────────────────────────
def flatline_mask(sig, min_run):
    n = len(sig)
    if n < 2:
        return np.zeros(n, dtype=bool)
    same = np.concatenate(([False], sig[1:] == sig[:-1]))
    change_points = np.where(~same)[0]
    change_points = np.concatenate(([0], change_points, [n]))
    mask = np.zeros(n, dtype=bool)
    for i in range(len(change_points) - 1):
        s, e = change_points[i], change_points[i + 1]
        if e - s >= min_run:
            mask[s:e] = True
    return mask


def clean_signal(sig, valid_range=None, flatline_min_run=FLATLINE_MIN_RUN):
    sig = sig.astype(np.float32).copy()
    bad = np.zeros(len(sig), dtype=bool)
    if valid_range is not None:
        lo, hi = valid_range
        bad |= (sig < lo) | (sig > hi)
    bad |= flatline_mask(sig, flatline_min_run)
    if bad.any() and not bad.all():
        good_idx = np.where(~bad)[0]
        bad_idx = np.where(bad)[0]
        sig[bad_idx] = np.interp(bad_idx, good_idx, sig[good_idx])
    return sig


def clean_spo2_pleth(window_8ch):
    window_8ch = window_8ch.copy()
    window_8ch[SPO2_IDX]  = clean_signal(window_8ch[SPO2_IDX], valid_range=SPO2_VALID_RANGE)
    window_8ch[PLETH_IDX] = clean_signal(window_8ch[PLETH_IDX], valid_range=None)
    return window_8ch

# ─────────────────────────────────────────────────────────────────────────
# 7. BOUNDARY LABEL SMOOTHING (train only)
# ─────────────────────────────────────────────────────────────────────────
def compute_boundary_soft_labels(y, soft_conf=BOUNDARY_SOFT_CONF):
    n = len(y)
    soft = y.astype(np.float32).copy()
    for i in range(n):
        left_diff  = (i > 0)     and (y[i - 1] != y[i])
        right_diff = (i < n - 1) and (y[i + 1] != y[i])
        if left_diff or right_diff:
            soft[i] = (1.0 - soft_conf) if y[i] == 1 else soft_conf
    return soft

# ─────────────────────────────────────────────────────────────────────────
# 8. DATASET  (parameterized by context radius)
# ─────────────────────────────────────────────────────────────────────────
class ApneaDataset(Dataset):
    def __init__(self, data_dir, patient_ids, mean, std,
                 context_radius, augment=False, use_soft_labels=False):
        self.data_dir = data_dir
        self._cache = {}
        self.index = []
        self.mean = mean.reshape(NUM_CHANNELS, 1)
        self.std  = std.reshape(NUM_CHANNELS, 1)
        self.context_radius = context_radius
        self.augment = augment
        self.use_soft_labels = use_soft_labels
        self._soft_label_cache = {}

        for pid in patient_ids:
            xp = os.path.join(data_dir, f"X_{pid}.npy")
            yp = os.path.join(data_dir, f"y_{pid}.npy")
            if not (os.path.exists(xp) and os.path.exists(yp)):
                continue
            y = np.load(yp, mmap_mode="r")
            n = len(y)
            self.index.extend([(pid, i, n) for i in range(n)])
            if use_soft_labels:
                self._soft_label_cache[pid] = compute_boundary_soft_labels(np.asarray(y))
            del y

    def _arrays(self, pid):
        if pid not in self._cache:
            self._cache[pid] = (
                np.load(os.path.join(self.data_dir, f"X_{pid}.npy"), mmap_mode="r"),
                np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r"),
            )
        return self._cache[pid]

    def __len__(self):
        return len(self.index)

    def labels(self):
        out = np.zeros(len(self.index), dtype=np.int8)
        cache = {}
        for i, (pid, li, _) in enumerate(self.index):
            if pid not in cache:
                cache[pid] = np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r")
            out[i] = cache[pid][li]
        return out

    def __getitem__(self, idx):
        pid, local_idx, n = self.index[idx]
        X_arr, y_arr = self._arrays(pid)

        r = self.context_radius
        neighbor_idxs = np.clip(np.arange(local_idx - r, local_idx + r + 1), 0, n - 1)
        segs = [X_arr[j].astype(np.float32) for j in neighbor_idxs]
        x = np.concatenate(segs, axis=1)

        x = clean_spo2_pleth(x)
        x = (x - self.mean) / self.std

        if self.augment:
            if NOISE_STD_FRAC > 0:
                x = x + np.random.randn(*x.shape).astype(np.float32) * NOISE_STD_FRAC
            if SCALE_JITTER > 0:
                scale = 1.0 + (np.random.rand(NUM_CHANNELS, 1).astype(np.float32) * 2 - 1) * SCALE_JITTER
                x = x * scale
            if MAX_TIME_SHIFT > 0:
                shift = np.random.randint(-MAX_TIME_SHIFT, MAX_TIME_SHIFT + 1)
                if shift != 0:
                    x = np.roll(x, shift, axis=1)
            if CHANNEL_DROPOUT_P > 0 and np.random.rand() < CHANNEL_DROPOUT_P:
                ch = np.random.randint(0, NUM_CHANNELS)
                x[ch, :] = 0.0

        if self.use_soft_labels:
            y = float(self._soft_label_cache[pid][local_idx])
        else:
            y = float(y_arr[local_idx])

        return torch.from_numpy(x.copy()), torch.tensor(y, dtype=torch.float32)

# ─────────────────────────────────────────────────────────────────────────
# 9. 1D RESNET ARCHITECTURE
# ─────────────────────────────────────────────────────────────────────────
class BasicBlock1D(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, downsample=None, k=7):
        super().__init__()
        pad = k // 2
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=k, stride=stride, padding=pad, bias=False)
        self.bn1   = nn.BatchNorm1d(planes)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=k, stride=1, padding=pad, bias=False)
        self.bn2   = nn.BatchNorm1d(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out = out + identity
        return self.relu(out)


class ResNet1D(nn.Module):
    def __init__(self, in_channels=8, base_width=32, dropout=0.4, layers=(2, 2, 2, 2)):
        super().__init__()
        self.in_planes = base_width
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, base_width, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(base_width),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )
        self.layer1 = self._make_layer(base_width,     layers[0], stride=1)
        self.layer2 = self._make_layer(base_width * 2, layers[1], stride=2)
        self.layer3 = self._make_layer(base_width * 4, layers[2], stride=2)
        self.layer4 = self._make_layer(base_width * 8, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(base_width * 8, 1)   # single logit

        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, planes, num_blocks, stride):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes),
            )
        layers = [BasicBlock1D(self.in_planes, planes, stride, downsample)]
        self.in_planes = planes
        for _ in range(1, num_blocks):
            layers.append(BasicBlock1D(self.in_planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        x = self.dropout(x)
        return self.fc(x).squeeze(-1)   # (B,) single logit per window

# ─────────────────────────────────────────────────────────────────────────
# 10. LOSS FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────
class FocalLossBinary(nn.Module):
    """Binary focal loss on top of BCEWithLogitsLoss. If pos_weight is
    given, class weighting is applied INSIDE the BCE term before the focal
    modulation -- this is "weighted focal loss". If pos_weight is None,
    it's plain (unweighted) focal loss."""
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction="none")
        p = torch.sigmoid(logits)
        pt = p * targets + (1 - p) * (1 - targets)   # prob assigned to the (possibly soft) target
        focal_weight = (1 - pt).clamp(min=1e-6) ** self.gamma
        return (focal_weight * bce).mean()


def get_criterion(loss_type, pos_weight_tensor):
    if loss_type == "bce":
        return nn.BCEWithLogitsLoss()
    elif loss_type == "weighted_bce":
        return nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    elif loss_type == "focal":
        return FocalLossBinary(gamma=FOCAL_GAMMA, pos_weight=None)
    elif loss_type == "weighted_focal":
        return FocalLossBinary(gamma=FOCAL_GAMMA, pos_weight=pos_weight_tensor)
    else:
        raise ValueError(loss_type)

# ─────────────────────────────────────────────────────────────────────────
# 11. TRAIN / EVAL HELPERS
# ─────────────────────────────────────────────────────────────────────────
def best_f1_threshold(labels, probs):
    prec, rec, thr = precision_recall_curve(labels, probs)
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    if len(thr) == 0:
        return 0.5, 0.0
    best_idx = np.nanargmax(f1s[:-1])
    return float(thr[best_idx]), float(f1s[best_idx])


def run_epoch(model, loader, criterion, optimizer, scaler, train_mode: bool):
    model.train(mode=train_mode)
    total_loss = 0.0
    all_labels, all_probs = [], []

    for X, y in loader:
        X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                logits = model(X)
                loss = criterion(logits, y)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * X.size(0)
        probs = torch.sigmoid(logits.float()).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(loader.dataset))
    return avg_loss, np.array(all_labels), np.array(all_probs)


def report_at_threshold(probs, labels, thr):
    preds = (probs >= thr).astype(int)
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)
    cm   = confusion_matrix(labels, preds)
    rep  = classification_report(labels, preds, target_names=["Normal", "Apnea"], zero_division=0)
    return dict(threshold=thr, accuracy=acc, precision=prec, recall=rec, f1_score=f1,
                tn=int(cm[0, 0]), fp=int(cm[0, 1]), fn=int(cm[1, 0]), tp=int(cm[1, 1])), rep, cm

# ─────────────────────────────────────────────────────────────────────────
# 12. TRAIN + EVALUATE ONE MODEL  (radius, loss_type)
# ─────────────────────────────────────────────────────────────────────────
def train_and_evaluate(radius, loss_type, train_pids, val_pids, test_pids):
    model_id = f"radius{radius}_{loss_type}"
    print(f"\n{'#'*70}\n#  MODEL: {model_id}\n{'#'*70}")

    train_ds = ApneaDataset(DATA_DIR, train_pids, CH_MEAN, CH_STD, radius, augment=AUGMENT, use_soft_labels=True)
    val_ds   = ApneaDataset(DATA_DIR, val_pids, CH_MEAN, CH_STD, radius, augment=False, use_soft_labels=False)
    test_ds  = ApneaDataset(DATA_DIR, test_pids, CH_MEAN, CH_STD, radius, augment=False, use_soft_labels=False)

    batch_size = 32 if radius == 1 else 16   # radius=2 sequences are longer -> smaller batch
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)

    train_labels = train_ds.labels()
    n0, n1 = int((train_labels == 0).sum()), int((train_labels == 1).sum())
    pos_weight_tensor = torch.tensor([n0 / max(1, n1)], dtype=torch.float32).to(DEVICE)
    print(f"  Train windows: {len(train_ds):,} (normal={n0:,} apnea={n1:,}) | "
          f"input length: {(2*radius+1)*6000} samples")

    model = ResNet1D(NUM_CHANNELS, BASE_WIDTH, DROPOUT).to(DEVICE)
    criterion = get_criterion(loss_type, pos_weight_tensor)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    best_val_f1, best_state, best_threshold, no_improve = -1.0, None, 0.5, 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss, _, _ = run_epoch(model, train_loader, criterion, optimizer, scaler, True)
        val_loss, val_labels, val_probs = run_epoch(model, val_loader, criterion, optimizer, scaler, False)
        thr, f1 = best_f1_threshold(val_labels, val_probs)
        scheduler.step(f1)
        dt = time.time() - t0

        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                         "val_f1_tuned": f1, "threshold": thr, "lr": optimizer.param_groups[0]["lr"]})
        print(f"  [{model_id}] epoch {epoch:2d}/{EPOCHS} | train_loss={train_loss:.4f} "
              f"val_loss={val_loss:.4f} val_f1={f1:.4f} (thr={thr:.2f}) | {dt:.1f}s")

        if f1 > best_val_f1:
            best_val_f1, best_threshold = f1, thr
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"  [{model_id}] early stopping at epoch {epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    _, test_labels, test_probs = run_epoch(model, test_loader, criterion, optimizer, scaler, False)
    try:
        auc = roc_auc_score(test_labels, test_probs)
    except ValueError:
        auc = float("nan")

    metrics_05, rep_05, cm_05 = report_at_threshold(test_probs, test_labels, 0.5)
    metrics_tuned, rep_tuned, cm_tuned = report_at_threshold(test_probs, test_labels, best_threshold)

    print(f"\n  [{model_id}] TEST RESULTS: AUC={auc:.4f} | "
          f"F1@0.5={metrics_05['f1_score']:.4f} | F1@tuned={metrics_tuned['f1_score']:.4f}")

    fpr, tpr, _ = roc_curve(test_labels, test_probs) if not np.isnan(auc) else (None, None, None)

    return {
        "model_id": model_id, "radius": radius, "loss_type": loss_type,
        "auc": auc, "history": history,
        "metrics_05": metrics_05, "metrics_tuned": metrics_tuned,
        "cm_05": cm_05, "cm_tuned": cm_tuned,
        "rep_tuned": rep_tuned,
        "fpr": fpr, "tpr": tpr,
        "test_labels": test_labels, "test_probs": test_probs,
    }

# ─────────────────────────────────────────────────────────────────────────
# 13. RUN ALL 8 MODELS
# ─────────────────────────────────────────────────────────────────────────
all_results = []
results_csv_path = os.path.join(OUTPUT_DIR, "all_models_results.csv")

for radius in RADII:
    for loss_type in LOSS_TYPES:
        result = train_and_evaluate(radius, loss_type, train_pids, val_pids, test_pids)
        all_results.append(result)

        # save incrementally so partial progress survives a timeout
        rows = []
        for r in all_results:
            for thr_label, m in [("default_0.5", r["metrics_05"]), ("val_tuned", r["metrics_tuned"])]:
                rows.append({
                    "model_id": r["model_id"], "radius": r["radius"], "loss_type": r["loss_type"],
                    "threshold_type": thr_label, "auc": r["auc"], **m,
                })
        pd.DataFrame(rows).to_csv(results_csv_path, index=False)

        torch.cuda.empty_cache()
        gc.collect()

print(f"\nAll 8 models trained. Results saved incrementally to {results_csv_path}")

# ─────────────────────────────────────────────────────────────────────────
# 14. SAVE PER-MODEL CLASSIFICATION REPORTS
# ─────────────────────────────────────────────────────────────────────────
for r in all_results:
    with open(os.path.join(OUTPUT_DIR, f"classification_report_{r['model_id']}.txt"), "w") as fh:
        fh.write(r["rep_tuned"])

# ─────────────────────────────────────────────────────────────────────────
# 15. COMPARISON PLOTS
# ─────────────────────────────────────────────────────────────────────────
loss_labels = {"bce": "BCE", "weighted_bce": "Weighted BCE",
               "focal": "Focal", "weighted_focal": "Weighted Focal"}
colors = {"bce": "#4C72B0", "weighted_bce": "#55A868",
          "focal": "#C44E52", "weighted_focal": "#8172B2"}

# --- 15a. Grouped bar chart: accuracy/precision/recall/F1 across all 8 models (tuned threshold) ---
fig, ax = plt.subplots(figsize=(16, 6))
model_order = [r["model_id"] for r in all_results]
metric_names = ["accuracy", "precision", "recall", "f1_score"]
x = np.arange(len(model_order))
width = 0.2
for i, metric in enumerate(metric_names):
    vals = [r["metrics_tuned"][metric] for r in all_results]
    ax.bar(x + i * width, vals, width, label=metric)
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(model_order, rotation=30, ha="right")
ax.set_ylabel("Score")
ax.set_title("Accuracy / Precision / Recall / F1 across all 8 models (val-tuned threshold)")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_bar_chart.png"), dpi=150)
plt.show()

# --- 15b. Overlaid ROC curves ---
fig, ax = plt.subplots(figsize=(9, 8))
for r in all_results:
    if r["fpr"] is None:
        continue
    style = "-" if r["radius"] == 1 else "--"
    ax.plot(r["fpr"], r["tpr"], style, color=colors[r["loss_type"]],
            label=f"{r['model_id']} (AUC={r['auc']:.3f})")
ax.plot([0, 1], [0, 1], "k:", alpha=0.4)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — all 8 models (solid=radius1, dashed=radius2)")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_roc_curves.png"), dpi=150)
plt.show()

# --- 15c. Confusion matrix grid (2 rows [radius] x 4 cols [loss]) ---
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for r in all_results:
    row = RADII.index(r["radius"])
    col = LOSS_TYPES.index(r["loss_type"])
    ax = axes[row, col]
    cm = r["cm_tuned"]
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(f"radius={r['radius']} | {loss_labels[r['loss_type']]}\nF1={r['metrics_tuned']['f1_score']:.3f}",
                 fontsize=10)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Normal", "Apnea"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Normal", "Apnea"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.suptitle("Confusion Matrices — all 8 models (val-tuned threshold)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_confusion_matrices.png"), dpi=150)
plt.show()

# --- 15d. Training curves grid (2 rows [radius] x 1, 4 lines [loss] each) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for r in all_results:
    ax = axes[RADII.index(r["radius"])]
    hist_df = pd.DataFrame(r["history"])
    ax.plot(hist_df["epoch"], hist_df["val_f1_tuned"], label=loss_labels[r["loss_type"]],
            color=colors[r["loss_type"]])
for i, radius in enumerate(RADII):
    axes[i].set_title(f"Validation F1 by epoch — radius={radius}")
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel("Val F1 (tuned threshold)")
    axes[i].legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_training_curves.png"), dpi=150)
plt.show()

# ─────────────────────────────────────────────────────────────────────────
# 16. FINAL SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────
summary_rows = []
for r in all_results:
    summary_rows.append({
        "model_id": r["model_id"], "radius": r["radius"], "loss_type": loss_labels[r["loss_type"]],
        "AUC": round(r["auc"], 4),
        "Accuracy": round(r["metrics_tuned"]["accuracy"], 4),
        "Precision": round(r["metrics_tuned"]["precision"], 4),
        "Recall": round(r["metrics_tuned"]["recall"], 4),
        "F1": round(r["metrics_tuned"]["f1_score"], 4),
        "Threshold": round(r["metrics_tuned"]["threshold"], 3),
    })
summary_df = pd.DataFrame(summary_rows).sort_values("F1", ascending=False)
summary_df.to_csv(os.path.join(OUTPUT_DIR, "summary_table.csv"), index=False)

print("\n" + "=" * 100)
print("  FINAL SUMMARY — all 8 models, sorted by F1 (val-tuned threshold)")
print("=" * 100)
print(summary_df.to_string(index=False))

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(" - all_models_results.csv              (every model x every threshold variant, full metrics)")
print(" - summary_table.csv                    (clean 8-row summary table)")
print(" - classification_report_<model_id>.txt (per model)")
print(" - comparison_bar_chart.png")
print(" - comparison_roc_curves.png")
print(" - comparison_confusion_matrices.png")
print(" - comparison_training_curves.png")

In [ ]:
"""
=============================================================================
 SLEEP APNEA DETECTION — 1D SE-RESNET ABLATION STUDY
 8 models = 2 context radii (1, 2) x 4 loss functions
   Model A: radius=1, BCE                Model E: radius=2, BCE
   Model B: radius=1, weighted BCE        Model F: radius=2, weighted BCE
   Model C: radius=1, focal loss          Model G: radius=2, focal loss
   Model D: radius=1, weighted focal      Model H: radius=2, weighted focal
=============================================================================
 ARCHITECTURE: 1D ResNet with SQUEEZE-AND-EXCITATION (SE) attention blocks
 after every residual block (SEBasicBlock1D) -- the same backbone used in
 the earlier ResNet1D-SE classifier. SE blocks learn a per-channel
 attention gate so the network can up/down-weight which of the 8
 physiological channels (flow, chest, abd, SpO2, pleth, ECG, pulse, flow2)
 matter most for a given window, instead of treating all channels equally.
 The head is a SINGLE LOGIT (not 2-class softmax) so the same architecture
 plugs directly into all four binary loss variants being compared.

 EXPERIMENTAL DESIGN NOTE (read this before changing anything):
 To make this a FAIR ablation of "does loss function / context radius
 matter", every model uses the IDENTICAL architecture, hyperparameters,
 preprocessing, and training schedule -- the ONLY things that differ
 between the 8 models are (a) context radius and (b) loss function. No
 per-model hyperparameter tuning is done here on purpose: tuning each
 model separately would confound "this loss function is better" with
 "this model got luckier hyperparameters", which defeats the point of an
 ablation study. If you want to further tune whichever model wins here,
 that's a good next step -- just don't do it inside this comparison.

 Also, on purpose, NO WeightedRandomSampler is used for training (unlike
 earlier scripts). The whole point of Models B/D vs A/C is to isolate
 what the LOSS FUNCTION alone contributes to handling class imbalance --
 if a sampler pre-balanced every batch, it would swamp any difference the
 loss weighting makes and the comparison would be meaningless.

 Handles both radius=1 (18000 samples) and radius=2 (30000 samples) inputs
 natively via adaptive pooling -- no architecture changes needed between
 radii.

 LOSSES:
   BCE              : nn.BCEWithLogitsLoss()                      -- no imbalance handling
   Weighted BCE      : nn.BCEWithLogitsLoss(pos_weight=...)         -- class-weighted
   Focal             : focal loss, gamma=2, no class weighting
   Weighted Focal    : focal loss, gamma=2, WITH class weighting (pos_weight baked into the BCE term)

 SPLIT: patient-level, stratified by apnea ratio (interleaved across bins
 so the split hits exact counts with no leftover), 60 train / 15 test /
 15 validation patients -- exact absolute counts, matching the ~90 usable
 patients left after skipping 10 from the ~100-patient dataset -- same
 skip list as previous scripts.

 EVALUATION (for every one of the 8 models): default 0.5 threshold,
 val-tuned threshold, AND SMOOTHED + val-tuned threshold (moving-average
 smoothing of each patient's probability sequence before thresholding --
 real apnea events span multiple consecutive windows, so this removes
 single-window noise and is typically the best-performing variant).
 AHI-style EVENT-LEVEL evaluation (precision/recall/F1 on detected events,
 not individual windows) is also computed for every model, using the
 smoothed probabilities.

 OUTPUTS: per-model confusion matrices (both val-tuned AND smoothed),
 ROC curve, classification reports (tuned AND smoothed), training curves;
 a master CSV with every model's metrics across all three threshold
 strategies, a separate event-level CSV, and 6 comparison figures.

 Input  : X_SN{n}.npy -> (num_windows, 8, 6000) float16
          y_SN{n}.npy -> (num_windows,)         int8  (0=normal, 1=apnea)
=============================================================================
"""

import os
import gc
import time
import random
import warnings
from itertools import zip_longest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_recall_curve
)

warnings.filterwarnings("ignore")
import kagglehub

# ─────────────────────────────────────────────────────────────────────────
# 0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ─────────────────────────────────────────────────────────────────────────
# 0b. GPU / KERNEL-COMPATIBILITY CHECK  (P100 = compute capability 6.0)
# ─────────────────────────────────────────────────────────────────────────
def _gpu_kernels_usable():
    if not torch.cuda.is_available():
        return False
    try:
        x = torch.randn(2, 4, 16, device="cuda")
        conv = nn.Conv1d(4, 4, 3, padding=1).cuda()
        _ = conv(x)
        return True
    except RuntimeError as e:
        print("\n⚠️  GPU kernel check failed:", str(e).splitlines()[0])
        print("⚠️  Falling back to CPU. Switch accelerator to T4/L4 to use GPU.")
        return False

_USE_CUDA = _gpu_kernels_usable()

# ─────────────────────────────────────────────────────────────────────────
# 1. DOWNLOAD DATASET
# ─────────────────────────────────────────────────────────────────────────
DATA_DIR = kagglehub.dataset_download(
    "nishantomhare/detection-of-disordered-breathing-dataset2"
)
print("Dataset path:", DATA_DIR)

def locate_npy_root(root):
    for dirpath, _, filenames in os.walk(root):
        if any(f.startswith("X_SN") and f.endswith(".npy") for f in filenames):
            return dirpath
    return root

DATA_DIR = locate_npy_root(DATA_DIR)
print("Resolved npy root:", DATA_DIR)

OUTPUT_DIR = "/kaggle/working/resnet1d_se_ablation_output_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────
# 2. CONFIG
# ─────────────────────────────────────────────────────────────────────────
SKIP_PATIENTS = {
    "SN1", "SN28", "SN14", "SN13", "SN17", "SN18",
    "SN19", "SN20", "SN27", "SN89"
}

# Exact ABSOLUTE patient counts (not fractions of some other total): out of
# the ~100-patient dataset minus the 10 skipped = ~90 usable patients,
# 60 go to train, 15 to test, 15 to validation -- these three numbers are
# expected to sum to the total usable patient count exactly (60+15+15=90),
# so every usable patient is assigned to a split with none left over.
TRAIN_N = 60
TEST_N  = 15
VAL_N   = 15
N_STRAT_BINS = 5

FIXED_CHANNEL_KEYS = ["flow", "chest", "abd", "spo2", "pleth", "ecg", "pulse", "flow2"]
SPO2_IDX  = FIXED_CHANNEL_KEYS.index("spo2")
PLETH_IDX = FIXED_CHANNEL_KEYS.index("pleth")
FS = 200

NUM_CHANNELS = 8
RADII = [1, 2]                 # context radii to compare
LOSS_TYPES = ["bce", "weighted_bce", "focal", "weighted_focal"]
FOCAL_GAMMA = 2.0

EPOCHS    = 16
PATIENCE  = 6
GRAD_CLIP = 3.0
NUM_WORKERS = 2

# fixed architecture/optimizer hyperparameters -- identical for all 8 models
BASE_WIDTH   = 32
DROPOUT      = 0.4
LR           = 7e-4
WEIGHT_DECAY = 8e-5

AUGMENT           = True
NOISE_STD_FRAC    = 0.03
SCALE_JITTER      = 0.10
MAX_TIME_SHIFT    = 100
CHANNEL_DROPOUT_P = 0.05

NORM_SAMPLE_WINDOWS = 8000
BOUNDARY_SOFT_CONF  = 0.15
SPO2_VALID_RANGE    = (50.0, 100.0)
FLATLINE_MIN_SEC    = 1.0
FLATLINE_MIN_RUN    = int(FLATLINE_MIN_SEC * FS)

DEVICE = torch.device("cuda" if _USE_CUDA else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| capability:", torch.cuda.get_device_capability(0),
          "| usable:", _USE_CUDA)

# ─────────────────────────────────────────────────────────────────────────
# 3. DISCOVER PATIENTS
# ─────────────────────────────────────────────────────────────────────────
all_x_files = sorted(f for f in os.listdir(DATA_DIR)
                      if f.startswith("X_SN") and f.endswith(".npy"))
all_pids = [f[len("X_"):-len(".npy")] for f in all_x_files]
pids = [p for p in all_pids if p not in SKIP_PATIENTS
        and os.path.exists(os.path.join(DATA_DIR, f"y_{p}.npy"))]
print(f"\nTotal X files: {len(all_x_files)} | Skipped: {sorted(SKIP_PATIENTS & set(all_pids))} "
      f"| Usable patients: {len(pids)}")

if len(pids) < 5:
    raise RuntimeError("Not enough usable patients to train/val/test split.")

# ─────────────────────────────────────────────────────────────────────────
# 4. STRATIFIED PATIENT-LEVEL SPLIT -- exact counts: 60 train / 15 test / 15 val
# ─────────────────────────────────────────────────────────────────────────
def patient_apnea_ratio(pid):
    y = np.load(os.path.join(DATA_DIR, f"y_{pid}.npy"), mmap_mode="r")
    n = len(y)
    apnea = int(np.sum(y))
    del y
    return apnea / max(1, n), n, apnea

patient_stats = {pid: patient_apnea_ratio(pid) for pid in pids}
ratios = np.array([patient_stats[p][0] for p in pids])

expected_total = TRAIN_N + TEST_N + VAL_N
if len(pids) != expected_total:
    print(f"\n⚠️  WARNING: usable patient count ({len(pids)}) does not match "
          f"TRAIN_N+TEST_N+VAL_N ({expected_total}). The split below will still "
          f"use exactly {TRAIN_N}/{TEST_N}/{VAL_N} patients where possible, but "
          f"{'some patients will be left unassigned' if len(pids) > expected_total else 'there are not enough patients to fill all three splits'} "
          f"-- check SKIP_PATIENTS / the dataset if this is unexpected.")

# Stratify by apnea ratio into N_STRAT_BINS bins, shuffle within each bin,
# then INTERLEAVE across bins (round-robin) before slicing into exact
# train/test/val counts. Interleaving (rather than per-bin fractional
# rounding) guarantees the requested absolute counts are hit exactly with
# zero leftover, while still spreading each apnea-ratio bin roughly evenly
# across all three splits.
rng = np.random.default_rng(SEED)
order = np.argsort(ratios)
sorted_pids = np.array(pids)[order]
bin_edges = np.linspace(0, len(sorted_pids), N_STRAT_BINS + 1).astype(int)
bins = []
for b in range(N_STRAT_BINS):
    bucket = list(sorted_pids[bin_edges[b]:bin_edges[b + 1]])
    rng.shuffle(bucket)
    bins.append(bucket)

combined = []
for group in zip_longest(*bins, fillvalue=None):
    combined.extend(pid for pid in group if pid is not None)

train_pids = combined[:TRAIN_N]
test_pids  = combined[TRAIN_N:TRAIN_N + TEST_N]
val_pids   = combined[TRAIN_N + TEST_N:TRAIN_N + TEST_N + VAL_N]
unused_pids = combined[TRAIN_N + TEST_N + VAL_N:]   # only non-empty if len(pids) > expected_total

print(f"\nTrain  patients ({len(train_pids)}): {sorted(train_pids)}")
print(f"Test   patients ({len(test_pids)}): {sorted(test_pids)}")
print(f"Val    patients ({len(val_pids)}): {sorted(val_pids)}")
if unused_pids:
    print(f"UNUSED patients ({len(unused_pids)}) -- more usable patients than "
          f"TRAIN_N+TEST_N+VAL_N accounts for: {sorted(unused_pids)}")

# ─────────────────────────────────────────────────────────────────────────
# 5. PER-CHANNEL NORMALIZATION STATS (radius-independent, from TRAIN only)
# ─────────────────────────────────────────────────────────────────────────
def estimate_norm_stats(patient_ids, n_sample_windows):
    sums, sqsums, count = np.zeros(NUM_CHANNELS, np.float64), np.zeros(NUM_CHANNELS, np.float64), 0
    per_patient_budget = max(1, n_sample_windows // max(1, len(patient_ids)))
    for pid in patient_ids:
        X = np.load(os.path.join(DATA_DIR, f"X_{pid}.npy"), mmap_mode="r")
        n = X.shape[0]
        if n == 0:
            continue
        k = min(per_patient_budget, n)
        idx = np.random.default_rng(SEED).choice(n, size=k, replace=False)
        sample = X[idx].astype(np.float64)
        sums   += sample.sum(axis=(0, 2))
        sqsums += (sample ** 2).sum(axis=(0, 2))
        count  += k * sample.shape[2]
        del X, sample
        gc.collect()
    mean = sums / count
    var  = sqsums / count - mean ** 2
    std  = np.sqrt(np.clip(var, 1e-8, None))
    return mean.astype(np.float32), std.astype(np.float32)

print("\nEstimating per-channel normalization stats from train set...")
CH_MEAN, CH_STD = estimate_norm_stats(train_pids, NORM_SAMPLE_WINDOWS)
np.save(os.path.join(OUTPUT_DIR, "channel_mean.npy"), CH_MEAN)
np.save(os.path.join(OUTPUT_DIR, "channel_std.npy"), CH_STD)

# ─────────────────────────────────────────────────────────────────────────
# 6. SIGNAL CLEANING (SpO2 / pleth)
# ─────────────────────────────────────────────────────────────────────────
def flatline_mask(sig, min_run):
    n = len(sig)
    if n < 2:
        return np.zeros(n, dtype=bool)
    same = np.concatenate(([False], sig[1:] == sig[:-1]))
    change_points = np.where(~same)[0]
    change_points = np.concatenate(([0], change_points, [n]))
    mask = np.zeros(n, dtype=bool)
    for i in range(len(change_points) - 1):
        s, e = change_points[i], change_points[i + 1]
        if e - s >= min_run:
            mask[s:e] = True
    return mask


def clean_signal(sig, valid_range=None, flatline_min_run=FLATLINE_MIN_RUN):
    sig = sig.astype(np.float32).copy()
    bad = np.zeros(len(sig), dtype=bool)
    if valid_range is not None:
        lo, hi = valid_range
        bad |= (sig < lo) | (sig > hi)
    bad |= flatline_mask(sig, flatline_min_run)
    if bad.any() and not bad.all():
        good_idx = np.where(~bad)[0]
        bad_idx = np.where(bad)[0]
        sig[bad_idx] = np.interp(bad_idx, good_idx, sig[good_idx])
    return sig


def clean_spo2_pleth(window_8ch):
    window_8ch = window_8ch.copy()
    window_8ch[SPO2_IDX]  = clean_signal(window_8ch[SPO2_IDX], valid_range=SPO2_VALID_RANGE)
    window_8ch[PLETH_IDX] = clean_signal(window_8ch[PLETH_IDX], valid_range=None)
    return window_8ch

# ─────────────────────────────────────────────────────────────────────────
# 7. BOUNDARY LABEL SMOOTHING (train only)
# ─────────────────────────────────────────────────────────────────────────
def compute_boundary_soft_labels(y, soft_conf=BOUNDARY_SOFT_CONF):
    n = len(y)
    soft = y.astype(np.float32).copy()
    for i in range(n):
        left_diff  = (i > 0)     and (y[i - 1] != y[i])
        right_diff = (i < n - 1) and (y[i + 1] != y[i])
        if left_diff or right_diff:
            soft[i] = (1.0 - soft_conf) if y[i] == 1 else soft_conf
    return soft

# ─────────────────────────────────────────────────────────────────────────
# 8. DATASET  (parameterized by context radius)
# ─────────────────────────────────────────────────────────────────────────
class ApneaDataset(Dataset):
    def __init__(self, data_dir, patient_ids, mean, std,
                 context_radius, augment=False, use_soft_labels=False):
        self.data_dir = data_dir
        self._cache = {}
        self.index = []
        self.mean = mean.reshape(NUM_CHANNELS, 1)
        self.std  = std.reshape(NUM_CHANNELS, 1)
        self.context_radius = context_radius
        self.augment = augment
        self.use_soft_labels = use_soft_labels
        self._soft_label_cache = {}

        for pid in patient_ids:
            xp = os.path.join(data_dir, f"X_{pid}.npy")
            yp = os.path.join(data_dir, f"y_{pid}.npy")
            if not (os.path.exists(xp) and os.path.exists(yp)):
                continue
            y = np.load(yp, mmap_mode="r")
            n = len(y)
            self.index.extend([(pid, i, n) for i in range(n)])
            if use_soft_labels:
                self._soft_label_cache[pid] = compute_boundary_soft_labels(np.asarray(y))
            del y

    def _arrays(self, pid):
        if pid not in self._cache:
            self._cache[pid] = (
                np.load(os.path.join(self.data_dir, f"X_{pid}.npy"), mmap_mode="r"),
                np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r"),
            )
        return self._cache[pid]

    def __len__(self):
        return len(self.index)

    def labels(self):
        out = np.zeros(len(self.index), dtype=np.int8)
        cache = {}
        for i, (pid, li, _) in enumerate(self.index):
            if pid not in cache:
                cache[pid] = np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r")
            out[i] = cache[pid][li]
        return out

    def __getitem__(self, idx):
        pid, local_idx, n = self.index[idx]
        X_arr, y_arr = self._arrays(pid)

        r = self.context_radius
        neighbor_idxs = np.clip(np.arange(local_idx - r, local_idx + r + 1), 0, n - 1)
        segs = [X_arr[j].astype(np.float32) for j in neighbor_idxs]
        x = np.concatenate(segs, axis=1)

        x = clean_spo2_pleth(x)
        x = (x - self.mean) / self.std

        if self.augment:
            if NOISE_STD_FRAC > 0:
                x = x + np.random.randn(*x.shape).astype(np.float32) * NOISE_STD_FRAC
            if SCALE_JITTER > 0:
                scale = 1.0 + (np.random.rand(NUM_CHANNELS, 1).astype(np.float32) * 2 - 1) * SCALE_JITTER
                x = x * scale
            if MAX_TIME_SHIFT > 0:
                shift = np.random.randint(-MAX_TIME_SHIFT, MAX_TIME_SHIFT + 1)
                if shift != 0:
                    x = np.roll(x, shift, axis=1)
            if CHANNEL_DROPOUT_P > 0 and np.random.rand() < CHANNEL_DROPOUT_P:
                ch = np.random.randint(0, NUM_CHANNELS)
                x[ch, :] = 0.0

        if self.use_soft_labels:
            y = float(self._soft_label_cache[pid][local_idx])
        else:
            y = float(y_arr[local_idx])

        return torch.from_numpy(x.copy()), torch.tensor(y, dtype=torch.float32)

# ─────────────────────────────────────────────────────────────────────────
# 9. 1D RESNET + SE-ATTENTION ARCHITECTURE
#    (SE = squeeze-and-excitation: a learned per-channel attention gate
#     after each residual block, letting the network up/down-weight which
#     of the 8 physiological channels matter most for a given window,
#     rather than treating all channels with equal importance)
# ─────────────────────────────────────────────────────────────────────────
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(4, channels // reduction)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels), nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1)
        return x * s


class SEBasicBlock1D(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, downsample=None, k=7):
        super().__init__()
        pad = k // 2
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=k, stride=stride, padding=pad, bias=False)
        self.bn1   = nn.BatchNorm1d(planes)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=k, stride=1, padding=pad, bias=False)
        self.bn2   = nn.BatchNorm1d(planes)
        self.se    = SEBlock1D(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out = out + identity
        return self.relu(out)


class ResNet1D(nn.Module):
    """SE-ResNet1D with a SINGLE-LOGIT output head (not 2-class softmax),
    so it plugs directly into all four binary loss variants (BCE, weighted
    BCE, focal, weighted focal) via the same sigmoid-based interface."""
    def __init__(self, in_channels=8, base_width=32, dropout=0.4, layers=(2, 2, 3, 2)):
        super().__init__()
        self.in_planes = base_width
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, base_width, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(base_width),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )
        self.layer1 = self._make_layer(base_width,     layers[0], stride=1)
        self.layer2 = self._make_layer(base_width * 2, layers[1], stride=2)
        self.layer3 = self._make_layer(base_width * 4, layers[2], stride=2)
        self.layer4 = self._make_layer(base_width * 8, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(base_width * 8, 1)   # single logit

        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, planes, num_blocks, stride):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes),
            )
        layers = [SEBasicBlock1D(self.in_planes, planes, stride, downsample)]
        self.in_planes = planes
        for _ in range(1, num_blocks):
            layers.append(SEBasicBlock1D(self.in_planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        x = self.dropout(x)
        return self.fc(x).squeeze(-1)   # (B,) single logit per window


# ─────────────────────────────────────────────────────────────────────────
# 10. LOSS FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────
class FocalLossBinary(nn.Module):
    """Binary focal loss on top of BCEWithLogitsLoss. If pos_weight is
    given, class weighting is applied INSIDE the BCE term before the focal
    modulation -- this is "weighted focal loss". If pos_weight is None,
    it's plain (unweighted) focal loss."""
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction="none")
        p = torch.sigmoid(logits)
        pt = p * targets + (1 - p) * (1 - targets)   # prob assigned to the (possibly soft) target
        focal_weight = (1 - pt).clamp(min=1e-6) ** self.gamma
        return (focal_weight * bce).mean()


def get_criterion(loss_type, pos_weight_tensor):
    if loss_type == "bce":
        return nn.BCEWithLogitsLoss()
    elif loss_type == "weighted_bce":
        return nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    elif loss_type == "focal":
        return FocalLossBinary(gamma=FOCAL_GAMMA, pos_weight=None)
    elif loss_type == "weighted_focal":
        return FocalLossBinary(gamma=FOCAL_GAMMA, pos_weight=pos_weight_tensor)
    else:
        raise ValueError(loss_type)

# ─────────────────────────────────────────────────────────────────────────
# 10b. TEMPORAL SMOOTHING  (post-hoc, applied to every one of the 8 models)
# ─────────────────────────────────────────────────────────────────────────
SMOOTH_RADIUS = 2   # moving-average window, in # of windows, each side

def smooth_probs_grouped(dataset, probs, radius=SMOOTH_RADIUS):
    """Moving-average smoothing of per-window probabilities, computed
    separately within each patient's (already time-ordered) window sequence
    -- never smooths across a patient boundary. Real apnea events span
    multiple consecutive windows, so smoothing removes single-window noise
    before thresholding."""
    probs = np.asarray(probs)
    smoothed = np.zeros_like(probs)
    pids_seq = [p for p, _, _ in dataset.index]
    n = len(pids_seq)
    i = 0
    kernel = np.ones(2 * radius + 1) / (2 * radius + 1)
    while i < n:
        j = i
        while j < n and pids_seq[j] == pids_seq[i]:
            j += 1
        seg = probs[i:j]
        padded = np.pad(seg, (radius, radius), mode="edge")
        smoothed[i:j] = np.convolve(padded, kernel, mode="valid")
        i = j
    return smoothed

# ─────────────────────────────────────────────────────────────────────────
# 10c. EVENT-LEVEL EVALUATION  (AHI-style: did we detect the EVENT, not the
#      individual 30s window -- applied to every one of the 8 models, using
#      the SMOOTHED probabilities + smoothed val-tuned threshold)
# ─────────────────────────────────────────────────────────────────────────
def extract_events(binary_seq):
    events, start = [], None
    for i, v in enumerate(binary_seq):
        if v == 1 and start is None:
            start = i
        elif v == 0 and start is not None:
            events.append((start, i - 1)); start = None
    if start is not None:
        events.append((start, len(binary_seq) - 1))
    return events


def match_events(pred_events, true_events, min_overlap=1):
    """A predicted event can legitimately overlap (and should credit)
    MULTIPLE ground-truth events if they're closely spaced -- no `break`
    after the first match, so one long predicted block correctly covering
    several nearby short true events gets credit for all of them."""
    matched_true = set()
    pred_hit = [False] * len(pred_events)
    for pi, (ps, pe) in enumerate(pred_events):
        for ti, (ts, te) in enumerate(true_events):
            overlap = max(0, min(pe, te) - max(ps, ts) + 1)
            if overlap >= min_overlap:
                matched_true.add(ti)
                pred_hit[pi] = True
    tp_true = len(matched_true)
    tp_pred = sum(pred_hit)
    fp = len(pred_events) - tp_pred
    fn = len(true_events) - tp_true
    return tp_true, tp_pred, fp, fn, len(pred_events), len(true_events)


def event_level_evaluation(dataset, labels, probs, threshold, min_overlap=1):
    preds = (probs >= threshold).astype(int)
    pids_seq = [p for p, _, _ in dataset.index]
    n = len(pids_seq)
    agg = dict(tp_true=0, tp_pred=0, fp=0, fn=0, n_pred=0, n_true=0)
    i = 0
    while i < n:
        j = i
        while j < n and pids_seq[j] == pids_seq[i]:
            j += 1
        pe = extract_events(preds[i:j])
        te = extract_events(labels[i:j])
        tp_true, tp_pred, fp, fn, npe, nte = match_events(pe, te, min_overlap)
        agg["tp_true"] += tp_true; agg["tp_pred"] += tp_pred
        agg["fp"] += fp; agg["fn"] += fn
        agg["n_pred"] += npe; agg["n_true"] += nte
        i = j
    precision = agg["tp_pred"] / agg["n_pred"] if agg["n_pred"] > 0 else 0.0
    recall    = agg["tp_true"] / agg["n_true"] if agg["n_true"] > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    agg["precision"], agg["recall"], agg["f1_score"] = precision, recall, f1
    return agg

# ─────────────────────────────────────────────────────────────────────────
# 11. TRAIN / EVAL HELPERS
# ─────────────────────────────────────────────────────────────────────────
def best_f1_threshold(labels, probs):
    prec, rec, thr = precision_recall_curve(labels, probs)
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    if len(thr) == 0:
        return 0.5, 0.0
    best_idx = np.nanargmax(f1s[:-1])
    return float(thr[best_idx]), float(f1s[best_idx])


def run_epoch(model, loader, criterion, optimizer, scaler, train_mode: bool):
    model.train(mode=train_mode)
    total_loss = 0.0
    all_labels, all_probs = [], []

    for X, y in loader:
        X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                logits = model(X)
                loss = criterion(logits, y)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * X.size(0)
        probs = torch.sigmoid(logits.float()).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(loader.dataset))
    return avg_loss, np.array(all_labels), np.array(all_probs)


def report_at_threshold(probs, labels, thr):
    preds = (probs >= thr).astype(int)
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)
    cm   = confusion_matrix(labels, preds)
    rep  = classification_report(labels, preds, target_names=["Normal", "Apnea"], zero_division=0)
    return dict(threshold=thr, accuracy=acc, precision=prec, recall=rec, f1_score=f1,
                tn=int(cm[0, 0]), fp=int(cm[0, 1]), fn=int(cm[1, 0]), tp=int(cm[1, 1])), rep, cm

# ─────────────────────────────────────────────────────────────────────────
# 12. TRAIN + EVALUATE ONE MODEL  (radius, loss_type)
# ─────────────────────────────────────────────────────────────────────────
def train_and_evaluate(radius, loss_type, train_pids, val_pids, test_pids):
    model_id = f"radius{radius}_{loss_type}"
    print(f"\n{'#'*70}\n#  MODEL: {model_id}\n{'#'*70}")

    train_ds = ApneaDataset(DATA_DIR, train_pids, CH_MEAN, CH_STD, radius, augment=AUGMENT, use_soft_labels=True)
    val_ds   = ApneaDataset(DATA_DIR, val_pids, CH_MEAN, CH_STD, radius, augment=False, use_soft_labels=False)
    test_ds  = ApneaDataset(DATA_DIR, test_pids, CH_MEAN, CH_STD, radius, augment=False, use_soft_labels=False)

    batch_size = 32 if radius == 1 else 16   # radius=2 sequences are longer -> smaller batch
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)

    train_labels = train_ds.labels()
    n0, n1 = int((train_labels == 0).sum()), int((train_labels == 1).sum())
    pos_weight_tensor = torch.tensor([n0 / max(1, n1)], dtype=torch.float32).to(DEVICE)
    print(f"  Train windows: {len(train_ds):,} (normal={n0:,} apnea={n1:,}) | "
          f"input length: {(2*radius+1)*6000} samples")

    model = ResNet1D(NUM_CHANNELS, BASE_WIDTH, DROPOUT).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Model: SE-ResNet1D | Trainable params: {n_params:,}")
    criterion = get_criterion(loss_type, pos_weight_tensor)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    best_val_f1, best_state, best_threshold, no_improve = -1.0, None, 0.5, 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss, _, _ = run_epoch(model, train_loader, criterion, optimizer, scaler, True)
        val_loss, val_labels, val_probs = run_epoch(model, val_loader, criterion, optimizer, scaler, False)
        thr, f1 = best_f1_threshold(val_labels, val_probs)
        scheduler.step(f1)
        dt = time.time() - t0

        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                         "val_f1_tuned": f1, "threshold": thr, "lr": optimizer.param_groups[0]["lr"]})
        print(f"  [{model_id}] epoch {epoch:2d}/{EPOCHS} | train_loss={train_loss:.4f} "
              f"val_loss={val_loss:.4f} val_f1={f1:.4f} (thr={thr:.2f}) | {dt:.1f}s")

        if f1 > best_val_f1:
            best_val_f1, best_threshold = f1, thr
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"  [{model_id}] early stopping at epoch {epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # recompute val probs with the BEST weights (the training loop's last
    # epoch isn't necessarily the best one due to early stopping) -- needed
    # to tune the smoothed threshold correctly
    _, val_labels_final, val_probs_final = run_epoch(model, val_loader, criterion, optimizer, scaler, False)
    val_probs_smooth = smooth_probs_grouped(val_ds, val_probs_final, radius=SMOOTH_RADIUS)
    smooth_thr, smooth_val_f1 = best_f1_threshold(val_labels_final, val_probs_smooth)

    _, test_labels, test_probs = run_epoch(model, test_loader, criterion, optimizer, scaler, False)
    try:
        auc = roc_auc_score(test_labels, test_probs)
    except ValueError:
        auc = float("nan")

    metrics_05, rep_05, cm_05 = report_at_threshold(test_probs, test_labels, 0.5)
    metrics_tuned, rep_tuned, cm_tuned = report_at_threshold(test_probs, test_labels, best_threshold)

    # --- SMOOTHED @ val-tuned threshold (the headline "best" result) ---
    test_probs_smooth = smooth_probs_grouped(test_ds, test_probs, radius=SMOOTH_RADIUS)
    metrics_smooth, rep_smooth, cm_smooth = report_at_threshold(test_probs_smooth, test_labels, smooth_thr)

    # --- event-level (AHI-style) evaluation, using smoothed probs + smoothed threshold ---
    event_metrics = event_level_evaluation(test_ds, test_labels, test_probs_smooth, smooth_thr, min_overlap=1)

    print(f"\n  [{model_id}] TEST RESULTS: AUC={auc:.4f} | "
          f"F1@0.5={metrics_05['f1_score']:.4f} | F1@tuned={metrics_tuned['f1_score']:.4f} | "
          f"F1@smoothed_tuned={metrics_smooth['f1_score']:.4f} | Event F1={event_metrics['f1_score']:.4f}")

    fpr, tpr, _ = roc_curve(test_labels, test_probs) if not np.isnan(auc) else (None, None, None)

    return {
        "model_id": model_id, "radius": radius, "loss_type": loss_type,
        "auc": auc, "history": history,
        "metrics_05": metrics_05, "metrics_tuned": metrics_tuned, "metrics_smooth": metrics_smooth,
        "cm_05": cm_05, "cm_tuned": cm_tuned, "cm_smooth": cm_smooth,
        "rep_tuned": rep_tuned, "rep_smooth": rep_smooth,
        "smooth_threshold": smooth_thr, "smooth_val_f1": smooth_val_f1,
        "event_metrics": event_metrics,
        "fpr": fpr, "tpr": tpr,
        "test_labels": test_labels, "test_probs": test_probs, "test_probs_smooth": test_probs_smooth,
    }

# ─────────────────────────────────────────────────────────────────────────
# 13. RUN ALL 8 MODELS
# ─────────────────────────────────────────────────────────────────────────
all_results = []
results_csv_path = os.path.join(OUTPUT_DIR, "all_models_results.csv")
event_csv_path = os.path.join(OUTPUT_DIR, "event_level_all_models.csv")

for radius in RADII:
    for loss_type in LOSS_TYPES:
        result = train_and_evaluate(radius, loss_type, train_pids, val_pids, test_pids)
        all_results.append(result)

        # save incrementally so partial progress survives a timeout
        rows = []
        for r in all_results:
            for thr_label, m in [("default_0.5", r["metrics_05"]),
                                  ("val_tuned", r["metrics_tuned"]),
                                  ("smoothed_val_tuned", r["metrics_smooth"])]:
                rows.append({
                    "model_id": r["model_id"], "radius": r["radius"], "loss_type": r["loss_type"],
                    "threshold_type": thr_label, "auc": r["auc"], **m,
                })
        pd.DataFrame(rows).to_csv(results_csv_path, index=False)

        event_rows = [{"model_id": r["model_id"], "radius": r["radius"], "loss_type": r["loss_type"],
                        "smooth_threshold": r["smooth_threshold"], **r["event_metrics"]}
                       for r in all_results]
        pd.DataFrame(event_rows).to_csv(event_csv_path, index=False)

        torch.cuda.empty_cache()
        gc.collect()

print(f"\nAll 8 models trained. Results saved incrementally to {results_csv_path} and {event_csv_path}")

# ─────────────────────────────────────────────────────────────────────────
# 14. SAVE PER-MODEL CLASSIFICATION REPORTS  (tuned AND smoothed)
# ─────────────────────────────────────────────────────────────────────────
for r in all_results:
    with open(os.path.join(OUTPUT_DIR, f"classification_report_{r['model_id']}_tuned.txt"), "w") as fh:
        fh.write(r["rep_tuned"])
    with open(os.path.join(OUTPUT_DIR, f"classification_report_{r['model_id']}_smoothed.txt"), "w") as fh:
        fh.write(r["rep_smooth"])

# ─────────────────────────────────────────────────────────────────────────
# 15. COMPARISON PLOTS
# ─────────────────────────────────────────────────────────────────────────
loss_labels = {"bce": "BCE", "weighted_bce": "Weighted BCE",
               "focal": "Focal", "weighted_focal": "Weighted Focal"}
colors = {"bce": "#4C72B0", "weighted_bce": "#55A868",
          "focal": "#C44E52", "weighted_focal": "#8172B2"}

# --- 15a. Grouped bar chart: accuracy/precision/recall/F1 across all 8 models (tuned threshold) ---
fig, ax = plt.subplots(figsize=(16, 6))
model_order = [r["model_id"] for r in all_results]
metric_names = ["accuracy", "precision", "recall", "f1_score"]
x = np.arange(len(model_order))
width = 0.2
for i, metric in enumerate(metric_names):
    vals = [r["metrics_tuned"][metric] for r in all_results]
    ax.bar(x + i * width, vals, width, label=metric)
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(model_order, rotation=30, ha="right")
ax.set_ylabel("Score")
ax.set_title("Accuracy / Precision / Recall / F1 across all 8 models (val-tuned threshold)")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_bar_chart.png"), dpi=150)
plt.show()

# --- 15b. Overlaid ROC curves ---
fig, ax = plt.subplots(figsize=(9, 8))
for r in all_results:
    if r["fpr"] is None:
        continue
    style = "-" if r["radius"] == 1 else "--"
    ax.plot(r["fpr"], r["tpr"], style, color=colors[r["loss_type"]],
            label=f"{r['model_id']} (AUC={r['auc']:.3f})")
ax.plot([0, 1], [0, 1], "k:", alpha=0.4)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — all 8 models (solid=radius1, dashed=radius2)")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_roc_curves.png"), dpi=150)
plt.show()

# --- 15c. Confusion matrix grid (2 rows [radius] x 4 cols [loss]) ---
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for r in all_results:
    row = RADII.index(r["radius"])
    col = LOSS_TYPES.index(r["loss_type"])
    ax = axes[row, col]
    cm = r["cm_tuned"]
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(f"radius={r['radius']} | {loss_labels[r['loss_type']]}\nF1={r['metrics_tuned']['f1_score']:.3f}",
                 fontsize=10)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Normal", "Apnea"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Normal", "Apnea"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.suptitle("Confusion Matrices — all 8 models (val-tuned threshold, RAW probs)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_confusion_matrices.png"), dpi=150)
plt.show()

# --- 15c2. SMOOTHED confusion matrix grid (2 rows [radius] x 4 cols [loss]) ---
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for r in all_results:
    row = RADII.index(r["radius"])
    col = LOSS_TYPES.index(r["loss_type"])
    ax = axes[row, col]
    cm = r["cm_smooth"]
    im = ax.imshow(cm, cmap="Greens")
    ax.set_title(f"radius={r['radius']} | {loss_labels[r['loss_type']]}\n"
                 f"F1(smoothed)={r['metrics_smooth']['f1_score']:.3f}",
                 fontsize=10)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Normal", "Apnea"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Normal", "Apnea"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.suptitle("Confusion Matrices — all 8 models (SMOOTHED @ val-tuned threshold)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_confusion_matrices_smoothed.png"), dpi=150)
plt.show()

# --- 15c3. F1 comparison across threshold types (default / tuned / smoothed / event-level) ---
fig, ax = plt.subplots(figsize=(16, 6))
f1_default = [r["metrics_05"]["f1_score"] for r in all_results]
f1_tuned   = [r["metrics_tuned"]["f1_score"] for r in all_results]
f1_smooth  = [r["metrics_smooth"]["f1_score"] for r in all_results]
f1_event   = [r["event_metrics"]["f1_score"] for r in all_results]
x = np.arange(len(model_order))
width = 0.2
ax.bar(x - 1.5 * width, f1_default, width, label="F1 @ default 0.5")
ax.bar(x - 0.5 * width, f1_tuned, width, label="F1 @ val-tuned")
ax.bar(x + 0.5 * width, f1_smooth, width, label="F1 @ smoothed + val-tuned")
ax.bar(x + 1.5 * width, f1_event, width, label="Event-level F1")
ax.set_xticks(x)
ax.set_xticklabels(model_order, rotation=30, ha="right")
ax.set_ylabel("F1 Score")
ax.set_title("F1 Score across threshold strategies — all 8 models")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_f1_by_threshold_strategy.png"), dpi=150)
plt.show()

# --- 15d. Training curves grid (2 rows [radius] x 1, 4 lines [loss] each) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for r in all_results:
    ax = axes[RADII.index(r["radius"])]
    hist_df = pd.DataFrame(r["history"])
    ax.plot(hist_df["epoch"], hist_df["val_f1_tuned"], label=loss_labels[r["loss_type"]],
            color=colors[r["loss_type"]])
for i, radius in enumerate(RADII):
    axes[i].set_title(f"Validation F1 by epoch — radius={radius}")
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel("Val F1 (tuned threshold)")
    axes[i].legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_training_curves.png"), dpi=150)
plt.show()

# ─────────────────────────────────────────────────────────────────────────
# 16. FINAL SUMMARY TABLE  (includes smoothed + event-level results)
# ─────────────────────────────────────────────────────────────────────────
summary_rows = []
for r in all_results:
    summary_rows.append({
        "model_id": r["model_id"], "radius": r["radius"], "loss_type": loss_labels[r["loss_type"]],
        "AUC": round(r["auc"], 4),
        "Accuracy_tuned": round(r["metrics_tuned"]["accuracy"], 4),
        "Precision_tuned": round(r["metrics_tuned"]["precision"], 4),
        "Recall_tuned": round(r["metrics_tuned"]["recall"], 4),
        "F1_tuned": round(r["metrics_tuned"]["f1_score"], 4),
        "Threshold_tuned": round(r["metrics_tuned"]["threshold"], 3),
        "Accuracy_smoothed": round(r["metrics_smooth"]["accuracy"], 4),
        "Precision_smoothed": round(r["metrics_smooth"]["precision"], 4),
        "Recall_smoothed": round(r["metrics_smooth"]["recall"], 4),
        "F1_smoothed": round(r["metrics_smooth"]["f1_score"], 4),
        "Threshold_smoothed": round(r["smooth_threshold"], 3),
        "Event_Precision": round(r["event_metrics"]["precision"], 4),
        "Event_Recall": round(r["event_metrics"]["recall"], 4),
        "Event_F1": round(r["event_metrics"]["f1_score"], 4),
    })
summary_df = pd.DataFrame(summary_rows).sort_values("F1_smoothed", ascending=False)
summary_df.to_csv(os.path.join(OUTPUT_DIR, "summary_table.csv"), index=False)

print("\n" + "=" * 130)
print("  FINAL SUMMARY — all 8 models, sorted by F1 (SMOOTHED @ val-tuned threshold)")
print("=" * 130)
print(summary_df.to_string(index=False))

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(" - all_models_results.csv                        (every model x default/tuned/smoothed thresholds, full metrics)")
print(" - event_level_all_models.csv                     (AHI-style event precision/recall/F1 per model)")
print(" - summary_table.csv                              (clean 8-row summary: tuned + smoothed + event-level side by side)")
print(" - classification_report_<model_id>_tuned.txt     (per model)")
print(" - classification_report_<model_id>_smoothed.txt  (per model)")
print(" - comparison_bar_chart.png                       (accuracy/precision/recall/F1, val-tuned threshold)")
print(" - comparison_roc_curves.png")
print(" - comparison_confusion_matrices.png              (val-tuned threshold, raw probs)")
print(" - comparison_confusion_matrices_smoothed.png     (SMOOTHED @ val-tuned threshold)")
print(" - comparison_f1_by_threshold_strategy.png        (F1 across default/tuned/smoothed/event-level)")
print(" - comparison_training_curves.png")
print(f"\nNOTE: at inference time, replicate BOTH the context-window concatenation "
      f"AND the temporal smoothing (radius={SMOOTH_RADIUS}) to reproduce the 'smoothed' metrics above.")

# Sleep Apnea Detection — 1D SE-ResNet Loss-Function Ablation Study

## Documentation of Pipeline, Architecture, and Evaluation Methodology

---

## 1. Overview

This document explains the full mechanism of the sleep apnea detection pipeline: how raw polysomnography (PSG) signals become model input, how the model is built and trained, and — most importantly — what every reported evaluation number actually means. The study trains **four models**, identical in every respect except the loss function used, to isolate the effect of loss-function choice on class-imbalanced apnea/hypopnea detection.

| Model | Loss Function |
|---|---|
| A | Binary Cross-Entropy (BCE) |
| B | Weighted BCE |
| C | Focal Loss |
| D | Weighted Focal Loss |

All four share the same architecture, hyperparameters, preprocessing, and training schedule. The **only** thing that differs is the loss function — this is a deliberate experimental design choice, explained in Section 8.

---

## 2. The Raw Data

Each patient contributes two files:

- **`X_SN{n}.npy`** — shape `(num_windows, 8, 6000)`, `float16`. Every patient's overnight recording is chopped into non-overlapping **30-second windows**, each sampled at 200 Hz across 8 channels: `flow, chest, abd, spo2, pleth, ecg, pulse, flow2`.
- **`y_SN{n}.npy`** — shape `(num_windows,)`, `int8`. A binary label per window: `1` if any apnea/hypopnea event overlaps that window, `0` otherwise.

A patient with, say, 8 hours of recording produces roughly 960 windows (8 hrs × 3600s / 30s).

Ten patients (`SN1, SN13, SN14, SN17, SN18, SN19, SN20, SN27, SN28, SN89`) are excluded up front — these were identified in earlier exploratory work as having corrupted, missing, or otherwise unusable channel data.

---

## 3. Patient-Level Train/Val/Test Split

### Why patient-level, not window-level?

Windows from the same patient are highly correlated (shared baseline physiology, breathing rhythm, recording equipment). If windows were split randomly across train/val/test, the model could effectively "memorize" a patient's signature and the test score would be inflated — it wouldn't reflect performance on a genuinely unseen patient. **Every split boundary in this pipeline is drawn between patients, never between windows of the same patient.**

### Exact counts: 60 train / 15 test / 15 validation

Out of the ~90 usable patients (100 total minus 10 skipped), exactly:
- **60** patients → training set
- **15** patients → test set
- **15** patients → validation set

### Stratification by apnea severity

Patients vary enormously in how much apnea they have (some nights are almost entirely normal breathing, others have apnea in 40%+ of windows). To make sure train/val/test all see a representative mix of mild-to-severe patients:

1. Every patient's **apnea ratio** (fraction of their windows labeled apnea) is computed.
2. Patients are sorted by this ratio and divided into 5 bins (bin 0 = least apnea, bin 4 = most apnea).
3. Within each bin, patients are shuffled.
4. The 5 bins are **interleaved round-robin** (one patient from bin 0, one from bin 1, ..., one from bin 4, then repeat) into a single combined list.
5. The first 60 patients in this interleaved list → train; next 15 → test; next 15 → validation.

Interleaving (rather than taking a fixed fraction from each bin) guarantees the exact 60/15/15 counts are hit with zero leftover, while still spreading each severity bin roughly evenly across all three sets.

---

## 4. Per-Channel Normalization

Each of the 8 channels has wildly different raw scales — SpO2 lives in the 90–100% range, flow signals might be in arbitrary sensor units, ECG has yet another scale. Feeding these directly into a neural network causes unstable training (large-scale channels dominate the loss gradient).

**Fix: per-channel z-score normalization.**

For each channel *c*:

```
normalized_value = (raw_value − mean_c) / std_c
```

`mean_c` and `std_c` are estimated **once, from the training set only** (never from validation/test, which would leak information), using a random subsample of windows for efficiency rather than scanning every window of every training patient.

---

## 5. SpO2 / Pleth Signal Cleaning

Pulse-oximetry channels (SpO2 and pleth/plethysmography) are notoriously prone to sensor artifacts:

- **Out-of-range values**: SpO2 outside the physiologically possible 50–100% range is almost certainly a sensor error, not a real reading.
- **Flatline dropout**: if a sensor loses contact (finger movement, poor perfusion), the signal often "freezes" at a constant value for a stretch — a run of ≥1 second (200 samples) of an exactly repeated value is flagged as a dropout.

Both problems are fixed the same way: **mask the bad samples, then linearly interpolate across the gap** using the nearest valid samples on either side. This prevents the model from learning spurious patterns like "flat SpO2 signal = normal breathing" when the flatness is actually just a broken sensor.

---

## 6. Context Windows (Radius = 1)

A single isolated 30-second window often lacks enough information to tell "this is an apnea event" from "this looks similar but isn't" — desaturation (the SpO2 drop that confirms an apnea event) can lag the actual breathing pause by 10–30+ seconds, spilling into neighboring windows.

**Fix: each training/evaluation sample is the target window concatenated with its immediate neighbor windows** (one window before, one window after — "radius 1"), giving the model 90 seconds of context (18,000 raw samples) instead of just 30 seconds. The **label stays the target (center) window's label** — only the input gets wider, not the prediction target. At the very start or end of a patient's recording, where a neighbor doesn't exist, the boundary window is repeated (edge-clamping) instead of leaving a gap.

---

## 7. Boundary Label Smoothing (Training Only)

Windows are labeled `1` if **any** apnea event overlaps them at all — so a window containing just 2 seconds of an event at its very edge gets the exact same hard label as a window that's 100% inside the event. This is genuine label noise, not a modeling problem, and it caps achievable performance regardless of architecture.

**Fix: soft labels for ambiguous "boundary" windows, used during training only.** For each patient's window sequence, any window whose immediate left *or* right neighbor has a **different** hard label is flagged as a boundary window (it likely only partially overlaps the true event). Instead of the hard `0` or `1`, it gets a softened target:

- Apnea window near a boundary → labeled `0.85` instead of `1.0`
- Normal window near a boundary → labeled `0.15` instead of `0.0`

This tells the loss function "be less confident here" without ever touching how validation/test accuracy is measured — evaluation always uses the true, hard ground-truth labels, so reported metrics remain directly comparable across models and to earlier experiments.

---

## 8. The Experimental Design: Why Everything Else Is Held Constant

This is an **ablation study** — its entire purpose is to answer "does the choice of loss function matter, all else being equal?" Two design choices protect the validity of that comparison:

1. **No per-model hyperparameter tuning.** All four models use the identical architecture, learning rate, weight decay, dropout, batch size, and training schedule. If each model were tuned separately, a result difference could just mean "this model got a luckier hyperparameter draw," not "this loss function is genuinely better." Every difference you see in the results table is attributable to the loss function alone.

2. **No `WeightedRandomSampler`.** A sampler that pre-balances every training batch (oversampling the minority apnea class) would handle class imbalance *before* the loss function ever sees the data — swamping any difference the loss weighting itself contributes. Removing the sampler means the loss function is the *only* imbalance-handling mechanism in play, which is exactly what's being measured.

---

## 9. Architecture: SE-ResNet1D

### Why a ResNet backbone

Residual (skip) connections let gradients flow directly through a network without having to pass through every intervening layer, which is what makes deep convolutional networks trainable at all — without them, very deep stacks of convolutions tend to degrade rather than improve as depth increases.

### Structure

```
Input (8 channels, 18000 samples)
   │
   ▼
Stem: Conv1d(k=15, stride=2) → BatchNorm → ReLU → MaxPool(stride=2)
   │
   ▼
Layer 1: SE-Residual Block(s) @ base_width channels,   stride 1
Layer 2: SE-Residual Block(s) @ base_width×2 channels, stride 2
Layer 3: SE-Residual Block(s) @ base_width×4 channels, stride 2
Layer 4: SE-Residual Block(s) @ base_width×8 channels, stride 2
   │
   ▼
Global Average Pool → Dropout → Linear → 1 logit
```

### The residual block (with SE attention)

Each block: `Conv1d → BatchNorm → ReLU → Conv1d → BatchNorm`, with the block's *input* added back to its output (the residual/skip connection) before a final ReLU. If the number of channels changes between input and output, a 1×1 convolution projects the skip path to match.

### Squeeze-and-Excitation (SE) attention

After every residual block, an **SE block** learns a per-channel attention gate:

1. **Squeeze**: global-average-pool the block's output across time, collapsing each channel to a single number — a compact summary of "how active is this channel, overall, in this window?"
2. **Excitation**: pass that summary through a small two-layer bottleneck (`Linear → ReLU → Linear → Sigmoid`) to produce one weight per channel, between 0 and 1.
3. **Reweight**: multiply the original block output by these learned weights, channel-by-channel.

In plain terms: the network learns to *turn up* the channels that matter most for a given window (e.g., emphasize SpO2/pleth when there's a desaturation pattern) and *turn down* channels that aren't currently informative, rather than treating all 8 physiological signals as equally important all the time.

### Single-logit output

The final layer outputs **one number** (a logit), not two class scores. Passed through a sigmoid, this becomes `P(apnea)` directly — the natural output format for binary cross-entropy-family losses (all four losses compared here are binary, sigmoid-based losses).

### Handling both context lengths without architecture changes

Because the final pooling layer is an *adaptive* average pool (it always collapses to a single value per channel, regardless of the input's time length), the exact same architecture works whether the input is 18,000 samples (radius=1) or any other window length — no manual resizing or architecture edits needed.

---

## 10. The Four Loss Functions, Explained

All four operate on the same single logit output. Let `p = sigmoid(logit)` be the model's predicted probability of apnea, and `y ∈ {0, 1}` (or a soft value in `[0,1]` for boundary-smoothed training labels) be the target.

### A. Binary Cross-Entropy (BCE)

```
BCE = −[ y·log(p) + (1−y)·log(1−p) ]
```

The standard classification loss. Every sample contributes equally regardless of class — with ~76% normal / ~24% apnea windows, this loss implicitly lets the majority (normal) class dominate the average gradient, since there are simply more of them.

### B. Weighted BCE

```
Weighted BCE = −[ w_pos·y·log(p) + (1−y)·log(1−p) ]
```

Identical to BCE, but the positive-class (apnea) term is multiplied by `w_pos = (# normal windows) / (# apnea windows)` — a number greater than 1 that inflates the apnea class's contribution to the loss, directly compensating for its rarity in the training data.

### C. Focal Loss

```
Focal = (1 − p_t)^γ · BCE,   where p_t = p if y=1, else (1−p)
```

Focal loss down-weights *easy, already-confident* predictions (`p_t` close to 1) via the `(1 − p_t)^γ` term, and up-weights *hard, currently-wrong* predictions (`p_t` close to 0). With `γ = 2`, a confidently-correct prediction (`p_t = 0.9`) is scaled down by `(1−0.9)² = 0.01` — contributing 100× less to the loss than a completely-wrong one. This forces training to keep focusing on the examples the model is still getting wrong, rather than the (numerous) easy normal windows it already classifies correctly.

### D. Weighted Focal Loss

```
Weighted Focal = (1 − p_t)^γ · Weighted BCE
```

Combines both mechanisms: the `pos_weight` class-imbalance correction from (B), *and* the hard-example-focusing behavior of (C), applied together in a single loss.

---

## 11. Training Procedure

- **Optimizer**: AdamW (Adam with decoupled weight decay), identical learning rate/weight decay across all 4 models.
- **Mixed precision (AMP)**: forward/backward passes run in float16 where safe, roughly halving memory use and speeding up training on the GPU, with automatic loss-scaling to prevent numerical underflow.
- **Gradient clipping**: gradient norms are clipped to a fixed maximum before each optimizer step, preventing rare large gradients (e.g., from an unusual batch) from destabilizing training.
- **Learning rate schedule**: `ReduceLROnPlateau` — the learning rate is halved whenever validation F1 stops improving for a set number of epochs, allowing finer convergence once progress stalls.
- **Early stopping**: training halts once validation F1 hasn't improved for a set patience window, and the **best-ever** epoch's weights (not the final epoch's) are what get evaluated on the test set.

---

## 12. Evaluation Methodology — What Each Reported Number Means

Every model is evaluated **four different ways**, each answering a slightly different question:

### 12.1 Default threshold (0.5)

The model's probability output is thresholded at the "obvious" cutoff: `predict apnea if P(apnea) ≥ 0.5`. This is the naive baseline — useful as a reference point, but rarely the *right* operating point for an imbalanced problem, since the model's raw probabilities are calibrated to the training distribution, not necessarily to maximize a chosen metric.

### 12.2 Validation-tuned threshold

Instead of assuming 0.5 is right, the threshold that **maximizes F1 score on the validation set** is found (by scanning the precision-recall curve) and then applied, unchanged, to the test set. This answers: "if we pick the best possible decision boundary ahead of time (without peeking at test data), how well does the model do?"

### 12.3 Smoothed + validation-tuned threshold ("smoothed" results)

Real apnea events last much longer than a single 30-second window and produce a *run* of consecutive high-probability windows — but the raw per-window probability sequence is often noisy, flickering above and below the threshold within a single true event. **Temporal smoothing** applies a moving average (radius=2, i.e., averaging each window's probability with its 2 neighbors on each side) to every patient's probability sequence *before* thresholding — smoothing is always computed separately per patient, never blending across a patient boundary. The threshold is then **re-tuned specifically on the smoothed validation probabilities** (not reusing the raw-tuned threshold, since smoothing shifts the probability distribution). This is typically the best-performing variant, since it removes single-window noise that the raw predictions can't avoid.

### 12.4 Event-level (AHI-style) evaluation

Window-level scoring has a structural unfairness: a real apnea event spanning 4 windows, where the model correctly flags 3 of them, is penalized as "1 false negative" even though the event was clearly detected. Clinically, sleep studies score the **Apnea-Hypopnea Index (AHI)** — events per hour — not individual 30-second segments.

**Event-level evaluation** converts both the ground truth and the (smoothed, thresholded) predictions into lists of *events* — runs of consecutive positive windows — per patient, then matches predicted events against true events by **overlap**:

- A predicted event counts as a **true positive** if it overlaps *any* unmatched true event by at least one window.
- Crucially, **one predicted event can be matched to multiple nearby true events** if they're closely spaced — a single long predicted block correctly spanning several short real events gets credit for detecting all of them, rather than being penalized for "merging" them.
- **Event Precision** = (predicted events that matched something real) / (total predicted events)
- **Event Recall** = (true events that were detected by something) / (total true events)
- **Event F1** = harmonic mean of the two

This is the metric closest to how a real clinical apnea-detection system would actually be judged.

---

## 13. Metric Definitions (Plain Language)

| Metric | Formula | What it answers |
|---|---|---|
| **Accuracy** | (correct predictions) / (all predictions) | Overall fraction correct — misleading here since ~76% of windows are normal, so a model that *always* predicts "normal" would already score ~76% accuracy while being clinically useless. |
| **Precision** | TP / (TP + FP) | "Of all the windows/events the model flagged as apnea, what fraction actually were?" High precision = few false alarms. |
| **Recall** | TP / (TP + FN) | "Of all the actual apnea windows/events, what fraction did the model catch?" High recall = few missed events. |
| **F1 Score** | 2·(Precision·Recall)/(Precision+Recall) | Harmonic mean of precision and recall — a single number balancing both, heavily penalized if *either* is low. The primary metric used to compare models and tune thresholds in this study. |
| **ROC-AUC** | Area under the ROC curve (TPR vs. FPR across all thresholds) | A **threshold-independent** measure of how well-separated the model's apnea and normal probability distributions are. AUC=0.5 is random guessing; AUC=1.0 is perfect separation. Unlike the other metrics, AUC doesn't depend on picking a decision threshold. |
| **Confusion Matrix** | 2×2 grid of TN / FP / FN / TP counts | The raw counts underlying every metric above — shows exactly how many normal windows were misclassified as apnea (FP) and vice versa (FN), not just a summary ratio. |

---

## 14. Outputs Produced

| File | Contents |
|---|---|
| `all_models_results.csv` | Every model × every threshold strategy (default/tuned/smoothed), full metrics |
| `event_level_all_models.csv` | AHI-style event precision/recall/F1 per model |
| `summary_table.csv` | Clean one-row-per-model comparison table, sorted by smoothed F1 |
| `classification_report_<model>_tuned.txt` / `_smoothed.txt` | Per-class precision/recall/F1/support, sklearn-style |
| `comparison_bar_chart.png` | Accuracy/precision/recall/F1 across all 4 models |
| `comparison_roc_curves.png` | Overlaid ROC curves, one per loss function |
| `comparison_confusion_matrices.png` / `_smoothed.png` | 1×4 grid of confusion matrices |
| `comparison_f1_by_threshold_strategy.png` | F1 compared across all four evaluation strategies, for all 4 models |
| `comparison_training_curves.png` | Validation F1 vs. epoch, all 4 loss functions overlaid |
| `channel_mean.npy` / `channel_std.npy` | Normalization statistics — **must be reapplied at inference time** |

---

## 15. Key Takeaway on Reproducing Results at Inference Time

The "smoothed" metrics are **not** a property of the trained model weights alone — they depend on the inference pipeline replicating two extra steps beyond a plain forward pass:

1. **Context-window concatenation** (radius=1: target window + 1 neighbor each side)
2. **Temporal smoothing** (radius=2: moving average across each patient's probability sequence)

Applying the saved model to a single raw window with neither of these steps will reproduce the "default 0.5" or "tuned" numbers, but not the smoothed ones.

In [ ]:
"""
=============================================================================
 SLEEP APNEA DETECTION — 1D SE-RESNET ABLATION STUDY (radius=1 only)
 4 models = 4 loss functions, context radius fixed at 1 (90s input)
   Model A: BCE
   Model B: weighted BCE
   Model C: focal loss
   Model D: weighted focal
 (The radius=2 variants from the earlier 8-model version have been
 removed, along with everything specific to comparing radii.)
=============================================================================
 ARCHITECTURE: 1D ResNet with SQUEEZE-AND-EXCITATION (SE) attention blocks
 after every residual block (SEBasicBlock1D) -- the same backbone used in
 the earlier ResNet1D-SE classifier. SE blocks learn a per-channel
 attention gate so the network can up/down-weight which of the 8
 physiological channels (flow, chest, abd, SpO2, pleth, ECG, pulse, flow2)
 matter most for a given window, instead of treating all channels equally.
 The head is a SINGLE LOGIT (not 2-class softmax) so the same architecture
 plugs directly into all four binary loss variants being compared.

 EXPERIMENTAL DESIGN NOTE (read this before changing anything):
 To make this a FAIR ablation of "does loss function matter", every model
 uses the IDENTICAL architecture, hyperparameters, preprocessing, and
 training schedule -- the ONLY thing that differs between the 4 models is
 the loss function. No per-model hyperparameter tuning is done here on
 purpose: tuning each model separately would confound "this loss function
 is better" with "this model got luckier hyperparameters", which defeats
 the point of an ablation study. If you want to further tune whichever
 model wins here, that's a good next step -- just don't do it inside this
 comparison.

 Also, on purpose, NO WeightedRandomSampler is used for training (unlike
 earlier scripts). The whole point of Models B/D vs A/C is to isolate
 what the LOSS FUNCTION alone contributes to handling class imbalance --
 if a sampler pre-balanced every batch, it would swamp any difference the
 loss weighting makes and the comparison would be meaningless.

 Context radius=1 -> 18000-sample input (90s), handled natively via
 adaptive pooling.

 LOSSES:
   BCE              : nn.BCEWithLogitsLoss()                      -- no imbalance handling
   Weighted BCE      : nn.BCEWithLogitsLoss(pos_weight=...)         -- class-weighted
   Focal             : focal loss, gamma=2, no class weighting
   Weighted Focal    : focal loss, gamma=2, WITH class weighting (pos_weight baked into the BCE term)

 SPLIT: patient-level, stratified by apnea ratio (interleaved across bins
 so the split hits exact counts with no leftover), 60 train / 15 test /
 15 validation patients -- exact absolute counts, matching the ~90 usable
 patients left after skipping 10 from the ~100-patient dataset -- same
 skip list as previous scripts.

 EVALUATION (for every one of the 4 models): default 0.5 threshold,
 val-tuned threshold, AND SMOOTHED + val-tuned threshold (moving-average
 smoothing of each patient's probability sequence before thresholding --
 real apnea events span multiple consecutive windows, so this removes
 single-window noise and is typically the best-performing variant).
 AHI-style EVENT-LEVEL evaluation (precision/recall/F1 on detected events,
 not individual windows) is also computed for every model, using the
 smoothed probabilities.

 OUTPUTS: per-model confusion matrices (both val-tuned AND smoothed),
 ROC curve, classification reports (tuned AND smoothed), training curves;
 a master CSV with every model's metrics across all three threshold
 strategies, a separate event-level CSV, and 6 comparison figures.

 Input  : X_SN{n}.npy -> (num_windows, 8, 6000) float16
          y_SN{n}.npy -> (num_windows,)         int8  (0=normal, 1=apnea)
=============================================================================
"""

import os
import gc
import time
import random
import warnings
from itertools import zip_longest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_recall_curve
)

warnings.filterwarnings("ignore")
import kagglehub

# ─────────────────────────────────────────────────────────────────────────
# 0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ─────────────────────────────────────────────────────────────────────────
# 0b. GPU / KERNEL-COMPATIBILITY CHECK  (P100 = compute capability 6.0)
# ─────────────────────────────────────────────────────────────────────────
def _gpu_kernels_usable():
    if not torch.cuda.is_available():
        return False
    try:
        x = torch.randn(2, 4, 16, device="cuda")
        conv = nn.Conv1d(4, 4, 3, padding=1).cuda()
        _ = conv(x)
        return True
    except RuntimeError as e:
        print("\n⚠️  GPU kernel check failed:", str(e).splitlines()[0])
        print("⚠️  Falling back to CPU. Switch accelerator to T4/L4 to use GPU.")
        return False

_USE_CUDA = _gpu_kernels_usable()

# ─────────────────────────────────────────────────────────────────────────
# 1. DOWNLOAD DATASET
# ─────────────────────────────────────────────────────────────────────────
DATA_DIR = kagglehub.dataset_download(
    "nishantomhare/detection-of-disordered-breathing-dataset2"
)
print("Dataset path:", DATA_DIR)

def locate_npy_root(root):
    for dirpath, _, filenames in os.walk(root):
        if any(f.startswith("X_SN") and f.endswith(".npy") for f in filenames):
            return dirpath
    return root

DATA_DIR = locate_npy_root(DATA_DIR)
print("Resolved npy root:", DATA_DIR)

OUTPUT_DIR = "/kaggle/working/resnet1d_se_ablation_output_radius1_only"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────
# 2. CONFIG
# ─────────────────────────────────────────────────────────────────────────
SKIP_PATIENTS = {
    "SN1", "SN28", "SN14", "SN13", "SN17", "SN18",
    "SN19", "SN20", "SN27", "SN89"
}

# Exact ABSOLUTE patient counts (not fractions of some other total): out of
# the ~100-patient dataset minus the 10 skipped = ~90 usable patients,
# 60 go to train, 15 to test, 15 to validation -- these three numbers are
# expected to sum to the total usable patient count exactly (60+15+15=90),
# so every usable patient is assigned to a split with none left over.
TRAIN_N = 60
TEST_N  = 15
VAL_N   = 15
N_STRAT_BINS = 5

FIXED_CHANNEL_KEYS = ["flow", "chest", "abd", "spo2", "pleth", "ecg", "pulse", "flow2"]
SPO2_IDX  = FIXED_CHANNEL_KEYS.index("spo2")
PLETH_IDX = FIXED_CHANNEL_KEYS.index("pleth")
FS = 200

NUM_CHANNELS = 8
RADII = [1]                    # radius=2 variants removed as requested
LOSS_TYPES = ["bce", "weighted_bce", "focal", "weighted_focal"]
FOCAL_GAMMA = 2.0

EPOCHS    = 16
PATIENCE  = 6
GRAD_CLIP = 3.0
NUM_WORKERS = 2

# fixed architecture/optimizer hyperparameters -- identical for all 4 models
BASE_WIDTH   = 32
DROPOUT      = 0.4
LR           = 7e-4
WEIGHT_DECAY = 8e-5

AUGMENT           = True
NOISE_STD_FRAC    = 0.03
SCALE_JITTER      = 0.10
MAX_TIME_SHIFT    = 100
CHANNEL_DROPOUT_P = 0.05

NORM_SAMPLE_WINDOWS = 8000
BOUNDARY_SOFT_CONF  = 0.15
SPO2_VALID_RANGE    = (50.0, 100.0)
FLATLINE_MIN_SEC    = 1.0
FLATLINE_MIN_RUN    = int(FLATLINE_MIN_SEC * FS)

DEVICE = torch.device("cuda" if _USE_CUDA else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| capability:", torch.cuda.get_device_capability(0),
          "| usable:", _USE_CUDA)

# ─────────────────────────────────────────────────────────────────────────
# 3. DISCOVER PATIENTS
# ─────────────────────────────────────────────────────────────────────────
all_x_files = sorted(f for f in os.listdir(DATA_DIR)
                      if f.startswith("X_SN") and f.endswith(".npy"))
all_pids = [f[len("X_"):-len(".npy")] for f in all_x_files]
pids = [p for p in all_pids if p not in SKIP_PATIENTS
        and os.path.exists(os.path.join(DATA_DIR, f"y_{p}.npy"))]
print(f"\nTotal X files: {len(all_x_files)} | Skipped: {sorted(SKIP_PATIENTS & set(all_pids))} "
      f"| Usable patients: {len(pids)}")

if len(pids) < 5:
    raise RuntimeError("Not enough usable patients to train/val/test split.")

# ─────────────────────────────────────────────────────────────────────────
# 4. STRATIFIED PATIENT-LEVEL SPLIT -- exact counts: 60 train / 15 test / 15 val
# ─────────────────────────────────────────────────────────────────────────
def patient_apnea_ratio(pid):
    y = np.load(os.path.join(DATA_DIR, f"y_{pid}.npy"), mmap_mode="r")
    n = len(y)
    apnea = int(np.sum(y))
    del y
    return apnea / max(1, n), n, apnea

patient_stats = {pid: patient_apnea_ratio(pid) for pid in pids}
ratios = np.array([patient_stats[p][0] for p in pids])

expected_total = TRAIN_N + TEST_N + VAL_N
if len(pids) != expected_total:
    print(f"\n⚠️  WARNING: usable patient count ({len(pids)}) does not match "
          f"TRAIN_N+TEST_N+VAL_N ({expected_total}). The split below will still "
          f"use exactly {TRAIN_N}/{TEST_N}/{VAL_N} patients where possible, but "
          f"{'some patients will be left unassigned' if len(pids) > expected_total else 'there are not enough patients to fill all three splits'} "
          f"-- check SKIP_PATIENTS / the dataset if this is unexpected.")

# Stratify by apnea ratio into N_STRAT_BINS bins, shuffle within each bin,
# then INTERLEAVE across bins (round-robin) before slicing into exact
# train/test/val counts. Interleaving (rather than per-bin fractional
# rounding) guarantees the requested absolute counts are hit exactly with
# zero leftover, while still spreading each apnea-ratio bin roughly evenly
# across all three splits.
rng = np.random.default_rng(SEED)
order = np.argsort(ratios)
sorted_pids = np.array(pids)[order]
bin_edges = np.linspace(0, len(sorted_pids), N_STRAT_BINS + 1).astype(int)
bins = []
for b in range(N_STRAT_BINS):
    bucket = list(sorted_pids[bin_edges[b]:bin_edges[b + 1]])
    rng.shuffle(bucket)
    bins.append(bucket)

combined = []
for group in zip_longest(*bins, fillvalue=None):
    combined.extend(pid for pid in group if pid is not None)

train_pids = combined[:TRAIN_N]
test_pids  = combined[TRAIN_N:TRAIN_N + TEST_N]
val_pids   = combined[TRAIN_N + TEST_N:TRAIN_N + TEST_N + VAL_N]
unused_pids = combined[TRAIN_N + TEST_N + VAL_N:]   # only non-empty if len(pids) > expected_total

print(f"\nTrain  patients ({len(train_pids)}): {sorted(train_pids)}")
print(f"Test   patients ({len(test_pids)}): {sorted(test_pids)}")
print(f"Val    patients ({len(val_pids)}): {sorted(val_pids)}")
if unused_pids:
    print(f"UNUSED patients ({len(unused_pids)}) -- more usable patients than "
          f"TRAIN_N+TEST_N+VAL_N accounts for: {sorted(unused_pids)}")

# ─────────────────────────────────────────────────────────────────────────
# 5. PER-CHANNEL NORMALIZATION STATS (radius-independent, from TRAIN only)
# ─────────────────────────────────────────────────────────────────────────
def estimate_norm_stats(patient_ids, n_sample_windows):
    sums, sqsums, count = np.zeros(NUM_CHANNELS, np.float64), np.zeros(NUM_CHANNELS, np.float64), 0
    per_patient_budget = max(1, n_sample_windows // max(1, len(patient_ids)))
    for pid in patient_ids:
        X = np.load(os.path.join(DATA_DIR, f"X_{pid}.npy"), mmap_mode="r")
        n = X.shape[0]
        if n == 0:
            continue
        k = min(per_patient_budget, n)
        idx = np.random.default_rng(SEED).choice(n, size=k, replace=False)
        sample = X[idx].astype(np.float64)
        sums   += sample.sum(axis=(0, 2))
        sqsums += (sample ** 2).sum(axis=(0, 2))
        count  += k * sample.shape[2]
        del X, sample
        gc.collect()
    mean = sums / count
    var  = sqsums / count - mean ** 2
    std  = np.sqrt(np.clip(var, 1e-8, None))
    return mean.astype(np.float32), std.astype(np.float32)

print("\nEstimating per-channel normalization stats from train set...")
CH_MEAN, CH_STD = estimate_norm_stats(train_pids, NORM_SAMPLE_WINDOWS)
np.save(os.path.join(OUTPUT_DIR, "channel_mean.npy"), CH_MEAN)
np.save(os.path.join(OUTPUT_DIR, "channel_std.npy"), CH_STD)

# ─────────────────────────────────────────────────────────────────────────
# 6. SIGNAL CLEANING (SpO2 / pleth)
# ─────────────────────────────────────────────────────────────────────────
def flatline_mask(sig, min_run):
    n = len(sig)
    if n < 2:
        return np.zeros(n, dtype=bool)
    same = np.concatenate(([False], sig[1:] == sig[:-1]))
    change_points = np.where(~same)[0]
    change_points = np.concatenate(([0], change_points, [n]))
    mask = np.zeros(n, dtype=bool)
    for i in range(len(change_points) - 1):
        s, e = change_points[i], change_points[i + 1]
        if e - s >= min_run:
            mask[s:e] = True
    return mask


def clean_signal(sig, valid_range=None, flatline_min_run=FLATLINE_MIN_RUN):
    sig = sig.astype(np.float32).copy()
    bad = np.zeros(len(sig), dtype=bool)
    if valid_range is not None:
        lo, hi = valid_range
        bad |= (sig < lo) | (sig > hi)
    bad |= flatline_mask(sig, flatline_min_run)
    if bad.any() and not bad.all():
        good_idx = np.where(~bad)[0]
        bad_idx = np.where(bad)[0]
        sig[bad_idx] = np.interp(bad_idx, good_idx, sig[good_idx])
    return sig


def clean_spo2_pleth(window_8ch):
    window_8ch = window_8ch.copy()
    window_8ch[SPO2_IDX]  = clean_signal(window_8ch[SPO2_IDX], valid_range=SPO2_VALID_RANGE)
    window_8ch[PLETH_IDX] = clean_signal(window_8ch[PLETH_IDX], valid_range=None)
    return window_8ch

# ─────────────────────────────────────────────────────────────────────────
# 7. BOUNDARY LABEL SMOOTHING (train only)
# ─────────────────────────────────────────────────────────────────────────
def compute_boundary_soft_labels(y, soft_conf=BOUNDARY_SOFT_CONF):
    n = len(y)
    soft = y.astype(np.float32).copy()
    for i in range(n):
        left_diff  = (i > 0)     and (y[i - 1] != y[i])
        right_diff = (i < n - 1) and (y[i + 1] != y[i])
        if left_diff or right_diff:
            soft[i] = (1.0 - soft_conf) if y[i] == 1 else soft_conf
    return soft

# ─────────────────────────────────────────────────────────────────────────
# 8. DATASET  (parameterized by context radius)
# ─────────────────────────────────────────────────────────────────────────
class ApneaDataset(Dataset):
    def __init__(self, data_dir, patient_ids, mean, std,
                 context_radius, augment=False, use_soft_labels=False):
        self.data_dir = data_dir
        self._cache = {}
        self.index = []
        self.mean = mean.reshape(NUM_CHANNELS, 1)
        self.std  = std.reshape(NUM_CHANNELS, 1)
        self.context_radius = context_radius
        self.augment = augment
        self.use_soft_labels = use_soft_labels
        self._soft_label_cache = {}

        for pid in patient_ids:
            xp = os.path.join(data_dir, f"X_{pid}.npy")
            yp = os.path.join(data_dir, f"y_{pid}.npy")
            if not (os.path.exists(xp) and os.path.exists(yp)):
                continue
            y = np.load(yp, mmap_mode="r")
            n = len(y)
            self.index.extend([(pid, i, n) for i in range(n)])
            if use_soft_labels:
                self._soft_label_cache[pid] = compute_boundary_soft_labels(np.asarray(y))
            del y

    def _arrays(self, pid):
        if pid not in self._cache:
            self._cache[pid] = (
                np.load(os.path.join(self.data_dir, f"X_{pid}.npy"), mmap_mode="r"),
                np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r"),
            )
        return self._cache[pid]

    def __len__(self):
        return len(self.index)

    def labels(self):
        out = np.zeros(len(self.index), dtype=np.int8)
        cache = {}
        for i, (pid, li, _) in enumerate(self.index):
            if pid not in cache:
                cache[pid] = np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r")
            out[i] = cache[pid][li]
        return out

    def __getitem__(self, idx):
        pid, local_idx, n = self.index[idx]
        X_arr, y_arr = self._arrays(pid)

        r = self.context_radius
        neighbor_idxs = np.clip(np.arange(local_idx - r, local_idx + r + 1), 0, n - 1)
        segs = [X_arr[j].astype(np.float32) for j in neighbor_idxs]
        x = np.concatenate(segs, axis=1)

        x = clean_spo2_pleth(x)
        x = (x - self.mean) / self.std

        if self.augment:
            if NOISE_STD_FRAC > 0:
                x = x + np.random.randn(*x.shape).astype(np.float32) * NOISE_STD_FRAC
            if SCALE_JITTER > 0:
                scale = 1.0 + (np.random.rand(NUM_CHANNELS, 1).astype(np.float32) * 2 - 1) * SCALE_JITTER
                x = x * scale
            if MAX_TIME_SHIFT > 0:
                shift = np.random.randint(-MAX_TIME_SHIFT, MAX_TIME_SHIFT + 1)
                if shift != 0:
                    x = np.roll(x, shift, axis=1)
            if CHANNEL_DROPOUT_P > 0 and np.random.rand() < CHANNEL_DROPOUT_P:
                ch = np.random.randint(0, NUM_CHANNELS)
                x[ch, :] = 0.0

        if self.use_soft_labels:
            y = float(self._soft_label_cache[pid][local_idx])
        else:
            y = float(y_arr[local_idx])

        return torch.from_numpy(x.copy()), torch.tensor(y, dtype=torch.float32)

# ─────────────────────────────────────────────────────────────────────────
# 9. 1D RESNET + SE-ATTENTION ARCHITECTURE
#    (SE = squeeze-and-excitation: a learned per-channel attention gate
#     after each residual block, letting the network up/down-weight which
#     of the 8 physiological channels matter most for a given window,
#     rather than treating all channels with equal importance)
# ─────────────────────────────────────────────────────────────────────────
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(4, channels // reduction)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels), nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1)
        return x * s


class SEBasicBlock1D(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, downsample=None, k=7):
        super().__init__()
        pad = k // 2
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=k, stride=stride, padding=pad, bias=False)
        self.bn1   = nn.BatchNorm1d(planes)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=k, stride=1, padding=pad, bias=False)
        self.bn2   = nn.BatchNorm1d(planes)
        self.se    = SEBlock1D(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out = out + identity
        return self.relu(out)


class ResNet1D(nn.Module):
    """SE-ResNet1D with a SINGLE-LOGIT output head (not 2-class softmax),
    so it plugs directly into all four binary loss variants (BCE, weighted
    BCE, focal, weighted focal) via the same sigmoid-based interface."""
    def __init__(self, in_channels=8, base_width=32, dropout=0.4, layers=(2, 2, 3, 2)):
        super().__init__()
        self.in_planes = base_width
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, base_width, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(base_width),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )
        self.layer1 = self._make_layer(base_width,     layers[0], stride=1)
        self.layer2 = self._make_layer(base_width * 2, layers[1], stride=2)
        self.layer3 = self._make_layer(base_width * 4, layers[2], stride=2)
        self.layer4 = self._make_layer(base_width * 8, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(base_width * 8, 1)   # single logit

        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, planes, num_blocks, stride):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes),
            )
        layers = [SEBasicBlock1D(self.in_planes, planes, stride, downsample)]
        self.in_planes = planes
        for _ in range(1, num_blocks):
            layers.append(SEBasicBlock1D(self.in_planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        x = self.dropout(x)
        return self.fc(x).squeeze(-1)   # (B,) single logit per window


# ─────────────────────────────────────────────────────────────────────────
# 10. LOSS FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────
class FocalLossBinary(nn.Module):
    """Binary focal loss on top of BCEWithLogitsLoss. If pos_weight is
    given, class weighting is applied INSIDE the BCE term before the focal
    modulation -- this is "weighted focal loss". If pos_weight is None,
    it's plain (unweighted) focal loss."""
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction="none")
        p = torch.sigmoid(logits)
        pt = p * targets + (1 - p) * (1 - targets)   # prob assigned to the (possibly soft) target
        focal_weight = (1 - pt).clamp(min=1e-6) ** self.gamma
        return (focal_weight * bce).mean()


def get_criterion(loss_type, pos_weight_tensor):
    if loss_type == "bce":
        return nn.BCEWithLogitsLoss()
    elif loss_type == "weighted_bce":
        return nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    elif loss_type == "focal":
        return FocalLossBinary(gamma=FOCAL_GAMMA, pos_weight=None)
    elif loss_type == "weighted_focal":
        return FocalLossBinary(gamma=FOCAL_GAMMA, pos_weight=pos_weight_tensor)
    else:
        raise ValueError(loss_type)

# ─────────────────────────────────────────────────────────────────────────
# 10b. TEMPORAL SMOOTHING  (post-hoc, applied to every one of the 4 models)
# ─────────────────────────────────────────────────────────────────────────
SMOOTH_RADIUS = 2   # moving-average window, in # of windows, each side

def smooth_probs_grouped(dataset, probs, radius=SMOOTH_RADIUS):
    """Moving-average smoothing of per-window probabilities, computed
    separately within each patient's (already time-ordered) window sequence
    -- never smooths across a patient boundary. Real apnea events span
    multiple consecutive windows, so smoothing removes single-window noise
    before thresholding."""
    probs = np.asarray(probs)
    smoothed = np.zeros_like(probs)
    pids_seq = [p for p, _, _ in dataset.index]
    n = len(pids_seq)
    i = 0
    kernel = np.ones(2 * radius + 1) / (2 * radius + 1)
    while i < n:
        j = i
        while j < n and pids_seq[j] == pids_seq[i]:
            j += 1
        seg = probs[i:j]
        padded = np.pad(seg, (radius, radius), mode="edge")
        smoothed[i:j] = np.convolve(padded, kernel, mode="valid")
        i = j
    return smoothed

# ─────────────────────────────────────────────────────────────────────────
# 10c. EVENT-LEVEL EVALUATION  (AHI-style: did we detect the EVENT, not the
#      individual 30s window -- applied to every one of the 4 models, using
#      the SMOOTHED probabilities + smoothed val-tuned threshold)
# ─────────────────────────────────────────────────────────────────────────
def extract_events(binary_seq):
    events, start = [], None
    for i, v in enumerate(binary_seq):
        if v == 1 and start is None:
            start = i
        elif v == 0 and start is not None:
            events.append((start, i - 1)); start = None
    if start is not None:
        events.append((start, len(binary_seq) - 1))
    return events


def match_events(pred_events, true_events, min_overlap=1):
    """A predicted event can legitimately overlap (and should credit)
    MULTIPLE ground-truth events if they're closely spaced -- no `break`
    after the first match, so one long predicted block correctly covering
    several nearby short true events gets credit for all of them."""
    matched_true = set()
    pred_hit = [False] * len(pred_events)
    for pi, (ps, pe) in enumerate(pred_events):
        for ti, (ts, te) in enumerate(true_events):
            overlap = max(0, min(pe, te) - max(ps, ts) + 1)
            if overlap >= min_overlap:
                matched_true.add(ti)
                pred_hit[pi] = True
    tp_true = len(matched_true)
    tp_pred = sum(pred_hit)
    fp = len(pred_events) - tp_pred
    fn = len(true_events) - tp_true
    return tp_true, tp_pred, fp, fn, len(pred_events), len(true_events)


def event_level_evaluation(dataset, labels, probs, threshold, min_overlap=1):
    preds = (probs >= threshold).astype(int)
    pids_seq = [p for p, _, _ in dataset.index]
    n = len(pids_seq)
    agg = dict(tp_true=0, tp_pred=0, fp=0, fn=0, n_pred=0, n_true=0)
    i = 0
    while i < n:
        j = i
        while j < n and pids_seq[j] == pids_seq[i]:
            j += 1
        pe = extract_events(preds[i:j])
        te = extract_events(labels[i:j])
        tp_true, tp_pred, fp, fn, npe, nte = match_events(pe, te, min_overlap)
        agg["tp_true"] += tp_true; agg["tp_pred"] += tp_pred
        agg["fp"] += fp; agg["fn"] += fn
        agg["n_pred"] += npe; agg["n_true"] += nte
        i = j
    precision = agg["tp_pred"] / agg["n_pred"] if agg["n_pred"] > 0 else 0.0
    recall    = agg["tp_true"] / agg["n_true"] if agg["n_true"] > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    agg["precision"], agg["recall"], agg["f1_score"] = precision, recall, f1
    return agg

# ─────────────────────────────────────────────────────────────────────────
# 11. TRAIN / EVAL HELPERS
# ─────────────────────────────────────────────────────────────────────────
def best_f1_threshold(labels, probs):
    prec, rec, thr = precision_recall_curve(labels, probs)
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    if len(thr) == 0:
        return 0.5, 0.0
    best_idx = np.nanargmax(f1s[:-1])
    return float(thr[best_idx]), float(f1s[best_idx])


def run_epoch(model, loader, criterion, optimizer, scaler, train_mode: bool):
    model.train(mode=train_mode)
    total_loss = 0.0
    all_labels, all_probs = [], []

    for X, y in loader:
        X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                logits = model(X)
                loss = criterion(logits, y)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * X.size(0)
        probs = torch.sigmoid(logits.float()).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(loader.dataset))
    return avg_loss, np.array(all_labels), np.array(all_probs)


def report_at_threshold(probs, labels, thr):
    preds = (probs >= thr).astype(int)
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)
    cm   = confusion_matrix(labels, preds)
    rep  = classification_report(labels, preds, target_names=["Normal", "Apnea"], zero_division=0)
    return dict(threshold=thr, accuracy=acc, precision=prec, recall=rec, f1_score=f1,
                tn=int(cm[0, 0]), fp=int(cm[0, 1]), fn=int(cm[1, 0]), tp=int(cm[1, 1])), rep, cm

# ─────────────────────────────────────────────────────────────────────────
# 12. TRAIN + EVALUATE ONE MODEL  (radius, loss_type)
# ─────────────────────────────────────────────────────────────────────────
def train_and_evaluate(radius, loss_type, train_pids, val_pids, test_pids):
    model_id = f"radius{radius}_{loss_type}"
    print(f"\n{'#'*70}\n#  MODEL: {model_id}\n{'#'*70}")

    train_ds = ApneaDataset(DATA_DIR, train_pids, CH_MEAN, CH_STD, radius, augment=AUGMENT, use_soft_labels=True)
    val_ds   = ApneaDataset(DATA_DIR, val_pids, CH_MEAN, CH_STD, radius, augment=False, use_soft_labels=False)
    test_ds  = ApneaDataset(DATA_DIR, test_pids, CH_MEAN, CH_STD, radius, augment=False, use_soft_labels=False)

    batch_size = 32   # radius=1 sequences (18000 samples) -- batch size fixed since only one radius is used
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)

    train_labels = train_ds.labels()
    n0, n1 = int((train_labels == 0).sum()), int((train_labels == 1).sum())
    pos_weight_tensor = torch.tensor([n0 / max(1, n1)], dtype=torch.float32).to(DEVICE)
    print(f"  Train windows: {len(train_ds):,} (normal={n0:,} apnea={n1:,}) | "
          f"input length: {(2*radius+1)*6000} samples")

    model = ResNet1D(NUM_CHANNELS, BASE_WIDTH, DROPOUT).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Model: SE-ResNet1D | Trainable params: {n_params:,}")
    criterion = get_criterion(loss_type, pos_weight_tensor)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    best_val_f1, best_state, best_threshold, no_improve = -1.0, None, 0.5, 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss, _, _ = run_epoch(model, train_loader, criterion, optimizer, scaler, True)
        val_loss, val_labels, val_probs = run_epoch(model, val_loader, criterion, optimizer, scaler, False)
        thr, f1 = best_f1_threshold(val_labels, val_probs)
        scheduler.step(f1)
        dt = time.time() - t0

        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                         "val_f1_tuned": f1, "threshold": thr, "lr": optimizer.param_groups[0]["lr"]})
        print(f"  [{model_id}] epoch {epoch:2d}/{EPOCHS} | train_loss={train_loss:.4f} "
              f"val_loss={val_loss:.4f} val_f1={f1:.4f} (thr={thr:.2f}) | {dt:.1f}s")

        if f1 > best_val_f1:
            best_val_f1, best_threshold = f1, thr
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"  [{model_id}] early stopping at epoch {epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # recompute val probs with the BEST weights (the training loop's last
    # epoch isn't necessarily the best one due to early stopping) -- needed
    # to tune the smoothed threshold correctly
    _, val_labels_final, val_probs_final = run_epoch(model, val_loader, criterion, optimizer, scaler, False)
    val_probs_smooth = smooth_probs_grouped(val_ds, val_probs_final, radius=SMOOTH_RADIUS)
    smooth_thr, smooth_val_f1 = best_f1_threshold(val_labels_final, val_probs_smooth)

    _, test_labels, test_probs = run_epoch(model, test_loader, criterion, optimizer, scaler, False)
    try:
        auc = roc_auc_score(test_labels, test_probs)
    except ValueError:
        auc = float("nan")

    metrics_05, rep_05, cm_05 = report_at_threshold(test_probs, test_labels, 0.5)
    metrics_tuned, rep_tuned, cm_tuned = report_at_threshold(test_probs, test_labels, best_threshold)

    # --- SMOOTHED @ val-tuned threshold (the headline "best" result) ---
    test_probs_smooth = smooth_probs_grouped(test_ds, test_probs, radius=SMOOTH_RADIUS)
    metrics_smooth, rep_smooth, cm_smooth = report_at_threshold(test_probs_smooth, test_labels, smooth_thr)

    # --- event-level (AHI-style) evaluation, using smoothed probs + smoothed threshold ---
    event_metrics = event_level_evaluation(test_ds, test_labels, test_probs_smooth, smooth_thr, min_overlap=1)

    print(f"\n  [{model_id}] TEST RESULTS: AUC={auc:.4f} | "
          f"F1@0.5={metrics_05['f1_score']:.4f} | F1@tuned={metrics_tuned['f1_score']:.4f} | "
          f"F1@smoothed_tuned={metrics_smooth['f1_score']:.4f} | Event F1={event_metrics['f1_score']:.4f}")

    fpr, tpr, _ = roc_curve(test_labels, test_probs) if not np.isnan(auc) else (None, None, None)

    return {
        "model_id": model_id, "radius": radius, "loss_type": loss_type,
        "auc": auc, "history": history,
        "metrics_05": metrics_05, "metrics_tuned": metrics_tuned, "metrics_smooth": metrics_smooth,
        "cm_05": cm_05, "cm_tuned": cm_tuned, "cm_smooth": cm_smooth,
        "rep_tuned": rep_tuned, "rep_smooth": rep_smooth,
        "smooth_threshold": smooth_thr, "smooth_val_f1": smooth_val_f1,
        "event_metrics": event_metrics,
        "fpr": fpr, "tpr": tpr,
        "test_labels": test_labels, "test_probs": test_probs, "test_probs_smooth": test_probs_smooth,
    }

# ─────────────────────────────────────────────────────────────────────────
# 13. RUN ALL 8 MODELS
# ─────────────────────────────────────────────────────────────────────────
all_results = []
results_csv_path = os.path.join(OUTPUT_DIR, "all_models_results.csv")
event_csv_path = os.path.join(OUTPUT_DIR, "event_level_all_models.csv")

for radius in RADII:
    for loss_type in LOSS_TYPES:
        result = train_and_evaluate(radius, loss_type, train_pids, val_pids, test_pids)
        all_results.append(result)

        # save incrementally so partial progress survives a timeout
        rows = []
        for r in all_results:
            for thr_label, m in [("default_0.5", r["metrics_05"]),
                                  ("val_tuned", r["metrics_tuned"]),
                                  ("smoothed_val_tuned", r["metrics_smooth"])]:
                rows.append({
                    "model_id": r["model_id"], "radius": r["radius"], "loss_type": r["loss_type"],
                    "threshold_type": thr_label, "auc": r["auc"], **m,
                })
        pd.DataFrame(rows).to_csv(results_csv_path, index=False)

        event_rows = [{"model_id": r["model_id"], "radius": r["radius"], "loss_type": r["loss_type"],
                        "smooth_threshold": r["smooth_threshold"], **r["event_metrics"]}
                       for r in all_results]
        pd.DataFrame(event_rows).to_csv(event_csv_path, index=False)

        torch.cuda.empty_cache()
        gc.collect()

print(f"\nAll 4 models trained. Results saved incrementally to {results_csv_path} and {event_csv_path}")

# ─────────────────────────────────────────────────────────────────────────
# 14. SAVE PER-MODEL CLASSIFICATION REPORTS  (tuned AND smoothed)
# ─────────────────────────────────────────────────────────────────────────
for r in all_results:
    with open(os.path.join(OUTPUT_DIR, f"classification_report_{r['model_id']}_tuned.txt"), "w") as fh:
        fh.write(r["rep_tuned"])
    with open(os.path.join(OUTPUT_DIR, f"classification_report_{r['model_id']}_smoothed.txt"), "w") as fh:
        fh.write(r["rep_smooth"])

# ─────────────────────────────────────────────────────────────────────────
# 15. COMPARISON PLOTS
# ─────────────────────────────────────────────────────────────────────────
loss_labels = {"bce": "BCE", "weighted_bce": "Weighted BCE",
               "focal": "Focal", "weighted_focal": "Weighted Focal"}
colors = {"bce": "#4C72B0", "weighted_bce": "#55A868",
          "focal": "#C44E52", "weighted_focal": "#8172B2"}

# --- 15a. Grouped bar chart: accuracy/precision/recall/F1 across all 8 models (tuned threshold) ---
fig, ax = plt.subplots(figsize=(16, 6))
model_order = [r["model_id"] for r in all_results]
metric_names = ["accuracy", "precision", "recall", "f1_score"]
x = np.arange(len(model_order))
width = 0.2
for i, metric in enumerate(metric_names):
    vals = [r["metrics_tuned"][metric] for r in all_results]
    ax.bar(x + i * width, vals, width, label=metric)
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(model_order, rotation=30, ha="right")
ax.set_ylabel("Score")
ax.set_title("Accuracy / Precision / Recall / F1 across all 4 models (val-tuned threshold)")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_bar_chart.png"), dpi=150)
plt.show()

# --- 15b. Overlaid ROC curves ---
fig, ax = plt.subplots(figsize=(9, 8))
for r in all_results:
    if r["fpr"] is None:
        continue
    ax.plot(r["fpr"], r["tpr"], "-", color=colors[r["loss_type"]],
            label=f"{r['model_id']} (AUC={r['auc']:.3f})")
ax.plot([0, 1], [0, 1], "k:", alpha=0.4)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — all 4 models")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_roc_curves.png"), dpi=150)
plt.show()

# --- 15c. Confusion matrix grid (1 row x 4 cols [loss]) ---
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for r in all_results:
    col = LOSS_TYPES.index(r["loss_type"])
    ax = axes[col]
    cm = r["cm_tuned"]
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(f"{loss_labels[r['loss_type']]}\nF1={r['metrics_tuned']['f1_score']:.3f}",
                 fontsize=10)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Normal", "Apnea"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Normal", "Apnea"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.suptitle("Confusion Matrices — all 4 models (val-tuned threshold, RAW probs)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_confusion_matrices.png"), dpi=150)
plt.show()

# --- 15c2. SMOOTHED confusion matrix grid (1 row x 4 cols [loss]) ---
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for r in all_results:
    col = LOSS_TYPES.index(r["loss_type"])
    ax = axes[col]
    cm = r["cm_smooth"]
    im = ax.imshow(cm, cmap="Greens")
    ax.set_title(f"{loss_labels[r['loss_type']]}\n"
                 f"F1(smoothed)={r['metrics_smooth']['f1_score']:.3f}",
                 fontsize=10)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Normal", "Apnea"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Normal", "Apnea"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.suptitle("Confusion Matrices — all 4 models (SMOOTHED @ val-tuned threshold)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_confusion_matrices_smoothed.png"), dpi=150)
plt.show()

# --- 15c3. F1 comparison across threshold types (default / tuned / smoothed / event-level) ---
fig, ax = plt.subplots(figsize=(16, 6))
f1_default = [r["metrics_05"]["f1_score"] for r in all_results]
f1_tuned   = [r["metrics_tuned"]["f1_score"] for r in all_results]
f1_smooth  = [r["metrics_smooth"]["f1_score"] for r in all_results]
f1_event   = [r["event_metrics"]["f1_score"] for r in all_results]
x = np.arange(len(model_order))
width = 0.2
ax.bar(x - 1.5 * width, f1_default, width, label="F1 @ default 0.5")
ax.bar(x - 0.5 * width, f1_tuned, width, label="F1 @ val-tuned")
ax.bar(x + 0.5 * width, f1_smooth, width, label="F1 @ smoothed + val-tuned")
ax.bar(x + 1.5 * width, f1_event, width, label="Event-level F1")
ax.set_xticks(x)
ax.set_xticklabels(model_order, rotation=30, ha="right")
ax.set_ylabel("F1 Score")
ax.set_title("F1 Score across threshold strategies — all 4 models")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_f1_by_threshold_strategy.png"), dpi=150)
plt.show()

# --- 15d. Training curves (single plot, 4 lines [loss]) ---
fig, ax = plt.subplots(figsize=(10, 6))
for r in all_results:
    hist_df = pd.DataFrame(r["history"])
    ax.plot(hist_df["epoch"], hist_df["val_f1_tuned"], label=loss_labels[r["loss_type"]],
            color=colors[r["loss_type"]])
ax.set_title("Validation F1 by epoch — all 4 models")
ax.set_xlabel("Epoch")
ax.set_ylabel("Val F1 (tuned threshold)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_training_curves.png"), dpi=150)
plt.show()

# ─────────────────────────────────────────────────────────────────────────
# 16. FINAL SUMMARY TABLE  (includes smoothed + event-level results)
# ─────────────────────────────────────────────────────────────────────────
summary_rows = []
for r in all_results:
    summary_rows.append({
        "model_id": r["model_id"], "radius": r["radius"], "loss_type": loss_labels[r["loss_type"]],
        "AUC": round(r["auc"], 4),
        "Accuracy_tuned": round(r["metrics_tuned"]["accuracy"], 4),
        "Precision_tuned": round(r["metrics_tuned"]["precision"], 4),
        "Recall_tuned": round(r["metrics_tuned"]["recall"], 4),
        "F1_tuned": round(r["metrics_tuned"]["f1_score"], 4),
        "Threshold_tuned": round(r["metrics_tuned"]["threshold"], 3),
        "Accuracy_smoothed": round(r["metrics_smooth"]["accuracy"], 4),
        "Precision_smoothed": round(r["metrics_smooth"]["precision"], 4),
        "Recall_smoothed": round(r["metrics_smooth"]["recall"], 4),
        "F1_smoothed": round(r["metrics_smooth"]["f1_score"], 4),
        "Threshold_smoothed": round(r["smooth_threshold"], 3),
        "Event_Precision": round(r["event_metrics"]["precision"], 4),
        "Event_Recall": round(r["event_metrics"]["recall"], 4),
        "Event_F1": round(r["event_metrics"]["f1_score"], 4),
    })
summary_df = pd.DataFrame(summary_rows).sort_values("F1_smoothed", ascending=False)
summary_df.to_csv(os.path.join(OUTPUT_DIR, "summary_table.csv"), index=False)

print("\n" + "=" * 130)
print("  FINAL SUMMARY — all 4 models, sorted by F1 (SMOOTHED @ val-tuned threshold)")
print("=" * 130)
print(summary_df.to_string(index=False))

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(" - all_models_results.csv                        (every model x default/tuned/smoothed thresholds, full metrics)")
print(" - event_level_all_models.csv                     (AHI-style event precision/recall/F1 per model)")
print(" - summary_table.csv                              (clean 8-row summary: tuned + smoothed + event-level side by side)")
print(" - classification_report_<model_id>_tuned.txt     (per model)")
print(" - classification_report_<model_id>_smoothed.txt  (per model)")
print(" - comparison_bar_chart.png                       (accuracy/precision/recall/F1, val-tuned threshold)")
print(" - comparison_roc_curves.png")
print(" - comparison_confusion_matrices.png              (val-tuned threshold, raw probs)")
print(" - comparison_confusion_matrices_smoothed.png     (SMOOTHED @ val-tuned threshold)")
print(" - comparison_f1_by_threshold_strategy.png        (F1 across default/tuned/smoothed/event-level)")
print(" - comparison_training_curves.png")
print(f"\nNOTE: at inference time, replicate BOTH the context-window concatenation "
      f"AND the temporal smoothing (radius={SMOOTH_RADIUS}) to reproduce the 'smoothed' metrics above.")

In [ ]:
"""
=============================================================================
 SLEEP APNEA DETECTION — SE-RESNET1D  (radius=2, weighted focal loss only,
 fixed threshold = 0.5)
=============================================================================
 This is the 4-model ablation script (radius=1, BCE/weighted BCE/focal/
 weighted focal) collapsed down to a SINGLE model:
   - Context radius CHANGED from 1 -> 2  (target window + 2 neighbors each
     side = 5 windows total = 150s / 30000-sample input, instead of 90s)
   - Loss function FIXED to weighted focal loss only (no BCE/weighted-BCE/
     plain-focal comparison -- this is the only loss used)
   - Decision threshold FIXED at 0.5 everywhere (default eval, smoothed
     eval, and event-level eval) -- no val-tuned threshold search anywhere
     in this script

 EVERYTHING ELSE is unchanged from the ablation study: same SE-ResNet1D
 architecture, same 60/15/15 exact-count patient split, same SpO2/pleth
 cleaning, same boundary label smoothing, same augmentation, same
 normalization, same temporal smoothing mechanism, same AHI-style
 event-level evaluation -- just applied to one model instead of eight.

 ARCHITECTURE: 1D ResNet with SQUEEZE-AND-EXCITATION (SE) attention blocks
 after every residual block. SE blocks learn a per-channel attention gate
 so the network can up/down-weight which of the 8 physiological channels
 (flow, chest, abd, SpO2, pleth, ECG, pulse, flow2) matter most for a given
 window. Single-logit output head (sigmoid-based), matching the weighted
 focal loss's binary formulation.

 LOSS -- WEIGHTED FOCAL LOSS ONLY:
   Weighted Focal = (1 - p_t)^gamma * BCEWithLogits(pos_weight=...)
 Combines class-imbalance correction (pos_weight, same idea as weighted
 BCE) with focal loss's hard-example focusing ((1-p_t)^gamma down-weights
 already-confident predictions).

 SPLIT: patient-level, stratified by apnea ratio (interleaved across bins
 so the split hits exact counts with no leftover), 60 train / 15 test /
 15 validation patients -- exact absolute counts, matching the ~90 usable
 patients left after skipping 10 from the ~100-patient dataset.

 Input  : X_SN{n}.npy -> (num_windows, 8, 6000) float16
          y_SN{n}.npy -> (num_windows,)         int8  (0=normal, 1=apnea)
=============================================================================
"""

import os
import gc
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

warnings.filterwarnings("ignore")
import kagglehub

# ─────────────────────────────────────────────────────────────────────────
# 0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ─────────────────────────────────────────────────────────────────────────
# 0b. GPU / KERNEL-COMPATIBILITY CHECK  (P100 = compute capability 6.0)
# ─────────────────────────────────────────────────────────────────────────
def _gpu_kernels_usable():
    if not torch.cuda.is_available():
        return False
    try:
        x = torch.randn(2, 4, 16, device="cuda")
        conv = nn.Conv1d(4, 4, 3, padding=1).cuda()
        _ = conv(x)
        return True
    except RuntimeError as e:
        print("\n⚠️  GPU kernel check failed:", str(e).splitlines()[0])
        print("⚠️  Falling back to CPU. Switch accelerator to T4/L4 to use GPU.")
        return False

_USE_CUDA = _gpu_kernels_usable()

# ─────────────────────────────────────────────────────────────────────────
# 1. DOWNLOAD DATASET
# ─────────────────────────────────────────────────────────────────────────
DATA_DIR = kagglehub.dataset_download(
    "nishantomhare/detection-of-disordered-breathing-dataset2"
)
print("Dataset path:", DATA_DIR)

def locate_npy_root(root):
    for dirpath, _, filenames in os.walk(root):
        if any(f.startswith("X_SN") and f.endswith(".npy") for f in filenames):
            return dirpath
    return root

DATA_DIR = locate_npy_root(DATA_DIR)
print("Resolved npy root:", DATA_DIR)

OUTPUT_DIR = "/kaggle/working/resnet1d_se_radius2_weightedfocal_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────
# 2. CONFIG
# ─────────────────────────────────────────────────────────────────────────
SKIP_PATIENTS = {
    "SN1", "SN28", "SN14", "SN13", "SN17", "SN18",
    "SN19", "SN20", "SN27", "SN89"
}

# Exact ABSOLUTE patient counts: out of the ~90 usable patients, 60 go to
# train, 15 to test, 15 to validation.
TRAIN_N = 60
TEST_N  = 15
VAL_N   = 15
N_STRAT_BINS = 5

FIXED_CHANNEL_KEYS = ["flow", "chest", "abd", "spo2", "pleth", "ecg", "pulse", "flow2"]
SPO2_IDX  = FIXED_CHANNEL_KEYS.index("spo2")
PLETH_IDX = FIXED_CHANNEL_KEYS.index("pleth")
FS = 200

NUM_CHANNELS  = 8
CONTEXT_RADIUS = 2          # CHANGED from 1 -> 2: target window + 2 neighbors each side = 150s input
FIXED_THRESHOLD = 0.5       # used everywhere: default eval, smoothed eval, event-level eval
SMOOTH_RADIUS = 2           # temporal smoothing window (in # of windows), each side

EPOCHS    = 16
PATIENCE  = 6
GRAD_CLIP = 3.0
NUM_WORKERS = 2
BATCH_SIZE = 16              # radius=2 sequences (30000 samples) are long -> smaller batch than radius=1's 32

# fixed architecture/optimizer hyperparameters (same values as the ablation study)
BASE_WIDTH   = 32
DROPOUT      = 0.4
LR           = 7e-4
WEIGHT_DECAY = 8e-5
FOCAL_GAMMA  = 2.0

AUGMENT           = True
NOISE_STD_FRAC    = 0.03
SCALE_JITTER      = 0.10
MAX_TIME_SHIFT    = 100
CHANNEL_DROPOUT_P = 0.05

NORM_SAMPLE_WINDOWS = 8000
BOUNDARY_SOFT_CONF  = 0.15
SPO2_VALID_RANGE    = (50.0, 100.0)
FLATLINE_MIN_SEC    = 1.0
FLATLINE_MIN_RUN    = int(FLATLINE_MIN_SEC * FS)

DEVICE = torch.device("cuda" if _USE_CUDA else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| capability:", torch.cuda.get_device_capability(0),
          "| usable:", _USE_CUDA)

# ─────────────────────────────────────────────────────────────────────────
# 3. DISCOVER PATIENTS
# ─────────────────────────────────────────────────────────────────────────
all_x_files = sorted(f for f in os.listdir(DATA_DIR)
                      if f.startswith("X_SN") and f.endswith(".npy"))
all_pids = [f[len("X_"):-len(".npy")] for f in all_x_files]
pids = [p for p in all_pids if p not in SKIP_PATIENTS
        and os.path.exists(os.path.join(DATA_DIR, f"y_{p}.npy"))]
print(f"\nTotal X files: {len(all_x_files)} | Skipped: {sorted(SKIP_PATIENTS & set(all_pids))} "
      f"| Usable patients: {len(pids)}")

if len(pids) < 5:
    raise RuntimeError("Not enough usable patients to train/val/test split.")

# ─────────────────────────────────────────────────────────────────────────
# 4. STRATIFIED PATIENT-LEVEL SPLIT -- exact counts: 60 train / 15 test / 15 val
# ─────────────────────────────────────────────────────────────────────────
def patient_apnea_ratio(pid):
    y = np.load(os.path.join(DATA_DIR, f"y_{pid}.npy"), mmap_mode="r")
    n = len(y)
    apnea = int(np.sum(y))
    del y
    return apnea / max(1, n), n, apnea

patient_stats = {pid: patient_apnea_ratio(pid) for pid in pids}
ratios = np.array([patient_stats[p][0] for p in pids])

expected_total = TRAIN_N + TEST_N + VAL_N
if len(pids) != expected_total:
    print(f"\n⚠️  WARNING: usable patient count ({len(pids)}) does not match "
          f"TRAIN_N+TEST_N+VAL_N ({expected_total}). The split below will still "
          f"use exactly {TRAIN_N}/{TEST_N}/{VAL_N} patients where possible.")

rng = np.random.default_rng(SEED)
order = np.argsort(ratios)
sorted_pids = np.array(pids)[order]
bin_edges = np.linspace(0, len(sorted_pids), N_STRAT_BINS + 1).astype(int)
bins = []
for b in range(N_STRAT_BINS):
    bucket = list(sorted_pids[bin_edges[b]:bin_edges[b + 1]])
    rng.shuffle(bucket)
    bins.append(bucket)

from itertools import zip_longest
combined = []
for group in zip_longest(*bins, fillvalue=None):
    combined.extend(pid for pid in group if pid is not None)

train_pids = combined[:TRAIN_N]
test_pids  = combined[TRAIN_N:TRAIN_N + TEST_N]
val_pids   = combined[TRAIN_N + TEST_N:TRAIN_N + TEST_N + VAL_N]
unused_pids = combined[TRAIN_N + TEST_N + VAL_N:]

print(f"\nTrain  patients ({len(train_pids)}): {sorted(train_pids)}")
print(f"Test   patients ({len(test_pids)}): {sorted(test_pids)}")
print(f"Val    patients ({len(val_pids)}): {sorted(val_pids)}")
if unused_pids:
    print(f"UNUSED patients ({len(unused_pids)}): {sorted(unused_pids)}")

# ─────────────────────────────────────────────────────────────────────────
# 5. PER-CHANNEL NORMALIZATION STATS (radius-independent, from TRAIN only)
# ─────────────────────────────────────────────────────────────────────────
def estimate_norm_stats(patient_ids, n_sample_windows):
    sums, sqsums, count = np.zeros(NUM_CHANNELS, np.float64), np.zeros(NUM_CHANNELS, np.float64), 0
    per_patient_budget = max(1, n_sample_windows // max(1, len(patient_ids)))
    for pid in patient_ids:
        X = np.load(os.path.join(DATA_DIR, f"X_{pid}.npy"), mmap_mode="r")
        n = X.shape[0]
        if n == 0:
            continue
        k = min(per_patient_budget, n)
        idx = np.random.default_rng(SEED).choice(n, size=k, replace=False)
        sample = X[idx].astype(np.float64)
        sums   += sample.sum(axis=(0, 2))
        sqsums += (sample ** 2).sum(axis=(0, 2))
        count  += k * sample.shape[2]
        del X, sample
        gc.collect()
    mean = sums / count
    var  = sqsums / count - mean ** 2
    std  = np.sqrt(np.clip(var, 1e-8, None))
    return mean.astype(np.float32), std.astype(np.float32)

print("\nEstimating per-channel normalization stats from train set...")
CH_MEAN, CH_STD = estimate_norm_stats(train_pids, NORM_SAMPLE_WINDOWS)
np.save(os.path.join(OUTPUT_DIR, "channel_mean.npy"), CH_MEAN)
np.save(os.path.join(OUTPUT_DIR, "channel_std.npy"), CH_STD)

# ─────────────────────────────────────────────────────────────────────────
# 6. SIGNAL CLEANING (SpO2 / pleth)
# ─────────────────────────────────────────────────────────────────────────
def flatline_mask(sig, min_run):
    n = len(sig)
    if n < 2:
        return np.zeros(n, dtype=bool)
    same = np.concatenate(([False], sig[1:] == sig[:-1]))
    change_points = np.where(~same)[0]
    change_points = np.concatenate(([0], change_points, [n]))
    mask = np.zeros(n, dtype=bool)
    for i in range(len(change_points) - 1):
        s, e = change_points[i], change_points[i + 1]
        if e - s >= min_run:
            mask[s:e] = True
    return mask


def clean_signal(sig, valid_range=None, flatline_min_run=FLATLINE_MIN_RUN):
    sig = sig.astype(np.float32).copy()
    bad = np.zeros(len(sig), dtype=bool)
    if valid_range is not None:
        lo, hi = valid_range
        bad |= (sig < lo) | (sig > hi)
    bad |= flatline_mask(sig, flatline_min_run)
    if bad.any() and not bad.all():
        good_idx = np.where(~bad)[0]
        bad_idx = np.where(bad)[0]
        sig[bad_idx] = np.interp(bad_idx, good_idx, sig[good_idx])
    return sig


def clean_spo2_pleth(window_8ch):
    window_8ch = window_8ch.copy()
    window_8ch[SPO2_IDX]  = clean_signal(window_8ch[SPO2_IDX], valid_range=SPO2_VALID_RANGE)
    window_8ch[PLETH_IDX] = clean_signal(window_8ch[PLETH_IDX], valid_range=None)
    return window_8ch

# ─────────────────────────────────────────────────────────────────────────
# 7. BOUNDARY LABEL SMOOTHING (train only)
# ─────────────────────────────────────────────────────────────────────────
def compute_boundary_soft_labels(y, soft_conf=BOUNDARY_SOFT_CONF):
    n = len(y)
    soft = y.astype(np.float32).copy()
    for i in range(n):
        left_diff  = (i > 0)     and (y[i - 1] != y[i])
        right_diff = (i < n - 1) and (y[i + 1] != y[i])
        if left_diff or right_diff:
            soft[i] = (1.0 - soft_conf) if y[i] == 1 else soft_conf
    return soft

# ─────────────────────────────────────────────────────────────────────────
# 8. DATASET  (context radius = 2, fixed)
# ─────────────────────────────────────────────────────────────────────────
class ApneaDataset(Dataset):
    def __init__(self, data_dir, patient_ids, mean, std,
                 context_radius=CONTEXT_RADIUS, augment=False, use_soft_labels=False):
        self.data_dir = data_dir
        self._cache = {}
        self.index = []
        self.mean = mean.reshape(NUM_CHANNELS, 1)
        self.std  = std.reshape(NUM_CHANNELS, 1)
        self.context_radius = context_radius
        self.augment = augment
        self.use_soft_labels = use_soft_labels
        self._soft_label_cache = {}

        for pid in patient_ids:
            xp = os.path.join(data_dir, f"X_{pid}.npy")
            yp = os.path.join(data_dir, f"y_{pid}.npy")
            if not (os.path.exists(xp) and os.path.exists(yp)):
                continue
            y = np.load(yp, mmap_mode="r")
            n = len(y)
            self.index.extend([(pid, i, n) for i in range(n)])
            if use_soft_labels:
                self._soft_label_cache[pid] = compute_boundary_soft_labels(np.asarray(y))
            del y

    def _arrays(self, pid):
        if pid not in self._cache:
            self._cache[pid] = (
                np.load(os.path.join(self.data_dir, f"X_{pid}.npy"), mmap_mode="r"),
                np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r"),
            )
        return self._cache[pid]

    def __len__(self):
        return len(self.index)

    def labels(self):
        out = np.zeros(len(self.index), dtype=np.int8)
        cache = {}
        for i, (pid, li, _) in enumerate(self.index):
            if pid not in cache:
                cache[pid] = np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r")
            out[i] = cache[pid][li]
        return out

    def __getitem__(self, idx):
        pid, local_idx, n = self.index[idx]
        X_arr, y_arr = self._arrays(pid)

        r = self.context_radius
        neighbor_idxs = np.clip(np.arange(local_idx - r, local_idx + r + 1), 0, n - 1)
        segs = [X_arr[j].astype(np.float32) for j in neighbor_idxs]
        x = np.concatenate(segs, axis=1)

        x = clean_spo2_pleth(x)
        x = (x - self.mean) / self.std

        if self.augment:
            if NOISE_STD_FRAC > 0:
                x = x + np.random.randn(*x.shape).astype(np.float32) * NOISE_STD_FRAC
            if SCALE_JITTER > 0:
                scale = 1.0 + (np.random.rand(NUM_CHANNELS, 1).astype(np.float32) * 2 - 1) * SCALE_JITTER
                x = x * scale
            if MAX_TIME_SHIFT > 0:
                shift = np.random.randint(-MAX_TIME_SHIFT, MAX_TIME_SHIFT + 1)
                if shift != 0:
                    x = np.roll(x, shift, axis=1)
            if CHANNEL_DROPOUT_P > 0 and np.random.rand() < CHANNEL_DROPOUT_P:
                ch = np.random.randint(0, NUM_CHANNELS)
                x[ch, :] = 0.0

        if self.use_soft_labels:
            y = float(self._soft_label_cache[pid][local_idx])
        else:
            y = float(y_arr[local_idx])

        return torch.from_numpy(x.copy()), torch.tensor(y, dtype=torch.float32)


train_ds = ApneaDataset(DATA_DIR, train_pids, CH_MEAN, CH_STD, augment=AUGMENT, use_soft_labels=True)
val_ds   = ApneaDataset(DATA_DIR, val_pids, CH_MEAN, CH_STD, augment=False, use_soft_labels=False)
test_ds  = ApneaDataset(DATA_DIR, test_pids, CH_MEAN, CH_STD, augment=False, use_soft_labels=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

train_labels = train_ds.labels()
n0, n1 = int((train_labels == 0).sum()), int((train_labels == 1).sum())
pos_weight_tensor = torch.tensor([n0 / max(1, n1)], dtype=torch.float32).to(DEVICE)
print(f"\nTrain windows: {len(train_ds):,} (normal={n0:,} apnea={n1:,}) | "
      f"input length: {(2*CONTEXT_RADIUS+1)*6000} samples ({(2*CONTEXT_RADIUS+1)*30}s)")
print(f"Val   windows: {len(val_ds):,}")
print(f"Test  windows: {len(test_ds):,}")
print(f"pos_weight for weighted focal loss: {pos_weight_tensor.item():.3f}")

# ─────────────────────────────────────────────────────────────────────────
# 9. 1D RESNET + SE-ATTENTION ARCHITECTURE
# ─────────────────────────────────────────────────────────────────────────
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(4, channels // reduction)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels), nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1)
        return x * s


class SEBasicBlock1D(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, downsample=None, k=7):
        super().__init__()
        pad = k // 2
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=k, stride=stride, padding=pad, bias=False)
        self.bn1   = nn.BatchNorm1d(planes)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=k, stride=1, padding=pad, bias=False)
        self.bn2   = nn.BatchNorm1d(planes)
        self.se    = SEBlock1D(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out = out + identity
        return self.relu(out)


class ResNet1D(nn.Module):
    """SE-ResNet1D with a single-logit output head."""
    def __init__(self, in_channels=8, base_width=32, dropout=0.4, layers=(2, 2, 3, 2)):
        super().__init__()
        self.in_planes = base_width
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, base_width, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(base_width),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )
        self.layer1 = self._make_layer(base_width,     layers[0], stride=1)
        self.layer2 = self._make_layer(base_width * 2, layers[1], stride=2)
        self.layer3 = self._make_layer(base_width * 4, layers[2], stride=2)
        self.layer4 = self._make_layer(base_width * 8, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(base_width * 8, 1)   # single logit

        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, planes, num_blocks, stride):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes),
            )
        layers = [SEBasicBlock1D(self.in_planes, planes, stride, downsample)]
        self.in_planes = planes
        for _ in range(1, num_blocks):
            layers.append(SEBasicBlock1D(self.in_planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        x = self.dropout(x)
        return self.fc(x).squeeze(-1)   # (B,) single logit per window


model = ResNet1D(NUM_CHANNELS, BASE_WIDTH, DROPOUT).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel: SE-ResNet1D | Trainable params: {n_params:,}")

# ─────────────────────────────────────────────────────────────────────────
# 10. LOSS -- WEIGHTED FOCAL LOSS ONLY
# ─────────────────────────────────────────────────────────────────────────
class FocalLossBinary(nn.Module):
    """Binary focal loss with pos_weight baked into the BCE term
    ("weighted focal loss")."""
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction="none")
        p = torch.sigmoid(logits)
        pt = p * targets + (1 - p) * (1 - targets)
        focal_weight = (1 - pt).clamp(min=1e-6) ** self.gamma
        return (focal_weight * bce).mean()


criterion = FocalLossBinary(gamma=FOCAL_GAMMA, pos_weight=pos_weight_tensor)
print(f"Loss: Weighted Focal Loss (gamma={FOCAL_GAMMA}, pos_weight={pos_weight_tensor.item():.3f})")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

# ─────────────────────────────────────────────────────────────────────────
# 11. TRAIN / EVAL HELPERS
# ─────────────────────────────────────────────────────────────────────────
def run_epoch(loader, train_mode: bool):
    model.train(mode=train_mode)
    total_loss = 0.0
    all_labels, all_probs = [], []

    for X, y in loader:
        X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                logits = model(X)
                loss = criterion(logits, y)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * X.size(0)
        probs = torch.sigmoid(logits.float()).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(loader.dataset))
    return avg_loss, np.array(all_labels), np.array(all_probs)


def report_at_threshold(probs, labels, thr):
    preds = (probs >= thr).astype(int)
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)
    cm   = confusion_matrix(labels, preds)
    rep  = classification_report(labels, preds, target_names=["Normal", "Apnea"], zero_division=0)
    return dict(threshold=thr, accuracy=acc, precision=prec, recall=rec, f1_score=f1,
                tn=int(cm[0, 0]), fp=int(cm[0, 1]), fn=int(cm[1, 0]), tp=int(cm[1, 1])), rep, cm


def smooth_probs_grouped(dataset, probs, radius=SMOOTH_RADIUS):
    """Moving-average smoothing of per-window probabilities, computed
    separately within each patient's (already time-ordered) window
    sequence -- never smooths across a patient boundary."""
    probs = np.asarray(probs)
    smoothed = np.zeros_like(probs)
    pids_seq = [p for p, _, _ in dataset.index]
    n = len(pids_seq)
    i = 0
    kernel = np.ones(2 * radius + 1) / (2 * radius + 1)
    while i < n:
        j = i
        while j < n and pids_seq[j] == pids_seq[i]:
            j += 1
        seg = probs[i:j]
        padded = np.pad(seg, (radius, radius), mode="edge")
        smoothed[i:j] = np.convolve(padded, kernel, mode="valid")
        i = j
    return smoothed


def extract_events(binary_seq):
    events, start = [], None
    for i, v in enumerate(binary_seq):
        if v == 1 and start is None:
            start = i
        elif v == 0 and start is not None:
            events.append((start, i - 1)); start = None
    if start is not None:
        events.append((start, len(binary_seq) - 1))
    return events


def match_events(pred_events, true_events, min_overlap=1):
    matched_true = set()
    pred_hit = [False] * len(pred_events)
    for pi, (ps, pe) in enumerate(pred_events):
        for ti, (ts, te) in enumerate(true_events):
            overlap = max(0, min(pe, te) - max(ps, ts) + 1)
            if overlap >= min_overlap:
                matched_true.add(ti)
                pred_hit[pi] = True
    tp_true = len(matched_true)
    tp_pred = sum(pred_hit)
    fp = len(pred_events) - tp_pred
    fn = len(true_events) - tp_true
    return tp_true, tp_pred, fp, fn, len(pred_events), len(true_events)


def event_level_evaluation(dataset, labels, probs, threshold, min_overlap=1):
    preds = (probs >= threshold).astype(int)
    pids_seq = [p for p, _, _ in dataset.index]
    n = len(pids_seq)
    agg = dict(tp_true=0, tp_pred=0, fp=0, fn=0, n_pred=0, n_true=0)
    i = 0
    while i < n:
        j = i
        while j < n and pids_seq[j] == pids_seq[i]:
            j += 1
        pe = extract_events(preds[i:j])
        te = extract_events(labels[i:j])
        tp_true, tp_pred, fp, fn, npe, nte = match_events(pe, te, min_overlap)
        agg["tp_true"] += tp_true; agg["tp_pred"] += tp_pred
        agg["fp"] += fp; agg["fn"] += fn
        agg["n_pred"] += npe; agg["n_true"] += nte
        i = j
    precision = agg["tp_pred"] / agg["n_pred"] if agg["n_pred"] > 0 else 0.0
    recall    = agg["tp_true"] / agg["n_true"] if agg["n_true"] > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    agg["precision"], agg["recall"], agg["f1_score"] = precision, recall, f1
    return agg

# ─────────────────────────────────────────────────────────────────────────
# 12. TRAINING LOOP  (early stopping on val F1 @ FIXED THRESHOLD 0.5)
# ─────────────────────────────────────────────────────────────────────────
best_val_f1, best_state, no_improve = -1.0, None, 0
history = []
ckpt_path = os.path.join(OUTPUT_DIR, "best_model.pt")

print("\n" + "=" * 65)
print("  TRAINING")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, _, _ = run_epoch(train_loader, True)
    val_loss, val_labels, val_probs = run_epoch(val_loader, False)

    val_preds = (val_probs >= FIXED_THRESHOLD).astype(int)
    val_f1 = f1_score(val_labels, val_preds, zero_division=0)
    scheduler.step(val_f1)
    dt = time.time() - t0

    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                     "val_f1@0.5": val_f1, "lr": optimizer.param_groups[0]["lr"]})
    print(f"Epoch {epoch:2d}/{EPOCHS} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
          f"val_f1@0.5={val_f1:.4f} | lr={optimizer.param_groups[0]['lr']:.2e} | {dt:.1f}s")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
        torch.save({"state_dict": best_state, "threshold": FIXED_THRESHOLD,
                    "context_radius": CONTEXT_RADIUS}, ckpt_path)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break

pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)
if best_state is not None:
    model.load_state_dict(best_state)
print(f"\nBest val F1 @ threshold=0.5: {best_val_f1:.4f}")

# ─────────────────────────────────────────────────────────────────────────
# 13. TEST SET EVALUATION  (all at fixed threshold = 0.5)
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  TEST SET EVALUATION  (threshold = 0.5 everywhere)")
print("=" * 65)

_, test_labels, test_probs = run_epoch(test_loader, False)
try:
    auc = roc_auc_score(test_labels, test_probs)
except ValueError:
    auc = float("nan")
print(f"\nROC-AUC (threshold-independent): {auc:.4f}")

metrics_raw, rep_raw, cm_raw = report_at_threshold(test_probs, test_labels, FIXED_THRESHOLD)
print(f"\n--- RAW @ threshold=0.5 ---")
print(f"Accuracy  : {metrics_raw['accuracy']:.4f}")
print(f"Precision : {metrics_raw['precision']:.4f}")
print(f"Recall    : {metrics_raw['recall']:.4f}")
print(f"F1 Score  : {metrics_raw['f1_score']:.4f}")
print(f"Confusion Matrix:\n{cm_raw}")
print(f"\nClassification Report:\n{rep_raw}")

# --- smoothed, still at threshold = 0.5 ---
test_probs_smooth = smooth_probs_grouped(test_ds, test_probs, radius=SMOOTH_RADIUS)
metrics_smooth, rep_smooth, cm_smooth = report_at_threshold(test_probs_smooth, test_labels, FIXED_THRESHOLD)
print(f"\n--- SMOOTHED @ threshold=0.5 ---")
print(f"Accuracy  : {metrics_smooth['accuracy']:.4f}")
print(f"Precision : {metrics_smooth['precision']:.4f}")
print(f"Recall    : {metrics_smooth['recall']:.4f}")
print(f"F1 Score  : {metrics_smooth['f1_score']:.4f}")
print(f"Confusion Matrix:\n{cm_smooth}")
print(f"\nClassification Report:\n{rep_smooth}")

pd.DataFrame([{"variant": "raw", **metrics_raw}, {"variant": "smoothed", **metrics_smooth}]).to_csv(
    os.path.join(OUTPUT_DIR, "test_metrics.csv"), index=False)
with open(os.path.join(OUTPUT_DIR, "classification_report_raw.txt"), "w") as fh:
    fh.write(rep_raw)
with open(os.path.join(OUTPUT_DIR, "classification_report_smoothed.txt"), "w") as fh:
    fh.write(rep_smooth)

# --- event-level, using smoothed probs @ threshold = 0.5 ---
print("\n" + "=" * 65)
print("  EVENT-LEVEL EVALUATION (AHI-style, smoothed probs, threshold=0.5)")
print("=" * 65)
event_metrics = event_level_evaluation(test_ds, test_labels, test_probs_smooth, FIXED_THRESHOLD, min_overlap=1)
print(f"\nGround-truth events : {event_metrics['n_true']}")
print(f"Predicted events    : {event_metrics['n_pred']}")
print(f"Event Precision     : {event_metrics['precision']:.4f}")
print(f"Event Recall        : {event_metrics['recall']:.4f}")
print(f"Event F1 Score      : {event_metrics['f1_score']:.4f}")
pd.DataFrame([event_metrics]).to_csv(os.path.join(OUTPUT_DIR, "event_level_metrics.csv"), index=False)

# ─────────────────────────────────────────────────────────────────────────
# 14. PLOTS
# ─────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im = axes[0].imshow(cm_smooth, cmap="Blues")
axes[0].set_title("Confusion Matrix (smoothed, thr=0.5)")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["Normal", "Apnea"])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(["Normal", "Apnea"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm_smooth[i, j]), ha="center", va="center",
                      color="white" if cm_smooth[i, j] > cm_smooth.max() / 2 else "black")
fig.colorbar(im, ax=axes[0])

if not np.isnan(auc):
    fpr, tpr, _ = roc_curve(test_labels, test_probs)
    axes[1].plot(fpr, tpr, label=f"AUC = {auc:.3f}")
    axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
    axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title("ROC Curve"); axes[1].legend()

hist_df = pd.DataFrame(history)
axes[2].plot(hist_df["epoch"], hist_df["train_loss"], label="Train Loss")
axes[2].plot(hist_df["epoch"], hist_df["val_f1@0.5"], label="Val F1 @ 0.5")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Value")
axes[2].set_title("Training Curves"); axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "evaluation_plots.png"), dpi=150)
plt.show()

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(" - best_model.pt                          (weights, threshold=0.5, context_radius=2)")
print(" - channel_mean.npy / channel_std.npy")
print(" - training_history.csv")
print(" - test_metrics.csv                       (raw @0.5 and smoothed @0.5)")
print(" - classification_report_raw.txt / classification_report_smoothed.txt")
print(" - event_level_metrics.csv")
print(" - evaluation_plots.png")

In [ ]:
"""
=============================================================================
 SLEEP APNEA DETECTION — SE-RESNET1D  (radius=3, weighted focal loss only,
 fixed threshold = 0.5)
=============================================================================
 This is the 4-model ablation script (radius=1, BCE/weighted BCE/focal/
 weighted focal) collapsed down to a SINGLE model:
   - Context radius CHANGED from 1 -> 3  (target window + 3 neighbors each
     side = 7 windows total = 210s / 42000-sample input, instead of 90s)
   - Loss function FIXED to weighted focal loss only (no BCE/weighted-BCE/
     plain-focal comparison -- this is the only loss used)
   - Decision threshold FIXED at 0.5 everywhere (default eval, smoothed
     eval, and event-level eval) -- no val-tuned threshold search anywhere
     in this script

 EVERYTHING ELSE is unchanged from the ablation study: same SE-ResNet1D
 architecture, same 60/15/15 exact-count patient split, same SpO2/pleth
 cleaning, same boundary label smoothing, same augmentation, same
 normalization, same temporal smoothing mechanism, same AHI-style
 event-level evaluation -- just applied to one model instead of eight.

 ARCHITECTURE: 1D ResNet with SQUEEZE-AND-EXCITATION (SE) attention blocks
 after every residual block. SE blocks learn a per-channel attention gate
 so the network can up/down-weight which of the 8 physiological channels
 (flow, chest, abd, SpO2, pleth, ECG, pulse, flow2) matter most for a given
 window. Single-logit output head (sigmoid-based), matching the weighted
 focal loss's binary formulation.

 LOSS -- WEIGHTED FOCAL LOSS ONLY:
   Weighted Focal = (1 - p_t)^gamma * BCEWithLogits(pos_weight=...)
 Combines class-imbalance correction (pos_weight, same idea as weighted
 BCE) with focal loss's hard-example focusing ((1-p_t)^gamma down-weights
 already-confident predictions).

 SPLIT: patient-level, stratified by apnea ratio (interleaved across bins
 so the split hits exact counts with no leftover), 60 train / 15 test /
 15 validation patients -- exact absolute counts, matching the ~90 usable
 patients left after skipping 10 from the ~100-patient dataset.

 Input  : X_SN{n}.npy -> (num_windows, 8, 6000) float16
          y_SN{n}.npy -> (num_windows,)         int8  (0=normal, 1=apnea)
=============================================================================
"""

import os
import gc
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

warnings.filterwarnings("ignore")
import kagglehub

# ─────────────────────────────────────────────────────────────────────────
# 0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ─────────────────────────────────────────────────────────────────────────
# 0b. GPU / KERNEL-COMPATIBILITY CHECK  (P100 = compute capability 6.0)
# ─────────────────────────────────────────────────────────────────────────
def _gpu_kernels_usable():
    if not torch.cuda.is_available():
        return False
    try:
        x = torch.randn(2, 4, 16, device="cuda")
        conv = nn.Conv1d(4, 4, 3, padding=1).cuda()
        _ = conv(x)
        return True
    except RuntimeError as e:
        print("\n⚠️  GPU kernel check failed:", str(e).splitlines()[0])
        print("⚠️  Falling back to CPU. Switch accelerator to T4/L4 to use GPU.")
        return False

_USE_CUDA = _gpu_kernels_usable()

# ─────────────────────────────────────────────────────────────────────────
# 1. DOWNLOAD DATASET
# ─────────────────────────────────────────────────────────────────────────
DATA_DIR = kagglehub.dataset_download(
    "nishantomhare/detection-of-disordered-breathing-dataset2"
)
print("Dataset path:", DATA_DIR)

def locate_npy_root(root):
    for dirpath, _, filenames in os.walk(root):
        if any(f.startswith("X_SN") and f.endswith(".npy") for f in filenames):
            return dirpath
    return root

DATA_DIR = locate_npy_root(DATA_DIR)
print("Resolved npy root:", DATA_DIR)

OUTPUT_DIR = "/kaggle/working/resnet1d_se_radius2_weightedfocal_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────
# 2. CONFIG
# ─────────────────────────────────────────────────────────────────────────
SKIP_PATIENTS = {
    "SN1", "SN28", "SN14", "SN13", "SN17", "SN18",
    "SN19", "SN20", "SN27", "SN89"
}

# Exact ABSOLUTE patient counts: out of the ~90 usable patients, 60 go to
# train, 15 to test, 15 to validation.
TRAIN_N = 60
TEST_N  = 15
VAL_N   = 15
N_STRAT_BINS = 5

FIXED_CHANNEL_KEYS = ["flow", "chest", "abd", "spo2", "pleth", "ecg", "pulse", "flow2"]
SPO2_IDX  = FIXED_CHANNEL_KEYS.index("spo2")
PLETH_IDX = FIXED_CHANNEL_KEYS.index("pleth")
FS = 200

NUM_CHANNELS  = 8
CONTEXT_RADIUS = 3        # CHANGED from 1 -> 3: target window + 3 neighbors each side = 210s input
FIXED_THRESHOLD = 0.5       # used everywhere: default eval, smoothed eval, event-level eval
SMOOTH_RADIUS = 2           # temporal smoothing window (in # of windows), each side

EPOCHS    = 16
PATIENCE  = 6
GRAD_CLIP = 3.0
NUM_WORKERS = 2
BATCH_SIZE = 16              # radius=3 sequences (42000 samples) are long -> smaller batch than radius=1's 32

# fixed architecture/optimizer hyperparameters (same values as the ablation study)
BASE_WIDTH   = 32
DROPOUT      = 0.4
LR           = 7e-4
WEIGHT_DECAY = 8e-5
FOCAL_GAMMA  = 2.0

AUGMENT           = True
NOISE_STD_FRAC    = 0.03
SCALE_JITTER      = 0.10
MAX_TIME_SHIFT    = 100
CHANNEL_DROPOUT_P = 0.05

NORM_SAMPLE_WINDOWS = 8000
BOUNDARY_SOFT_CONF  = 0.15
SPO2_VALID_RANGE    = (50.0, 100.0)
FLATLINE_MIN_SEC    = 1.0
FLATLINE_MIN_RUN    = int(FLATLINE_MIN_SEC * FS)

DEVICE = torch.device("cuda" if _USE_CUDA else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| capability:", torch.cuda.get_device_capability(0),
          "| usable:", _USE_CUDA)

# ─────────────────────────────────────────────────────────────────────────
# 3. DISCOVER PATIENTS
# ─────────────────────────────────────────────────────────────────────────
all_x_files = sorted(f for f in os.listdir(DATA_DIR)
                      if f.startswith("X_SN") and f.endswith(".npy"))
all_pids = [f[len("X_"):-len(".npy")] for f in all_x_files]
pids = [p for p in all_pids if p not in SKIP_PATIENTS
        and os.path.exists(os.path.join(DATA_DIR, f"y_{p}.npy"))]
print(f"\nTotal X files: {len(all_x_files)} | Skipped: {sorted(SKIP_PATIENTS & set(all_pids))} "
      f"| Usable patients: {len(pids)}")

if len(pids) < 5:
    raise RuntimeError("Not enough usable patients to train/val/test split.")

# ─────────────────────────────────────────────────────────────────────────
# 4. STRATIFIED PATIENT-LEVEL SPLIT -- exact counts: 60 train / 15 test / 15 val
# ─────────────────────────────────────────────────────────────────────────
def patient_apnea_ratio(pid):
    y = np.load(os.path.join(DATA_DIR, f"y_{pid}.npy"), mmap_mode="r")
    n = len(y)
    apnea = int(np.sum(y))
    del y
    return apnea / max(1, n), n, apnea

patient_stats = {pid: patient_apnea_ratio(pid) for pid in pids}
ratios = np.array([patient_stats[p][0] for p in pids])

expected_total = TRAIN_N + TEST_N + VAL_N
if len(pids) != expected_total:
    print(f"\n⚠️  WARNING: usable patient count ({len(pids)}) does not match "
          f"TRAIN_N+TEST_N+VAL_N ({expected_total}). The split below will still "
          f"use exactly {TRAIN_N}/{TEST_N}/{VAL_N} patients where possible.")

rng = np.random.default_rng(SEED)
order = np.argsort(ratios)
sorted_pids = np.array(pids)[order]
bin_edges = np.linspace(0, len(sorted_pids), N_STRAT_BINS + 1).astype(int)
bins = []
for b in range(N_STRAT_BINS):
    bucket = list(sorted_pids[bin_edges[b]:bin_edges[b + 1]])
    rng.shuffle(bucket)
    bins.append(bucket)

from itertools import zip_longest
combined = []
for group in zip_longest(*bins, fillvalue=None):
    combined.extend(pid for pid in group if pid is not None)

train_pids = combined[:TRAIN_N]
test_pids  = combined[TRAIN_N:TRAIN_N + TEST_N]
val_pids   = combined[TRAIN_N + TEST_N:TRAIN_N + TEST_N + VAL_N]
unused_pids = combined[TRAIN_N + TEST_N + VAL_N:]

print(f"\nTrain  patients ({len(train_pids)}): {sorted(train_pids)}")
print(f"Test   patients ({len(test_pids)}): {sorted(test_pids)}")
print(f"Val    patients ({len(val_pids)}): {sorted(val_pids)}")
if unused_pids:
    print(f"UNUSED patients ({len(unused_pids)}): {sorted(unused_pids)}")

# ─────────────────────────────────────────────────────────────────────────
# 5. PER-CHANNEL NORMALIZATION STATS (radius-independent, from TRAIN only)
# ─────────────────────────────────────────────────────────────────────────
def estimate_norm_stats(patient_ids, n_sample_windows):
    sums, sqsums, count = np.zeros(NUM_CHANNELS, np.float64), np.zeros(NUM_CHANNELS, np.float64), 0
    per_patient_budget = max(1, n_sample_windows // max(1, len(patient_ids)))
    for pid in patient_ids:
        X = np.load(os.path.join(DATA_DIR, f"X_{pid}.npy"), mmap_mode="r")
        n = X.shape[0]
        if n == 0:
            continue
        k = min(per_patient_budget, n)
        idx = np.random.default_rng(SEED).choice(n, size=k, replace=False)
        sample = X[idx].astype(np.float64)
        sums   += sample.sum(axis=(0, 2))
        sqsums += (sample ** 2).sum(axis=(0, 2))
        count  += k * sample.shape[2]
        del X, sample
        gc.collect()
    mean = sums / count
    var  = sqsums / count - mean ** 2
    std  = np.sqrt(np.clip(var, 1e-8, None))
    return mean.astype(np.float32), std.astype(np.float32)

print("\nEstimating per-channel normalization stats from train set...")
CH_MEAN, CH_STD = estimate_norm_stats(train_pids, NORM_SAMPLE_WINDOWS)
np.save(os.path.join(OUTPUT_DIR, "channel_mean.npy"), CH_MEAN)
np.save(os.path.join(OUTPUT_DIR, "channel_std.npy"), CH_STD)

# ─────────────────────────────────────────────────────────────────────────
# 6. SIGNAL CLEANING (SpO2 / pleth)
# ─────────────────────────────────────────────────────────────────────────
def flatline_mask(sig, min_run):
    n = len(sig)
    if n < 2:
        return np.zeros(n, dtype=bool)
    same = np.concatenate(([False], sig[1:] == sig[:-1]))
    change_points = np.where(~same)[0]
    change_points = np.concatenate(([0], change_points, [n]))
    mask = np.zeros(n, dtype=bool)
    for i in range(len(change_points) - 1):
        s, e = change_points[i], change_points[i + 1]
        if e - s >= min_run:
            mask[s:e] = True
    return mask


def clean_signal(sig, valid_range=None, flatline_min_run=FLATLINE_MIN_RUN):
    sig = sig.astype(np.float32).copy()
    bad = np.zeros(len(sig), dtype=bool)
    if valid_range is not None:
        lo, hi = valid_range
        bad |= (sig < lo) | (sig > hi)
    bad |= flatline_mask(sig, flatline_min_run)
    if bad.any() and not bad.all():
        good_idx = np.where(~bad)[0]
        bad_idx = np.where(bad)[0]
        sig[bad_idx] = np.interp(bad_idx, good_idx, sig[good_idx])
    return sig


def clean_spo2_pleth(window_8ch):
    window_8ch = window_8ch.copy()
    window_8ch[SPO2_IDX]  = clean_signal(window_8ch[SPO2_IDX], valid_range=SPO2_VALID_RANGE)
    window_8ch[PLETH_IDX] = clean_signal(window_8ch[PLETH_IDX], valid_range=None)
    return window_8ch

# ─────────────────────────────────────────────────────────────────────────
# 7. BOUNDARY LABEL SMOOTHING (train only)
# ─────────────────────────────────────────────────────────────────────────
def compute_boundary_soft_labels(y, soft_conf=BOUNDARY_SOFT_CONF):
    n = len(y)
    soft = y.astype(np.float32).copy()
    for i in range(n):
        left_diff  = (i > 0)     and (y[i - 1] != y[i])
        right_diff = (i < n - 1) and (y[i + 1] != y[i])
        if left_diff or right_diff:
            soft[i] = (1.0 - soft_conf) if y[i] == 1 else soft_conf
    return soft

# ─────────────────────────────────────────────────────────────────────────
# 8. DATASET  (context radius = 2, fixed)
# ─────────────────────────────────────────────────────────────────────────
class ApneaDataset(Dataset):
    def __init__(self, data_dir, patient_ids, mean, std,
                 context_radius=CONTEXT_RADIUS, augment=False, use_soft_labels=False):
        self.data_dir = data_dir
        self._cache = {}
        self.index = []
        self.mean = mean.reshape(NUM_CHANNELS, 1)
        self.std  = std.reshape(NUM_CHANNELS, 1)
        self.context_radius = context_radius
        self.augment = augment
        self.use_soft_labels = use_soft_labels
        self._soft_label_cache = {}

        for pid in patient_ids:
            xp = os.path.join(data_dir, f"X_{pid}.npy")
            yp = os.path.join(data_dir, f"y_{pid}.npy")
            if not (os.path.exists(xp) and os.path.exists(yp)):
                continue
            y = np.load(yp, mmap_mode="r")
            n = len(y)
            self.index.extend([(pid, i, n) for i in range(n)])
            if use_soft_labels:
                self._soft_label_cache[pid] = compute_boundary_soft_labels(np.asarray(y))
            del y

    def _arrays(self, pid):
        if pid not in self._cache:
            self._cache[pid] = (
                np.load(os.path.join(self.data_dir, f"X_{pid}.npy"), mmap_mode="r"),
                np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r"),
            )
        return self._cache[pid]

    def __len__(self):
        return len(self.index)

    def labels(self):
        out = np.zeros(len(self.index), dtype=np.int8)
        cache = {}
        for i, (pid, li, _) in enumerate(self.index):
            if pid not in cache:
                cache[pid] = np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r")
            out[i] = cache[pid][li]
        return out

    def __getitem__(self, idx):
        pid, local_idx, n = self.index[idx]
        X_arr, y_arr = self._arrays(pid)

        r = self.context_radius
        neighbor_idxs = np.clip(np.arange(local_idx - r, local_idx + r + 1), 0, n - 1)
        segs = [X_arr[j].astype(np.float32) for j in neighbor_idxs]
        x = np.concatenate(segs, axis=1)

        x = clean_spo2_pleth(x)
        x = (x - self.mean) / self.std

        if self.augment:
            if NOISE_STD_FRAC > 0:
                x = x + np.random.randn(*x.shape).astype(np.float32) * NOISE_STD_FRAC
            if SCALE_JITTER > 0:
                scale = 1.0 + (np.random.rand(NUM_CHANNELS, 1).astype(np.float32) * 2 - 1) * SCALE_JITTER
                x = x * scale
            if MAX_TIME_SHIFT > 0:
                shift = np.random.randint(-MAX_TIME_SHIFT, MAX_TIME_SHIFT + 1)
                if shift != 0:
                    x = np.roll(x, shift, axis=1)
            if CHANNEL_DROPOUT_P > 0 and np.random.rand() < CHANNEL_DROPOUT_P:
                ch = np.random.randint(0, NUM_CHANNELS)
                x[ch, :] = 0.0

        if self.use_soft_labels:
            y = float(self._soft_label_cache[pid][local_idx])
        else:
            y = float(y_arr[local_idx])

        return torch.from_numpy(x.copy()), torch.tensor(y, dtype=torch.float32)


train_ds = ApneaDataset(DATA_DIR, train_pids, CH_MEAN, CH_STD, augment=AUGMENT, use_soft_labels=True)
val_ds   = ApneaDataset(DATA_DIR, val_pids, CH_MEAN, CH_STD, augment=False, use_soft_labels=False)
test_ds  = ApneaDataset(DATA_DIR, test_pids, CH_MEAN, CH_STD, augment=False, use_soft_labels=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

train_labels = train_ds.labels()
n0, n1 = int((train_labels == 0).sum()), int((train_labels == 1).sum())
pos_weight_tensor = torch.tensor([n0 / max(1, n1)], dtype=torch.float32).to(DEVICE)
print(f"\nTrain windows: {len(train_ds):,} (normal={n0:,} apnea={n1:,}) | "
      f"input length: {(2*CONTEXT_RADIUS+1)*6000} samples ({(2*CONTEXT_RADIUS+1)*30}s)")
print(f"Val   windows: {len(val_ds):,}")
print(f"Test  windows: {len(test_ds):,}")
print(f"pos_weight for weighted focal loss: {pos_weight_tensor.item():.3f}")

# ─────────────────────────────────────────────────────────────────────────
# 9. 1D RESNET + SE-ATTENTION ARCHITECTURE
# ─────────────────────────────────────────────────────────────────────────
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(4, channels // reduction)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels), nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1)
        return x * s


class SEBasicBlock1D(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, downsample=None, k=7):
        super().__init__()
        pad = k // 2
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=k, stride=stride, padding=pad, bias=False)
        self.bn1   = nn.BatchNorm1d(planes)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=k, stride=1, padding=pad, bias=False)
        self.bn2   = nn.BatchNorm1d(planes)
        self.se    = SEBlock1D(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out = out + identity
        return self.relu(out)


class ResNet1D(nn.Module):
    """SE-ResNet1D with a single-logit output head."""
    def __init__(self, in_channels=8, base_width=32, dropout=0.4, layers=(2, 2, 3, 2)):
        super().__init__()
        self.in_planes = base_width
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, base_width, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(base_width),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )
        self.layer1 = self._make_layer(base_width,     layers[0], stride=1)
        self.layer2 = self._make_layer(base_width * 2, layers[1], stride=2)
        self.layer3 = self._make_layer(base_width * 4, layers[2], stride=2)
        self.layer4 = self._make_layer(base_width * 8, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(base_width * 8, 1)   # single logit

        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, planes, num_blocks, stride):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes),
            )
        layers = [SEBasicBlock1D(self.in_planes, planes, stride, downsample)]
        self.in_planes = planes
        for _ in range(1, num_blocks):
            layers.append(SEBasicBlock1D(self.in_planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        x = self.dropout(x)
        return self.fc(x).squeeze(-1)   # (B,) single logit per window


model = ResNet1D(NUM_CHANNELS, BASE_WIDTH, DROPOUT).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel: SE-ResNet1D | Trainable params: {n_params:,}")

# ─────────────────────────────────────────────────────────────────────────
# 10. LOSS -- WEIGHTED FOCAL LOSS ONLY
# ─────────────────────────────────────────────────────────────────────────
class FocalLossBinary(nn.Module):
    """Binary focal loss with pos_weight baked into the BCE term
    ("weighted focal loss")."""
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction="none")
        p = torch.sigmoid(logits)
        pt = p * targets + (1 - p) * (1 - targets)
        focal_weight = (1 - pt).clamp(min=1e-6) ** self.gamma
        return (focal_weight * bce).mean()


criterion = FocalLossBinary(gamma=FOCAL_GAMMA, pos_weight=pos_weight_tensor)
print(f"Loss: Weighted Focal Loss (gamma={FOCAL_GAMMA}, pos_weight={pos_weight_tensor.item():.3f})")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

# ─────────────────────────────────────────────────────────────────────────
# 11. TRAIN / EVAL HELPERS
# ─────────────────────────────────────────────────────────────────────────
def run_epoch(loader, train_mode: bool):
    model.train(mode=train_mode)
    total_loss = 0.0
    all_labels, all_probs = [], []

    for X, y in loader:
        X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                logits = model(X)
                loss = criterion(logits, y)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * X.size(0)
        probs = torch.sigmoid(logits.float()).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(loader.dataset))
    return avg_loss, np.array(all_labels), np.array(all_probs)


def report_at_threshold(probs, labels, thr):
    preds = (probs >= thr).astype(int)
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)
    cm   = confusion_matrix(labels, preds)
    rep  = classification_report(labels, preds, target_names=["Normal", "Apnea"], zero_division=0)
    return dict(threshold=thr, accuracy=acc, precision=prec, recall=rec, f1_score=f1,
                tn=int(cm[0, 0]), fp=int(cm[0, 1]), fn=int(cm[1, 0]), tp=int(cm[1, 1])), rep, cm


def smooth_probs_grouped(dataset, probs, radius=SMOOTH_RADIUS):
    """Moving-average smoothing of per-window probabilities, computed
    separately within each patient's (already time-ordered) window
    sequence -- never smooths across a patient boundary."""
    probs = np.asarray(probs)
    smoothed = np.zeros_like(probs)
    pids_seq = [p for p, _, _ in dataset.index]
    n = len(pids_seq)
    i = 0
    kernel = np.ones(2 * radius + 1) / (2 * radius + 1)
    while i < n:
        j = i
        while j < n and pids_seq[j] == pids_seq[i]:
            j += 1
        seg = probs[i:j]
        padded = np.pad(seg, (radius, radius), mode="edge")
        smoothed[i:j] = np.convolve(padded, kernel, mode="valid")
        i = j
    return smoothed


def extract_events(binary_seq):
    events, start = [], None
    for i, v in enumerate(binary_seq):
        if v == 1 and start is None:
            start = i
        elif v == 0 and start is not None:
            events.append((start, i - 1)); start = None
    if start is not None:
        events.append((start, len(binary_seq) - 1))
    return events


def match_events(pred_events, true_events, min_overlap=1):
    matched_true = set()
    pred_hit = [False] * len(pred_events)
    for pi, (ps, pe) in enumerate(pred_events):
        for ti, (ts, te) in enumerate(true_events):
            overlap = max(0, min(pe, te) - max(ps, ts) + 1)
            if overlap >= min_overlap:
                matched_true.add(ti)
                pred_hit[pi] = True
    tp_true = len(matched_true)
    tp_pred = sum(pred_hit)
    fp = len(pred_events) - tp_pred
    fn = len(true_events) - tp_true
    return tp_true, tp_pred, fp, fn, len(pred_events), len(true_events)


def event_level_evaluation(dataset, labels, probs, threshold, min_overlap=1):
    preds = (probs >= threshold).astype(int)
    pids_seq = [p for p, _, _ in dataset.index]
    n = len(pids_seq)
    agg = dict(tp_true=0, tp_pred=0, fp=0, fn=0, n_pred=0, n_true=0)
    i = 0
    while i < n:
        j = i
        while j < n and pids_seq[j] == pids_seq[i]:
            j += 1
        pe = extract_events(preds[i:j])
        te = extract_events(labels[i:j])
        tp_true, tp_pred, fp, fn, npe, nte = match_events(pe, te, min_overlap)
        agg["tp_true"] += tp_true; agg["tp_pred"] += tp_pred
        agg["fp"] += fp; agg["fn"] += fn
        agg["n_pred"] += npe; agg["n_true"] += nte
        i = j
    precision = agg["tp_pred"] / agg["n_pred"] if agg["n_pred"] > 0 else 0.0
    recall    = agg["tp_true"] / agg["n_true"] if agg["n_true"] > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    agg["precision"], agg["recall"], agg["f1_score"] = precision, recall, f1
    return agg

# ─────────────────────────────────────────────────────────────────────────
# 12. TRAINING LOOP  (early stopping on val F1 @ FIXED THRESHOLD 0.5)
# ─────────────────────────────────────────────────────────────────────────
best_val_f1, best_state, no_improve = -1.0, None, 0
history = []
ckpt_path = os.path.join(OUTPUT_DIR, "best_model.pt")

print("\n" + "=" * 65)
print("  TRAINING")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, _, _ = run_epoch(train_loader, True)
    val_loss, val_labels, val_probs = run_epoch(val_loader, False)

    val_preds = (val_probs >= FIXED_THRESHOLD).astype(int)
    val_f1 = f1_score(val_labels, val_preds, zero_division=0)
    scheduler.step(val_f1)
    dt = time.time() - t0

    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                     "val_f1@0.5": val_f1, "lr": optimizer.param_groups[0]["lr"]})
    print(f"Epoch {epoch:2d}/{EPOCHS} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
          f"val_f1@0.5={val_f1:.4f} | lr={optimizer.param_groups[0]['lr']:.2e} | {dt:.1f}s")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
        torch.save({"state_dict": best_state, "threshold": FIXED_THRESHOLD,
                    "context_radius": CONTEXT_RADIUS}, ckpt_path)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break

pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)
if best_state is not None:
    model.load_state_dict(best_state)
print(f"\nBest val F1 @ threshold=0.5: {best_val_f1:.4f}")

# ─────────────────────────────────────────────────────────────────────────
# 13. TEST SET EVALUATION  (all at fixed threshold = 0.5)
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  TEST SET EVALUATION  (threshold = 0.5 everywhere)")
print("=" * 65)

_, test_labels, test_probs = run_epoch(test_loader, False)
try:
    auc = roc_auc_score(test_labels, test_probs)
except ValueError:
    auc = float("nan")
print(f"\nROC-AUC (threshold-independent): {auc:.4f}")

metrics_raw, rep_raw, cm_raw = report_at_threshold(test_probs, test_labels, FIXED_THRESHOLD)
print(f"\n--- RAW @ threshold=0.5 ---")
print(f"Accuracy  : {metrics_raw['accuracy']:.4f}")
print(f"Precision : {metrics_raw['precision']:.4f}")
print(f"Recall    : {metrics_raw['recall']:.4f}")
print(f"F1 Score  : {metrics_raw['f1_score']:.4f}")
print(f"Confusion Matrix:\n{cm_raw}")
print(f"\nClassification Report:\n{rep_raw}")

# --- smoothed, still at threshold = 0.5 ---
test_probs_smooth = smooth_probs_grouped(test_ds, test_probs, radius=SMOOTH_RADIUS)
metrics_smooth, rep_smooth, cm_smooth = report_at_threshold(test_probs_smooth, test_labels, FIXED_THRESHOLD)
print(f"\n--- SMOOTHED @ threshold=0.5 ---")
print(f"Accuracy  : {metrics_smooth['accuracy']:.4f}")
print(f"Precision : {metrics_smooth['precision']:.4f}")
print(f"Recall    : {metrics_smooth['recall']:.4f}")
print(f"F1 Score  : {metrics_smooth['f1_score']:.4f}")
print(f"Confusion Matrix:\n{cm_smooth}")
print(f"\nClassification Report:\n{rep_smooth}")

pd.DataFrame([{"variant": "raw", **metrics_raw}, {"variant": "smoothed", **metrics_smooth}]).to_csv(
    os.path.join(OUTPUT_DIR, "test_metrics.csv"), index=False)
with open(os.path.join(OUTPUT_DIR, "classification_report_raw.txt"), "w") as fh:
    fh.write(rep_raw)
with open(os.path.join(OUTPUT_DIR, "classification_report_smoothed.txt"), "w") as fh:
    fh.write(rep_smooth)

# --- event-level, using smoothed probs @ threshold = 0.5 ---
print("\n" + "=" * 65)
print("  EVENT-LEVEL EVALUATION (AHI-style, smoothed probs, threshold=0.5)")
print("=" * 65)
event_metrics = event_level_evaluation(test_ds, test_labels, test_probs_smooth, FIXED_THRESHOLD, min_overlap=1)
print(f"\nGround-truth events : {event_metrics['n_true']}")
print(f"Predicted events    : {event_metrics['n_pred']}")
print(f"Event Precision     : {event_metrics['precision']:.4f}")
print(f"Event Recall        : {event_metrics['recall']:.4f}")
print(f"Event F1 Score      : {event_metrics['f1_score']:.4f}")
pd.DataFrame([event_metrics]).to_csv(os.path.join(OUTPUT_DIR, "event_level_metrics.csv"), index=False)

# ─────────────────────────────────────────────────────────────────────────
# 14. PLOTS
# ─────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im = axes[0].imshow(cm_smooth, cmap="Blues")
axes[0].set_title("Confusion Matrix (smoothed, thr=0.5)")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["Normal", "Apnea"])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(["Normal", "Apnea"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm_smooth[i, j]), ha="center", va="center",
                      color="white" if cm_smooth[i, j] > cm_smooth.max() / 2 else "black")
fig.colorbar(im, ax=axes[0])

if not np.isnan(auc):
    fpr, tpr, _ = roc_curve(test_labels, test_probs)
    axes[1].plot(fpr, tpr, label=f"AUC = {auc:.3f}")
    axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
    axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title("ROC Curve"); axes[1].legend()

hist_df = pd.DataFrame(history)
axes[2].plot(hist_df["epoch"], hist_df["train_loss"], label="Train Loss")
axes[2].plot(hist_df["epoch"], hist_df["val_f1@0.5"], label="Val F1 @ 0.5")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Value")
axes[2].set_title("Training Curves"); axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "evaluation_plots.png"), dpi=150)
plt.show()

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(" - best_model.pt                          (weights, threshold=0.5, context_radius=2)")
print(" - channel_mean.npy / channel_std.npy")
print(" - training_history.csv")
print(" - test_metrics.csv                       (raw @0.5 and smoothed @0.5)")
print(" - classification_report_raw.txt / classification_report_smoothed.txt")
print(" - event_level_metrics.csv")
print(" - evaluation_plots.png")

In [ ]:
"""
=============================================================================
 SLEEP APNEA DETECTION — SE-RESNET1D  (radius=4, weighted focal loss only,
 fixed threshold = 0.5)
=============================================================================
 This is the 4-model ablation script (radius=1, BCE/weighted BCE/focal/
 weighted focal) collapsed down to a SINGLE model:
   - Context radius CHANGED from 1 -> 4  (target window + 4 neighbors each
     side = 9 windows total = 270s / 54000-sample input, instead of 90s)
   - Loss function FIXED to weighted focal loss only (no BCE/weighted-BCE/
     plain-focal comparison -- this is the only loss used)
   - Decision threshold FIXED at 0.5 everywhere (default eval, smoothed
     eval, and event-level eval) -- no val-tuned threshold search anywhere
     in this script

 EVERYTHING ELSE is unchanged from the ablation study: same SE-ResNet1D
 architecture, same 60/15/15 exact-count patient split, same SpO2/pleth
 cleaning, same boundary label smoothing, same augmentation, same
 normalization, same temporal smoothing mechanism, same AHI-style
 event-level evaluation -- just applied to one model instead of eight.

 NOTE ON RADIUS=4: a 54000-sample input is 3x longer than radius=2's
 30000 samples and 6x longer than radius=1's 18000 samples. Batch size is
 reduced accordingly (8, down from 16 at radius=2) to keep memory usage
 manageable -- this is the main practical cost of a much larger radius,
 alongside proportionally longer per-epoch training time.

 ARCHITECTURE: 1D ResNet with SQUEEZE-AND-EXCITATION (SE) attention blocks
 after every residual block. SE blocks learn a per-channel attention gate
 so the network can up/down-weight which of the 8 physiological channels
 (flow, chest, abd, SpO2, pleth, ECG, pulse, flow2) matter most for a given
 window. Single-logit output head (sigmoid-based), matching the weighted
 focal loss's binary formulation.

 LOSS -- WEIGHTED FOCAL LOSS ONLY:
   Weighted Focal = (1 - p_t)^gamma * BCEWithLogits(pos_weight=...)
 Combines class-imbalance correction (pos_weight, same idea as weighted
 BCE) with focal loss's hard-example focusing ((1-p_t)^gamma down-weights
 already-confident predictions).

 SPLIT: patient-level, stratified by apnea ratio (interleaved across bins
 so the split hits exact counts with no leftover), 60 train / 15 test /
 15 validation patients -- exact absolute counts, matching the ~90 usable
 patients left after skipping 10 from the ~100-patient dataset.

 Input  : X_SN{n}.npy -> (num_windows, 8, 6000) float16
          y_SN{n}.npy -> (num_windows,)         int8  (0=normal, 1=apnea)
=============================================================================
"""

import os
import gc
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

warnings.filterwarnings("ignore")
import kagglehub

# ─────────────────────────────────────────────────────────────────────────
# 0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ─────────────────────────────────────────────────────────────────────────
# 0b. GPU / KERNEL-COMPATIBILITY CHECK  (P100 = compute capability 6.0)
# ─────────────────────────────────────────────────────────────────────────
def _gpu_kernels_usable():
    if not torch.cuda.is_available():
        return False
    try:
        x = torch.randn(2, 4, 16, device="cuda")
        conv = nn.Conv1d(4, 4, 3, padding=1).cuda()
        _ = conv(x)
        return True
    except RuntimeError as e:
        print("\n⚠️  GPU kernel check failed:", str(e).splitlines()[0])
        print("⚠️  Falling back to CPU. Switch accelerator to T4/L4 to use GPU.")
        return False

_USE_CUDA = _gpu_kernels_usable()

# ─────────────────────────────────────────────────────────────────────────
# 1. DOWNLOAD DATASET
# ─────────────────────────────────────────────────────────────────────────
DATA_DIR = kagglehub.dataset_download(
    "nishantomhare/detection-of-disordered-breathing-dataset2"
)
print("Dataset path:", DATA_DIR)

def locate_npy_root(root):
    for dirpath, _, filenames in os.walk(root):
        if any(f.startswith("X_SN") and f.endswith(".npy") for f in filenames):
            return dirpath
    return root

DATA_DIR = locate_npy_root(DATA_DIR)
print("Resolved npy root:", DATA_DIR)

OUTPUT_DIR = "/kaggle/working/resnet1d_se_radius4_weightedfocal_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────
# 2. CONFIG
# ─────────────────────────────────────────────────────────────────────────
SKIP_PATIENTS = {
    "SN1", "SN28", "SN14", "SN13", "SN17", "SN18",
    "SN19", "SN20", "SN27", "SN89"
}

# Exact ABSOLUTE patient counts: out of the ~90 usable patients, 60 go to
# train, 15 to test, 15 to validation.
TRAIN_N = 60
TEST_N  = 15
VAL_N   = 15
N_STRAT_BINS = 5

FIXED_CHANNEL_KEYS = ["flow", "chest", "abd", "spo2", "pleth", "ecg", "pulse", "flow2"]
SPO2_IDX  = FIXED_CHANNEL_KEYS.index("spo2")
PLETH_IDX = FIXED_CHANNEL_KEYS.index("pleth")
FS = 200

NUM_CHANNELS  = 8
CONTEXT_RADIUS = 4          # CHANGED from 1 -> 4: target window + 4 neighbors each side = 270s input
FIXED_THRESHOLD = 0.5       # used everywhere: default eval, smoothed eval, event-level eval
SMOOTH_RADIUS = 2           # temporal smoothing window (in # of windows), each side

EPOCHS    = 16
PATIENCE  = 6
GRAD_CLIP = 3.0
NUM_WORKERS = 2
BATCH_SIZE = 8                # radius=4 sequences (54000 samples) are very long -> smaller batch than radius=2's 16

# fixed architecture/optimizer hyperparameters (same values as the ablation study)
BASE_WIDTH   = 32
DROPOUT      = 0.4
LR           = 7e-4
WEIGHT_DECAY = 8e-5
FOCAL_GAMMA  = 2.0

AUGMENT           = True
NOISE_STD_FRAC    = 0.03
SCALE_JITTER      = 0.10
MAX_TIME_SHIFT    = 100
CHANNEL_DROPOUT_P = 0.05

NORM_SAMPLE_WINDOWS = 8000
BOUNDARY_SOFT_CONF  = 0.15
SPO2_VALID_RANGE    = (50.0, 100.0)
FLATLINE_MIN_SEC    = 1.0
FLATLINE_MIN_RUN    = int(FLATLINE_MIN_SEC * FS)

DEVICE = torch.device("cuda" if _USE_CUDA else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| capability:", torch.cuda.get_device_capability(0),
          "| usable:", _USE_CUDA)

# ─────────────────────────────────────────────────────────────────────────
# 3. DISCOVER PATIENTS
# ─────────────────────────────────────────────────────────────────────────
all_x_files = sorted(f for f in os.listdir(DATA_DIR)
                      if f.startswith("X_SN") and f.endswith(".npy"))
all_pids = [f[len("X_"):-len(".npy")] for f in all_x_files]
pids = [p for p in all_pids if p not in SKIP_PATIENTS
        and os.path.exists(os.path.join(DATA_DIR, f"y_{p}.npy"))]
print(f"\nTotal X files: {len(all_x_files)} | Skipped: {sorted(SKIP_PATIENTS & set(all_pids))} "
      f"| Usable patients: {len(pids)}")

if len(pids) < 5:
    raise RuntimeError("Not enough usable patients to train/val/test split.")

# ─────────────────────────────────────────────────────────────────────────
# 4. STRATIFIED PATIENT-LEVEL SPLIT -- exact counts: 60 train / 15 test / 15 val
# ─────────────────────────────────────────────────────────────────────────
def patient_apnea_ratio(pid):
    y = np.load(os.path.join(DATA_DIR, f"y_{pid}.npy"), mmap_mode="r")
    n = len(y)
    apnea = int(np.sum(y))
    del y
    return apnea / max(1, n), n, apnea

patient_stats = {pid: patient_apnea_ratio(pid) for pid in pids}
ratios = np.array([patient_stats[p][0] for p in pids])

expected_total = TRAIN_N + TEST_N + VAL_N
if len(pids) != expected_total:
    print(f"\n⚠️  WARNING: usable patient count ({len(pids)}) does not match "
          f"TRAIN_N+TEST_N+VAL_N ({expected_total}). The split below will still "
          f"use exactly {TRAIN_N}/{TEST_N}/{VAL_N} patients where possible.")

rng = np.random.default_rng(SEED)
order = np.argsort(ratios)
sorted_pids = np.array(pids)[order]
bin_edges = np.linspace(0, len(sorted_pids), N_STRAT_BINS + 1).astype(int)
bins = []
for b in range(N_STRAT_BINS):
    bucket = list(sorted_pids[bin_edges[b]:bin_edges[b + 1]])
    rng.shuffle(bucket)
    bins.append(bucket)

from itertools import zip_longest
combined = []
for group in zip_longest(*bins, fillvalue=None):
    combined.extend(pid for pid in group if pid is not None)

train_pids = combined[:TRAIN_N]
test_pids  = combined[TRAIN_N:TRAIN_N + TEST_N]
val_pids   = combined[TRAIN_N + TEST_N:TRAIN_N + TEST_N + VAL_N]
unused_pids = combined[TRAIN_N + TEST_N + VAL_N:]

print(f"\nTrain  patients ({len(train_pids)}): {sorted(train_pids)}")
print(f"Test   patients ({len(test_pids)}): {sorted(test_pids)}")
print(f"Val    patients ({len(val_pids)}): {sorted(val_pids)}")
if unused_pids:
    print(f"UNUSED patients ({len(unused_pids)}): {sorted(unused_pids)}")

# ─────────────────────────────────────────────────────────────────────────
# 5. PER-CHANNEL NORMALIZATION STATS (radius-independent, from TRAIN only)
# ─────────────────────────────────────────────────────────────────────────
def estimate_norm_stats(patient_ids, n_sample_windows):
    sums, sqsums, count = np.zeros(NUM_CHANNELS, np.float64), np.zeros(NUM_CHANNELS, np.float64), 0
    per_patient_budget = max(1, n_sample_windows // max(1, len(patient_ids)))
    for pid in patient_ids:
        X = np.load(os.path.join(DATA_DIR, f"X_{pid}.npy"), mmap_mode="r")
        n = X.shape[0]
        if n == 0:
            continue
        k = min(per_patient_budget, n)
        idx = np.random.default_rng(SEED).choice(n, size=k, replace=False)
        sample = X[idx].astype(np.float64)
        sums   += sample.sum(axis=(0, 2))
        sqsums += (sample ** 2).sum(axis=(0, 2))
        count  += k * sample.shape[2]
        del X, sample
        gc.collect()
    mean = sums / count
    var  = sqsums / count - mean ** 2
    std  = np.sqrt(np.clip(var, 1e-8, None))
    return mean.astype(np.float32), std.astype(np.float32)

print("\nEstimating per-channel normalization stats from train set...")
CH_MEAN, CH_STD = estimate_norm_stats(train_pids, NORM_SAMPLE_WINDOWS)
np.save(os.path.join(OUTPUT_DIR, "channel_mean.npy"), CH_MEAN)
np.save(os.path.join(OUTPUT_DIR, "channel_std.npy"), CH_STD)

# ─────────────────────────────────────────────────────────────────────────
# 6. SIGNAL CLEANING (SpO2 / pleth)
# ─────────────────────────────────────────────────────────────────────────
def flatline_mask(sig, min_run):
    n = len(sig)
    if n < 2:
        return np.zeros(n, dtype=bool)
    same = np.concatenate(([False], sig[1:] == sig[:-1]))
    change_points = np.where(~same)[0]
    change_points = np.concatenate(([0], change_points, [n]))
    mask = np.zeros(n, dtype=bool)
    for i in range(len(change_points) - 1):
        s, e = change_points[i], change_points[i + 1]
        if e - s >= min_run:
            mask[s:e] = True
    return mask


def clean_signal(sig, valid_range=None, flatline_min_run=FLATLINE_MIN_RUN):
    sig = sig.astype(np.float32).copy()
    bad = np.zeros(len(sig), dtype=bool)
    if valid_range is not None:
        lo, hi = valid_range
        bad |= (sig < lo) | (sig > hi)
    bad |= flatline_mask(sig, flatline_min_run)
    if bad.any() and not bad.all():
        good_idx = np.where(~bad)[0]
        bad_idx = np.where(bad)[0]
        sig[bad_idx] = np.interp(bad_idx, good_idx, sig[good_idx])
    return sig


def clean_spo2_pleth(window_8ch):
    window_8ch = window_8ch.copy()
    window_8ch[SPO2_IDX]  = clean_signal(window_8ch[SPO2_IDX], valid_range=SPO2_VALID_RANGE)
    window_8ch[PLETH_IDX] = clean_signal(window_8ch[PLETH_IDX], valid_range=None)
    return window_8ch

# ─────────────────────────────────────────────────────────────────────────
# 7. BOUNDARY LABEL SMOOTHING (train only)
# ─────────────────────────────────────────────────────────────────────────
def compute_boundary_soft_labels(y, soft_conf=BOUNDARY_SOFT_CONF):
    n = len(y)
    soft = y.astype(np.float32).copy()
    for i in range(n):
        left_diff  = (i > 0)     and (y[i - 1] != y[i])
        right_diff = (i < n - 1) and (y[i + 1] != y[i])
        if left_diff or right_diff:
            soft[i] = (1.0 - soft_conf) if y[i] == 1 else soft_conf
    return soft

# ─────────────────────────────────────────────────────────────────────────
# 8. DATASET  (context radius = 2, fixed)
# ─────────────────────────────────────────────────────────────────────────
class ApneaDataset(Dataset):
    def __init__(self, data_dir, patient_ids, mean, std,
                 context_radius=CONTEXT_RADIUS, augment=False, use_soft_labels=False):
        self.data_dir = data_dir
        self._cache = {}
        self.index = []
        self.mean = mean.reshape(NUM_CHANNELS, 1)
        self.std  = std.reshape(NUM_CHANNELS, 1)
        self.context_radius = context_radius
        self.augment = augment
        self.use_soft_labels = use_soft_labels
        self._soft_label_cache = {}

        for pid in patient_ids:
            xp = os.path.join(data_dir, f"X_{pid}.npy")
            yp = os.path.join(data_dir, f"y_{pid}.npy")
            if not (os.path.exists(xp) and os.path.exists(yp)):
                continue
            y = np.load(yp, mmap_mode="r")
            n = len(y)
            self.index.extend([(pid, i, n) for i in range(n)])
            if use_soft_labels:
                self._soft_label_cache[pid] = compute_boundary_soft_labels(np.asarray(y))
            del y

    def _arrays(self, pid):
        if pid not in self._cache:
            self._cache[pid] = (
                np.load(os.path.join(self.data_dir, f"X_{pid}.npy"), mmap_mode="r"),
                np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r"),
            )
        return self._cache[pid]

    def __len__(self):
        return len(self.index)

    def labels(self):
        out = np.zeros(len(self.index), dtype=np.int8)
        cache = {}
        for i, (pid, li, _) in enumerate(self.index):
            if pid not in cache:
                cache[pid] = np.load(os.path.join(self.data_dir, f"y_{pid}.npy"), mmap_mode="r")
            out[i] = cache[pid][li]
        return out

    def __getitem__(self, idx):
        pid, local_idx, n = self.index[idx]
        X_arr, y_arr = self._arrays(pid)

        r = self.context_radius
        neighbor_idxs = np.clip(np.arange(local_idx - r, local_idx + r + 1), 0, n - 1)
        segs = [X_arr[j].astype(np.float32) for j in neighbor_idxs]
        x = np.concatenate(segs, axis=1)

        x = clean_spo2_pleth(x)
        x = (x - self.mean) / self.std

        if self.augment:
            if NOISE_STD_FRAC > 0:
                x = x + np.random.randn(*x.shape).astype(np.float32) * NOISE_STD_FRAC
            if SCALE_JITTER > 0:
                scale = 1.0 + (np.random.rand(NUM_CHANNELS, 1).astype(np.float32) * 2 - 1) * SCALE_JITTER
                x = x * scale
            if MAX_TIME_SHIFT > 0:
                shift = np.random.randint(-MAX_TIME_SHIFT, MAX_TIME_SHIFT + 1)
                if shift != 0:
                    x = np.roll(x, shift, axis=1)
            if CHANNEL_DROPOUT_P > 0 and np.random.rand() < CHANNEL_DROPOUT_P:
                ch = np.random.randint(0, NUM_CHANNELS)
                x[ch, :] = 0.0

        if self.use_soft_labels:
            y = float(self._soft_label_cache[pid][local_idx])
        else:
            y = float(y_arr[local_idx])

        return torch.from_numpy(x.copy()), torch.tensor(y, dtype=torch.float32)


train_ds = ApneaDataset(DATA_DIR, train_pids, CH_MEAN, CH_STD, augment=AUGMENT, use_soft_labels=True)
val_ds   = ApneaDataset(DATA_DIR, val_pids, CH_MEAN, CH_STD, augment=False, use_soft_labels=False)
test_ds  = ApneaDataset(DATA_DIR, test_pids, CH_MEAN, CH_STD, augment=False, use_soft_labels=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

train_labels = train_ds.labels()
n0, n1 = int((train_labels == 0).sum()), int((train_labels == 1).sum())
pos_weight_tensor = torch.tensor([n0 / max(1, n1)], dtype=torch.float32).to(DEVICE)
print(f"\nTrain windows: {len(train_ds):,} (normal={n0:,} apnea={n1:,}) | "
      f"input length: {(2*CONTEXT_RADIUS+1)*6000} samples ({(2*CONTEXT_RADIUS+1)*30}s)")
print(f"Val   windows: {len(val_ds):,}")
print(f"Test  windows: {len(test_ds):,}")
print(f"pos_weight for weighted focal loss: {pos_weight_tensor.item():.3f}")

# ─────────────────────────────────────────────────────────────────────────
# 9. 1D RESNET + SE-ATTENTION ARCHITECTURE
# ─────────────────────────────────────────────────────────────────────────
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(4, channels // reduction)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels), nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1)
        return x * s


class SEBasicBlock1D(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, downsample=None, k=7):
        super().__init__()
        pad = k // 2
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=k, stride=stride, padding=pad, bias=False)
        self.bn1   = nn.BatchNorm1d(planes)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=k, stride=1, padding=pad, bias=False)
        self.bn2   = nn.BatchNorm1d(planes)
        self.se    = SEBlock1D(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out = out + identity
        return self.relu(out)


class ResNet1D(nn.Module):
    """SE-ResNet1D with a single-logit output head."""
    def __init__(self, in_channels=8, base_width=32, dropout=0.4, layers=(2, 2, 3, 2)):
        super().__init__()
        self.in_planes = base_width
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, base_width, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(base_width),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )
        self.layer1 = self._make_layer(base_width,     layers[0], stride=1)
        self.layer2 = self._make_layer(base_width * 2, layers[1], stride=2)
        self.layer3 = self._make_layer(base_width * 4, layers[2], stride=2)
        self.layer4 = self._make_layer(base_width * 8, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(base_width * 8, 1)   # single logit

        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, planes, num_blocks, stride):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes),
            )
        layers = [SEBasicBlock1D(self.in_planes, planes, stride, downsample)]
        self.in_planes = planes
        for _ in range(1, num_blocks):
            layers.append(SEBasicBlock1D(self.in_planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        x = self.dropout(x)
        return self.fc(x).squeeze(-1)   # (B,) single logit per window


model = ResNet1D(NUM_CHANNELS, BASE_WIDTH, DROPOUT).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel: SE-ResNet1D | Trainable params: {n_params:,}")

# ─────────────────────────────────────────────────────────────────────────
# 10. LOSS -- WEIGHTED FOCAL LOSS ONLY
# ─────────────────────────────────────────────────────────────────────────
class FocalLossBinary(nn.Module):
    """Binary focal loss with pos_weight baked into the BCE term
    ("weighted focal loss")."""
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction="none")
        p = torch.sigmoid(logits)
        pt = p * targets + (1 - p) * (1 - targets)
        focal_weight = (1 - pt).clamp(min=1e-6) ** self.gamma
        return (focal_weight * bce).mean()


criterion = FocalLossBinary(gamma=FOCAL_GAMMA, pos_weight=pos_weight_tensor)
print(f"Loss: Weighted Focal Loss (gamma={FOCAL_GAMMA}, pos_weight={pos_weight_tensor.item():.3f})")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

# ─────────────────────────────────────────────────────────────────────────
# 11. TRAIN / EVAL HELPERS
# ─────────────────────────────────────────────────────────────────────────
def run_epoch(loader, train_mode: bool):
    model.train(mode=train_mode)
    total_loss = 0.0
    all_labels, all_probs = [], []

    for X, y in loader:
        X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                logits = model(X)
                loss = criterion(logits, y)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * X.size(0)
        probs = torch.sigmoid(logits.float()).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(loader.dataset))
    return avg_loss, np.array(all_labels), np.array(all_probs)


def report_at_threshold(probs, labels, thr):
    preds = (probs >= thr).astype(int)
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)
    cm   = confusion_matrix(labels, preds)
    rep  = classification_report(labels, preds, target_names=["Normal", "Apnea"], zero_division=0)
    return dict(threshold=thr, accuracy=acc, precision=prec, recall=rec, f1_score=f1,
                tn=int(cm[0, 0]), fp=int(cm[0, 1]), fn=int(cm[1, 0]), tp=int(cm[1, 1])), rep, cm


def smooth_probs_grouped(dataset, probs, radius=SMOOTH_RADIUS):
    """Moving-average smoothing of per-window probabilities, computed
    separately within each patient's (already time-ordered) window
    sequence -- never smooths across a patient boundary."""
    probs = np.asarray(probs)
    smoothed = np.zeros_like(probs)
    pids_seq = [p for p, _, _ in dataset.index]
    n = len(pids_seq)
    i = 0
    kernel = np.ones(2 * radius + 1) / (2 * radius + 1)
    while i < n:
        j = i
        while j < n and pids_seq[j] == pids_seq[i]:
            j += 1
        seg = probs[i:j]
        padded = np.pad(seg, (radius, radius), mode="edge")
        smoothed[i:j] = np.convolve(padded, kernel, mode="valid")
        i = j
    return smoothed


def extract_events(binary_seq):
    events, start = [], None
    for i, v in enumerate(binary_seq):
        if v == 1 and start is None:
            start = i
        elif v == 0 and start is not None:
            events.append((start, i - 1)); start = None
    if start is not None:
        events.append((start, len(binary_seq) - 1))
    return events


def match_events(pred_events, true_events, min_overlap=1):
    matched_true = set()
    pred_hit = [False] * len(pred_events)
    for pi, (ps, pe) in enumerate(pred_events):
        for ti, (ts, te) in enumerate(true_events):
            overlap = max(0, min(pe, te) - max(ps, ts) + 1)
            if overlap >= min_overlap:
                matched_true.add(ti)
                pred_hit[pi] = True
    tp_true = len(matched_true)
    tp_pred = sum(pred_hit)
    fp = len(pred_events) - tp_pred
    fn = len(true_events) - tp_true
    return tp_true, tp_pred, fp, fn, len(pred_events), len(true_events)


def event_level_evaluation(dataset, labels, probs, threshold, min_overlap=1):
    preds = (probs >= threshold).astype(int)
    pids_seq = [p for p, _, _ in dataset.index]
    n = len(pids_seq)
    agg = dict(tp_true=0, tp_pred=0, fp=0, fn=0, n_pred=0, n_true=0)
    i = 0
    while i < n:
        j = i
        while j < n and pids_seq[j] == pids_seq[i]:
            j += 1
        pe = extract_events(preds[i:j])
        te = extract_events(labels[i:j])
        tp_true, tp_pred, fp, fn, npe, nte = match_events(pe, te, min_overlap)
        agg["tp_true"] += tp_true; agg["tp_pred"] += tp_pred
        agg["fp"] += fp; agg["fn"] += fn
        agg["n_pred"] += npe; agg["n_true"] += nte
        i = j
    precision = agg["tp_pred"] / agg["n_pred"] if agg["n_pred"] > 0 else 0.0
    recall    = agg["tp_true"] / agg["n_true"] if agg["n_true"] > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    agg["precision"], agg["recall"], agg["f1_score"] = precision, recall, f1
    return agg

# ─────────────────────────────────────────────────────────────────────────
# 12. TRAINING LOOP  (early stopping on val F1 @ FIXED THRESHOLD 0.5)
# ─────────────────────────────────────────────────────────────────────────
best_val_f1, best_state, no_improve = -1.0, None, 0
history = []
ckpt_path = os.path.join(OUTPUT_DIR, "best_model.pt")

print("\n" + "=" * 65)
print("  TRAINING")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, _, _ = run_epoch(train_loader, True)
    val_loss, val_labels, val_probs = run_epoch(val_loader, False)

    val_preds = (val_probs >= FIXED_THRESHOLD).astype(int)
    val_f1 = f1_score(val_labels, val_preds, zero_division=0)
    scheduler.step(val_f1)
    dt = time.time() - t0

    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                     "val_f1@0.5": val_f1, "lr": optimizer.param_groups[0]["lr"]})
    print(f"Epoch {epoch:2d}/{EPOCHS} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
          f"val_f1@0.5={val_f1:.4f} | lr={optimizer.param_groups[0]['lr']:.2e} | {dt:.1f}s")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
        torch.save({"state_dict": best_state, "threshold": FIXED_THRESHOLD,
                    "context_radius": CONTEXT_RADIUS}, ckpt_path)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break

pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)
if best_state is not None:
    model.load_state_dict(best_state)
print(f"\nBest val F1 @ threshold=0.5: {best_val_f1:.4f}")

# ─────────────────────────────────────────────────────────────────────────
# 13. TEST SET EVALUATION  (all at fixed threshold = 0.5)
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  TEST SET EVALUATION  (threshold = 0.5 everywhere)")
print("=" * 65)

_, test_labels, test_probs = run_epoch(test_loader, False)
try:
    auc = roc_auc_score(test_labels, test_probs)
except ValueError:
    auc = float("nan")
print(f"\nROC-AUC (threshold-independent): {auc:.4f}")

metrics_raw, rep_raw, cm_raw = report_at_threshold(test_probs, test_labels, FIXED_THRESHOLD)
print(f"\n--- RAW @ threshold=0.5 ---")
print(f"Accuracy  : {metrics_raw['accuracy']:.4f}")
print(f"Precision : {metrics_raw['precision']:.4f}")
print(f"Recall    : {metrics_raw['recall']:.4f}")
print(f"F1 Score  : {metrics_raw['f1_score']:.4f}")
print(f"Confusion Matrix:\n{cm_raw}")
print(f"\nClassification Report:\n{rep_raw}")

# --- smoothed, still at threshold = 0.5 ---
test_probs_smooth = smooth_probs_grouped(test_ds, test_probs, radius=SMOOTH_RADIUS)
metrics_smooth, rep_smooth, cm_smooth = report_at_threshold(test_probs_smooth, test_labels, FIXED_THRESHOLD)
print(f"\n--- SMOOTHED @ threshold=0.5 ---")
print(f"Accuracy  : {metrics_smooth['accuracy']:.4f}")
print(f"Precision : {metrics_smooth['precision']:.4f}")
print(f"Recall    : {metrics_smooth['recall']:.4f}")
print(f"F1 Score  : {metrics_smooth['f1_score']:.4f}")
print(f"Confusion Matrix:\n{cm_smooth}")
print(f"\nClassification Report:\n{rep_smooth}")

pd.DataFrame([{"variant": "raw", **metrics_raw}, {"variant": "smoothed", **metrics_smooth}]).to_csv(
    os.path.join(OUTPUT_DIR, "test_metrics.csv"), index=False)
with open(os.path.join(OUTPUT_DIR, "classification_report_raw.txt"), "w") as fh:
    fh.write(rep_raw)
with open(os.path.join(OUTPUT_DIR, "classification_report_smoothed.txt"), "w") as fh:
    fh.write(rep_smooth)

# --- event-level, using smoothed probs @ threshold = 0.5 ---
print("\n" + "=" * 65)
print("  EVENT-LEVEL EVALUATION (AHI-style, smoothed probs, threshold=0.5)")
print("=" * 65)
event_metrics = event_level_evaluation(test_ds, test_labels, test_probs_smooth, FIXED_THRESHOLD, min_overlap=1)
print(f"\nGround-truth events : {event_metrics['n_true']}")
print(f"Predicted events    : {event_metrics['n_pred']}")
print(f"Event Precision     : {event_metrics['precision']:.4f}")
print(f"Event Recall        : {event_metrics['recall']:.4f}")
print(f"Event F1 Score      : {event_metrics['f1_score']:.4f}")
pd.DataFrame([event_metrics]).to_csv(os.path.join(OUTPUT_DIR, "event_level_metrics.csv"), index=False)

# ─────────────────────────────────────────────────────────────────────────
# 14. PLOTS
# ─────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im = axes[0].imshow(cm_smooth, cmap="Blues")
axes[0].set_title("Confusion Matrix (smoothed, thr=0.5)")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["Normal", "Apnea"])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(["Normal", "Apnea"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm_smooth[i, j]), ha="center", va="center",
                      color="white" if cm_smooth[i, j] > cm_smooth.max() / 2 else "black")
fig.colorbar(im, ax=axes[0])

if not np.isnan(auc):
    fpr, tpr, _ = roc_curve(test_labels, test_probs)
    axes[1].plot(fpr, tpr, label=f"AUC = {auc:.3f}")
    axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
    axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title("ROC Curve"); axes[1].legend()

hist_df = pd.DataFrame(history)
axes[2].plot(hist_df["epoch"], hist_df["train_loss"], label="Train Loss")
axes[2].plot(hist_df["epoch"], hist_df["val_f1@0.5"], label="Val F1 @ 0.5")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Value")
axes[2].set_title("Training Curves"); axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "evaluation_plots.png"), dpi=150)
plt.show()

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(" - best_model.pt                          (weights, threshold=0.5, context_radius=4)")
print(" - channel_mean.npy / channel_std.npy")
print(" - training_history.csv")
print(" - test_metrics.csv                       (raw @0.5 and smoothed @0.5)")
print(" - classification_report_raw.txt / classification_report_smoothed.txt")
print(" - event_level_metrics.csv")
print(" - evaluation_plots.png")

In [ ]:
"""
=============================================================================
 SLEEP APNEA / HYPOPNEA DETECTION — v5
 (supersedes v4; every change below is a response to a specific defect
  visible in the v4 run log, not a blind hyperparameter tweak)
=============================================================================

WHAT WAS WRONG IN v4, AND WHAT v5 DOES ABOUT IT
-----------------------------------------------------------------------------
[D1] THE AHI PROXY WAS MEASURING THE WRONG THING.
     v4 defined an "event" as a contiguous run of positive epochs. In the
     ground truth those runs average 3.87 epochs = 116 s -- far longer than
     any single apnea (10-90 s). The labels therefore encode CLUSTERS of
     merged events, and counting clusters gave 7.4/h, which bins literally
     every patient as Normal or Mild. That is why quadratic-kappa was 0.041
     and ICC 0.157 while Spearman rho was a healthy 0.815: the RANKING was
     fine, the SCALE was wrong.
     -> v5 uses the APNEA-EPOCH INDEX as the primary AHI proxy: positive
        epochs per hour. A 30 s epoch is flagged when an event overlaps it,
        and events average ~25 s, so one positive epoch ~ one event. On the
        v4 test set this gives 28.6/h pooled, which spans the real severity
        range. The cluster index is still reported as a secondary column.

[D2] EVENT RECALL WAS CAPPED BY POST-PROCESSING, NOT BY THE MODEL.
     v4 predicted 290 events against 805 true ones, so one-to-one matching
     could not score recall above 0.360; it got 0.278. MERGE_GAP_EPOCHS=1
     was actively fusing neighbouring distinct events.
     -> v5 (a) defaults to no merging, (b) SPLITS predicted runs longer than
        a tuned maximum, since a run of 5+ epochs cannot be one event, and
        (c) tunes the event operating point SEPARATELY from the window one.
        Both strict one-to-one and lenient any-overlap matching are reported.

[D3] SMOOTHING WAS ASSUMED TO HELP. IT HURT.
     v4 hard-coded radius-1 smoothing and lost F1: 0.6734 at the default
     threshold vs 0.6511 smoothed. It also tuned the threshold on val and
     landed on a worse test point than 0.5.
     -> v5 SEARCHES the post-processing space on validation (smoothing
        radius x threshold x min length x merge gap x split length) and
        applies the winner unchanged to test. Nothing is assumed.

[D4] THE MODEL OVERFIT FROM EPOCH ~7.
     Train loss kept falling (0.316 -> 0.120) while val F1 peaked at epoch
     6-8 and decayed. 762k params against 60 patients.
     -> v5 adds stochastic depth, higher dropout/weight decay, stronger
        augmentation, a Tversky term that optimises overlap directly, an
        auxiliary chunk-burden head, and selects checkpoints on val AUPRC
        (threshold-free) instead of val F1 (threshold-noisy).

[D5] THE ENSEMBLE ADDED ALMOST NOTHING.
     0.6511 vs a per-seed mean of 0.6457: three identical configs make
     correlated errors.
     -> v5 varies the CONTEXT LENGTH across ensemble members (20 / 40 / 60
        epochs = 10 / 20 / 30 min) as well as the seed. Different receptive
        fields make genuinely different mistakes, which is what averaging
        needs to pay off.

[D6] NORMAL-SEVERITY PATIENTS WERE NEAR-RANDOM (window F1 0.273).
     -> v5 feeds an explicit PHYSIOLOGICAL FEATURE STREAM alongside the
        learned tokens: airflow amplitude relative to the patient's own
        5-minute rolling baseline, thoraco-abdominal paradox, SpO2
        desaturation depth and slope, pulse surge. These are the quantities
        a scorer actually reads, they are patient-relative by construction,
        and they carry most of the signal on low-prevalence patients where
        the learned features have little positive data to latch onto.

OUTPUT: every table is PRINTED in full and saved to CSV; every figure is
both saved and shown inline.

  Smoke test  :  SMOKE_TEST=1 python apnea_v5.py
  Real run    :  python apnea_v5.py
  Grouped CV  :  CV_FOLDS=5 python apnea_v5.py
=============================================================================
"""

import os
import gc
import json
import time
import math
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import resample_poly
from scipy.ndimage import uniform_filter1d, maximum_filter1d, median_filter
from scipy.stats import pearsonr, spearmanr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score, brier_score_loss,
)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)

# =========================================================================
# 1.  CONFIGURATION
# =========================================================================
SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"
CV_FOLDS = int(os.environ.get("CV_FOLDS", "0"))
CACHE_VERSION = "v5"                     # bump -> cache rebuilds

SOURCE_DATASET = "nishantomhare/detection-of-disordered-breathing-dataset2"

if SMOKE_TEST:
    WORK_DIR = os.environ.get("SMOKE_WORK", "/tmp/apnea_v5_smoke")
else:
    WORK_DIR = "/kaggle/working"
OUTPUT_DIR = os.path.join(WORK_DIR, "apnea_v5_output")
CACHE_DIR = os.path.join(WORK_DIR, f"apnea_cache_{CACHE_VERSION}")
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
for d in (OUTPUT_DIR, CACHE_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

# ---- source layout ------------------------------------------------------
SRC_FS = 200
WINDOW_SEC = 30
SRC_SPE = SRC_FS * WINDOW_SEC
FIXED_CHANNEL_KEYS = ["flow", "chest", "abd", "spo2", "pleth", "ecg", "pulse", "flow2"]

# ---- multi-rate cache ---------------------------------------------------
RESP_KEYS = ["flow", "chest", "abd", "pleth", "flow2"]
SLOW_KEYS = ["spo2", "spo2_rel", "pulse", "ecg_env"]
RESP_FS, SLOW_FS = 32, 4
RESP_SPE, SLOW_SPE = RESP_FS * WINDOW_SEC, SLOW_FS * WINDOW_SEC   # 960, 120
N_RESP, N_SLOW = len(RESP_KEYS), len(SLOW_KEYS)

FEATURE_NAMES = [
    "flow_amp_ratio", "flow2_amp_ratio", "chest_amp_ratio", "abd_amp_ratio",
    "pleth_amp_ratio", "flow_min_sub_ratio", "chest_abd_corr", "paradox",
    "spo2_desat_depth", "spo2_min_slope", "spo2_rel_mean",
    "pulse_delta", "pulse_surge", "ecg_env_var_ratio",
]
N_FEAT = len(FEATURE_NAMES)

SPO2_VALID_RANGE = (50.0, 100.0)
SPO2_BASELINE_SEC = 180
BASELINE_EPOCHS = 10                     # +/- 10 epochs = 5 min rolling baseline
FLATLINE_MIN_SEC = 1.0

# ---- exclusions ---------------------------------------------------------
SKIP_PATIENTS = {
    "SN1": "corrupted / missing channels", "SN13": "corrupted / missing channels",
    "SN14": "corrupted / missing channels", "SN17": "corrupted / missing channels",
    "SN18": "corrupted / missing channels", "SN19": "corrupted / missing channels",
    "SN20": "corrupted / missing channels", "SN27": "corrupted / missing channels",
    "SN28": "corrupted / missing channels", "SN89": "corrupted / missing channels",
    "SN53": "zero labelled events across ~900 epochs -> annotation failure",
}
MIN_POSITIVE_EPOCHS = 5

# ---- split --------------------------------------------------------------
SEED = 42
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.68, 0.16, 0.16
N_STRAT_BINS = 5
SPLIT_JSON = os.path.join(OUTPUT_DIR, "patient_split.json")

# ---- ensemble members: (seed, context length in epochs) -----------------
# [D5] varying the receptive field is what makes averaging worth anything
RUN_CONFIGS = [(42, 20)] if SMOKE_TEST else [(42, 40), (1337, 20), (2024, 60)]

TRAIN_STRIDE_FRAC = 0.25                 # 4x more training chunks than v4
INFER_STRIDE_FRAC = 0.25

# ---- model --------------------------------------------------------------
TOKEN_DIM = 128
RESP_WIDTH, SLOW_WIDTH, FEAT_DIM = 32, 16, 32
GRU_HIDDEN, GRU_LAYERS = 96, 2
ATTN_HEADS, ATTN_LAYERS = 4, 1
DROPOUT = 0.30                            # [D4] 0.25 -> 0.30
STOCH_DEPTH = 0.10                        # [D4] new

# ---- training -----------------------------------------------------------
EPOCHS = 6 if SMOKE_TEST else 26
PATIENCE = 7
BATCH_SIZE = 16
LR = 1.5e-3
WEIGHT_DECAY = 0.02                       # [D4] 1e-4 -> 0.02
GRAD_CLIP = 2.0
WARMUP_EPOCHS = 2
NUM_WORKERS = 0 if SMOKE_TEST else 2
EMA_DECAY = 0.995
FOCAL_GAMMA = 1.5
POS_WEIGHT_CAP = 3.0
TVERSKY_WEIGHT = 0.5                      # [D4] optimise overlap directly
TVERSKY_ALPHA, TVERSKY_BETA = 0.3, 0.7    # beta > alpha -> penalise FN harder
AUX_WEIGHT = 0.2                          # chunk-burden auxiliary head
LABEL_SMOOTH_BOUNDARY = 0.10
SELECT_METRIC = "auprc"                   # [D4] threshold-free selection

# ---- augmentation -------------------------------------------------------
AUG_NOISE_STD = 0.04
AUG_SCALE_JITTER = 0.15
AUG_CHANNEL_DROP_P = 0.15
AUG_SIGN_FLIP_P = 0.30
AUG_TIME_MASK_P = 0.30
AUG_TIME_MASK_MAX_EP = 4

# ---- post-processing search space [D2][D3] ------------------------------
PP_SMOOTH_RADII = [0, 1, 2]
PP_THRESHOLDS = np.round(np.arange(0.15, 0.86, 0.02), 3)
PP_MIN_LENS = [1, 2]
PP_MERGE_GAPS = [0, 1]
PP_SPLIT_MAX = [2, 3, 4, 999]             # 999 = never split
EVENT_MIN_OVERLAP = 1

# ---- AHI / severity [D1] ------------------------------------------------
AHI_MODE = "epoch_index"                  # positive epochs per hour
SEVERITY_EDGES = [5.0, 15.0, 30.0]
SEVERITY_NAMES = ["Normal", "Mild", "Moderate", "Severe"]

SEP = "=" * 110


# =========================================================================
# 2.  REPRODUCIBILITY / DEVICE
# =========================================================================
def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


set_all_seeds(SEED)


def gpu_usable():
    if not torch.cuda.is_available():
        return False
    try:
        conv = nn.Conv1d(4, 4, 3, padding=1).cuda()
        _ = conv(torch.randn(2, 4, 16, device="cuda"))
        return True
    except RuntimeError as e:
        print("GPU kernel check failed:", str(e).splitlines()[0])
        return False


DEVICE = torch.device("cuda" if gpu_usable() else "cpu")


# =========================================================================
# 3.  SIGNAL UTILITIES
# =========================================================================
def fit_length(x, n):
    if len(x) == n:
        return x
    if len(x) > n:
        return x[:n]
    if len(x) == 0:
        return np.zeros(n, dtype=np.float32)
    return np.concatenate([x, np.full(n - len(x), x[-1], dtype=x.dtype)])


def decimate_to(x, fs_in, fs_out):
    g = math.gcd(int(fs_in), int(fs_out))
    up, down = int(fs_out // g), int(fs_in // g)
    if up == 1 and down == 1:
        return x.astype(np.float32)
    return resample_poly(x.astype(np.float32), up, down).astype(np.float32)


def block_mean(x, factor):
    n = (len(x) // factor) * factor
    if n == 0:
        return np.zeros(1, dtype=np.float32)
    return x[:n].reshape(-1, factor).mean(axis=1).astype(np.float32)


def flatline_mask(sig, min_run):
    n = len(sig)
    if n < 2:
        return np.zeros(n, dtype=bool)
    changed = np.concatenate(([True], sig[1:] != sig[:-1]))
    bounds = np.concatenate((np.flatnonzero(changed), [n]))
    mask = np.zeros(n, dtype=bool)
    for start, run in zip(bounds[:-1], np.diff(bounds)):
        if run >= min_run:
            mask[start:start + run] = True
    return mask


def interp_bad(sig, bad):
    if not bad.any() or bad.all():
        return sig
    good, bad_i = np.flatnonzero(~bad), np.flatnonzero(bad)
    sig = sig.copy()
    sig[bad_i] = np.interp(bad_i, good, sig[good])
    return sig


def robust_normalize(x, clip=8.0):
    med = np.median(x)
    scale = (np.percentile(x, 75) - np.percentile(x, 25)) / 1.349
    if not np.isfinite(scale) or scale < 1e-6:
        scale = np.std(x) + 1e-6
    return np.clip((x - med) / scale, -clip, clip).astype(np.float32)


def clean_spo2(sig):
    lo, hi = SPO2_VALID_RANGE
    bad = ~np.isfinite(sig) | (sig < lo) | (sig > hi)
    bad |= flatline_mask(np.nan_to_num(sig), int(30 * SRC_FS))
    return interp_bad(np.nan_to_num(sig, nan=lo), bad), bad


def clean_pleth(sig):
    bad = ~np.isfinite(sig)
    bad |= flatline_mask(np.nan_to_num(sig), int(FLATLINE_MIN_SEC * SRC_FS))
    return interp_bad(np.nan_to_num(sig), bad), bad


def ecg_qrs_envelope(sig, fs_in, fs_out):
    d = np.abs(np.diff(sig, prepend=sig[0]))
    env = uniform_filter1d(d, size=max(3, int(0.15 * fs_in)))
    return decimate_to(env, fs_in, fs_out)


def rolling_baseline(per_epoch, half_width=BASELINE_EPOCHS):
    """Median over +/- half_width epochs. [D6] Every amplitude feature is
    expressed relative to THIS, not to a cohort-wide constant -- an airflow
    signal has no absolute units."""
    size = 2 * half_width + 1
    if len(per_epoch) < 3:
        return np.full_like(per_epoch, np.median(per_epoch) if len(per_epoch) else 1.0)
    return median_filter(per_epoch, size=min(size, len(per_epoch)), mode="nearest")


# =========================================================================
# 4.  PHYSIOLOGICAL FEATURE STREAM  [D6]
# =========================================================================
def build_features(resp_ep, slow_ep):
    """resp_ep (n, N_RESP, RESP_SPE) float32, slow_ep (n, N_SLOW, SLOW_SPE)
       -> (n, N_FEAT) float32. All quantities are patient-relative."""
    n = resp_ep.shape[0]
    feats = np.zeros((n, N_FEAT), dtype=np.float32)
    eps = 1e-6

    def amp_ratio(ch):
        amp = np.abs(resp_ep[:, ch, :] - resp_ep[:, ch, :].mean(axis=1, keepdims=True)).mean(axis=1)
        return np.clip(amp / (rolling_baseline(amp) + eps), 0, 3.0)

    # RESP_KEYS order is flow(0) chest(1) abd(2) pleth(3) flow2(4)
    feats[:, 0] = amp_ratio(0)
    feats[:, 1] = amp_ratio(4)
    feats[:, 2] = amp_ratio(1)
    feats[:, 3] = amp_ratio(2)
    feats[:, 4] = amp_ratio(3)

    # worst 5 s sub-window of airflow: a 12 s apnea inside a 30 s epoch is
    # invisible to the epoch mean but obvious here
    sub = resp_ep[:, 0, :].reshape(n, 6, RESP_SPE // 6)
    sub_amp = np.abs(sub - sub.mean(axis=2, keepdims=True)).mean(axis=2)
    base = rolling_baseline(sub_amp.mean(axis=1))
    feats[:, 5] = np.clip(sub_amp.min(axis=1) / (base + eps), 0, 3.0)

    # thoraco-abdominal paradox: effort continues but out of phase -> obstruction
    c = resp_ep[:, 1, :] - resp_ep[:, 1, :].mean(axis=1, keepdims=True)
    a = resp_ep[:, 2, :] - resp_ep[:, 2, :].mean(axis=1, keepdims=True)
    corr = (c * a).sum(axis=1) / (np.sqrt((c ** 2).sum(axis=1) * (a ** 2).sum(axis=1)) + eps)
    feats[:, 6] = np.clip(corr, -1, 1)
    feats[:, 7] = np.clip(-corr, 0, 1)

    spo2_rel = slow_ep[:, 1, :]
    feats[:, 8] = np.clip(spo2_rel.min(axis=1), -6, 1)                 # desat depth
    feats[:, 9] = np.clip(np.diff(slow_ep[:, 0, :], axis=1).min(axis=1) * SLOW_FS, -3, 1)
    feats[:, 10] = np.clip(spo2_rel.mean(axis=1), -6, 1)

    pulse = slow_ep[:, 2, :]
    pm = pulse.mean(axis=1)
    feats[:, 11] = np.clip(pm - rolling_baseline(pm), -4, 4)
    feats[:, 12] = np.clip(pulse.max(axis=1) - rolling_baseline(pm), -4, 6)

    ev = slow_ep[:, 3, :].std(axis=1)
    feats[:, 13] = np.clip(ev / (rolling_baseline(ev) + eps), 0, 3.0)

    return np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


# =========================================================================
# 5.  STAGE 1 — MULTI-RATE CACHE
# =========================================================================
def cache_paths(pid):
    return {k: os.path.join(CACHE_DIR, f"{pid}_{k}.npy")
            for k in ("resp", "slow", "feat", "y", "mask")}


def cache_complete(pid):
    return all(os.path.exists(p) for p in cache_paths(pid).values())


def build_cache_for_patient(src_dir, pid):
    p = cache_paths(pid)
    X = np.load(os.path.join(src_dir, f"X_{pid}.npy"), mmap_mode="r")
    y = np.load(os.path.join(src_dir, f"y_{pid}.npy")).astype(np.int8)
    n_ep = min(X.shape[0], len(y))
    y = y[:n_ep]

    resp = np.zeros((n_ep, N_RESP, RESP_SPE), dtype=np.float32)
    slow = np.zeros((n_ep, N_SLOW, SLOW_SPE), dtype=np.float32)
    mask = np.ones(N_RESP + N_SLOW, dtype=np.float32)

    def continuous(key):
        ch = FIXED_CHANNEL_KEYS.index(key)
        return np.asarray(X[:n_ep, ch, :], dtype=np.float32).reshape(-1)

    for i, key in enumerate(RESP_KEYS):
        sig = continuous(key)
        if not np.isfinite(sig).any() or np.allclose(sig, 0.0):
            mask[i] = 0.0
            continue
        sig = np.nan_to_num(sig)
        if key == "pleth":
            sig, _ = clean_pleth(sig)
        ds = fit_length(decimate_to(sig, SRC_FS, RESP_FS), n_ep * RESP_SPE)
        resp[:, i, :] = robust_normalize(ds).reshape(n_ep, RESP_SPE)
        del sig, ds

    spo2_ds = None
    for j, key in enumerate(SLOW_KEYS):
        src_key = {"ecg_env": "ecg", "spo2_rel": "spo2"}.get(key, key)
        sig = continuous(src_key)
        if not np.isfinite(sig).any() or np.allclose(sig, 0.0):
            mask[N_RESP + j] = 0.0
            continue
        sig = np.nan_to_num(sig)
        if key == "spo2":
            sig, _ = clean_spo2(sig)
            spo2_ds = fit_length(block_mean(sig, SRC_FS // SLOW_FS), n_ep * SLOW_SPE)
            ds = np.clip((spo2_ds - 100.0) / 10.0, -4.0, 0.5)
        elif key == "spo2_rel":
            if spo2_ds is None:
                mask[N_RESP + j] = 0.0
                continue
            base = maximum_filter1d(spo2_ds, size=SPO2_BASELINE_SEC * SLOW_FS, mode="nearest")
            ds = np.clip((spo2_ds - base) / 3.0, -6.0, 0.5)
        elif key == "ecg_env":
            ds = robust_normalize(fit_length(ecg_qrs_envelope(sig, SRC_FS, SLOW_FS),
                                             n_ep * SLOW_SPE))
        else:
            ds = robust_normalize(fit_length(block_mean(sig, SRC_FS // SLOW_FS),
                                             n_ep * SLOW_SPE))
        slow[:, j, :] = fit_length(np.asarray(ds, np.float32), n_ep * SLOW_SPE).reshape(n_ep, SLOW_SPE)
        del sig, ds

    resp *= mask[None, :N_RESP, None]
    slow *= mask[None, N_RESP:, None]
    feat = build_features(resp, slow)

    np.save(p["resp"], resp.astype(np.float16))
    np.save(p["slow"], slow.astype(np.float16))
    np.save(p["feat"], feat)
    np.save(p["y"], y)
    np.save(p["mask"], mask)
    del X, resp, slow
    gc.collect()
    return n_ep, int(y.sum()), mask


def build_cache(src_dir, pids, force=False):
    rows, t0 = [], time.time()
    for k, pid in enumerate(pids, 1):
        if cache_complete(pid) and not force:
            y = np.load(cache_paths(pid)["y"])
            mask = np.load(cache_paths(pid)["mask"])
            n_ep, n_pos, status = len(y), int(y.sum()), "cached"
        else:
            n_ep, n_pos, mask = build_cache_for_patient(src_dir, pid)
            status = "built"
        hours = n_ep * WINDOW_SEC / 3600.0
        rows.append({
            "patient": pid, "epochs": n_ep, "hours": round(hours, 2),
            "positive_epochs": n_pos, "apnea_ratio": round(n_pos / max(1, n_ep), 4),
            "AHI_epoch_index": round(n_pos / max(1e-6, hours), 2),
            "severity": SEVERITY_NAMES[int(np.searchsorted(
                SEVERITY_EDGES, n_pos / max(1e-6, hours), side="right"))],
            "missing_channels": ",".join(
                [k2 for k2, m in zip(RESP_KEYS + SLOW_KEYS, mask) if m == 0.0]) or "none",
            "status": status,
        })
        if k % 10 == 0 or k == len(pids):
            print(f"    cache {k}/{len(pids)} patients  ({time.time() - t0:.0f}s)")
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(OUTPUT_DIR, "cache_manifest.csv"), index=False)
    return df


# =========================================================================
# 6.  SPLITS
# =========================================================================
def stratified_split(pid_ratio, seed, fracs=(TRAIN_FRAC, VAL_FRAC, TEST_FRAC)):
    rng = np.random.default_rng(seed)
    pids = sorted(pid_ratio, key=lambda p: pid_ratio[p])
    edges = np.linspace(0, len(pids), N_STRAT_BINS + 1).astype(int)
    train, val, test = [], [], []
    for b in range(N_STRAT_BINS):
        bucket = list(rng.permutation(pids[edges[b]:edges[b + 1]]))
        nb = len(bucket)
        n_tr = min(int(round(fracs[0] * nb)), nb)
        n_va = min(int(round(fracs[1] * nb)), nb - n_tr)
        train += bucket[:n_tr]
        val += bucket[n_tr:n_tr + n_va]
        test += bucket[n_tr + n_va:]
    return sorted(train), sorted(val), sorted(test)


def grouped_folds(pid_ratio, n_folds, seed):
    rng = np.random.default_rng(seed)
    pids = sorted(pid_ratio, key=lambda p: pid_ratio[p])
    folds = [[] for _ in range(n_folds)]
    for i in range(0, len(pids), n_folds):
        for j, pid in enumerate(rng.permutation(pids[i:i + n_folds])):
            folds[j % n_folds].append(pid)
    return folds


# =========================================================================
# 7.  DATASET
# =========================================================================
class ChunkDataset(Dataset):
    def __init__(self, pids, chunk_len, stride, augment=False,
                 soft_boundary=False, rng_seed=0):
        self.pids, self.L, self.stride = list(pids), chunk_len, stride
        self.augment, self.soft_boundary = augment, soft_boundary
        self.rng = np.random.default_rng(rng_seed)
        self._mm = {}
        self.n_epochs = {p: len(np.load(cache_paths(p)["y"], mmap_mode="r"))
                         for p in self.pids}
        self.resample(jitter=False)

    def resample(self, jitter=True):
        self.index = []
        for pid in self.pids:
            n = self.n_epochs[pid]
            off = int(self.rng.integers(0, self.stride)) if jitter else 0
            starts = list(range(off, max(1, n - self.L + 1), self.stride))
            if not starts or starts[-1] + self.L < n:
                starts.append(max(0, n - self.L))
            self.index += [(pid, s) for s in starts]

    def _arrays(self, pid):
        if pid not in self._mm:
            p = cache_paths(pid)
            self._mm[pid] = (np.load(p["resp"], mmap_mode="r"),
                             np.load(p["slow"], mmap_mode="r"),
                             np.load(p["feat"], mmap_mode="r"),
                             np.load(p["y"], mmap_mode="r"),
                             np.load(p["mask"]))
        return self._mm[pid]

    def __len__(self):
        return len(self.index)

    def labels(self):
        return np.concatenate([np.load(cache_paths(p)["y"]) for p in self.pids]) \
            if self.pids else np.array([], dtype=np.int8)

    def __getitem__(self, i):
        pid, s = self.index[i]
        resp_mm, slow_mm, feat_mm, y_mm, mask = self._arrays(pid)
        n, L = len(y_mm), self.L
        e = min(s + L, n)
        k = e - s

        resp = np.zeros((N_RESP, L * RESP_SPE), dtype=np.float32)
        slow = np.zeros((N_SLOW, L * SLOW_SPE), dtype=np.float32)
        feat = np.zeros((N_FEAT, L), dtype=np.float32)
        y = np.zeros(L, dtype=np.float32)
        valid = np.zeros(L, dtype=np.float32)

        resp[:, :k * RESP_SPE] = np.asarray(resp_mm[s:e], np.float32).transpose(1, 0, 2).reshape(N_RESP, -1)
        slow[:, :k * SLOW_SPE] = np.asarray(slow_mm[s:e], np.float32).transpose(1, 0, 2).reshape(N_SLOW, -1)
        feat[:, :k] = np.asarray(feat_mm[s:e], np.float32).T
        y_raw = np.asarray(y_mm[s:e], np.float32)
        y[:k], valid[:k] = y_raw, 1.0

        if self.soft_boundary and k > 1:
            tr = np.zeros(k, dtype=bool)
            tr[:-1] |= y_raw[:-1] != y_raw[1:]
            tr[1:] |= y_raw[1:] != y_raw[:-1]
            y[:k] = np.where(tr, np.where(y_raw > .5, 1 - LABEL_SMOOTH_BOUNDARY,
                                          LABEL_SMOOTH_BOUNDARY), y_raw)

        resp *= mask[:N_RESP, None]
        slow *= mask[N_RESP:, None]

        if self.augment:
            resp, slow, feat, valid = self._augment(resp, slow, feat, valid)

        return (torch.from_numpy(resp), torch.from_numpy(slow),
                torch.from_numpy(feat), torch.from_numpy(y), torch.from_numpy(valid))

    def _augment(self, resp, slow, feat, valid):
        r = self.rng
        resp *= (1 + (r.random((N_RESP, 1)) * 2 - 1) * AUG_SCALE_JITTER).astype(np.float32)
        slow *= (1 + (r.random((N_SLOW, 1)) * 2 - 1) * AUG_SCALE_JITTER).astype(np.float32)
        resp += r.normal(0, AUG_NOISE_STD, resp.shape).astype(np.float32)
        slow += r.normal(0, AUG_NOISE_STD * .5, slow.shape).astype(np.float32)
        feat += r.normal(0, AUG_NOISE_STD, feat.shape).astype(np.float32)
        if r.random() < AUG_SIGN_FLIP_P:
            resp *= -1.0
        if r.random() < AUG_CHANNEL_DROP_P:
            if r.random() < .7:
                resp[r.integers(0, N_RESP)] = 0.0
            else:
                slow[r.integers(0, N_SLOW)] = 0.0
        if r.random() < AUG_TIME_MASK_P:
            w = int(r.integers(1, AUG_TIME_MASK_MAX_EP + 1))
            st = int(r.integers(0, max(1, self.L - w)))
            resp[:, st * RESP_SPE:(st + w) * RESP_SPE] = 0.0
            slow[:, st * SLOW_SPE:(st + w) * SLOW_SPE] = 0.0
            feat[:, st:st + w] = 0.0
            valid[st:st + w] = 0.0
        return resp, slow, feat, valid


def _worker_init(worker_id):
    info = torch.utils.data.get_worker_info()
    if info is not None and hasattr(info.dataset, "rng"):
        info.dataset.rng = np.random.default_rng((info.seed + worker_id * 9973) % (2 ** 32))


# =========================================================================
# 8.  MODEL
# =========================================================================
class SE1D(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        h = max(4, c // r)
        self.fc = nn.Sequential(nn.Linear(c, h), nn.SiLU(), nn.Linear(h, c), nn.Sigmoid())

    def forward(self, x):
        return x * self.fc(x.mean(dim=-1)).unsqueeze(-1)


def conv_block(cin, cout, k, stride):
    return nn.Sequential(
        nn.Conv1d(cin, cout, k, stride=stride, padding=(k - 1) // 2, bias=False),
        nn.BatchNorm1d(cout), nn.SiLU(inplace=True), SE1D(cout))


class StreamEncoder(nn.Module):
    def __init__(self, cin, width, strides, kernels, out_dim):
        super().__init__()
        layers, c = [], cin
        for i, (s, k) in enumerate(zip(strides, kernels)):
            cout = out_dim if i == len(strides) - 1 else width * (2 ** min(i, 2))
            layers.append(conv_block(c, cout, k, s))
            c = cout
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class DilatedBlock(nn.Module):
    """[D4] with stochastic depth: the whole branch is dropped at random
    during training, which is a strong regulariser for deep residual stacks
    trained on only 60 patients."""
    def __init__(self, d, k=5, dilation=1, dropout=DROPOUT, drop_path=STOCH_DEPTH):
        super().__init__()
        pad = (k - 1) // 2 * dilation
        self.dw = nn.Conv1d(d, d, k, padding=pad, dilation=dilation, groups=d, bias=False)
        self.pw = nn.Conv1d(d, d, 1, bias=False)
        self.bn = nn.BatchNorm1d(d)
        self.act = nn.SiLU(inplace=True)
        self.drop = nn.Dropout(dropout)
        self.drop_path = drop_path

    def forward(self, x):
        out = self.drop(self.act(self.bn(self.pw(self.dw(x)))))
        if self.training and self.drop_path > 0:
            keep = torch.rand(x.shape[0], 1, 1, device=x.device) >= self.drop_path
            out = out * keep / (1 - self.drop_path)
        return x + out


class ApneaSegmenterV5(nn.Module):
    """(resp, slow, feat) -> one logit per 30 s epoch, plus an auxiliary
    chunk-level burden logit."""
    def __init__(self, token_dim=TOKEN_DIM):
        super().__init__()
        self.resp_enc = StreamEncoder(N_RESP, RESP_WIDTH, [4, 4, 4, 5, 3],
                                      [15, 9, 9, 7, 7], token_dim)          # 960 -> 1
        self.slow_enc = StreamEncoder(N_SLOW, SLOW_WIDTH, [4, 5, 6],
                                      [7, 5, 5], token_dim // 2)            # 120 -> 1
        self.feat_enc = nn.Sequential(
            nn.Conv1d(N_FEAT, FEAT_DIM, 1, bias=False),
            nn.BatchNorm1d(FEAT_DIM), nn.SiLU(inplace=True),
            nn.Conv1d(FEAT_DIM, FEAT_DIM, 3, padding=1, bias=False),
            nn.BatchNorm1d(FEAT_DIM), nn.SiLU(inplace=True))
        d = token_dim + token_dim // 2 + FEAT_DIM
        self.proj = nn.Sequential(nn.Conv1d(d, token_dim, 1, bias=False),
                                  nn.BatchNorm1d(token_dim), nn.SiLU(inplace=True))
        self.temporal = nn.Sequential(*[DilatedBlock(token_dim, dilation=dl)
                                        for dl in (1, 2, 4, 8, 16)])
        self.gru = nn.GRU(token_dim, GRU_HIDDEN, num_layers=GRU_LAYERS, batch_first=True,
                          bidirectional=True, dropout=DROPOUT if GRU_LAYERS > 1 else 0.0)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=2 * GRU_HIDDEN, nhead=ATTN_HEADS, dim_feedforward=4 * GRU_HIDDEN,
            dropout=DROPOUT, batch_first=True, norm_first=True, activation="gelu")
        self.attn = nn.TransformerEncoder(enc_layer, num_layers=ATTN_LAYERS)
        self.head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(2 * GRU_HIDDEN, 1))
        self.aux = nn.Linear(2 * GRU_HIDDEN, 1)

    def forward(self, resp, slow, feat):
        rt = self.resp_enc(resp)
        st = self.slow_enc(slow)
        ft = self.feat_enc(feat)
        L = min(rt.shape[-1], st.shape[-1], ft.shape[-1])
        t = self.proj(torch.cat([rt[..., :L], st[..., :L], ft[..., :L]], dim=1))
        t = self.temporal(t).transpose(1, 2)
        t, _ = self.gru(t)
        t = self.attn(t)
        return self.head(t).squeeze(-1), self.aux(t.mean(dim=1)).squeeze(-1)


class EMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()}

    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone().float()

    def copy_to(self, model):
        sd = model.state_dict()
        model.load_state_dict({k: self.shadow[k].to(sd[k].dtype) for k in sd})


# =========================================================================
# 9.  LOSS  [D4]
# =========================================================================
class CombinedLoss(nn.Module):
    """Weighted focal (per-epoch calibration) + Tversky (sequence overlap,
    beta>alpha so false negatives cost more than false positives) + an
    auxiliary chunk-burden term that pushes the model to get each patient's
    OVERALL event rate right, which is what the AHI actually measures."""
    def __init__(self, pos_weight, gamma=FOCAL_GAMMA):
        super().__init__()
        self.register_buffer("pos_weight", torch.as_tensor(pos_weight, dtype=torch.float32))
        self.gamma = gamma

    def forward(self, logits, aux_logit, targets, valid):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction="none")
        p = torch.sigmoid(logits)
        pt = p * targets + (1 - p) * (1 - targets)
        focal = (((1 - pt).clamp(min=1e-6) ** self.gamma) * bce * valid).sum() / valid.sum().clamp(min=1)

        tp = (p * targets * valid).sum()
        fp = (p * (1 - targets) * valid).sum()
        fn = ((1 - p) * targets * valid).sum()
        tversky = 1 - (tp + 1.0) / (tp + TVERSKY_ALPHA * fp + TVERSKY_BETA * fn + 1.0)

        burden = (targets * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1)
        aux = F.binary_cross_entropy_with_logits(aux_logit, burden)
        return focal + TVERSKY_WEIGHT * tversky + AUX_WEIGHT * aux


# =========================================================================
# 10.  INFERENCE
# =========================================================================
@torch.no_grad()
def predict_patients(model, pids, chunk_len, batch_size=BATCH_SIZE, tta=True):
    """Overlap-averaged inference, optionally averaged with a sign-flipped
    pass (airflow polarity is arbitrary, so the model should be invariant)."""
    model.eval()
    stride = max(1, int(chunk_len * INFER_STRIDE_FRAC))
    out = {}
    for pid in pids:
        p = cache_paths(pid)
        resp_mm = np.load(p["resp"], mmap_mode="r")
        slow_mm = np.load(p["slow"], mmap_mode="r")
        feat_mm = np.load(p["feat"], mmap_mode="r")
        y = np.load(p["y"]).astype(np.int8)
        mask = np.load(p["mask"])
        n = len(y)

        starts = list(range(0, max(1, n - chunk_len + 1), stride))
        if not starts or starts[-1] + chunk_len < n:
            starts.append(max(0, n - chunk_len))

        acc = np.zeros(n); cnt = np.zeros(n)
        for b0 in range(0, len(starts), batch_size):
            batch = starts[b0:b0 + batch_size]
            R = np.zeros((len(batch), N_RESP, chunk_len * RESP_SPE), np.float32)
            S = np.zeros((len(batch), N_SLOW, chunk_len * SLOW_SPE), np.float32)
            Fx = np.zeros((len(batch), N_FEAT, chunk_len), np.float32)
            spans = []
            for bi, s in enumerate(batch):
                e = min(s + chunk_len, n); k = e - s
                R[bi, :, :k * RESP_SPE] = np.asarray(resp_mm[s:e], np.float32).transpose(1, 0, 2).reshape(N_RESP, -1)
                S[bi, :, :k * SLOW_SPE] = np.asarray(slow_mm[s:e], np.float32).transpose(1, 0, 2).reshape(N_SLOW, -1)
                Fx[bi, :, :k] = np.asarray(feat_mm[s:e], np.float32).T
                spans.append((s, e, k))
            R *= mask[None, :N_RESP, None]
            S *= mask[None, N_RESP:, None]
            Rt, St, Ft = (torch.from_numpy(R).to(DEVICE), torch.from_numpy(S).to(DEVICE),
                          torch.from_numpy(Fx).to(DEVICE))
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                logits, _ = model(Rt, St, Ft)
                probs = torch.sigmoid(logits.float())
                if tta:
                    logits2, _ = model(-Rt, St, Ft)
                    probs = 0.5 * (probs + torch.sigmoid(logits2.float()))
            probs = probs.cpu().numpy()
            for bi, (s, e, k) in enumerate(spans):
                acc[s:e] += probs[bi, :k]; cnt[s:e] += 1
        out[pid] = (y, (acc / np.maximum(cnt, 1e-9)).astype(np.float32))
        del resp_mm, slow_mm, feat_mm
    return out


# =========================================================================
# 11.  POST-PROCESSING  [D2][D3]
# =========================================================================
def smooth_probs(prob, radius):
    if radius <= 0 or len(prob) == 0:
        return prob
    k = np.ones(2 * radius + 1) / (2 * radius + 1)
    return np.convolve(np.pad(prob, radius, mode="edge"), k, mode="valid")


def extract_runs(seq):
    seq = np.asarray(seq).astype(int)
    if seq.size == 0:
        return []
    d = np.diff(np.concatenate(([0], seq, [0])))
    return list(zip(np.flatnonzero(d == 1).tolist(), (np.flatnonzero(d == -1) - 1).tolist()))


def postprocess(prob, cfg):
    """-> (binary sequence for window metrics, event list for event metrics).
    A run longer than split_max cannot be a single respiratory event (they
    top out around 90 s = 3 epochs), so it is divided into equal parts.
    [D2] This is what lifts event recall off its structural ceiling."""
    p = smooth_probs(prob, cfg["smooth"])
    runs = extract_runs((p >= cfg["thr"]).astype(int))
    if not runs:
        return np.zeros(len(prob), dtype=int), []

    merged = [list(runs[0])]
    for s, e in runs[1:]:
        if s - merged[-1][1] - 1 <= cfg["merge_gap"]:
            merged[-1][1] = e
        else:
            merged.append([s, e])
    merged = [(s, e) for s, e in merged if e - s + 1 >= cfg["min_len"]]

    binary = np.zeros(len(prob), dtype=int)
    events = []
    for s, e in merged:
        binary[s:e + 1] = 1
        L = e - s + 1
        k = max(1, int(math.ceil(L / cfg["split_max"])))
        bounds = np.linspace(s, e + 1, k + 1).astype(int)
        events += [(int(bounds[i]), int(bounds[i + 1]) - 1) for i in range(k)
                   if bounds[i + 1] - 1 >= bounds[i]]
    return binary, events


def true_events(y, split_max=999):
    """Ground-truth runs, split the same way so predicted and true event
    counts are produced by the SAME rule -- otherwise the comparison is
    rigged in one direction or the other."""
    ev = []
    for s, e in extract_runs(y):
        L = e - s + 1
        k = max(1, int(math.ceil(L / split_max)))
        b = np.linspace(s, e + 1, k + 1).astype(int)
        ev += [(int(b[i]), int(b[i + 1]) - 1) for i in range(k) if b[i + 1] - 1 >= b[i]]
    return ev


def match_events(pred_events, gt_events, one_to_one=True, min_overlap=EVENT_MIN_OVERLAP):
    pairs = []
    for pi, (ps, pe) in enumerate(pred_events):
        for ti, (ts, te) in enumerate(gt_events):
            ov = min(pe, te) - max(ps, ts) + 1
            if ov >= min_overlap:
                pairs.append((ov, pi, ti))
    if one_to_one:
        pairs.sort(reverse=True)
        up, ut, onset = set(), set(), []
        for ov, pi, ti in pairs:
            if pi in up or ti in ut:
                continue
            up.add(pi); ut.add(ti)
            onset.append(pred_events[pi][0] - gt_events[ti][0])
        tp_pred, tp_true = len(up), len(ut)
    else:
        up = {pi for _, pi, _ in pairs}
        ut = {ti for _, _, ti in pairs}
        onset = []
        tp_pred, tp_true = len(up), len(ut)
    return dict(tp=tp_true, tp_pred=tp_pred,
                fp=len(pred_events) - tp_pred, fn=len(gt_events) - tp_true,
                n_pred=len(pred_events), n_true=len(gt_events), onset=onset)


def event_scores(preds_by_pid, labels_by_pid, split_max, one_to_one=True):
    agg = dict(tp=0, tp_pred=0, fp=0, fn=0, n_pred=0, n_true=0)
    onset = []
    for pid, ev in preds_by_pid.items():
        m = match_events(ev, true_events(labels_by_pid[pid], split_max), one_to_one)
        for k in agg:
            agg[k] += m[k]
        onset += m["onset"]
    prec = agg["tp_pred"] / max(1, agg["n_pred"])
    rec = agg["tp"] / max(1, agg["n_true"])
    agg.update(precision=prec, recall=rec,
               f1=2 * prec * rec / (prec + rec) if prec + rec > 0 else 0.0,
               median_onset_error=float(np.median(onset)) if onset else float("nan"))
    return agg


def search_postprocessing(val_preds, objective="window"):
    """[D3] Selects smoothing radius, threshold, minimum length, merge gap and
    split length on VALIDATION. v4 assumed smoothing helped; on that run it
    cost 2.2 F1 points. Nothing here is assumed."""
    pids = sorted(val_preds)
    best, best_score = None, -1.0
    cache = {}
    for sm in PP_SMOOTH_RADII:
        cache[sm] = {p: smooth_probs(val_preds[p][1], sm) for p in pids}

    if objective == "window":
        y = np.concatenate([val_preds[p][0] for p in pids])
        for sm in PP_SMOOTH_RADII:
            pv = np.concatenate([cache[sm][p] for p in pids])
            prec, rec, thr = precision_recall_curve(y, pv)
            f1s = 2 * prec * rec / (prec + rec + 1e-9)
            i = int(np.nanargmax(f1s[:-1])) if len(thr) else 0
            score = float(f1s[i])
            if score > best_score:
                best_score = score
                best = {"smooth": sm, "thr": float(thr[i]) if len(thr) else 0.5,
                        "min_len": 1, "merge_gap": 0, "split_max": 999}
        return best, best_score

    labels = {p: val_preds[p][0].astype(int) for p in pids}
    for sm in PP_SMOOTH_RADII:
        for thr in PP_THRESHOLDS:
            for min_len in PP_MIN_LENS:
                for gap in PP_MERGE_GAPS:
                    for sx in PP_SPLIT_MAX:
                        cfg = {"smooth": sm, "thr": float(thr), "min_len": min_len,
                               "merge_gap": gap, "split_max": sx}
                        ev = {p: postprocess(cache[sm][p], cfg)[1] for p in pids}
                        sc = event_scores(ev, labels, sx)["f1"]
                        if sc > best_score:
                            best_score, best = sc, cfg
    return best, best_score


# =========================================================================
# 12.  METRICS  [D1]
# =========================================================================
def ahi_epoch_index(binary_or_labels, n_epochs=None):
    """PRIMARY AHI proxy: positive 30 s epochs per hour. See [D1]."""
    seq = np.asarray(binary_or_labels)
    n = n_epochs or len(seq)
    return float(seq.sum()) / max(1e-6, n * WINDOW_SEC / 3600.0)


def ahi_cluster_index(binary):
    return len(extract_runs(binary)) / max(1e-6, len(binary) * WINDOW_SEC / 3600.0)


def severity_of(ahi):
    return int(np.searchsorted(SEVERITY_EDGES, ahi, side="right"))


def window_metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    try:
        auroc = roc_auc_score(y, p) if len(np.unique(y)) > 1 else float("nan")
        auprc = average_precision_score(y, p) if len(np.unique(y)) > 1 else float("nan")
    except ValueError:
        auroc = auprc = float("nan")
    return {"threshold": float(thr), "accuracy": accuracy_score(y, pred),
            "balanced_accuracy": balanced_accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "specificity": tn / max(1, tn + fp),
            "f1": f1_score(y, pred, zero_division=0),
            "mcc": matthews_corrcoef(y, pred) if len(np.unique(pred)) > 1 else 0.0,
            "kappa": cohen_kappa_score(y, pred), "auroc": auroc, "auprc": auprc,
            "brier": brier_score_loss(y, np.clip(p, 0, 1)),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "n_windows": int(len(y))}


def icc21(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    n = len(a)
    if n < 2:
        return float("nan")
    M = np.column_stack([a, b]); grand = M.mean()
    ms_r = 2 * ((M.mean(axis=1) - grand) ** 2).sum() / (n - 1)
    ms_c = n * ((M.mean(axis=0) - grand) ** 2).sum()
    ms_e = ((M - M.mean(axis=1, keepdims=True) - M.mean(axis=0, keepdims=True) + grand) ** 2).sum() / (n - 1)
    den = ms_r + ms_e + 2 * (ms_c - ms_e) / n
    return float((ms_r - ms_e) / den) if den > 0 else float("nan")


# =========================================================================
# 13.  FULL EVALUATION
# =========================================================================
def evaluate(preds, win_cfg, ev_cfg, split_name="test"):
    pids = sorted(preds)
    y_all = np.concatenate([preds[p][0] for p in pids])
    p_raw = np.concatenate([preds[p][1] for p in pids])
    p_win = np.concatenate([smooth_probs(preds[p][1], win_cfg["smooth"]) for p in pids])

    res = {"split": split_name, "n_patients": len(pids),
           "win_cfg": win_cfg, "ev_cfg": ev_cfg}
    res["window_default"] = window_metrics(y_all, p_raw, 0.5)
    res["window_tuned"] = window_metrics(y_all, p_win, win_cfg["thr"])

    win_bin = {p: postprocess(preds[p][1], win_cfg)[0] for p in pids}
    res["window_postprocessed"] = window_metrics(
        y_all, np.concatenate([win_bin[p].astype(float) for p in pids]), 0.5)
    res["window_postprocessed"].update(
        auroc=res["window_tuned"]["auroc"], auprc=res["window_tuned"]["auprc"],
        brier=res["window_tuned"]["brier"])

    ev_out = {p: postprocess(preds[p][1], ev_cfg) for p in pids}
    ev_by_pid = {p: v[1] for p, v in ev_out.items()}
    ev_bin = {p: v[0] for p, v in ev_out.items()}
    labels = {p: preds[p][0].astype(int) for p in pids}
    res["event_strict"] = event_scores(ev_by_pid, labels, ev_cfg["split_max"], True)
    res["event_lenient"] = event_scores(ev_by_pid, labels, ev_cfg["split_max"], False)

    # ---- per patient -----------------------------------------------------
    rows = []
    for pid in pids:
        y = preds[pid][0]
        wm = window_metrics(y, smooth_probs(preds[pid][1], win_cfg["smooth"]), win_cfg["thr"])
        em = match_events(ev_by_pid[pid], true_events(y, ev_cfg["split_max"]), True)
        ep = em["tp_pred"] / max(1, em["n_pred"])
        er = em["tp"] / max(1, em["n_true"])
        t_ahi = ahi_epoch_index(y)
        p_ahi = ahi_epoch_index(win_bin[pid])
        rows.append({
            "patient": pid, "n_epochs": len(y), "hours": round(len(y) * WINDOW_SEC / 3600, 2),
            "true_ratio": round(float(y.mean()), 4),
            "true_AHI": round(t_ahi, 2), "pred_AHI": round(p_ahi, 2),
            "AHI_error": round(p_ahi - t_ahi, 2),
            "true_severity": SEVERITY_NAMES[severity_of(t_ahi)],
            "pred_severity": SEVERITY_NAMES[severity_of(p_ahi)],
            "true_cluster_index": round(ahi_cluster_index(y), 2),
            "win_precision": round(wm["precision"], 4), "win_recall": round(wm["recall"], 4),
            "win_f1": round(wm["f1"], 4), "win_auroc": round(wm["auroc"], 4),
            "win_auprc": round(wm["auprc"], 4), "win_mcc": round(wm["mcc"], 4),
            "event_precision": round(ep, 4), "event_recall": round(er, 4),
            "event_f1": round(2 * ep * er / (ep + er), 4) if ep + er > 0 else 0.0,
            "n_true_events": em["n_true"], "n_pred_events": em["n_pred"],
        })
    pp = pd.DataFrame(rows).sort_values("true_AHI").reset_index(drop=True)
    res["per_patient"] = pp

    # ---- AHI agreement ---------------------------------------------------
    ta, pa = pp["true_AHI"].values, pp["pred_AHI"].values
    diff = pa - ta
    res["ahi"] = {
        "pearson_r": float(pearsonr(ta, pa)[0]) if len(ta) > 2 else float("nan"),
        "spearman_r": float(spearmanr(ta, pa)[0]) if len(ta) > 2 else float("nan"),
        "icc_2_1": icc21(ta, pa), "bias": float(diff.mean()),
        "sd_diff": float(diff.std(ddof=1)) if len(diff) > 1 else float("nan"),
        "loa_low": float(diff.mean() - 1.96 * diff.std(ddof=1)) if len(diff) > 1 else float("nan"),
        "loa_high": float(diff.mean() + 1.96 * diff.std(ddof=1)) if len(diff) > 1 else float("nan"),
        "mae": float(np.abs(diff).mean()), "rmse": float(np.sqrt((diff ** 2).mean())),
    }

    ts = [severity_of(v) for v in ta]
    ps = [severity_of(v) for v in pa]
    res["severity"] = {
        "accuracy": float(np.mean(np.array(ts) == np.array(ps))),
        "within_one_class": float(np.mean(np.abs(np.array(ts) - np.array(ps)) <= 1)),
        "quadratic_kappa": float(cohen_kappa_score(ts, ps, weights="quadratic", labels=[0, 1, 2, 3])),
        "confusion": confusion_matrix(ts, ps, labels=[0, 1, 2, 3]),
    }

    # ---- severity-stratified --------------------------------------------
    strat = []
    for si, sname in enumerate(SEVERITY_NAMES):
        grp = [p for p in pids if severity_of(ahi_epoch_index(preds[p][0])) == si]
        if not grp:
            continue
        yg = np.concatenate([preds[p][0] for p in grp])
        pg = np.concatenate([smooth_probs(preds[p][1], win_cfg["smooth"]) for p in grp])
        wm = window_metrics(yg, pg, win_cfg["thr"])
        em = event_scores({p: ev_by_pid[p] for p in grp},
                          {p: labels[p] for p in grp}, ev_cfg["split_max"], True)
        strat.append({"severity": sname, "n_patients": len(grp), "n_epochs": len(yg),
                      "prevalence": round(float(yg.mean()), 4),
                      "win_precision": round(wm["precision"], 4),
                      "win_recall": round(wm["recall"], 4), "win_f1": round(wm["f1"], 4),
                      "win_auroc": round(wm["auroc"], 4), "win_auprc": round(wm["auprc"], 4),
                      "event_precision": round(em["precision"], 4),
                      "event_recall": round(em["recall"], 4), "event_f1": round(em["f1"], 4),
                      "mean_AHI_error": round(float(pp.loc[pp.patient.isin(grp), "AHI_error"].mean()), 2)})
    res["stratified"] = pd.DataFrame(strat)

    res["classification_report"] = classification_report(
        y_all, np.concatenate([win_bin[p] for p in pids]),
        target_names=["Normal", "Apnea"], zero_division=0, digits=4)
    res["_arrays"] = {"y": y_all, "p_raw": p_raw, "p_win": p_win,
                      "p_by_pid": {p: smooth_probs(preds[p][1], win_cfg["smooth"]) for p in pids},
                      "bin_by_pid": win_bin, "ev_by_pid": ev_by_pid, "ev_bin": ev_bin}
    return res


# =========================================================================
# 14.  PRINTING  (everything is printed AND saved)
# =========================================================================
def show(fig_path):
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    try:
        plt.show()
    except Exception:
        pass
    plt.close()


def print_full_report(res, history_df, seed_df, title):
    print("\n" + SEP)
    print(f"  {title}")
    print(SEP)

    print("\n--- SELECTED POST-PROCESSING (tuned on VALIDATION, applied unchanged to TEST) ---")
    print(f"  window operating point : {res['win_cfg']}")
    print(f"  event  operating point : {res['ev_cfg']}")

    print("\n--- WINDOW-LEVEL METRICS ---")
    wrows = []
    for label, key in [("raw @ 0.5", "window_default"),
                       ("tuned (val)", "window_tuned"),
                       ("tuned + cleanup", "window_postprocessed")]:
        m = res[key]
        wrows.append({"operating_point": label, "thr": round(m["threshold"], 3),
                      "accuracy": round(m["accuracy"], 4), "precision": round(m["precision"], 4),
                      "recall": round(m["recall"], 4), "specificity": round(m["specificity"], 4),
                      "f1": round(m["f1"], 4), "balanced_acc": round(m["balanced_accuracy"], 4),
                      "mcc": round(m["mcc"], 4), "kappa": round(m["kappa"], 4),
                      "auroc": round(m["auroc"], 4), "auprc": round(m["auprc"], 4),
                      "brier": round(m["brier"], 4), "TP": m["tp"], "FP": m["fp"],
                      "FN": m["fn"], "TN": m["tn"]})
    win_df = pd.DataFrame(wrows)
    print(win_df.to_string(index=False))

    print("\n--- EVENT-LEVEL METRICS ---")
    erows = []
    for name, key in [("strict (one-to-one)", "event_strict"),
                      ("lenient (any overlap)", "event_lenient")]:
        e = res[key]
        erows.append({"matching": name, "precision": round(e["precision"], 4),
                      "recall": round(e["recall"], 4), "f1": round(e["f1"], 4),
                      "TP": e["tp"], "FP": e["fp"], "FN": e["fn"],
                      "n_pred": e["n_pred"], "n_true": e["n_true"],
                      "median_onset_err_epochs": e["median_onset_error"]})
    ev_df = pd.DataFrame(erows)
    print(ev_df.to_string(index=False))

    print("\n--- AHI AGREEMENT (apnea-epoch index, events/hour) ---")
    a = res["ahi"]
    ahi_df = pd.DataFrame([{k: round(v, 4) for k, v in a.items()}])
    print(ahi_df.to_string(index=False))

    print("\n--- SEVERITY CLASSIFICATION ---")
    s = res["severity"]
    print(f"  exact accuracy = {s['accuracy']:.4f} | within-one-class = "
          f"{s['within_one_class']:.4f} | quadratic kappa = {s['quadratic_kappa']:.4f}")
    print(pd.DataFrame(s["confusion"], index=[f"true_{n}" for n in SEVERITY_NAMES],
                       columns=[f"pred_{n}" for n in SEVERITY_NAMES]).to_string())

    print("\n--- METRICS BY TRUE SEVERITY STRATUM ---")
    print(res["stratified"].to_string(index=False) if len(res["stratified"]) else "  (empty)")

    print("\n--- PER-PATIENT RESULTS (all test patients) ---")
    print(res["per_patient"].to_string(index=False))

    print("\n--- WINDOW CLASSIFICATION REPORT ---")
    print(res["classification_report"])

    if seed_df is not None and len(seed_df):
        print("--- ENSEMBLE MEMBERS vs ENSEMBLE ---")
        print(seed_df.to_string(index=False))
        print(f"  ensemble F1 = {res['window_tuned']['f1']:.4f} | member mean = "
              f"{seed_df['f1'].mean():.4f} +/- "
              f"{seed_df['f1'].std(ddof=1) if len(seed_df) > 1 else 0:.4f}")

    if history_df is not None and len(history_df):
        print("\n--- TRAINING HISTORY (last epoch per member) ---")
        print(history_df.groupby("tag").tail(1).to_string(index=False))

    return win_df, ev_df, ahi_df


# =========================================================================
# 15.  FIGURES
# =========================================================================
def make_figures(res, history_df, prefix):
    A = res["_arrays"]
    y, p = A["y"], A["p_win"]

    fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))
    if len(np.unique(y)) > 1:
        fpr, tpr, _ = roc_curve(y, p)
        ax[0].plot(fpr, tpr, lw=2, label=f"AUROC={res['window_tuned']['auroc']:.4f}")
        ax[0].plot([0, 1], [0, 1], "k:", alpha=.4)
        pr, rc, _ = precision_recall_curve(y, p)
        ax[1].plot(rc, pr, lw=2, label=f"AUPRC={res['window_tuned']['auprc']:.4f}")
        ax[1].axhline(y.mean(), ls=":", c="k", alpha=.5, label=f"prevalence={y.mean():.3f}")
        ax[1].plot(res["window_tuned"]["recall"], res["window_tuned"]["precision"],
                   "r*", ms=14, label="chosen operating point")
    ax[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC")
    ax[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall")
    for a in ax:
        a.legend(); a.grid(alpha=.3)
    plt.tight_layout(); show(f"{FIG_DIR}/{prefix}_roc_pr.png")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
    for a, (name, key) in zip(axes, [("raw @ 0.5", "window_default"),
                                     ("tuned (val)", "window_tuned"),
                                     ("tuned + cleanup", "window_postprocessed")]):
        m = res[key]
        cm = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
        a.imshow(cm, cmap="Blues")
        a.set_title(f"{name}\nF1={m['f1']:.4f}", fontsize=10)
        a.set_xticks([0, 1]); a.set_xticklabels(["Normal", "Apnea"])
        a.set_yticks([0, 1]); a.set_yticklabels(["Normal", "Apnea"])
        a.set_xlabel("Predicted"); a.set_ylabel("True")
        for i in range(2):
            for j in range(2):
                a.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                       color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.suptitle("Window-level confusion matrices")
    plt.tight_layout(); show(f"{FIG_DIR}/{prefix}_confusion.png")

    pp = res["per_patient"]; a = res["ahi"]
    diff = pp["pred_AHI"] - pp["true_AHI"]
    fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))
    ax[0].scatter(pp["true_AHI"], pp["pred_AHI"], s=50, alpha=.85)
    lim = max(pp["true_AHI"].max(), pp["pred_AHI"].max()) * 1.1 + 1
    ax[0].plot([0, lim], [0, lim], "k--", alpha=.5)
    for e in SEVERITY_EDGES:
        ax[0].axvline(e, ls=":", c="grey", alpha=.6); ax[0].axhline(e, ls=":", c="grey", alpha=.6)
    ax[0].set(xlabel="True AHI (apnea-epoch index)", ylabel="Predicted AHI",
              title=f"AHI agreement  r={a['pearson_r']:.3f}  ICC={a['icc_2_1']:.3f}")
    ax[1].scatter((pp["true_AHI"] + pp["pred_AHI"]) / 2, diff, s=50, alpha=.85)
    ax[1].axhline(a["bias"], c="r", label=f"bias={a['bias']:.2f}")
    ax[1].axhline(a["loa_high"], c="r", ls="--",
                  label=f"LoA=[{a['loa_low']:.1f}, {a['loa_high']:.1f}]")
    ax[1].axhline(a["loa_low"], c="r", ls="--")
    ax[1].set(xlabel="Mean of true & predicted AHI", ylabel="Predicted - True",
              title="Bland-Altman")
    for x in ax:
        x.grid(alpha=.3); x.legend()
    plt.tight_layout(); show(f"{FIG_DIR}/{prefix}_ahi_agreement.png")

    cm = res["severity"]["confusion"]
    plt.figure(figsize=(6, 5.2))
    plt.imshow(cm, cmap="Oranges")
    plt.xticks(range(4), SEVERITY_NAMES, rotation=20); plt.yticks(range(4), SEVERITY_NAMES)
    plt.xlabel("Predicted severity"); plt.ylabel("True severity")
    plt.title(f"Severity  acc={res['severity']['accuracy']:.3f}  "
              f"quad-kappa={res['severity']['quadratic_kappa']:.3f}")
    for i in range(4):
        for j in range(4):
            plt.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.tight_layout(); show(f"{FIG_DIR}/{prefix}_severity_confusion.png")

    st = res["stratified"]
    if len(st):
        plt.figure(figsize=(11, 5))
        x = np.arange(len(st)); w = .2
        for i, col in enumerate(["win_f1", "win_auprc", "event_f1", "event_recall"]):
            plt.bar(x + (i - 1.5) * w, st[col], w, label=col)
        plt.xticks(x, [f"{r.severity}\n(n={r.n_patients})" for r in st.itertuples()])
        plt.ylim(0, 1); plt.ylabel("Score"); plt.legend(); plt.grid(alpha=.3, axis="y")
        plt.title("Performance by TRUE severity stratum")
        plt.tight_layout(); show(f"{FIG_DIR}/{prefix}_by_severity.png")

    bins = np.linspace(0, 1, 11); idx = np.digitize(p, bins) - 1
    xs, ys = [], []
    for b in range(10):
        m = idx == b
        if m.sum() > 20:
            xs.append(p[m].mean()); ys.append(y[m].mean())
    plt.figure(figsize=(6, 5.4))
    plt.plot([0, 1], [0, 1], "k--", alpha=.5, label="perfect")
    plt.plot(xs, ys, "o-", label=f"model (Brier={res['window_tuned']['brier']:.4f})")
    plt.xlabel("Predicted probability"); plt.ylabel("Observed frequency")
    plt.title("Calibration"); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); show(f"{FIG_DIR}/{prefix}_calibration.png")

    pids = list(res["per_patient"]["patient"])
    pick = pids[::max(1, len(pids) // 8)][:8]
    if pick:
        fig, axes = plt.subplots(len(pick), 1, figsize=(13, 1.8 * len(pick)))
        axes = np.atleast_1d(axes)
        info = res["per_patient"].set_index("patient")
        for a, pid in zip(axes, pick):
            prob = A["p_by_pid"][pid]
            a.plot(prob, lw=.8, label="p(apnea)")
            a.fill_between(np.arange(len(prob)), 0, A["bin_by_pid"][pid], alpha=.3,
                           step="mid", label="predicted")
            a.set_ylim(0, 1); a.set_yticks([0, 1]); a.set_ylabel(pid, fontsize=8)
            r = info.loc[pid]
            a.set_title(f"{r['true_severity']} | true AHI={r['true_AHI']:.1f} "
                        f"pred={r['pred_AHI']:.1f} | win F1={r['win_f1']:.3f} "
                        f"event F1={r['event_f1']:.3f}", fontsize=8)
        axes[0].legend(fontsize=7, loc="upper right")
        axes[-1].set_xlabel("30 s epoch index")
        plt.tight_layout(); show(f"{FIG_DIR}/{prefix}_patient_timelines.png")

    if history_df is not None and len(history_df):
        fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
        for tag, g in history_df.groupby("tag"):
            ax[0].plot(g["epoch"], g["train_loss"], label=tag)
            ax[1].plot(g["epoch"], g["val_auprc"], label=tag)
            ax[2].plot(g["epoch"], g["val_f1"], label=tag)
        ax[0].set(xlabel="Epoch", ylabel="Train loss", title="Training loss")
        ax[1].set(xlabel="Epoch", ylabel="Val AUPRC", title="Validation AUPRC (selection metric)")
        ax[2].set(xlabel="Epoch", ylabel="Val F1", title="Validation F1")
        for a in ax:
            a.legend(fontsize=8); a.grid(alpha=.3)
        plt.tight_layout(); show(f"{FIG_DIR}/{prefix}_training_curves.png")


# =========================================================================
# 16.  TRAINING
# =========================================================================
def train_one(seed, chunk_len, train_pids, val_pids, tag, log_rows):
    set_all_seeds(seed)
    stride = max(1, int(chunk_len * TRAIN_STRIDE_FRAC))
    ds = ChunkDataset(train_pids, chunk_len, stride, augment=True,
                      soft_boundary=True, rng_seed=seed)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                        pin_memory=(DEVICE.type == "cuda"),
                        drop_last=len(ds) > BATCH_SIZE, worker_init_fn=_worker_init)

    y_tr = ds.labels()
    n_pos, n_neg = int((y_tr == 1).sum()), int((y_tr == 0).sum())
    pos_weight = min(POS_WEIGHT_CAP, n_neg / max(1, n_pos))

    model = ApneaSegmenterV5().to(DEVICE)
    n_par = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n  [{tag}] context={chunk_len} epochs ({chunk_len * WINDOW_SEC}s) | "
          f"chunks={len(ds):,} | train epochs={len(y_tr):,} "
          f"(apnea {100 * n_pos / max(1, len(y_tr)):.1f}%) | params={n_par:,} | "
          f"pos_weight={pos_weight:.2f}")

    crit = CombinedLoss(pos_weight).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps = max(1, len(loader))
    total_steps = EPOCHS * (steps + 2)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR, total_steps=total_steps,
        pct_start=WARMUP_EPOCHS / max(1, EPOCHS), anneal_strategy="cos")
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    ema = EMA(model)

    best_score, best_state, bad = -1.0, None, 0
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        ds.resample(jitter=True)
        model.train()
        tot, nb = 0.0, 0
        for resp, slow, feat, y, valid in loader:
            resp, slow, feat = (resp.to(DEVICE, non_blocking=True),
                                slow.to(DEVICE, non_blocking=True),
                                feat.to(DEVICE, non_blocking=True))
            y, valid = y.to(DEVICE, non_blocking=True), valid.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                logits, aux = model(resp, slow, feat)
                loss = crit(logits, aux, y, valid)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update()
            if sched.last_epoch < total_steps - 1:
                sched.step()
            ema.update(model)
            tot += float(loss.item()); nb += 1

        backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
        ema.copy_to(model)
        vp = predict_patients(model, val_pids, chunk_len)
        yv = np.concatenate([v[0] for v in vp.values()])
        pv = np.concatenate([v[1] for v in vp.values()])
        auprc = average_precision_score(yv, pv) if len(np.unique(yv)) > 1 else 0.0
        auroc = roc_auc_score(yv, pv) if len(np.unique(yv)) > 1 else 0.0
        prec, rec, thr = precision_recall_curve(yv, pv)
        f1s = 2 * prec * rec / (prec + rec + 1e-9)
        bi = int(np.nanargmax(f1s[:-1])) if len(thr) else 0
        f1 = float(f1s[bi])

        score = auprc if SELECT_METRIC == "auprc" else f1
        if score > best_score:
            best_score = score
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
        model.load_state_dict(backup)

        dt = time.time() - t0
        log_rows.append({"tag": tag, "seed": seed, "context": chunk_len, "epoch": ep,
                         "train_loss": round(tot / max(1, nb), 4),
                         "val_auprc": round(auprc, 4), "val_auroc": round(auroc, 4),
                         "val_f1": round(f1, 4), "val_thr": round(float(thr[bi]) if len(thr) else .5, 3),
                         "lr": opt.param_groups[0]["lr"], "sec": round(dt, 1)})
        print(f"  [{tag}] ep {ep:2d}/{EPOCHS} | loss={tot / max(1, nb):.4f} | "
              f"val_AUPRC={auprc:.4f} val_AUROC={auroc:.4f} val_F1={f1:.4f} | {dt:.1f}s"
              + ("  <- best" if bad == 0 else ""))
        if bad >= PATIENCE:
            print(f"  [{tag}] early stop at epoch {ep}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    torch.save({"state_dict": model.state_dict(), "seed": seed, "context": chunk_len,
                f"val_{SELECT_METRIC}": best_score},
               os.path.join(OUTPUT_DIR, f"model_{tag}.pt"))
    return model, best_score


# =========================================================================
# 17.  ONE SPLIT END-TO-END
# =========================================================================
def run_split(train_pids, val_pids, test_pids, prefix="test"):
    log_rows, members = [], []
    for seed, clen in RUN_CONFIGS:
        tag = f"{prefix}_s{seed}_L{clen}"
        model, score = train_one(seed, clen, train_pids, val_pids, tag, log_rows)
        members.append((tag, model, clen, score))

    def ensemble(pids):
        acc = None
        for _, m, clen, _ in members:
            pr = predict_patients(m, pids, clen)
            if acc is None:
                acc = {p: (v[0], v[1].astype(np.float64)) for p, v in pr.items()}
            else:
                for p in acc:
                    acc[p] = (acc[p][0], acc[p][1] + pr[p][1])
        return {p: (v[0], (v[1] / len(members)).astype(np.float32)) for p, v in acc.items()}

    print("\n  searching post-processing on VALIDATION...")
    val_pred = ensemble(val_pids)
    win_cfg, win_val = search_postprocessing(val_pred, "window")
    ev_cfg, ev_val = search_postprocessing(val_pred, "event")
    print(f"    window: {win_cfg}  -> val F1 {win_val:.4f}")
    print(f"    event : {ev_cfg}  -> val event F1 {ev_val:.4f}")

    print("  predicting on TEST (ensemble, overlap + sign-flip TTA)...")
    test_pred = ensemble(test_pids)
    res = evaluate(test_pred, win_cfg, ev_cfg, prefix)

    seed_rows = []
    for tag, m, clen, score in members:
        sp = predict_patients(m, test_pids, clen)
        ys = np.concatenate([sp[p][0] for p in sorted(sp)])
        ps = np.concatenate([smooth_probs(sp[p][1], win_cfg["smooth"]) for p in sorted(sp)])
        wm = window_metrics(ys, ps, win_cfg["thr"])
        seed_rows.append({"member": tag, "context_epochs": clen,
                          f"val_{SELECT_METRIC}": round(score, 4),
                          "precision": round(wm["precision"], 4),
                          "recall": round(wm["recall"], 4), "f1": round(wm["f1"], 4),
                          "mcc": round(wm["mcc"], 4), "auroc": round(wm["auroc"], 4),
                          "auprc": round(wm["auprc"], 4)})
        del sp
    seed_df = pd.DataFrame(seed_rows)
    history_df = pd.DataFrame(log_rows)

    win_df, ev_df, ahi_df = print_full_report(
        res, history_df, seed_df,
        f"FINAL TEST RESULTS — {prefix} (ensemble of {len(members)} members)")
    make_figures(res, history_df, prefix)

    # ---- save everything -------------------------------------------------
    res["per_patient"].to_csv(f"{OUTPUT_DIR}/{prefix}_per_patient.csv", index=False)
    res["stratified"].to_csv(f"{OUTPUT_DIR}/{prefix}_by_severity.csv", index=False)
    win_df.to_csv(f"{OUTPUT_DIR}/{prefix}_window_metrics.csv", index=False)
    ev_df.to_csv(f"{OUTPUT_DIR}/{prefix}_event_metrics.csv", index=False)
    ahi_df.to_csv(f"{OUTPUT_DIR}/{prefix}_ahi_agreement.csv", index=False)
    seed_df.to_csv(f"{OUTPUT_DIR}/{prefix}_members.csv", index=False)
    history_df.to_csv(f"{OUTPUT_DIR}/{prefix}_training_history.csv", index=False)
    pd.DataFrame(res["severity"]["confusion"],
                 index=[f"true_{n}" for n in SEVERITY_NAMES],
                 columns=[f"pred_{n}" for n in SEVERITY_NAMES]
                 ).to_csv(f"{OUTPUT_DIR}/{prefix}_severity_confusion.csv")
    with open(f"{OUTPUT_DIR}/{prefix}_classification_report.txt", "w") as fh:
        fh.write(res["classification_report"])
    json.dump({"window": res["win_cfg"], "event": res["ev_cfg"]},
              open(f"{OUTPUT_DIR}/{prefix}_postprocessing.json", "w"), indent=2)
    np.savez_compressed(
        f"{OUTPUT_DIR}/{prefix}_probabilities.npz",
        **{f"prob_{p}": v.astype(np.float32) for p, v in res["_arrays"]["p_by_pid"].items()},
        **{f"pred_{p}": v.astype(np.int8) for p, v in res["_arrays"]["bin_by_pid"].items()},
        **{f"true_{p}": test_pred[p][0].astype(np.int8) for p in test_pred})

    flat = {"split": prefix, "n_patients": res["n_patients"]}
    for key, pre in [("window_default", "win_raw"), ("window_tuned", "win_tuned"),
                     ("window_postprocessed", "win_clean")]:
        flat.update({f"{pre}_{k}": v for k, v in res[key].items()})
    flat.update({f"event_{k}": v for k, v in res["event_strict"].items() if k != "onset"})
    flat.update({f"ahi_{k}": v for k, v in res["ahi"].items()})
    flat.update({f"sev_{k}": v for k, v in res["severity"].items() if k != "confusion"})
    pd.DataFrame([flat]).to_csv(f"{OUTPUT_DIR}/{prefix}_summary.csv", index=False)

    for _, m, _, _ in members:
        del m
    torch.cuda.empty_cache(); gc.collect()
    return res, flat


# =========================================================================
# 18.  SYNTHETIC SOURCE (smoke test)
# =========================================================================
def make_synthetic_source(path, n_patients=10, epochs=110):
    os.makedirs(path, exist_ok=True)
    rng = np.random.default_rng(0)
    for i in range(1, n_patients + 1):
        pid = f"SN{i}"
        n = epochs + int(rng.integers(0, 20))
        y = np.zeros(n, dtype=np.int8)
        j, rate = 0, 0.1 + 0.6 * rng.random()
        while j < n:
            j += int(rng.integers(2, max(3, int(8 / rate))))
            d = int(rng.integers(1, 4))
            y[j:j + d] = 1
            j += d
        X = rng.normal(0, 1, (n, 8, SRC_SPE)).astype(np.float32)
        t = np.arange(SRC_SPE) / SRC_FS
        for k in range(n):
            breath = np.sin(2 * np.pi * .25 * t + rng.random() * 6)
            amp = .2 if y[k] else 1.0
            for ch in (0, 1, 2, 7):
                X[k, ch] = amp * breath * 5 + .3 * X[k, ch]
            X[k, 2] *= -1 if y[k] else 1
            X[k, 3] = 97 - (3.5 if y[k] else 0) + .1 * X[k, 3]
            X[k, 6] = 60 + (9 if y[k] else 0) + X[k, 6]
        np.save(os.path.join(path, f"X_{pid}.npy"), X.astype(np.float16))
        np.save(os.path.join(path, f"y_{pid}.npy"), y)
    return path


# =========================================================================
# 19.  MAIN
# =========================================================================
def resolve_source():
    if SMOKE_TEST:
        return make_synthetic_source(os.path.join(WORK_DIR, "src"))
    import kagglehub
    root = kagglehub.dataset_download(SOURCE_DATASET)
    for dp, _, fns in os.walk(root):
        if any(f.startswith("X_SN") and f.endswith(".npy") for f in fns):
            return dp
    return root


def main():
    print(SEP)
    print("  SLEEP APNEA DETECTION — v5")
    print(SEP)
    print(f"  device={DEVICE} | smoke={SMOKE_TEST} | cv_folds={CV_FOLDS} | "
          f"cache={CACHE_VERSION}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)} "
              f"| capability {torch.cuda.get_device_capability(0)}")
    print(f"  ensemble members (seed, context epochs): {RUN_CONFIGS}")
    print(f"  AHI proxy = apnea-epoch index (positive epochs / hour)  [fix for D1]")

    src = resolve_source()
    print("  source:", src)

    all_pids = sorted({f[2:-4] for f in os.listdir(src)
                       if f.startswith("X_SN") and f.endswith(".npy")},
                      key=lambda p: int(p[2:]) if p[2:].isdigit() else 0)
    excluded = [p for p in all_pids if p in SKIP_PATIENTS]
    pids = [p for p in all_pids if p not in SKIP_PATIENTS
            and os.path.exists(os.path.join(src, f"y_{p}.npy"))]
    print(f"  found {len(all_pids)} | excluded {len(excluded)} | candidates {len(pids)}")
    pd.DataFrame([{"patient": p, "reason": SKIP_PATIENTS[p]} for p in excluded]
                 ).to_csv(f"{OUTPUT_DIR}/excluded_patients.csv", index=False)

    print("\n" + SEP)
    print(f"  STAGE 1 — multi-rate cache + physiological features "
          f"(resp @{RESP_FS} Hz, slow @{SLOW_FS} Hz, {N_FEAT} features/epoch)")
    print(SEP)
    manifest = build_cache(src, pids)

    bad = manifest[manifest["positive_epochs"] < MIN_POSITIVE_EPOCHS]["patient"].tolist()
    if bad:
        print(f"  dropping {len(bad)} patients with < {MIN_POSITIVE_EPOCHS} positives: {bad}")
        pids = [p for p in pids if p not in bad]
    manifest = manifest[manifest.patient.isin(pids)]

    print("\n--- COHORT MANIFEST ---")
    print(manifest.to_string(index=False))
    print("\n--- COHORT SEVERITY DISTRIBUTION (apnea-epoch index) ---")
    print(manifest.groupby("severity").agg(
        patients=("patient", "count"), mean_AHI=("AHI_epoch_index", "mean"),
        mean_ratio=("apnea_ratio", "mean")).round(3).to_string())
    print(f"\n  pooled prevalence = "
          f"{manifest['positive_epochs'].sum() / manifest['epochs'].sum():.4f} | "
          f"total {manifest['epochs'].sum():,} epochs / {manifest['hours'].sum():.0f} h")

    ratio = dict(zip(manifest["patient"], manifest["apnea_ratio"]))

    print("\n" + SEP)
    print("  STAGE 2 — patient-level splits")
    print(SEP)

    if CV_FOLDS and CV_FOLDS > 1:
        folds = grouped_folds(ratio, CV_FOLDS, SEED)
        json.dump({f"fold{i}": f for i, f in enumerate(folds)},
                  open(f"{OUTPUT_DIR}/cv_folds.json", "w"), indent=2)
        summaries = []
        for k in range(CV_FOLDS):
            test_pids = folds[k]
            val_pids = folds[(k + 1) % CV_FOLDS]
            train_pids = [p for i, f in enumerate(folds)
                          if i not in (k, (k + 1) % CV_FOLDS) for p in f]
            print(f"\n  FOLD {k}: train={len(train_pids)} val={len(val_pids)} test={len(test_pids)}")
            _, flat = run_split(train_pids, val_pids, test_pids, prefix=f"fold{k}")
            flat["fold"] = k
            summaries.append(flat)
        cv = pd.DataFrame(summaries)
        cv.to_csv(f"{OUTPUT_DIR}/cv_summary.csv", index=False)
        print("\n" + SEP)
        print("  CROSS-VALIDATION SUMMARY")
        print(SEP)
        cols = ["win_tuned_f1", "win_tuned_precision", "win_tuned_recall",
                "win_tuned_auroc", "win_tuned_auprc", "event_f1",
                "ahi_icc_2_1", "sev_quadratic_kappa"]
        print(cv[[c for c in cols if c in cv]].to_string(index=False))
        print("\n  mean +/- sd across folds:")
        for c in cols:
            if c in cv:
                print(f"    {c:28s} {cv[c].mean():.4f} +/- {cv[c].std(ddof=1):.4f}")
    else:
        train_pids, val_pids, test_pids = stratified_split(ratio, SEED)
        json.dump({"seed": SEED, "train": train_pids, "val": val_pids, "test": test_pids},
                  open(SPLIT_JSON, "w"), indent=2)
        sev = dict(zip(manifest["patient"], manifest["severity"]))
        for name, grp in [("Train", train_pids), ("Val", val_pids), ("Test", test_pids)]:
            r = np.array([ratio[p] for p in grp])
            counts = pd.Series([sev[p] for p in grp]).value_counts().to_dict()
            print(f"  {name:5s} {len(grp):3d} patients | apnea ratio {r.mean():.3f} "
                  f"+/- {r.std():.3f} | severity {counts}")
        print(f"  split -> {SPLIT_JSON}")

        print("\n" + SEP)
        print(f"  STAGE 3 — training {len(RUN_CONFIGS)} ensemble members")
        print(SEP)
        run_split(train_pids, val_pids, test_pids, prefix="test")

    print("\n" + SEP)
    print("  SAVED OUTPUTS")
    print(SEP)
    for f in sorted(os.listdir(OUTPUT_DIR)):
        p = os.path.join(OUTPUT_DIR, f)
        if os.path.isfile(p):
            print(f"   {f:48s} {os.path.getsize(p) / 1024:8.1f} KB")
    print(f"  figures ({FIG_DIR}):")
    for f in sorted(os.listdir(FIG_DIR)):
        print(f"   - {f}")


if __name__ == "__main__":
    main()